# 17 — Relation-Specific Evidence Representations

## Motivation

Notebook 16 established that trajectory context is useful, but its utility is
strongly dependent on the functional relation between events.

Key findings from Notebook 16:

- No-t1 macro-F1: 0.4548
- Full-t1 macro-F1: 0.5105
- Uncertainty-gated t1 macro-F1: 0.5105
- Uncertainty-gated t1 accuracy: 0.5406
- Structural cross-fit macro-F1: 0.4900

Trajectory-induced prediction changes were highly structured:

- TOOL_CALL -> TOOL_CALL:
    workflow_error -> tool_use_error was strongly beneficial

- TOOL_CALL -> ASSISTANT:
    workflow_error -> grounding_state_error was strongly beneficial

- ASSISTANT -> TOOL_CALL:
    workflow_error -> constraint_error was beneficial

However:

- raw t-1 L1/L2 distance was almost useless for distinguishing rescues from breaks
- generic rescue/break classifiers did not generalize
- relation structure generalized better than generic distance magnitude

## Central hypothesis

Trajectory information is useful because different event relations encode
different reliability mechanisms.

Generic current↔previous distance only captures these mechanisms indirectly.

We therefore replace generic temporal-distance reasoning with
relation-specific representations.

## Research questions

RQ1.
Does TOOL_CALL -> ASSISTANT consistency provide stronger evidence for
grounding_state_error than generic t-1 distance?

RQ2.
Does ASSISTANT -> TOOL_CALL consistency provide stronger evidence for
constraint_error / tool_use_error?

RQ3.
Does TOOL_CALL -> TOOL_CALL relational change provide stronger evidence
for tool_use_error?

RQ4.
Do relation-specific representations contain information beyond the
semantic classifier state?

RQ5.
Can relation-specific evidence explain trajectory rescues versus breaks?

RQ6.
Can these features eventually improve the uncertainty-gated t-1 baseline
without increasing trajectory-induced breaks?

In [2]:
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    GroupKFold,
    cross_val_predict,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Notebook 17 initialized.")
CLASS_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

N_CLASSES = len(CLASS_NAMES)

Notebook 17 initialized.


In [3]:
# ============================================================
# 2. Load canonical trajectory exports
# ============================================================

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [4]:
# ============================================================
# 3. Labels and group-safe split identifiers
# ============================================================

y_train = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

group_col = (
    "canonical_group"
    if "canonical_group" in train_targets.columns
    else "group_id"
)

groups_train = (
    train_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)

print(
    train_targets[
        "failure_family"
    ].value_counts()
)

failure_family
workflow_error           660
constraint_error         317
grounding_state_error    244
tool_use_error           237
reasoning_value_error     31
Name: count, dtype: int64


In [5]:
# ============================================================
# 4. Reconstruct event history for each target
# ============================================================

TRAJECTORY_KEY = [
    "dataset",
    "group_id",
]


def build_history_indices(
    events_df,
    targets_df,
):

    lookup = {}

    for key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        lookup[key] = (
            group
            .sort_values(
                "message_index"
            )
        )

    histories = []

    for _, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_index = int(
            row["message_index"]
        )

        events = lookup.get(key)

        if events is None:
            histories.append([])
            continue

        history = (
            events.loc[
                events[
                    "message_index"
                ] < target_index
            ]
            .index
            .tolist()
        )

        histories.append(
            history
        )

    return histories


train_history_indices = (
    build_history_indices(
        train_events,
        train_targets,
    )
)

history_count = np.array([
    len(x)
    for x in train_history_indices
])

np.testing.assert_array_equal(
    history_count,
    train_targets[
        "history_event_count"
    ].to_numpy(),
)

print(
    "✓ History reconstruction exact"
)

print(
    "Zero history:",
    (history_count == 0).sum()
)

✓ History reconstruction exact
Zero history: 56


In [6]:
# ============================================================
# 8. Previous-event role and transition type
# ============================================================

previous_role = []
previous_event_index = []

for indices in train_history_indices:

    if len(indices) == 0:

        previous_role.append(
            "NO_HISTORY"
        )

        previous_event_index.append(
            None
        )

    else:

        idx = indices[-1]

        previous_event_index.append(
            idx
        )

        previous_role.append(
            str(
                train_events.loc[
                    idx,
                    "event_role",
                ]
            )
        )


# ------------------------------------------------------------
# Determine current role if export contains it
# ------------------------------------------------------------

if "event_role" in train_targets.columns:

    current_role = (
        train_targets[
            "event_role"
        ]
        .fillna("UNKNOWN")
        .astype(str)
        .to_numpy()
    )

else:

    # Target export may not contain explicit role.
    # Infer from target text prefix when possible.
    target_text = (
        train_targets[
            target_text_col
        ]
        .fillna("")
        .astype(str)
    )

    current_role = np.where(
        target_text.str.startswith(
            "[TOOL_CALL]"
        ),
        "TOOL_CALL",
        np.where(
            target_text.str.startswith(
                "[ASSISTANT]"
            ),
            "ASSISTANT",
            "UNKNOWN",
        ),
    )


transition_type = np.array([
    f"{prev}->{cur}"
    for prev, cur in zip(
        previous_role,
        current_role,
    )
])


transition_structure_df = pd.DataFrame({
    "previous_role":
        previous_role,

    "current_role":
        current_role,

    "transition_type":
        transition_type,

    "history_event_count":
        history_count,

    "failure_family":
        [
            CLASS_NAMES[int(y)]
            for y in y_train
        ],
})


print(
    "Unique transition types:",
    transition_structure_df[
        "transition_type"
    ].nunique()
)

display(
    transition_structure_df[
        "transition_type"
    ]
    .value_counts()
    .head(20)
)

Unique transition types: 6


transition_type
TOOL_CALL->TOOL_CALL     590
TOOL_CALL->ASSISTANT     397
ASSISTANT->ASSISTANT     226
ASSISTANT->TOOL_CALL     220
NO_HISTORY->ASSISTANT     34
NO_HISTORY->TOOL_CALL     22
Name: count, dtype: int64

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd

from collections import Counter

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

RANDOM_STATE = 42

CLASS_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

N_CLASSES = len(CLASS_NAMES)

# ============================================================
# 5. Encode target and event text
# ============================================================

if "content" in train_targets.columns:
    target_text_col = "content"
elif "current_text" in train_targets.columns:
    target_text_col = "current_text"
else:
    raise ValueError(
        "No target text column found."
    )


def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(row["content"])
        else str(row["content"]).strip()
    )

    prefix = f"[{role}]"

    if content.upper().startswith(
        prefix.upper()
    ):
        return content

    return (
        prefix
        + "\n"
        + content
    )


train_events[
    "event_text"
] = train_events.apply(
    make_event_text,
    axis=1,
)

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

train_current_embeddings = encoder.encode(
    train_targets[
        target_text_col
    ]
    .fillna("")
    .astype(str)
    .tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

train_event_embeddings = encoder.encode(
    train_events[
        "event_text"
    ].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_sem_train = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

print(
    X_sem_train.shape,
    train_event_embeddings.shape,
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 6. Build last-three historical embeddings
# ============================================================

def get_last_k_embeddings(
    history_indices,
    event_embeddings,
    k=3,
):

    n = len(history_indices)
    d = event_embeddings.shape[1]

    output = np.zeros(
        (n, k, d),
        dtype=np.float32,
    )

    for i, indices in enumerate(
        history_indices
    ):

        recent = indices[-k:]

        if not recent:
            continue

        output[
            i,
            -len(recent):,
            :
        ] = event_embeddings[
            recent
        ]

    return output


H_train_3 = get_last_k_embeddings(
    train_history_indices,
    train_event_embeddings,
    k=3,
)

In [ ]:
# ============================================================
# 7. Six positional distance features
# ============================================================

def row_l1(a, b):
    return np.mean(
        np.abs(a - b),
        axis=1,
    )


def row_l2(a, b):
    return np.linalg.norm(
        a - b,
        axis=1,
    )


distance_data = {}

for pos, lag in [
    (0, 3),
    (1, 2),
    (2, 1),
]:

    h = H_train_3[
        :,
        pos,
        :
    ]

    present = (
        np.linalg.norm(
            h,
            axis=1,
        ) > 1e-8
    ).astype(float)

    distance_data[
        f"current_tminus{lag}_l1"
    ] = (
        row_l1(
            X_sem_train,
            h,
        )
        * present
    )

    distance_data[
        f"current_tminus{lag}_l2"
    ] = (
        row_l2(
            X_sem_train,
            h,
        )
        * present
    )


distance_df = pd.DataFrame(
    distance_data
)

distance_names = list(
    distance_df.columns
)

T_distance = distance_df.to_numpy(
    dtype=np.float32
)

print(
    distance_names
)

print(
    T_distance.shape
)

['current_tminus3_l1', 'current_tminus3_l2', 'current_tminus2_l1', 'current_tminus2_l2', 'current_tminus1_l1', 'current_tminus1_l2']
(1489, 6)


In [ ]:
# ============================================================
# 9. OOF full-distance vs remove-t1 intervention
# ============================================================

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

full_prob = np.zeros(
    (len(y_train), N_CLASSES)
)

remove_t1_prob = np.zeros(
    (len(y_train), N_CLASSES)
)

# T_distance order:
#
# t3_l1, t3_l2,
# t2_l1, t2_l2,
# t1_l1, t1_l2

T1_POSITIONS = [
    4,
    5,
]


for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    X_tr = np.hstack([
        X_sem_train[
            tr_idx
        ],
        T_distance[
            tr_idx
        ],
    ])

    scaler = StandardScaler()

    X_tr_scaled = (
        scaler.fit_transform(
            X_tr
        )
    )

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[
            tr_idx
        ],
    )

    # --------------------------------------------------------
    # Full recent-distance representation
    # --------------------------------------------------------

    X_va_full = np.hstack([
        X_sem_train[
            va_idx
        ],
        T_distance[
            va_idx
        ],
    ])

    full_prob[
        va_idx
    ] = model.predict_proba(
        scaler.transform(
            X_va_full
        )
    )

    # --------------------------------------------------------
    # Same model; remove t-1 only
    # --------------------------------------------------------

    T_remove = (
        T_distance[
            va_idx
        ].copy()
    )

    T_remove[
        :,
        T1_POSITIONS
    ] = 0.0

    X_va_remove = np.hstack([
        X_sem_train[
            va_idx
        ],
        T_remove,
    ])

    remove_t1_prob[
        va_idx
    ] = model.predict_proba(
        scaler.transform(
            X_va_remove
        )
    )

    print(
        f"Fold {fold} complete"
    )


full_pred = (
    full_prob.argmax(
        axis=1
    )
)

remove_t1_pred = (
    remove_t1_prob.argmax(
        axis=1
    )
)

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [ ]:
# ============================================================
# 3. Recover known objects from the current notebook state
# ============================================================

required_objects = [
    "train_targets",
    "transition_structure_df",
    "distance_df",
    "full_prob",
    "remove_t1_prob",
    "train_current_embeddings",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    print("Missing required objects:")
    for name in missing:
        print("  -", name)

        print(name)
    raise RuntimeError(
        "Load/recreate the Notebook 16 artifacts above before continuing."
    )

print("Core objects found.")

for name in required_objects:
    obj = globals()[name]

    if hasattr(obj, "shape"):
        print(f"{name:30s}", obj.shape)
    else:
        print(f"{name:30s}", type(obj))

Core objects found.
train_targets                  (1489, 14)
transition_structure_df        (1489, 5)
distance_df                    (1489, 6)
full_prob                      (1489, 5)
remove_t1_prob                 (1489, 5)
train_current_embeddings       (1489, 384)


In [ ]:
# ============================================================
# 4. Canonical multiclass order
# ============================================================

class_names = np.array([
    "constraint_error",
    "grounding_state_error",
    "reasoning_value_error",
    "tool_use_error",
    "workflow_error",
])

class_to_idx = {
    name: i
    for i, name in enumerate(class_names)
}

idx_to_class = {
    i: name
    for i, name in enumerate(class_names)
}

print("Class order:")
print(class_names)

print("\nNumber of classes:", len(class_names))

assert full_prob.shape[1] == len(class_names)
assert remove_t1_prob.shape[1] == len(class_names)

Class order:
['constraint_error' 'grounding_state_error' 'reasoning_value_error'
 'tool_use_error' 'workflow_error']

Number of classes: 5


In [ ]:
semantic_pred_idx = remove_t1_prob.argmax(axis=1)
trajectory_pred_idx = full_prob.argmax(axis=1)

semantic_pred = class_names[semantic_pred_idx]
trajectory_pred = class_names[trajectory_pred_idx]

true_family = (
    train_targets["failure_family"]
    .astype(str)
    .to_numpy()
)

print("\nSemantic prediction counts:")
print(pd.Series(semantic_pred).value_counts())

print("\nTrajectory prediction counts:")
print(pd.Series(trajectory_pred).value_counts())

print("\nTrue family counts:")
print(pd.Series(true_family).value_counts())


Semantic prediction counts:
constraint_error         873
grounding_state_error    303
tool_use_error           155
reasoning_value_error    132
workflow_error            26
Name: count, dtype: int64

Trajectory prediction counts:
constraint_error         714
grounding_state_error    325
tool_use_error           224
reasoning_value_error    199
workflow_error            27
Name: count, dtype: int64

True family counts:
workflow_error           660
constraint_error         317
grounding_state_error    244
tool_use_error           237
reasoning_value_error     31
Name: count, dtype: int64


In [ ]:
# ============================================================
# 5. Alignment checks
# ============================================================

n = len(train_targets)

objects_to_check = {
    "transition_structure_df": transition_structure_df,
    "distance_df": distance_df,
    "full_prob": full_prob,
    "remove_t1_prob": remove_t1_prob,
    "train_current_embeddings": train_current_embeddings,
}

for name, obj in objects_to_check.items():
    assert len(obj) == n, (
        f"{name} has {len(obj)} rows; expected {n}"
    )

assert np.isfinite(full_prob).all()
assert np.isfinite(remove_t1_prob).all()
assert np.isfinite(train_current_embeddings).all()

assert np.allclose(
    full_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    remove_t1_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

print("All row-alignment and probability checks passed.")

All row-alignment and probability checks passed.


In [ ]:
# ============================================================
# 4. Recover ACTUAL probability-column class order
# ============================================================

# IMPORTANT:
# This is the column order of full_prob / remove_t1_prob.
# It is NOT the alphabetical family order.
#
# Notebook 16 empirically established:
#   0 -> workflow_error
#   1 -> constraint_error
#   2 -> tool_use_error
#   3 -> grounding_state_error
#   4 -> reasoning_value_error

prob_class_names = np.array([
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
])

prob_class_to_idx = {
    name: i
    for i, name in enumerate(prob_class_names)
}

prob_idx_to_class = {
    i: name
    for i, name in enumerate(prob_class_names)
}

print("Probability-column order:")
for i, name in enumerate(prob_class_names):
    print(f"  {i} -> {name}")

print("\nNumber of classes:", len(prob_class_names))

assert full_prob.shape[1] == len(prob_class_names)
assert remove_t1_prob.shape[1] == len(prob_class_names)


# ------------------------------------------------------------
# Reconstruct predictions
# ------------------------------------------------------------

semantic_pred_idx = remove_t1_prob.argmax(axis=1)
trajectory_pred_idx = full_prob.argmax(axis=1)

semantic_pred = prob_class_names[semantic_pred_idx]
trajectory_pred = prob_class_names[trajectory_pred_idx]

true_family = (
    train_targets["failure_family"]
    .astype(str)
    .to_numpy()
)

print("\nSemantic prediction counts:")
print(pd.Series(semantic_pred).value_counts())

print("\nTrajectory prediction counts:")
print(pd.Series(trajectory_pred).value_counts())

print("\nTrue family counts:")
print(pd.Series(true_family).value_counts())

Probability-column order:
  0 -> workflow_error
  1 -> constraint_error
  2 -> tool_use_error
  3 -> grounding_state_error
  4 -> reasoning_value_error

Number of classes: 5

Semantic prediction counts:
workflow_error           873
constraint_error         303
grounding_state_error    155
tool_use_error           132
reasoning_value_error     26
Name: count, dtype: int64

Trajectory prediction counts:
workflow_error           714
constraint_error         325
grounding_state_error    224
tool_use_error           199
reasoning_value_error     27
Name: count, dtype: int64

True family counts:
workflow_error           660
constraint_error         317
grounding_state_error    244
tool_use_error           237
reasoning_value_error     31
Name: count, dtype: int64


In [ ]:
# ============================================================
# 6. Canonical transition table
# ============================================================

relation_df = pd.DataFrame({
    "true_family": true_family,

    "semantic_prediction": semantic_pred,
    "trajectory_prediction": trajectory_pred,

    "transition_type":
        transition_structure_df["transition_type"]
        .astype(str)
        .to_numpy(),

    "previous_role":
        transition_structure_df["previous_role"]
        .astype(str)
        .to_numpy(),

    "current_role":
        transition_structure_df["current_role"]
        .astype(str)
        .to_numpy(),

    "history_event_count":
        transition_structure_df["history_event_count"]
        .to_numpy(),
})

relation_df["changed"] = (
    relation_df["semantic_prediction"]
    != relation_df["trajectory_prediction"]
)

relation_df["semantic_correct"] = (
    relation_df["semantic_prediction"]
    == relation_df["true_family"]
)

relation_df["trajectory_correct"] = (
    relation_df["trajectory_prediction"]
    == relation_df["true_family"]
)

relation_df["effect"] = "unchanged"

relation_df.loc[
    relation_df["changed"]
    & (~relation_df["semantic_correct"])
    & relation_df["trajectory_correct"],
    "effect"
] = "rescue"

relation_df.loc[
    relation_df["changed"]
    & relation_df["semantic_correct"]
    & (~relation_df["trajectory_correct"]),
    "effect"
] = "break"

relation_df.loc[
    relation_df["changed"]
    & (~relation_df["semantic_correct"])
    & (~relation_df["trajectory_correct"]),
    "effect"
] = "wrong_to_wrong"

display(relation_df.head())

print("\nTransition distribution:")
print(
    relation_df["transition_type"]
    .value_counts()
)

print("\nTrajectory effect distribution:")
print(
    relation_df["effect"]
    .value_counts()
)

,true_family,semantic_prediction,trajectory_prediction,transition_type,previous_role,current_role,history_event_count,changed,semantic_correct,trajectory_correct,effect
0,constraint_error,grounding_state_error,grounding_state_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,2,False,False,False,unchanged
1,tool_use_error,workflow_error,workflow_error,ASSISTANT->TOOL_CALL,ASSISTANT,TOOL_CALL,1,False,False,False,unchanged
2,workflow_error,constraint_error,constraint_error,TOOL_CALL->TOOL_CALL,TOOL_CALL,TOOL_CALL,2,False,False,False,unchanged
3,constraint_error,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,3,False,True,True,unchanged
4,constraint_error,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,3,False,True,True,unchanged



Transition distribution:
transition_type
TOOL_CALL->TOOL_CALL     590
TOOL_CALL->ASSISTANT     397
ASSISTANT->ASSISTANT     226
ASSISTANT->TOOL_CALL     220
NO_HISTORY->ASSISTANT     34
NO_HISTORY->TOOL_CALL     22
Name: count, dtype: int64

Trajectory effect distribution:
effect
unchanged         1308
rescue              87
break               53
wrong_to_wrong      41
Name: count, dtype: int64


In [ ]:
# ============================================================
# 6. Canonical transition/intervention table
# ============================================================

relation_df = pd.DataFrame({
    "true_family": true_family,

    "semantic_prediction": semantic_pred,
    "trajectory_prediction": trajectory_pred,

    "transition_type":
        transition_structure_df["transition_type"]
        .astype(str)
        .to_numpy(),

    "previous_role":
        transition_structure_df["previous_role"]
        .astype(str)
        .to_numpy(),

    "current_role":
        transition_structure_df["current_role"]
        .astype(str)
        .to_numpy(),

    "history_event_count":
        transition_structure_df["history_event_count"]
        .to_numpy(),
})

relation_df["changed"] = (
    relation_df["semantic_prediction"]
    != relation_df["trajectory_prediction"]
)

relation_df["semantic_correct"] = (
    relation_df["semantic_prediction"]
    == relation_df["true_family"]
)

relation_df["trajectory_correct"] = (
    relation_df["trajectory_prediction"]
    == relation_df["true_family"]
)


# ------------------------------------------------------------
# Intervention effect
# ------------------------------------------------------------

relation_df["effect"] = "unchanged"

changed = relation_df["changed"]

relation_df.loc[
    changed
    & (~relation_df["semantic_correct"])
    & relation_df["trajectory_correct"],
    "effect"
] = "rescue"

relation_df.loc[
    changed
    & relation_df["semantic_correct"]
    & (~relation_df["trajectory_correct"]),
    "effect"
] = "break"

relation_df.loc[
    changed
    & (~relation_df["semantic_correct"])
    & (~relation_df["trajectory_correct"]),
    "effect"
] = "wrong_to_wrong"


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

display(relation_df.head())

print("\nTransition distribution:")
print(
    relation_df["transition_type"]
    .value_counts()
)

print("\nTrajectory effect distribution:")
print(
    relation_df["effect"]
    .value_counts()
)

print("\nChanged predictions:")
print(int(relation_df["changed"].sum()))


# ------------------------------------------------------------
# Hard regression checks against Notebook 16
# ------------------------------------------------------------

effect_counts = (
    relation_df["effect"]
    .value_counts()
    .to_dict()
)

expected = {
    "unchanged": 1308,
    "rescue": 87,
    "break": 53,
    "wrong_to_wrong": 41,
}

print("\nExpected from Notebook 16:")
print(expected)

print("\nObserved:")
print(effect_counts)

for effect, expected_count in expected.items():

    observed = effect_counts.get(effect, 0)

    assert observed == expected_count, (
        f"{effect}: expected {expected_count}, "
        f"observed {observed}"
    )

assert relation_df["changed"].sum() == 181

print(
    "\n✓ Notebook 16 intervention structure reproduced exactly."
)

,true_family,semantic_prediction,trajectory_prediction,transition_type,previous_role,current_role,history_event_count,changed,semantic_correct,trajectory_correct,effect
0,constraint_error,grounding_state_error,grounding_state_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,2,False,False,False,unchanged
1,tool_use_error,workflow_error,workflow_error,ASSISTANT->TOOL_CALL,ASSISTANT,TOOL_CALL,1,False,False,False,unchanged
2,workflow_error,constraint_error,constraint_error,TOOL_CALL->TOOL_CALL,TOOL_CALL,TOOL_CALL,2,False,False,False,unchanged
3,constraint_error,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,3,False,True,True,unchanged
4,constraint_error,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,TOOL_CALL,ASSISTANT,3,False,True,True,unchanged



Transition distribution:
transition_type
TOOL_CALL->TOOL_CALL     590
TOOL_CALL->ASSISTANT     397
ASSISTANT->ASSISTANT     226
ASSISTANT->TOOL_CALL     220
NO_HISTORY->ASSISTANT     34
NO_HISTORY->TOOL_CALL     22
Name: count, dtype: int64

Trajectory effect distribution:
effect
unchanged         1308
rescue              87
break               53
wrong_to_wrong      41
Name: count, dtype: int64

Changed predictions:
181

Expected from Notebook 16:
{'unchanged': 1308, 'rescue': 87, 'break': 53, 'wrong_to_wrong': 41}

Observed:
{'unchanged': 1308, 'rescue': 87, 'break': 53, 'wrong_to_wrong': 41}

✓ Notebook 16 intervention structure reproduced exactly.


In [ ]:
# ============================================================
# 7. Functional relation families
# ============================================================

RELATION_FAMILY_MAP = {
    "TOOL_CALL->ASSISTANT":
        "tool_to_assistant",

    "ASSISTANT->TOOL_CALL":
        "assistant_to_tool",

    "TOOL_CALL->TOOL_CALL":
        "tool_to_tool",

    "ASSISTANT->ASSISTANT":
        "assistant_to_assistant",

    "NO_HISTORY->ASSISTANT":
        "no_history",

    "NO_HISTORY->TOOL_CALL":
        "no_history",
}

relation_df["relation_family"] = (
    relation_df["transition_type"]
    .map(RELATION_FAMILY_MAP)
    .fillna("other")
)

display(
    relation_df[
        [
            "transition_type",
            "relation_family",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)

,transition_type,relation_family,count
0,TOOL_CALL->TOOL_CALL,tool_to_tool,590
1,TOOL_CALL->ASSISTANT,tool_to_assistant,397
2,ASSISTANT->ASSISTANT,assistant_to_assistant,226
3,ASSISTANT->TOOL_CALL,assistant_to_tool,220
4,NO_HISTORY->ASSISTANT,no_history,34
5,NO_HISTORY->TOOL_CALL,no_history,22


In [ ]:
# ============================================================
# 8. Inspect processed event representation
# ============================================================

print("train_targets columns:")
print(train_targets.columns.tolist())

print("\ntransition_structure_df columns:")
print(transition_structure_df.columns.tolist())

inspect_cols = [
    c for c in [
        "dataset",
        "group_id",
        "canonical_group",
        "split",
        "trajectory_index",
        "message_index",
        "event_position",
        "history_event_count",
        "has_history",
        "event_role",
        "primary_tool",
        "content",
        "failure_family",
    ]
    if c in train_targets.columns
]

display(
    train_targets[
        inspect_cols
    ].head(20)
)

train_targets columns:
['dataset', 'group_id', 'canonical_group', 'split', 'trajectory_index', 'message_index', 'event_position', 'history_event_count', 'has_history', 'event_role', 'primary_tool', 'content', 'family_label', 'failure_family']

transition_structure_df columns:
['previous_role', 'current_role', 'transition_type', 'history_event_count', 'failure_family']


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family
0,A,a_1,a_1,train,1,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Australian city founded in 18...,constraint_error
1,A,a_104,a_104,train,104,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",tool_use_error
2,A,a_105,a_105,train,105,6,2,2,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",workflow_error
3,A,a_105,a_105,train,105,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe university with its main camp...,constraint_error
4,A,a_107,a_107,train,107,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe search results clearly confir...,constraint_error
5,A,a_108,a_108,train,108,2,0,0,0,ASSISTANT,NO_TOOL,[ASSISTANT]\n<answer>I'm a Jayhawk</answer>,constraint_error
6,A,a_109,a_109,train,109,2,0,0,0,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>University with main campu...,constraint_error
7,A,a_109,a_109,train,109,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""University of...",constraint_error
8,A,a_109,a_109,train,109,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>University with main campu...,constraint_error
9,A,a_115,a_115,train,115,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe travel parody series featurin...,grounding_state_error


In [ ]:
# ============================================================
# 8B. Representative examples by transition type
# ============================================================

major_transitions = [
    "TOOL_CALL->TOOL_CALL",
    "TOOL_CALL->ASSISTANT",
    "ASSISTANT->TOOL_CALL",
    "ASSISTANT->ASSISTANT",
]

for transition in major_transitions:

    print("\n" + "=" * 100)
    print(transition)
    print("=" * 100)

    candidate_idx = (
        relation_df.index[
            relation_df[
                "transition_type"
            ].eq(transition)
        ]
        .tolist()
    )

    # deterministic small sample
    example_idx = candidate_idx[:8]

    display(
        train_targets.loc[
            example_idx,
            inspect_cols
        ]
    )


TOOL_CALL->TOOL_CALL


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family
2,A,a_105,a_105,train,105,6,2,2,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",workflow_error
31,A,a_184,a_184,train,184,6,2,2,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""Yi Guan resea...",workflow_error
35,A,a_198,a_198,train,198,8,3,3,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"":[""\""Dim Gr...",tool_use_error
36,A,a_198,a_198,train,198,10,4,4,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"":[""\""Dim Gr...",workflow_error
37,A,a_198,a_198,train,198,12,5,5,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"":[""\""Dim Gr...",workflow_error
38,A,a_198,a_198,train,198,14,6,6,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"":[""\""Dim Gr...",workflow_error
39,A,a_198,a_198,train,198,16,7,7,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"":""\""Dim Gray Bar...",tool_use_error
40,A,a_198,a_198,train,198,18,8,8,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"":[""Dim Gray...",grounding_state_error



TOOL_CALL->ASSISTANT


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family
0,A,a_1,a_1,train,1,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Australian city founded in 18...,constraint_error
3,A,a_105,a_105,train,105,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe university with its main camp...,constraint_error
4,A,a_107,a_107,train,107,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe search results clearly confir...,constraint_error
8,A,a_109,a_109,train,109,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>University with main campu...,constraint_error
9,A,a_115,a_115,train,115,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe travel parody series featurin...,grounding_state_error
10,A,a_116,a_116,train,116,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe travel parody series that fea...,grounding_state_error
11,A,a_119,a_119,train,119,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>We need the travel parody ...,grounding_state_error
12,A,a_120,a_120,train,120,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Apple Remote was originally d...,workflow_error



ASSISTANT->TOOL_CALL


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family
1,A,a_104,a_104,train,104,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",tool_use_error
7,A,a_109,a_109,train,109,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""University of...",constraint_error
30,A,a_184,a_184,train,184,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""Yi Guan resea...",tool_use_error
113,B,b_109,b_109,train,109,3,1,1,1,TOOL_CALL,find_user_id_by_name_zip,"[TOOL_CALL]\n\nfind_user_id_by_name_zip({""firs...",grounding_state_error
116,B,b_109,b_109,train,109,21,10,10,1,TOOL_CALL,modify_pending_order_payment,"[TOOL_CALL]\n\nmodify_pending_order_payment({""...",constraint_error
120,B,b_115,b_115,train,115,3,1,1,1,TOOL_CALL,find_user_id_by_email,"[TOOL_CALL]\n\nfind_user_id_by_email({""email"":...",tool_use_error
122,B,b_116,b_116,train,116,3,1,1,1,TOOL_CALL,To,[TOOL_CALL]\nTo assist you with returning the ...,tool_use_error
124,B,b_119,b_119,train,119,3,1,1,1,TOOL_CALL,find_user_id_by_email,"[TOOL_CALL]\n\nfind_user_id_by_email({""email"":...",tool_use_error



ASSISTANT->ASSISTANT


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family
115,B,b_109,b_109,train,109,19,9,9,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{""message"":""You want to apply gif...",constraint_error
118,B,b_109,b_109,train,109,25,12,12,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{""message"":""Confirmed. I've appli...",grounding_state_error
119,B,b_109,b_109,train,109,29,14,14,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{""message"":""You're welcome — glad...",workflow_error
123,B,b_116,b_116,train,116,23,11,11,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe return policy allows you to r...,constraint_error
132,B,b_13,b_13,train,13,27,13,13,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{\n ""message"": ""Here’s your prop...",reasoning_value_error
146,B,b_14,b_14,train,14,9,4,4,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{""message"":""You confirmed changes...",constraint_error
147,B,b_140,b_140,train,140,19,9,9,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\nI understand your concern, but I’...",workflow_error
153,B,b_144,b_144,train,144,11,5,5,1,ASSISTANT,NO_TOOL,"[ASSISTANT]\n{""message"":""Thanks — the system r...",grounding_state_error


In [ ]:
# ============================================================
# 9. Reconstruct within-trajectory neighbors
# ============================================================

trajectory_candidates = [
    "trajectory_index",
    "group_id",
    "canonical_group",
]

order_candidates = [
    "event_position",
    "message_index",
]

trajectory_col = next(
    (
        col
        for col in trajectory_candidates
        if col in train_targets.columns
    ),
    None,
)

order_col = next(
    (
        col
        for col in order_candidates
        if col in train_targets.columns
    ),
    None,
)

print("Trajectory column:", trajectory_col)
print("Ordering column:", order_col)

if trajectory_col is None:
    raise RuntimeError(
        "No trajectory identifier found."
    )

if order_col is None:
    raise RuntimeError(
        "No event ordering column found."
    )


tmp = train_targets[
    [
        trajectory_col,
        order_col,
    ]
].copy()

tmp["_row_idx"] = np.arange(
    len(tmp)
)

tmp = tmp.sort_values(
    [
        trajectory_col,
        order_col,
    ],
    kind="stable",
)


# ------------------------------------------------------------
# Previous event
# ------------------------------------------------------------

tmp["previous_idx"] = (
    tmp
    .groupby(
        trajectory_col
    )["_row_idx"]
    .shift(1)
)

tmp["tminus2_idx"] = (
    tmp
    .groupby(
        trajectory_col
    )["_row_idx"]
    .shift(2)
)

tmp["tminus3_idx"] = (
    tmp
    .groupby(
        trajectory_col
    )["_row_idx"]
    .shift(3)
)


previous_idx = np.full(
    len(train_targets),
    -1,
    dtype=int,
)

tminus2_idx = np.full(
    len(train_targets),
    -1,
    dtype=int,
)

tminus3_idx = np.full(
    len(train_targets),
    -1,
    dtype=int,
)


for source_col, target_array in [
    ("previous_idx", previous_idx),
    ("tminus2_idx", tminus2_idx),
    ("tminus3_idx", tminus3_idx),
]:

    valid = tmp[
        source_col
    ].notna()

    original_rows = (
        tmp.loc[
            valid,
            "_row_idx"
        ]
        .astype(int)
        .to_numpy()
    )

    historical_rows = (
        tmp.loc[
            valid,
            source_col
        ]
        .astype(int)
        .to_numpy()
    )

    target_array[
        original_rows
    ] = historical_rows


relation_df[
    "previous_idx"
] = previous_idx

relation_df[
    "tminus2_idx"
] = tminus2_idx

relation_df[
    "tminus3_idx"
] = tminus3_idx


print(
    "Previous available:",
    (previous_idx >= 0).sum(),
)

print(
    "t-2 available:",
    (tminus2_idx >= 0).sum(),
)

print(
    "t-3 available:",
    (tminus3_idx >= 0).sum(),
)

Trajectory column: trajectory_index
Ordering column: event_position
Previous available: 1284
t-2 available: 1109
t-3 available: 968


In [ ]:
# ============================================================
# 10. Verify reconstructed transitions
# ============================================================

event_roles = (
    train_targets[
        "event_role"
    ]
    .astype(str)
    .to_numpy()
)

reconstructed = np.empty(
    len(train_targets),
    dtype=object,
)

for i in range(
    len(train_targets)
):

    prev = previous_idx[i]

    if prev < 0:

        reconstructed[i] = (
            "NO_HISTORY->"
            + event_roles[i]
        )

    else:

        reconstructed[i] = (
            event_roles[prev]
            + "->"
            + event_roles[i]
        )


relation_df[
    "reconstructed_transition"
] = reconstructed

relation_df[
    "transition_match"
] = (
    relation_df[
        "transition_type"
    ]
    ==
    relation_df[
        "reconstructed_transition"
    ]
)


print(
    "Exact reconstruction rate:",
    relation_df[
        "transition_match"
    ].mean()
)

print(
    "\nMatched:",
    relation_df[
        "transition_match"
    ].sum(),
    "/",
    len(relation_df),
)


display(
    pd.crosstab(
        relation_df[
            "transition_type"
        ],
        relation_df[
            "reconstructed_transition"
        ],
    )
)

Exact reconstruction rate: 0.7689724647414372

Matched: 1145 / 1489


reconstructed_transition,ASSISTANT->ASSISTANT,ASSISTANT->TOOL_CALL,NO_HISTORY->ASSISTANT,NO_HISTORY->TOOL_CALL,TOOL_CALL->ASSISTANT,TOOL_CALL->TOOL_CALL
transition_type,,,,,,
ASSISTANT->ASSISTANT,183,0,15,0,28,0
ASSISTANT->TOOL_CALL,0,175,0,25,0,20
NO_HISTORY->ASSISTANT,2,0,32,0,0,0
NO_HISTORY->TOOL_CALL,0,3,0,19,0,0
TOOL_CALL->ASSISTANT,76,0,64,0,257,0
TOOL_CALL->TOOL_CALL,0,61,0,50,0,479


In [ ]:
# ============================================================
# 10B. Inspect transition mismatches
# ============================================================

transition_mismatches = (
    relation_df[
        ~relation_df[
            "transition_match"
        ]
    ]
    .copy()
)

print(
    "Mismatch count:",
    len(transition_mismatches)
)

if len(transition_mismatches) > 0:

    mismatch_idx = (
        transition_mismatches
        .index[:20]
    )

    display(
        pd.concat(
            [
                train_targets.loc[
                    mismatch_idx,
                    inspect_cols
                ],

                relation_df.loc[
                    mismatch_idx,
                    [
                        "transition_type",
                        "reconstructed_transition",
                        "previous_idx",
                        "history_event_count",
                    ]
                ],
            ],
            axis=1,
        )
    )

Mismatch count: 344


,dataset,group_id,canonical_group,split,trajectory_index,message_index,event_position,history_event_count,has_history,event_role,primary_tool,content,failure_family,transition_type,reconstructed_transition,previous_idx,history_event_count
0,A,a_1,a_1,train,1,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Australian city founded in 18...,constraint_error,TOOL_CALL->ASSISTANT,ASSISTANT->ASSISTANT,1072,2
1,A,a_104,a_104,train,104,4,1,1,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",tool_use_error,ASSISTANT->TOOL_CALL,NO_HISTORY->TOOL_CALL,-1,1
2,A,a_105,a_105,train,105,6,2,2,1,TOOL_CALL,search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",workflow_error,TOOL_CALL->TOOL_CALL,NO_HISTORY->TOOL_CALL,-1,2
4,A,a_107,a_107,train,107,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe search results clearly confir...,constraint_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,3
10,A,a_116,a_116,train,116,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe travel parody series that fea...,grounding_state_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,1
11,A,a_119,a_119,train,119,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>We need the travel parody ...,grounding_state_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,1
12,A,a_120,a_120,train,120,6,2,2,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Apple Remote was originally d...,workflow_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,2
13,A,a_121,a_121,train,121,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\nThe Apple Remote was originally d...,workflow_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,1
16,A,a_127,a_127,train,127,8,3,3,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n</think></think>The search result...,grounding_state_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,3
18,A,a_134,a_134,train,134,4,1,1,1,ASSISTANT,NO_TOOL,[ASSISTANT]\n<think>We need when the Western G...,constraint_error,TOOL_CALL->ASSISTANT,NO_HISTORY->ASSISTANT,-1,1


In [ ]:
# ============================================================
# 11. Find role-specific historical embedding arrays
# ============================================================

namespace_snapshot = list(globals().items())

array_candidates = []

for name, obj in namespace_snapshot:

    if name.startswith("_"):
        continue

    try:
        if isinstance(obj, np.ndarray):

            shape = obj.shape

            if (
                len(shape) >= 2
                and shape[0] == len(train_targets)
            ):

                array_candidates.append({
                    "name": name,
                    "shape": shape,
                    "dtype": str(obj.dtype),
                })

    except Exception:
        pass


candidate_df = pd.DataFrame(
    array_candidates
)

display(
    candidate_df.sort_values(
        ["shape", "name"],
        key=lambda x: x.astype(str)
    )
)

,name,shape,dtype
2,H_train_3,"(1489, 3, 384)",float32
1,X_sem_train,"(1489, 384)",float32
3,h,"(1489, 384)",float32
7,obj,"(1489, 384)",float32
0,train_current_embeddings,"(1489, 384)",float32
5,full_prob,"(1489, 5)",float64
6,remove_t1_prob,"(1489, 5)",float64
4,T_distance,"(1489, 6)",float32


In [ ]:
# ============================================================
# 11B. Search names related to role/history embeddings
# ============================================================

keywords = [
    "assistant",
    "tool",
    "history",
    "prev",
    "role",
    "embed",
]

for name, obj in namespace_snapshot:

    lower = name.lower()

    if not any(
        keyword in lower
        for keyword in keywords
    ):
        continue

    try:
        shape = obj.shape
    except Exception:
        shape = None

    print(
        f"{name:45s}",
        type(obj).__name__,
        shape
    )

previous_role                                 list None
previous_event_index                          list None
build_history_indices                         function None
train_history_indices                         list None
history_count                                 ndarray (1489,)
current_role                                  ndarray (1489,)
get_last_k_embeddings                         function None
train_current_embeddings                      ndarray (1489, 384)
train_event_embeddings                        ndarray (3792, 384)
previous_idx                                  ndarray (1489,)
event_roles                                   ndarray (1489,)
prev                                          int64 ()


In [ ]:
# ============================================================
# 11. Inspect true history-index structure
# ============================================================

print("Number of training targets:", len(train_targets))
print("History-index entries:", len(train_history_indices))
print("Event embeddings:", train_event_embeddings.shape)

print("\nFirst 10 history-index entries:")

for i in range(10):
    print(
        i,
        "history_count=",
        history_count[i],
        "history_indices=",
        train_history_indices[i],
    )

Number of training targets: 1489
History-index entries: 1489
Event embeddings: (3792, 384)

First 10 history-index entries:
0 history_count= 2 history_indices= [0, 1]
1 history_count= 1 history_indices= [3]
2 history_count= 2 history_indices= [6, 7]
3 history_count= 3 history_indices= [6, 7, 8]
4 history_count= 3 history_indices= [10, 11, 12]
5 history_count= 0 history_indices= []
6 history_count= 0 history_indices= []
7 history_count= 1 history_indices= [15]
8 history_count= 2 history_indices= [15, 16]
9 history_count= 3 history_indices= [18, 19, 20]


In [ ]:
# ============================================================
# 12. Find full-event metadata aligned to train_event_embeddings
# ============================================================

namespace_snapshot = list(globals().items())

print("Objects with length 3792:\n")

for name, obj in namespace_snapshot:

    if name.startswith("_"):
        continue

    try:
        if len(obj) == len(train_event_embeddings):
            print(
                f"{name:40s}",
                type(obj).__name__,
                getattr(obj, "shape", None),
            )
    except Exception:
        pass

Objects with length 3792:

train_events                             DataFrame (3792, 41)
train_event_embeddings                   ndarray (3792, 384)


In [ ]:
# ============================================================
# 13. Inspect existing history helper functions
# ============================================================

import inspect

for fn_name in [
    "build_history_indices",
    "get_last_k_embeddings",
]:

    fn = globals().get(fn_name)

    print("\n" + "=" * 80)
    print(fn_name)

    if fn is None:
        print("NOT FOUND")
        continue

    try:
        print("Signature:")
        print(inspect.signature(fn))
    except Exception as e:
        print("Could not inspect signature:", e)

    try:
        print("\nSource:")
        print(inspect.getsource(fn))
    except Exception as e:
        print("Could not recover source:", e)


build_history_indices
Signature:
(events_df, targets_df)

Source:
def build_history_indices(
    events_df,
    targets_df,
):

    lookup = {}

    for key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        lookup[key] = (
            group
            .sort_values(
                "message_index"
            )
        )

    histories = []

    for _, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_index = int(
            row["message_index"]
        )

        events = lookup.get(key)

        if events is None:
            histories.append([])
            continue

        history = (
            events.loc[
                events[
                    "message_index"
                ] < target_index
            ]
            .index
            .tolist()
        )

        histories.append(
            history
        )

    return histories


get_last_k_embeddin

In [ ]:
# ============================================================
# 14. Inspect full event-stream schema
# ============================================================

print("train_events shape:", train_events.shape)

print("\nColumns:")
for i, col in enumerate(train_events.columns):
    print(f"{i:2d}: {col}")

display(
    train_events.head(10)
)

train_events shape: (3792, 41)

Columns:
 0: dataset
 1: group_id
 2: trajectory_index
 3: message_index
 4: event_role
 5: current_role
 6: content
 7: primary_tool
 8: char_length
 9: word_count
10: has_content
11: has_error_signal
12: is_system
13: is_user
14: is_assistant
15: is_tool_call
16: is_tool_result
17: raw_step_label
18: raw_step_reason
19: context_text
20: previous_messages
21: previous_tool_calls
22: previous_tool_results
23: previous_user_messages
24: previous_assistant_messages
25: context_char_length
26: context_word_count
27: canonical_group
28: split
29: failure_family
30: family_label
31: is_taxonomy_target
32: event_position
33: trajectory_event_count
34: relative_event_position
35: previous_error_signals
36: previous_event_role
37: previous_event_tool
38: role_transition
39: tool_transition
40: event_text


,dataset,group_id,trajectory_index,message_index,event_role,current_role,content,primary_tool,char_length,word_count,has_content,has_error_signal,is_system,is_user,is_assistant,is_tool_call,is_tool_result,raw_step_label,raw_step_reason,context_text,previous_messages,previous_tool_calls,previous_tool_results,previous_user_messages,previous_assistant_messages,context_char_length,context_word_count,canonical_group,split,failure_family,family_label,is_taxonomy_target,event_position,trajectory_event_count,relative_event_position,previous_error_signals,previous_event_role,previous_event_tool,role_transition,tool_transition,event_text
0,A,a_1,1,2,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral...",search,201,25,1,0,0,0,0,1,0,1,Annotated +1: this assistant step is correct a...,NaN,0,0,0,0,0,0,0,a_1,train,NaN,NaN,0,0,3,0.000000,0,START,NO_TOOL,START->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral..."
1,A,a_1,1,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci...",search,152,20,1,0,0,0,0,1,0,1,Annotated +1: this assistant step is correct a...,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral...",2,1,0,0,0,4514,675,a_1,train,NaN,NaN,0,1,3,0.500000,0,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci..."
2,A,a_1,1,6,ASSISTANT,ASSISTANT,[ASSISTANT]\nThe Australian city founded in 18...,NO_TOOL,628,101,1,0,0,0,1,0,0,-1,"Annotated -1: Concludes Adelaide, but Adelaide...","[TOOL_CALL]\n\nsearch({""query"": ""Australian ci...",4,2,0,0,0,2334,345,a_1,train,constraint_error,1.0,1,2,3,1.000000,0,TOOL_CALL,search,TOOL_CALL->ASSISTANT,search->NO_TOOL,[ASSISTANT]\nThe Australian city founded in 18...
3,A,a_104,104,2,ASSISTANT,ASSISTANT,[ASSISTANT]\n<think>We need birth dates for Er...,NO_TOOL,518,82,1,0,0,0,1,0,0,1,Annotated +1: this assistant step is correct a...,NaN,0,0,0,0,0,0,0,a_104,train,NaN,NaN,0,0,3,0.000000,0,START,NO_TOOL,START->ASSISTANT,NO_TOOL->NO_TOOL,[ASSISTANT]\n<think>We need birth dates for Er...
4,A,a_104,104,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",search,212,31,1,0,0,0,0,1,0,-1,"Annotated -1: the assistant calls search({""que...",[ASSISTANT]\n<think>We need birth dates for Er...,1,0,0,0,1,518,82,a_104,train,tool_use_error,2.0,1,1,3,0.500000,0,ASSISTANT,NO_TOOL,ASSISTANT->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv..."
5,A,a_104,104,6,ASSISTANT,ASSISTANT,[ASSISTANT]\n<think>Erika Jayne (Erika Girardi...,NO_TOOL,488,76,1,0,0,0,1,0,0,1,Annotated +1: this assistant step is correct a...,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",3,1,0,0,1,4346,688,a_104,train,NaN,NaN,0,2,3,1.000000,0,TOOL_CALL,search,TOOL_CALL->ASSISTANT,search->NO_TOOL,[ASSISTANT]\n<think>Erika Jayne (Erika Girardi...
6,A,a_105,105,2,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers...",search,136,15,1,0,0,0,0,1,0,1,Annotated +1: this assistant step is correct a...,NaN,0,0,0,0,0,0,0,a_105,train,NaN,NaN,0,0,4,0.000000,0,START,NO_TOOL,START->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers..."
7,A,a_105,105,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",search,146,17,1,0,0,0,0,1,0,0,Annotated 0: this is neutral or exploratory ra...,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers...",2,1,0,0,0,2410,343,a_105,train,NaN,NaN,0,1,4,0.333333,0,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
8,A,a_105,105,6,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",search,139,16,1,0,0,0,0,1,0,-1,Annotated -1: Inefficient repetition of search...,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",4,2,0,0,0,2420,345,a_105,train,workflow_error,0.0,1,2,4,0.666667,0,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
9,A,a_105,105,8,ASSISTANT,ASSISTANT,[ASSIST

In [ ]:
# ============================================================
# 14B. Candidate role / content / trajectory columns
# ============================================================

keywords = [
    "role",
    "type",
    "event",
    "content",
    "text",
    "tool",
    "trajectory",
    "group",
    "message",
    "position",
    "index",
]

candidate_cols = [
    col
    for col in train_events.columns
    if any(
        keyword in col.lower()
        for keyword in keywords
    )
]

print(candidate_cols)

display(
    train_events[
        candidate_cols
    ].head(20)
)

['group_id', 'trajectory_index', 'message_index', 'event_role', 'current_role', 'content', 'primary_tool', 'has_content', 'is_tool_call', 'is_tool_result', 'context_text', 'previous_messages', 'previous_tool_calls', 'previous_tool_results', 'previous_user_messages', 'previous_assistant_messages', 'context_char_length', 'context_word_count', 'canonical_group', 'event_position', 'trajectory_event_count', 'relative_event_position', 'previous_event_role', 'previous_event_tool', 'role_transition', 'tool_transition', 'event_text']


,group_id,trajectory_index,message_index,event_role,current_role,content,primary_tool,has_content,is_tool_call,is_tool_result,context_text,previous_messages,previous_tool_calls,previous_tool_results,previous_user_messages,previous_assistant_messages,context_char_length,context_word_count,canonical_group,event_position,trajectory_event_count,relative_event_position,previous_event_role,previous_event_tool,role_transition,tool_transition,event_text
0,a_1,1,2,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral...",search,1,1,0,NaN,0,0,0,0,0,0,0,a_1,0,3,0.000000,START,NO_TOOL,START->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral..."
1,a_1,1,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci...",search,1,1,0,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral...",2,1,0,0,0,4514,675,a_1,1,3,0.500000,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci..."
2,a_1,1,6,ASSISTANT,ASSISTANT,[ASSISTANT]\nThe Australian city founded in 18...,NO_TOOL,1,0,0,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci...",4,2,0,0,0,2334,345,a_1,2,3,1.000000,TOOL_CALL,search,TOOL_CALL->ASSISTANT,search->NO_TOOL,[ASSISTANT]\nThe Australian city founded in 18...
3,a_104,104,2,ASSISTANT,ASSISTANT,[ASSISTANT]\n<think>We need birth dates for Er...,NO_TOOL,1,0,0,NaN,0,0,0,0,0,0,0,a_104,0,3,0.000000,START,NO_TOOL,START->ASSISTANT,NO_TOOL->NO_TOOL,[ASSISTANT]\n<think>We need birth dates for Er...
4,a_104,104,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",search,1,1,0,[ASSISTANT]\n<think>We need birth dates for Er...,1,0,0,0,1,518,82,a_104,1,3,0.500000,ASSISTANT,NO_TOOL,ASSISTANT->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv..."
5,a_104,104,6,ASSISTANT,ASSISTANT,[ASSISTANT]\n<think>Erika Jayne (Erika Girardi...,NO_TOOL,1,0,0,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv...",3,1,0,0,1,4346,688,a_104,2,3,1.000000,TOOL_CALL,search,TOOL_CALL->ASSISTANT,search->NO_TOOL,[ASSISTANT]\n<think>Erika Jayne (Erika Girardi...
6,a_105,105,2,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers...",search,1,1,0,NaN,0,0,0,0,0,0,0,a_105,0,4,0.000000,START,NO_TOOL,START->TOOL_CALL,NO_TOOL->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers..."
7,a_105,105,4,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",search,1,1,0,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers...",2,1,0,0,0,2410,343,a_105,1,4,0.333333,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
8,a_105,105,6,TOOL_CALL,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",search,1,1,0,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",4,2,0,0,0,2420,345,a_105,2,4,0.666667,TOOL_CALL,search,TOOL_CALL->TOOL_CALL,search->search,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
9,a_105,105,8,ASSISTANT,ASSISTANT,[ASSISTANT]\nThe university with its main camp...,NO_TOOL,1,0,0,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers...",6,3,0,0,0,2244,344,a_105,3,4,1.000000,TOOL_CALL,search,TOOL_CALL->ASSISTANT,search->NO_TOOL,[ASSISTANT]\nThe university with its main camp...


In [ ]:
# ============================================================
# 15. Find role column in full event stream
# ============================================================

role_candidates = []

for col in train_events.columns:

    try:
        values = (
            train_events[col]
            .dropna()
            .astype(str)
        )

        unique_values = set(
            values.unique()
        )

        score = len(
            unique_values.intersection(
                {
                    "ASSISTANT",
                    "TOOL_CALL",
                    "USER",
                    "TOOL_RESULT",
                    "SYSTEM",
                }
            )
        )

        if score > 0:
            role_candidates.append({
                "column": col,
                "score": score,
                "n_unique": len(unique_values),
                "examples": list(unique_values)[:10],
            })

    except Exception:
        pass


role_candidate_df = (
    pd.DataFrame(role_candidates)
    .sort_values(
        ["score", "n_unique"],
        ascending=[False, True],
    )
)

display(role_candidate_df)

,column,score,n_unique,examples
0,event_role,2,2,"[ASSISTANT, TOOL_CALL]"
1,current_role,2,2,"[ASSISTANT, TOOL_CALL]"
2,previous_event_role,2,3,"[ASSISTANT, TOOL_CALL, START]"


In [ ]:
if len(role_candidate_df) == 0:
    raise RuntimeError(
        "Could not identify event-role column."
    )

FULL_EVENT_ROLE_COL = (
    role_candidate_df.iloc[0]["column"]
)

print(
    "Selected full-event role column:",
    FULL_EVENT_ROLE_COL
)

print(
    train_events[
        FULL_EVENT_ROLE_COL
    ].value_counts(dropna=False)
)

Selected full-event role column: event_role
event_role
TOOL_CALL    2256
ASSISTANT    1536
Name: count, dtype: int64


In [ ]:
# ============================================================
# 16. Validate true historical role against transition metadata
# ============================================================

full_event_roles = (
    train_events[
        FULL_EVENT_ROLE_COL
    ]
    .astype(str)
    .to_numpy()
)

last_history_idx = np.full(
    len(train_targets),
    -1,
    dtype=int,
)

last_history_role = np.full(
    len(train_targets),
    "NO_HISTORY",
    dtype=object,
)

for i, hist in enumerate(
    train_history_indices
):

    if len(hist) == 0:
        continue

    j = int(hist[-1])

    last_history_idx[i] = j
    last_history_role[i] = (
        full_event_roles[j]
    )


expected_previous_role = (
    transition_structure_df[
        "previous_role"
    ]
    .astype(str)
    .to_numpy()
)

has_history = (
    np.asarray(history_count)
    > 0
)

history_role_match = np.where(
    has_history,
    last_history_role
    == expected_previous_role,
    True,
)

print(
    "History-role reconstruction rate:",
    history_role_match.mean()
)

print(
    "Historical rows:",
    has_history.sum()
)

print(
    "Historical-role matches:",
    history_role_match[
        has_history
    ].sum(),
    "/",
    has_history.sum()
)

History-role reconstruction rate: 1.0
Historical rows: 1433
Historical-role matches: 1433 / 1433


In [ ]:
display(
    pd.crosstab(
        pd.Series(
            expected_previous_role,
            name="metadata_previous_role",
        ),
        pd.Series(
            last_history_role,
            name="actual_last_history_role",
        ),
    )
)

actual_last_history_role,ASSISTANT,NO_HISTORY,TOOL_CALL
metadata_previous_role,,,
ASSISTANT,446,0,0
NO_HISTORY,0,56,0
TOOL_CALL,0,0,987


In [ ]:
# ============================================================
# 17. Recover actual role-specific historical events
# ============================================================

N = len(train_targets)

last_tool_idx = np.full(
    N, -1, dtype=int
)

second_last_tool_idx = np.full(
    N, -1, dtype=int
)

last_assistant_idx = np.full(
    N, -1, dtype=int
)

second_last_assistant_idx = np.full(
    N, -1, dtype=int
)


for i, hist in enumerate(
    train_history_indices
):

    if len(hist) == 0:
        continue

    hist = [
        int(j)
        for j in hist
    ]

    tool_events = [
        j
        for j in hist
        if full_event_roles[j]
        == "TOOL_CALL"
    ]

    assistant_events = [
        j
        for j in hist
        if full_event_roles[j]
        == "ASSISTANT"
    ]

    if len(tool_events) >= 1:
        last_tool_idx[i] = (
            tool_events[-1]
        )

    if len(tool_events) >= 2:
        second_last_tool_idx[i] = (
            tool_events[-2]
        )

    if len(assistant_events) >= 1:
        last_assistant_idx[i] = (
            assistant_events[-1]
        )

    if len(assistant_events) >= 2:
        second_last_assistant_idx[i] = (
            assistant_events[-2]
        )


print(
    "Previous tool available:",
    (last_tool_idx >= 0).mean()
)

print(
    "Previous assistant available:",
    (last_assistant_idx >= 0).mean()
)

print(
    "Second previous tool available:",
    (second_last_tool_idx >= 0).mean()
)

print(
    "Second previous assistant available:",
    (second_last_assistant_idx >= 0).mean()
)

Previous tool available: 0.9395567494963063
Previous assistant available: 0.8629952988582942
Second previous tool available: 0.8723975822699799
Second previous assistant available: 0.661517797179315


In [ ]:
# ============================================================
# 18. Validate relation-specific source events
# ============================================================

validation_rows = []

for transition in [
    "TOOL_CALL->ASSISTANT",
    "ASSISTANT->TOOL_CALL",
    "TOOL_CALL->TOOL_CALL",
    "ASSISTANT->ASSISTANT",
]:

    mask = (
        relation_df[
            "transition_type"
        ].eq(transition)
        .to_numpy()
    )

    previous_expected = (
        transition.split("->")[0]
    )

    actual_last_role = (
        last_history_role[mask]
    )

    validation_rows.append({
        "transition_type":
            transition,

        "support":
            int(mask.sum()),

        "expected_previous_role":
            previous_expected,

        "last_history_role_match_rate":
            np.mean(
                actual_last_role
                == previous_expected
            ),
    })


relation_validation_df = (
    pd.DataFrame(
        validation_rows
    )
)

display(
    relation_validation_df.round(4)
)

,transition_type,support,expected_previous_role,last_history_role_match_rate
0,TOOL_CALL->ASSISTANT,397,TOOL_CALL,1.0
1,ASSISTANT->TOOL_CALL,220,ASSISTANT,1.0
2,TOOL_CALL->TOOL_CALL,590,TOOL_CALL,1.0
3,ASSISTANT->ASSISTANT,226,ASSISTANT,1.0


In [ ]:
# ============================================================
# 19. Role-aware historical embedding matrices
# ============================================================

E_current = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

E_events = np.asarray(
    train_event_embeddings,
    dtype=np.float32,
)

D = E_current.shape[1]

assert E_events.shape[1] == D


def gather_history_embeddings(
    history_idx
):
    out = np.zeros(
        (N, D),
        dtype=np.float32,
    )

    present = (
        history_idx >= 0
    )

    out[present] = (
        E_events[
            history_idx[present]
        ]
    )

    return out, present


E_last_tool, has_last_tool = (
    gather_history_embeddings(
        last_tool_idx
    )
)

E_last_assistant, has_last_assistant = (
    gather_history_embeddings(
        last_assistant_idx
    )
)

E_second_tool, has_second_tool = (
    gather_history_embeddings(
        second_last_tool_idx
    )
)

E_second_assistant, has_second_assistant = (
    gather_history_embeddings(
        second_last_assistant_idx
    )
)


print(
    "Current:",
    E_current.shape
)

print(
    "Last tool:",
    E_last_tool.shape,
    has_last_tool.mean()
)

print(
    "Last assistant:",
    E_last_assistant.shape,
    has_last_assistant.mean()
)

Current: (1489, 384)
Last tool: (1489, 384) 0.9395567494963063
Last assistant: (1489, 384) 0.8629952988582942


In [ ]:
# ============================================================
# 20. Relation representation helper
# ============================================================

def build_relation_representation(
    current,
    previous,
    present,
):

    delta = (
        current - previous
    )

    abs_delta = np.abs(
        delta
    )

    product = (
        current * previous
    )

    l1 = np.zeros(
        len(current),
        dtype=np.float32,
    )

    l2 = np.zeros(
        len(current),
        dtype=np.float32,
    )

    cosine = np.zeros(
        len(current),
        dtype=np.float32,
    )

    l1[present] = np.mean(
        np.abs(
            delta[present]
        ),
        axis=1,
    )

    l2[present] = np.linalg.norm(
        delta[present],
        axis=1,
    )

    current_norm = np.linalg.norm(
        current[present],
        axis=1,
    )

    previous_norm = np.linalg.norm(
        previous[present],
        axis=1,
    )

    cosine[present] = (
        np.sum(
            current[present]
            *
            previous[present],
            axis=1,
        )
        /
        (
            current_norm
            *
            previous_norm
            +
            1e-12
        )
    )

    return {
        "delta":
            delta,

        "abs_delta":
            abs_delta,

        "product":
            product,

        "l1":
            l1,

        "l2":
            l2,

        "cosine":
            cosine,

        "present":
            present.astype(
                np.float32
            ),
    }


tool_relation = (
    build_relation_representation(
        E_current,
        E_last_tool,
        has_last_tool,
    )
)

assistant_relation = (
    build_relation_representation(
        E_current,
        E_last_assistant,
        has_last_assistant,
    )
)

In [ ]:
# ============================================================
# 21. TOOL_CALL -> ASSISTANT grounding task
# ============================================================

ground_mask = (
    relation_df[
        "transition_type"
    ]
    .eq(
        "TOOL_CALL->ASSISTANT"
    )
    .to_numpy()
)

ground_idx = np.where(
    ground_mask
)[0]

y_ground = (
    true_family[
        ground_idx
    ]
    ==
    "grounding_state_error"
).astype(int)

print(
    "Support:",
    len(ground_idx)
)

print(
    "Grounding positives:",
    y_ground.sum()
)

print(
    "Prevalence:",
    y_ground.mean()
)

print(
    "Last tool availability:",
    has_last_tool[
        ground_idx
    ].mean()
)

Support: 397
Grounding positives: 97
Prevalence: 0.24433249370277077
Last tool availability: 1.0


In [ ]:
# ============================================================
# 22. Grounding relation representation comparison
# ============================================================

X_ground_scalar = np.column_stack([
    tool_relation["l1"][
        ground_idx
    ],
    tool_relation["l2"][
        ground_idx
    ],
    tool_relation["cosine"][
        ground_idx
    ],
])

X_ground_delta = (
    tool_relation[
        "delta"
    ][
        ground_idx
    ]
)

X_ground_abs_delta = (
    tool_relation[
        "abs_delta"
    ][
        ground_idx
    ]
)

X_ground_product = (
    tool_relation[
        "product"
    ][
        ground_idx
    ]
)

X_ground_current = (
    E_current[
        ground_idx
    ]
)

X_ground_current_plus_delta = (
    np.concatenate(
        [
            E_current[
                ground_idx
            ],
            tool_relation[
                "delta"
            ][
                ground_idx
            ],
        ],
        axis=1,
    )
)

grounding_representations = {
    "scalar_tool_relation":
        X_ground_scalar,

    "current_semantic":
        X_ground_current,

    "tool_relation_delta":
        X_ground_delta,

    "tool_relation_abs_delta":
        X_ground_abs_delta,

    "tool_relation_product":
        X_ground_product,

    "current_plus_tool_delta":
        X_ground_current_plus_delta,
}

for name, X in (
    grounding_representations.items()
):
    print(
        f"{name:30s}",
        X.shape
    )

scalar_tool_relation           (397, 3)
current_semantic               (397, 384)
tool_relation_delta            (397, 384)
tool_relation_abs_delta        (397, 384)
tool_relation_product          (397, 384)
current_plus_tool_delta        (397, 768)


In [ ]:
# ============================================================
# 23. Grouped grounding representation evaluation
# ============================================================

from sklearn.model_selection import (
    StratifiedGroupKFold,
)

from sklearn.pipeline import (
    make_pipeline,
)

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


if "groups_train" in globals():

    ground_groups = (
        np.asarray(
            groups_train
        )[
            ground_idx
        ]
    )

else:

    group_col = (
        "canonical_group"
        if "canonical_group"
        in train_targets.columns
        else "group_id"
    )

    ground_groups = (
        train_targets.loc[
            ground_idx,
            group_col
        ]
        .to_numpy()
    )


rows = []

for name, X in (
    grounding_representations.items()
):

    oof_prob = np.zeros(
        len(y_ground),
        dtype=float,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        y_ground,
        groups=ground_groups,
    ):

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.03,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        )

        model.fit(
            X[tr],
            y_ground[tr],
        )

        oof_prob[va] = (
            model.predict_proba(
                X[va]
            )[:, 1]
        )

    rows.append({
        "representation":
            name,

        "n_features":
            X.shape[1],

        "support":
            len(y_ground),

        "positives":
            int(
                y_ground.sum()
            ),

        "prevalence":
            y_ground.mean(),

        "pr_auc":
            average_precision_score(
                y_ground,
                oof_prob,
            ),

        "roc_auc":
            roc_auc_score(
                y_ground,
                oof_prob,
            ),
    })


grounding_relation_results = (
    pd.DataFrame(rows)
    .sort_values(
        "pr_auc",
        ascending=False,
    )
)

display(
    grounding_relation_results
    .round(4)
)

,representation,n_features,support,positives,prevalence,pr_auc,roc_auc
4,tool_relation_product,384,397,97,0.2443,0.4449,0.6336
5,current_plus_tool_delta,768,397,97,0.2443,0.4162,0.6779
2,tool_relation_delta,384,397,97,0.2443,0.4091,0.7003
1,current_semantic,384,397,97,0.2443,0.3820,0.6484
3,tool_relation_abs_delta,384,397,97,0.2443,0.3546,0.6301
0,scalar_tool_relation,3,397,97,0.2443,0.2565,0.4862


In [ ]:
# ============================================================
# 24. Generic role-aware relation task evaluator
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score


def evaluate_relation_task(
    task_name,
    transition_type,
    target_family,
    relation,
    previous_embedding,
    previous_present,
    C=0.03,
    n_splits=5,
):

    # --------------------------------------------------------
    # Select transition
    # --------------------------------------------------------

    mask = (
        relation_df["transition_type"]
        .eq(transition_type)
        .to_numpy()
    )

    idx = np.where(mask)[0]

    y = (
        true_family[idx]
        == target_family
    ).astype(int)

    print("=" * 80)
    print(task_name)
    print("Transition:", transition_type)
    print("Target:", target_family)
    print("Support:", len(idx))
    print("Positives:", int(y.sum()))
    print("Prevalence:", y.mean())
    print(
        "Previous-role embedding availability:",
        previous_present[idx].mean(),
    )

    # --------------------------------------------------------
    # Representations
    # --------------------------------------------------------

    X_scalar = np.column_stack([
        relation["l1"][idx],
        relation["l2"][idx],
        relation["cosine"][idx],
    ])

    X_current = E_current[idx]

    X_previous = previous_embedding[idx]

    X_delta = relation["delta"][idx]

    X_abs_delta = relation["abs_delta"][idx]

    X_product = relation["product"][idx]

    X_current_plus_delta = np.concatenate(
        [
            E_current[idx],
            relation["delta"][idx],
        ],
        axis=1,
    )

    X_current_plus_product = np.concatenate(
        [
            E_current[idx],
            relation["product"][idx],
        ],
        axis=1,
    )

    X_current_previous = np.concatenate(
        [
            E_current[idx],
            previous_embedding[idx],
        ],
        axis=1,
    )

    X_full_relation = np.concatenate(
        [
            E_current[idx],
            previous_embedding[idx],
            relation["delta"][idx],
            relation["product"][idx],
        ],
        axis=1,
    )

    representations = {
        "scalar_relation":
            X_scalar,

        "current_semantic":
            X_current,

        "previous_semantic":
            X_previous,

        "relation_delta":
            X_delta,

        "relation_abs_delta":
            X_abs_delta,

        "relation_product":
            X_product,

        "current_plus_delta":
            X_current_plus_delta,

        "current_plus_product":
            X_current_plus_product,

        "current_plus_previous":
            X_current_previous,

        "full_relation":
            X_full_relation,
    }

    # --------------------------------------------------------
    # Groups
    # --------------------------------------------------------

    if "groups_train" in globals():

        groups = np.asarray(
            groups_train
        )[idx]

    else:

        group_col = (
            "canonical_group"
            if "canonical_group" in train_targets.columns
            else "group_id"
        )

        groups = (
            train_targets.loc[
                idx,
                group_col
            ]
            .to_numpy()
        )

    # --------------------------------------------------------
    # OOF evaluation
    # --------------------------------------------------------

    rows = []

    oof_predictions = {}

    for name, X in representations.items():

        oof_prob = np.zeros(
            len(y),
            dtype=float,
        )

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=42,
        )

        for tr, va in cv.split(
            X,
            y,
            groups=groups,
        ):

            clf = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42,
                ),
            )

            clf.fit(
                X[tr],
                y[tr],
            )

            oof_prob[va] = (
                clf.predict_proba(
                    X[va]
                )[:, 1]
            )

        pr_auc = average_precision_score(
            y,
            oof_prob,
        )

        roc_auc = roc_auc_score(
            y,
            oof_prob,
        )

        rows.append({
            "task":
                task_name,

            "transition_type":
                transition_type,

            "target_family":
                target_family,

            "representation":
                name,

            "n_features":
                X.shape[1],

            "support":
                len(y),

            "positives":
                int(y.sum()),

            "prevalence":
                y.mean(),

            "pr_auc":
                pr_auc,

            "pr_lift":
                pr_auc / y.mean(),

            "roc_auc":
                roc_auc,
        })

        oof_predictions[name] = oof_prob

    results = (
        pd.DataFrame(rows)
        .sort_values(
            "pr_auc",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    display(
        results.round(4)
    )

    return {
        "task_name": task_name,
        "idx": idx,
        "y": y,
        "groups": groups,
        "representations": representations,
        "results": results,
        "oof_predictions": oof_predictions,
    }

In [ ]:
# ============================================================
# 25. TOOL_CALL -> TOOL_CALL : tool-use mechanism
# ============================================================

tool_use_relation_results = evaluate_relation_task(
    task_name="tool_to_tool__tool_use",
    transition_type="TOOL_CALL->TOOL_CALL",
    target_family="tool_use_error",
    relation=tool_relation,
    previous_embedding=E_last_tool,
    previous_present=has_last_tool,
    C=0.03,
)

tool_use_relation_results["results"]

tool_to_tool__tool_use
Transition: TOOL_CALL->TOOL_CALL
Target: tool_use_error
Support: 590
Positives: 136
Prevalence: 0.2305084745762712
Previous-role embedding availability: 1.0


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_previous,768,590,136,0.2305,0.7019,3.0452,0.8638
1,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,full_relation,1536,590,136,0.2305,0.6942,3.0115,0.8598
2,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_product,768,590,136,0.2305,0.6908,2.9969,0.8623
3,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_delta,768,590,136,0.2305,0.6830,2.9631,0.8587
4,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_abs_delta,384,590,136,0.2305,0.6659,2.8890,0.8574
5,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_product,384,590,136,0.2305,0.6607,2.8663,0.8456
6,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,previous_semantic,384,590,136,0.2305,0.6340,2.7506,0.8036
7,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_semantic,384,590,136,0.2305,0.6142,2.6645,0.8471
8,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_delta,384,590,136,0.2305,0.5508,2.3894,0.7233
9,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,scalar_relation,3,590,136,0.2305,0.2863,1.2419,0.6289


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_previous,768,590,136,0.230508,0.701936,3.045164,0.863809
1,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,full_relation,1536,590,136,0.230508,0.694172,3.011481,0.859776
2,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_product,768,590,136,0.230508,0.690817,2.996927,0.862335
3,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_delta,768,590,136,0.230508,0.683022,2.963112,0.858723
4,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_abs_delta,384,590,136,0.230508,0.665940,2.889004,0.857395
5,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_product,384,590,136,0.230508,0.660695,2.866250,0.845572
6,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,previous_semantic,384,590,136,0.230508,0.634031,2.750576,0.803649
7,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,current_semantic,384,590,136,0.230508,0.614192,2.664509,0.847103
8,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,relation_delta,384,590,136,0.230508,0.550769,2.389365,0.723261
9,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,scalar_relation,3,590,136,0.230508,0.286280,1.241950,0.628863


In [ ]:
# ============================================================
# 26. ASSISTANT -> TOOL_CALL : constraint mechanism
# ============================================================

constraint_relation_results = evaluate_relation_task(
    task_name="assistant_to_tool__constraint",
    transition_type="ASSISTANT->TOOL_CALL",
    target_family="constraint_error",
    relation=assistant_relation,
    previous_embedding=E_last_assistant,
    previous_present=has_last_assistant,
    C=0.03,
)

constraint_relation_results["results"]

assistant_to_tool__constraint
Transition: ASSISTANT->TOOL_CALL
Target: constraint_error
Support: 220
Positives: 57
Prevalence: 0.2590909090909091
Previous-role embedding availability: 1.0


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,previous_semantic,384,220,57,0.2591,0.6039,2.3310,0.7770
1,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,full_relation,1536,220,57,0.2591,0.5673,2.1896,0.7689
2,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_previous,768,220,57,0.2591,0.5599,2.1609,0.7606
3,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_delta,768,220,57,0.2591,0.5406,2.0867,0.7548
4,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_product,768,220,57,0.2591,0.5034,1.9430,0.7599
5,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_delta,384,220,57,0.2591,0.4880,1.8836,0.7283
6,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_product,384,220,57,0.2591,0.4867,1.8784,0.7593
7,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_semantic,384,220,57,0.2591,0.4786,1.8473,0.7489
8,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_abs_delta,384,220,57,0.2591,0.4451,1.7180,0.6947
9,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,scalar_relation,3,220,57,0.2591,0.3859,1.4893,0.6727


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,previous_semantic,384,220,57,0.259091,0.603934,2.330974,0.776988
1,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,full_relation,1536,220,57,0.259091,0.567296,2.189562,0.768916
2,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_previous,768,220,57,0.259091,0.559882,2.160949,0.760629
3,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_delta,768,220,57,0.259091,0.540649,2.086714,0.754816
4,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_product,768,220,57,0.259091,0.503425,1.943045,0.759875
5,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_delta,384,220,57,0.259091,0.488014,1.883563,0.728339
6,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_product,384,220,57,0.259091,0.486665,1.878355,0.759337
7,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_semantic,384,220,57,0.259091,0.478629,1.847342,0.748897
8,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_abs_delta,384,220,57,0.259091,0.445119,1.718005,0.694651
9,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,scalar_relation,3,220,57,0.259091,0.385868,1.489316,0.672694


In [ ]:
# ============================================================
# 27. TOOL_CALL -> ASSISTANT : grounding mechanism
#     expanded representation experiment
# ============================================================

grounding_relation_results = evaluate_relation_task(
    task_name="tool_to_assistant__grounding",
    transition_type="TOOL_CALL->ASSISTANT",
    target_family="grounding_state_error",
    relation=tool_relation,
    previous_embedding=E_last_tool,
    previous_present=has_last_tool,
    C=0.03,
)

grounding_relation_results["results"]

tool_to_assistant__grounding
Transition: TOOL_CALL->ASSISTANT
Target: grounding_state_error
Support: 397
Positives: 97
Prevalence: 0.24433249370277077
Previous-role embedding availability: 1.0


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,previous_semantic,384,397,97,0.2443,0.4957,2.0287,0.6852
1,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,full_relation,1536,397,97,0.2443,0.4725,1.9338,0.6777
2,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_previous,768,397,97,0.2443,0.4637,1.8977,0.6737
3,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_product,384,397,97,0.2443,0.4449,1.8208,0.6336
4,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_product,768,397,97,0.2443,0.4440,1.8173,0.6621
5,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_delta,768,397,97,0.2443,0.4162,1.7033,0.6779
6,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_delta,384,397,97,0.2443,0.4091,1.6745,0.7003
7,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_semantic,384,397,97,0.2443,0.3820,1.5634,0.6484
8,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_abs_delta,384,397,97,0.2443,0.3546,1.4515,0.6301
9,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,scalar_relation,3,397,97,0.2443,0.2565,1.0500,0.4862


,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,previous_semantic,384,397,97,0.244332,0.495682,2.028719,0.685241
1,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,full_relation,1536,397,97,0.244332,0.472492,1.933808,0.677732
2,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_previous,768,397,97,0.244332,0.463658,1.897652,0.673746
3,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_product,384,397,97,0.244332,0.444885,1.820817,0.633608
4,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_product,768,397,97,0.244332,0.444014,1.817252,0.662062
5,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_plus_delta,768,397,97,0.244332,0.416175,1.703314,0.677869
6,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_delta,384,397,97,0.244332,0.409124,1.674454,0.700344
7,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,current_semantic,384,397,97,0.244332,0.381996,1.563428,0.648385
8,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,relation_abs_delta,384,397,97,0.244332,0.354648,1.451499,0.630103
9,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,scalar_relation,3,397,97,0.244332,0.256549,1.049998,0.486220


In [ ]:
# ============================================================
# 28. Compare representations across all three mechanisms
# ============================================================

all_relation_results = pd.concat(
    [
        tool_use_relation_results["results"],
        grounding_relation_results["results"],
        constraint_relation_results["results"],
    ],
    ignore_index=True,
)

display(
    all_relation_results
    .sort_values(
        ["task", "pr_auc"],
        ascending=[True, False],
    )
    .round(4)
)

,task,transition_type,target_family,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
20,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,previous_semantic,384,220,57,0.2591,0.6039,2.3310,0.7770
21,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,full_relation,1536,220,57,0.2591,0.5673,2.1896,0.7689
22,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_previous,768,220,57,0.2591,0.5599,2.1609,0.7606
23,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_delta,768,220,57,0.2591,0.5406,2.0867,0.7548
24,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_plus_product,768,220,57,0.2591,0.5034,1.9430,0.7599
25,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_delta,384,220,57,0.2591,0.4880,1.8836,0.7283
26,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_product,384,220,57,0.2591,0.4867,1.8784,0.7593
27,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,current_semantic,384,220,57,0.2591,0.4786,1.8473,0.7489
28,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,relation_abs_delta,384,220,57,0.2591,0.4451,1.7180,0.6947
29,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,scalar_relation,3,220,57,0.2591,0.3859,1.4893,0.6727


In [ ]:
pr_matrix = (
    all_relation_results
    .pivot(
        index="task",
        columns="representation",
        values="pr_auc",
    )
)

display(
    pr_matrix.round(4)
)

representation,current_plus_delta,current_plus_previous,current_plus_product,current_semantic,full_relation,previous_semantic,relation_abs_delta,relation_delta,relation_product,scalar_relation
task,,,,,,,,,,
assistant_to_tool__constraint,0.5406,0.5599,0.5034,0.4786,0.5673,0.6039,0.4451,0.4880,0.4867,0.3859
tool_to_assistant__grounding,0.4162,0.4637,0.4440,0.3820,0.4725,0.4957,0.3546,0.4091,0.4449,0.2565
tool_to_tool__tool_use,0.6830,0.7019,0.6908,0.6142,0.6942,0.6340,0.6659,0.5508,0.6607,0.2863


In [ ]:
pr_lift_matrix = (
    all_relation_results
    .pivot(
        index="task",
        columns="representation",
        values="pr_lift",
    )
)

display(
    pr_lift_matrix.round(3)
)

representation,current_plus_delta,current_plus_previous,current_plus_product,current_semantic,full_relation,previous_semantic,relation_abs_delta,relation_delta,relation_product,scalar_relation
task,,,,,,,,,,
assistant_to_tool__constraint,2.087,2.161,1.943,1.847,2.190,2.331,1.718,1.884,1.878,1.489
tool_to_assistant__grounding,1.703,1.898,1.817,1.563,1.934,2.029,1.451,1.674,1.821,1.050
tool_to_tool__tool_use,2.963,3.045,2.997,2.665,3.011,2.751,2.889,2.389,2.866,1.242


In [ ]:
roc_matrix = (
    all_relation_results
    .pivot(
        index="task",
        columns="representation",
        values="roc_auc",
    )
)

display(
    roc_matrix.round(4)
)

representation,current_plus_delta,current_plus_previous,current_plus_product,current_semantic,full_relation,previous_semantic,relation_abs_delta,relation_delta,relation_product,scalar_relation
task,,,,,,,,,,
assistant_to_tool__constraint,0.7548,0.7606,0.7599,0.7489,0.7689,0.7770,0.6947,0.7283,0.7593,0.6727
tool_to_assistant__grounding,0.6779,0.6737,0.6621,0.6484,0.6777,0.6852,0.6301,0.7003,0.6336,0.4862
tool_to_tool__tool_use,0.8587,0.8638,0.8623,0.8471,0.8598,0.8036,0.8574,0.7233,0.8456,0.6289


In [ ]:
# Best representation per task

best_by_pr = (
    all_relation_results
    .sort_values(
        "pr_auc",
        ascending=False,
    )
    .groupby(
        "task",
        as_index=False,
    )
    .first()
)

display(
    best_by_pr[
        [
            "task",
            "representation",
            "n_features",
            "support",
            "positives",
            "prevalence",
            "pr_auc",
            "pr_lift",
            "roc_auc",
        ]
    ].round(4)
)

,task,representation,n_features,support,positives,prevalence,pr_auc,pr_lift,roc_auc
0,assistant_to_tool__constraint,previous_semantic,384,220,57,0.2591,0.6039,2.3310,0.7770
1,tool_to_assistant__grounding,previous_semantic,384,397,97,0.2443,0.4957,2.0287,0.6852
2,tool_to_tool__tool_use,current_plus_previous,768,590,136,0.2305,0.7019,3.0452,0.8638


In [ ]:
# ============================================================
# 29. Incremental value of relation-specific history
# ============================================================

comparison_rows = []

for result_bundle in [
    tool_use_relation_results,
    grounding_relation_results,
    constraint_relation_results,
]:

    r = result_bundle["results"].copy()

    task = r["task"].iloc[0]

    current_row = r[
        r["representation"]
        == "current_semantic"
    ].iloc[0]

    best_row = (
        r.sort_values(
            "pr_auc",
            ascending=False
        )
        .iloc[0]
    )

    previous_row = r[
        r["representation"]
        == "previous_semantic"
    ].iloc[0]

    comparison_rows.append({
        "task":
            task,

        "prevalence":
            best_row["prevalence"],

        "current_pr_auc":
            current_row["pr_auc"],

        "previous_pr_auc":
            previous_row["pr_auc"],

        "best_representation":
            best_row["representation"],

        "best_pr_auc":
            best_row["pr_auc"],

        "delta_pr_vs_current":
            best_row["pr_auc"]
            - current_row["pr_auc"],

        "current_roc_auc":
            current_row["roc_auc"],

        "previous_roc_auc":
            previous_row["roc_auc"],

        "best_roc_auc":
            best_row["roc_auc"],

        "delta_roc_vs_current":
            best_row["roc_auc"]
            - current_row["roc_auc"],
    })


relation_increment_df = (
    pd.DataFrame(comparison_rows)
)

display(
    relation_increment_df
    .round(4)
)

,task,prevalence,current_pr_auc,previous_pr_auc,best_representation,best_pr_auc,delta_pr_vs_current,current_roc_auc,previous_roc_auc,best_roc_auc,delta_roc_vs_current
0,tool_to_tool__tool_use,0.2305,0.6142,0.6340,current_plus_previous,0.7019,0.0877,0.8471,0.8036,0.8638,0.0167
1,tool_to_assistant__grounding,0.2443,0.3820,0.4957,previous_semantic,0.4957,0.1137,0.6484,0.6852,0.6852,0.0369
2,assistant_to_tool__constraint,0.2591,0.4786,0.6039,previous_semantic,0.6039,0.1253,0.7489,0.7770,0.7770,0.0281


In [ ]:
# ============================================================
# 29. Incremental value of relation-specific history
# ============================================================

comparison_rows = []

for result_bundle in [
    tool_use_relation_results,
    grounding_relation_results,
    constraint_relation_results,
]:

    r = result_bundle["results"].copy()

    task = r["task"].iloc[0]

    current_row = r[
        r["representation"]
        == "current_semantic"
    ].iloc[0]

    best_row = (
        r.sort_values(
            "pr_auc",
            ascending=False
        )
        .iloc[0]
    )

    previous_row = r[
        r["representation"]
        == "previous_semantic"
    ].iloc[0]

    comparison_rows.append({
        "task":
            task,

        "prevalence":
            best_row["prevalence"],

        "current_pr_auc":
            current_row["pr_auc"],

        "previous_pr_auc":
            previous_row["pr_auc"],

        "best_representation":
            best_row["representation"],

        "best_pr_auc":
            best_row["pr_auc"],

        "delta_pr_vs_current":
            best_row["pr_auc"]
            - current_row["pr_auc"],

        "current_roc_auc":
            current_row["roc_auc"],

        "previous_roc_auc":
            previous_row["roc_auc"],

        "best_roc_auc":
            best_row["roc_auc"],

        "delta_roc_vs_current":
            best_row["roc_auc"]
            - current_row["roc_auc"],
    })


relation_increment_df = (
    pd.DataFrame(comparison_rows)
)

display(
    relation_increment_df
    .round(4)
)

,task,prevalence,current_pr_auc,previous_pr_auc,best_representation,best_pr_auc,delta_pr_vs_current,current_roc_auc,previous_roc_auc,best_roc_auc,delta_roc_vs_current
0,tool_to_tool__tool_use,0.2305,0.6142,0.6340,current_plus_previous,0.7019,0.0877,0.8471,0.8036,0.8638,0.0167
1,tool_to_assistant__grounding,0.2443,0.3820,0.4957,previous_semantic,0.4957,0.1137,0.6484,0.6852,0.6852,0.0369
2,assistant_to_tool__constraint,0.2591,0.4786,0.6039,previous_semantic,0.6039,0.1253,0.7489,0.7770,0.7770,0.0281


In [ ]:
# ============================================================
# 30. Correct-role vs wrong-role history control
# ============================================================

def evaluate_simple_embedding_task(
    transition_type,
    target_family,
    embeddings_dict,
    random_state=42,
):

    mask = (
        relation_df["transition_type"]
        .eq(transition_type)
        .to_numpy()
    )

    idx = np.where(mask)[0]

    y = (
        true_family[idx]
        == target_family
    ).astype(int)

    if "groups_train" in globals():
        groups = np.asarray(groups_train)[idx]

    else:
        group_col = (
            "canonical_group"
            if "canonical_group"
            in train_targets.columns
            else "group_id"
        )

        groups = (
            train_targets.loc[
                idx,
                group_col
            ]
            .to_numpy()
        )

    rows = []

    for name, X_all in embeddings_dict.items():

        X = X_all[idx]

        oof = np.zeros(
            len(idx),
            dtype=float
        )

        cv = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=random_state,
        )

        for tr, va in cv.split(
            X,
            y,
            groups=groups,
        ):

            clf = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=0.03,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )
            )

            clf.fit(
                X[tr],
                y[tr]
            )

            oof[va] = (
                clf.predict_proba(
                    X[va]
                )[:, 1]
            )

        rows.append({
            "transition_type":
                transition_type,

            "target_family":
                target_family,

            "representation":
                name,

            "support":
                len(idx),

            "positives":
                int(y.sum()),

            "prevalence":
                y.mean(),

            "pr_auc":
                average_precision_score(
                    y,
                    oof
                ),

            "roc_auc":
                roc_auc_score(
                    y,
                    oof
                ),
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 30B. Grounding: last tool vs last assistant
# ============================================================

grounding_role_control = (
    evaluate_simple_embedding_task(
        transition_type=
            "TOOL_CALL->ASSISTANT",

        target_family=
            "grounding_state_error",

        embeddings_dict={
            "current":
                E_current,

            "correct_previous_tool":
                E_last_tool,

            "wrong_previous_assistant":
                E_last_assistant,
        },
    )
)

display(
    grounding_role_control
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .round(4)
)

,transition_type,target_family,representation,support,positives,prevalence,pr_auc,roc_auc
1,TOOL_CALL->ASSISTANT,grounding_state_error,correct_previous_tool,397,97,0.2443,0.4957,0.6852
0,TOOL_CALL->ASSISTANT,grounding_state_error,current,397,97,0.2443,0.3820,0.6484
2,TOOL_CALL->ASSISTANT,grounding_state_error,wrong_previous_assistant,397,97,0.2443,0.3078,0.5609


In [ ]:
# ============================================================
# 30C. Constraint: last assistant vs last tool
# ============================================================

constraint_role_control = (
    evaluate_simple_embedding_task(
        transition_type=
            "ASSISTANT->TOOL_CALL",

        target_family=
            "constraint_error",

        embeddings_dict={
            "current":
                E_current,

            "correct_previous_assistant":
                E_last_assistant,

            "wrong_previous_tool":
                E_last_tool,
        },
    )
)

display(
    constraint_role_control
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .round(4)
)

,transition_type,target_family,representation,support,positives,prevalence,pr_auc,roc_auc
1,ASSISTANT->TOOL_CALL,constraint_error,correct_previous_assistant,220,57,0.2591,0.6039,0.7770
0,ASSISTANT->TOOL_CALL,constraint_error,current,220,57,0.2591,0.4786,0.7489
2,ASSISTANT->TOOL_CALL,constraint_error,wrong_previous_tool,220,57,0.2591,0.4782,0.7487


In [ ]:
# ============================================================
# 30D. Tool-use: previous tool vs previous assistant
# ============================================================

tool_use_role_control = (
    evaluate_simple_embedding_task(
        transition_type=
            "TOOL_CALL->TOOL_CALL",

        target_family=
            "tool_use_error",

        embeddings_dict={
            "current":
                E_current,

            "correct_previous_tool":
                E_last_tool,

            "wrong_previous_assistant":
                E_last_assistant,

            "current_plus_tool":
                np.concatenate(
                    [
                        E_current,
                        E_last_tool,
                    ],
                    axis=1
                ),

            "current_plus_assistant":
                np.concatenate(
                    [
                        E_current,
                        E_last_assistant,
                    ],
                    axis=1
                ),
        },
    )
)

display(
    tool_use_role_control
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .round(4)
)

,transition_type,target_family,representation,support,positives,prevalence,pr_auc,roc_auc
3,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_tool,590,136,0.2305,0.7019,0.8638
4,TOOL_CALL->TOOL_CALL,tool_use_error,current_plus_assistant,590,136,0.2305,0.6380,0.8469
1,TOOL_CALL->TOOL_CALL,tool_use_error,correct_previous_tool,590,136,0.2305,0.6340,0.8036
0,TOOL_CALL->TOOL_CALL,tool_use_error,current,590,136,0.2305,0.6142,0.8471
2,TOOL_CALL->TOOL_CALL,tool_use_error,wrong_previous_assistant,590,136,0.2305,0.4581,0.7831


In [ ]:
# ============================================================
# 31. Build relation-specialist OOF probability channels
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

n = len(train_targets)

# ------------------------------------------------------------
# Resolve grouping
# ------------------------------------------------------------

if "groups_train" in globals():
    all_groups = np.asarray(groups_train)

else:
    group_col = (
        "canonical_group"
        if "canonical_group" in train_targets.columns
        else "group_id"
    )

    all_groups = (
        train_targets[group_col]
        .astype(str)
        .to_numpy()
    )


# ------------------------------------------------------------
# Generic OOF specialist builder
# ------------------------------------------------------------

def build_relation_specialist_oof(
    name,
    transition_type,
    target_family,
    X_all,
    random_state=42,
):

    transition_mask = (
        relation_df["transition_type"]
        .eq(transition_type)
        .to_numpy()
    )

    idx = np.where(transition_mask)[0]

    X = np.asarray(X_all)[idx]

    y = (
        np.asarray(true_family)[idx]
        == target_family
    ).astype(int)

    groups = all_groups[idx]

    oof_local = np.full(
        len(idx),
        np.nan,
        dtype=float
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state,
    )

    for fold, (tr, va) in enumerate(
        cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.03,
                class_weight="balanced",
                max_iter=5000,
                random_state=random_state,
            )
        )

        clf.fit(
            X[tr],
            y[tr],
        )

        oof_local[va] = (
            clf.predict_proba(
                X[va]
            )[:, 1]
        )

    assert not np.isnan(oof_local).any()

    # Global channel:
    # NaN means "specialist not applicable"
    oof_global = np.full(
        n,
        np.nan,
        dtype=float
    )

    oof_global[idx] = oof_local

    metrics = {
        "specialist": name,
        "transition_type": transition_type,
        "target_family": target_family,
        "support": len(idx),
        "positives": int(y.sum()),
        "prevalence": float(y.mean()),
        "pr_auc": average_precision_score(
            y,
            oof_local,
        ),
        "roc_auc": roc_auc_score(
            y,
            oof_local,
        ),
    }

    return {
        "name": name,
        "idx": idx,
        "y": y,
        "oof_local": oof_local,
        "oof_global": oof_global,
        "metrics": metrics,
    }

In [ ]:
# ============================================================
# 31B. Fit the three mechanism specialists
# ============================================================

tool_use_specialist = (
    build_relation_specialist_oof(
        name="tool_use_relation",
        transition_type=
            "TOOL_CALL->TOOL_CALL",
        target_family=
            "tool_use_error",
        X_all=np.concatenate(
            [
                E_current,
                E_last_tool,
            ],
            axis=1,
        ),
    )
)


grounding_specialist = (
    build_relation_specialist_oof(
        name="grounding_relation",
        transition_type=
            "TOOL_CALL->ASSISTANT",
        target_family=
            "grounding_state_error",
        X_all=E_last_tool,
    )
)


constraint_specialist = (
    build_relation_specialist_oof(
        name="constraint_relation",
        transition_type=
            "ASSISTANT->TOOL_CALL",
        target_family=
            "constraint_error",
        X_all=E_last_assistant,
    )
)


specialist_metric_df = pd.DataFrame([
    tool_use_specialist["metrics"],
    grounding_specialist["metrics"],
    constraint_specialist["metrics"],
])

display(
    specialist_metric_df.round(4)
)

,specialist,transition_type,target_family,support,positives,prevalence,pr_auc,roc_auc
0,tool_use_relation,TOOL_CALL->TOOL_CALL,tool_use_error,590,136,0.2305,0.7019,0.8638
1,grounding_relation,TOOL_CALL->ASSISTANT,grounding_state_error,397,97,0.2443,0.4957,0.6852
2,constraint_relation,ASSISTANT->TOOL_CALL,constraint_error,220,57,0.2591,0.6039,0.7770


In [ ]:
# ============================================================
# 32. Specialist feature matrix
# ============================================================

specialists = [
    tool_use_specialist,
    grounding_specialist,
    constraint_specialist,
]

specialist_prob_cols = []
specialist_active_cols = []

for spec in specialists:

    p = spec["oof_global"]

    active = (
        ~np.isnan(p)
    ).astype(float)

    # Neutral fill only for numerical representation.
    # Applicability is carried separately.
    p_filled = np.where(
        active == 1,
        p,
        0.5,
    )

    specialist_prob_cols.append(
        p_filled
    )

    specialist_active_cols.append(
        active
    )


X_specialist_probs = np.column_stack(
    specialist_prob_cols
)

X_specialist_active = np.column_stack(
    specialist_active_cols
)

X_specialists = np.column_stack([
    X_specialist_probs,
    X_specialist_active,
])


print(
    "Specialist probability shape:",
    X_specialist_probs.shape
)

print(
    "Specialist applicability shape:",
    X_specialist_active.shape
)

print(
    "Combined specialist shape:",
    X_specialists.shape
)

print(
    "\nApplicability counts:"
)

for j, spec in enumerate(specialists):
    print(
        spec["name"],
        int(
            X_specialist_active[:, j]
            .sum()
        )
    )

Specialist probability shape: (1489, 3)
Specialist applicability shape: (1489, 3)
Combined specialist shape: (1489, 6)

Applicability counts:
tool_use_relation 590
grounding_relation 397
constraint_relation 220


In [ ]:
# ============================================================
# 33. Locate semantic OOF probability matrix
# ============================================================

semantic_candidates = []

for name, obj in globals().items():

    if isinstance(obj, np.ndarray):

        if (
            obj.ndim == 2
            and obj.shape[0] == n
            and obj.shape[1] == len(class_names)
        ):
            semantic_candidates.append(
                (
                    name,
                    obj.shape,
                    obj.dtype,
                )
            )

semantic_candidates

[('full_prob', (1489, 5), dtype('float64')),
 ('remove_t1_prob', (1489, 5), dtype('float64'))]

In [ ]:
# ============================================================
# 33. Recover semantic probability matrix safely
# ============================================================

candidate_names = [
    "remove_t1_prob",
    "oof_sem_prob",
    "semantic_prob",
    "semantic_probs",
    "semantic_oof_prob",
]

found = []

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if (
            isinstance(obj, np.ndarray)
            and obj.ndim == 2
            and obj.shape == (len(train_targets), 5)
        ):
            found.append(name)

print("Candidate semantic probability matrices:", found)

if "remove_t1_prob" in found:
    P_semantic = np.asarray(
        remove_t1_prob,
        dtype=float,
    )

    print(
        "Using remove_t1_prob as semantic/no-t1 probability state."
    )

elif "oof_sem_prob" in found:
    P_semantic = np.asarray(
        oof_sem_prob,
        dtype=float,
    )

    print(
        "Using oof_sem_prob."
    )

elif len(found) == 1:
    P_semantic = np.asarray(
        globals()[found[0]],
        dtype=float,
    )

    print(
        "Using:",
        found[0]
    )

else:
    raise RuntimeError(
        "Could not uniquely identify the semantic probability matrix."
    )

print("Shape:", P_semantic.shape)
print(
    "Row-sum min/max:",
    P_semantic.sum(axis=1).min(),
    P_semantic.sum(axis=1).max(),
)

Candidate semantic probability matrices: ['remove_t1_prob']
Using remove_t1_prob as semantic/no-t1 probability state.
Shape: (1489, 5)
Row-sum min/max: 0.9999998284038156 1.0000002043088898


In [ ]:
prob_class_names = np.array([
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
])

In [ ]:
# ============================================================
# 34. Verify semantic baseline with correct probability order
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

prob_class_names = np.array([
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
])

y_true_name = np.asarray(
    true_family
)

semantic_idx = (
    P_semantic.argmax(axis=1)
)

semantic_pred = (
    prob_class_names[
        semantic_idx
    ]
)

print("Semantic prediction counts:")
print(
    pd.Series(
        semantic_pred
    ).value_counts()
)

baseline_metrics = {
    "accuracy":
        accuracy_score(
            y_true_name,
            semantic_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_true_name,
            semantic_pred,
        ),

    "macro_f1":
        f1_score(
            y_true_name,
            semantic_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_true_name,
            semantic_pred,
            average="weighted",
            zero_division=0,
        ),
}

print("\nSemantic/no-t1 metrics:")
print(baseline_metrics)

Semantic prediction counts:
workflow_error           873
constraint_error         303
grounding_state_error    155
tool_use_error           132
reasoning_value_error     26
Name: count, dtype: int64

Semantic/no-t1 metrics:
{'accuracy': 0.5151108126259234, 'balanced_accuracy': 0.43550722847563206, 'macro_f1': 0.4548333618176154, 'weighted_f1': 0.4969282458539298}


In [ ]:
# ============================================================
# 35. Semantic probabilities + relation-specialist channels
# ============================================================

# P_semantic: (1489, 5)
# X_specialists:
#   3 specialist probabilities
#   3 applicability indicators
#
# total = 11 features

X_meta = np.column_stack([
    P_semantic,
    X_specialists,
])

print("Meta feature shape:", X_meta.shape)

assert X_meta.shape == (
    len(train_targets),
    11,
)

print("\nFeature blocks:")
print("semantic probabilities:", P_semantic.shape)
print("specialist features:", X_specialists.shape)

Meta feature shape: (1489, 11)

Feature blocks:
semantic probabilities: (1489, 5)
specialist features: (1489, 6)


In [ ]:
# ============================================================
# 35B. Cross-fitted meta-classifier
# ============================================================

from sklearn.preprocessing import LabelEncoder

le_meta = LabelEncoder()

y_meta = le_meta.fit_transform(
    y_true_name
)

print(
    "Meta class order:",
    le_meta.classes_
)


meta_oof_prob = np.zeros(
    (
        n,
        len(le_meta.classes_),
    ),
    dtype=float,
)


cv_meta = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for fold, (tr, va) in enumerate(
    cv_meta.split(
        X_meta,
        y_meta,
        groups=all_groups,
    ),
    start=1,
):

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )
    )

    clf.fit(
        X_meta[tr],
        y_meta[tr],
    )

    meta_oof_prob[va] = (
        clf.predict_proba(
            X_meta[va]
        )
    )

    print(
        f"Fold {fold} complete"
    )


meta_pred_idx = (
    meta_oof_prob.argmax(axis=1)
)

meta_pred = (
    le_meta.inverse_transform(
        meta_pred_idx
    )
)

Meta class order: ['constraint_error' 'grounding_state_error' 'reasoning_value_error'
 'tool_use_error' 'workflow_error']
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [ ]:
# ============================================================
# 36. Evaluate relation-specialist meta-model
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

def multiclass_metrics(
    y_true,
    y_pred,
):

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
    }


semantic_metrics = (
    multiclass_metrics(
        y_true_name,
        semantic_pred,
    )
)

meta_metrics = (
    multiclass_metrics(
        y_true_name,
        meta_pred,
    )
)

comparison_df = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",
        **semantic_metrics,
    },
    {
        "model":
            "relation_specialist_meta",
        **meta_metrics,
    },
])

display(
    comparison_df.round(4)
)


print("\nMetric deltas:")

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    print(
        f"{metric:20s}",
        f"{meta_metrics[metric] - semantic_metrics[metric]:+.4f}"
    )

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,relation_specialist_meta,0.5232,0.5399,0.4925,0.5299



Metric deltas:
accuracy             +0.0081
balanced_accuracy    +0.1044
macro_f1             +0.0376
weighted_f1          +0.0329


In [ ]:
# ============================================================
# 37. Relation-meta intervention analysis
# ============================================================

semantic_correct = (
    semantic_pred
    == y_true_name
)

meta_correct = (
    meta_pred
    == y_true_name
)

changed = (
    semantic_pred
    != meta_pred
)

effect = np.full(
    len(train_targets),
    "unchanged",
    dtype=object,
)

effect[
    changed
    &
    (~semantic_correct)
    &
    meta_correct
] = "rescue"

effect[
    changed
    &
    semantic_correct
    &
    (~meta_correct)
] = "break"

effect[
    changed
    &
    (~semantic_correct)
    &
    (~meta_correct)
] = "wrong_to_wrong"

effect[
    changed
    &
    semantic_correct
    &
    meta_correct
] = "correct_to_correct"


effect_counts = (
    pd.Series(effect)
    .value_counts()
)

print(
    "Changed predictions:",
    int(changed.sum())
)

print("\nEffect distribution:")
print(effect_counts)

rescues = int(
    (effect == "rescue").sum()
)

breaks = int(
    (effect == "break").sum()
)

print(
    "\nRescues:",
    rescues
)

print(
    "Breaks:",
    breaks
)

print(
    "Net:",
    rescues - breaks
)

Changed predictions: 473

Effect distribution:
unchanged         1016
rescue             179
break              167
wrong_to_wrong     127
Name: count, dtype: int64

Rescues: 179
Breaks: 167
Net: 12


In [ ]:
# ============================================================
# 38. Family-level effect of relation specialists
# ============================================================

family_effect_rows = []

for family in np.unique(
    y_true_name
):

    mask = (
        y_true_name
        == family
    )

    support = int(
        mask.sum()
    )

    sem_recall = (
        semantic_pred[mask]
        == family
    ).mean()

    meta_recall = (
        meta_pred[mask]
        == family
    ).mean()

    family_rescues = int(
        (
            mask
            &
            (effect == "rescue")
        ).sum()
    )

    family_breaks = int(
        (
            mask
            &
            (effect == "break")
        ).sum()
    )

    family_effect_rows.append({
        "failure_family":
            family,

        "support":
            support,

        "semantic_recall":
            sem_recall,

        "meta_recall":
            meta_recall,

        "recall_delta":
            meta_recall
            -
            sem_recall,

        "rescues":
            family_rescues,

        "breaks":
            family_breaks,

        "net":
            family_rescues
            -
            family_breaks,
    })


family_effect_df = (
    pd.DataFrame(
        family_effect_rows
    )
    .sort_values(
        "recall_delta",
        ascending=False,
    )
)

display(
    family_effect_df.round(4)
)

,failure_family,support,semantic_recall,meta_recall,recall_delta,rescues,breaks,net
3,tool_use_error,237,0.2911,0.5148,0.2236,53,0,53
2,reasoning_value_error,31,0.4516,0.6452,0.1935,6,0,6
1,grounding_state_error,244,0.2992,0.4590,0.1598,39,0,39
0,constraint_error,317,0.4038,0.5489,0.1451,69,23,46
4,workflow_error,660,0.7318,0.5318,-0.2000,12,144,-132


In [ ]:
# ============================================================
# 39. Effect by transition type
# ============================================================

transition_effect_rows = []

for transition in (
    relation_df[
        "transition_type"
    ].unique()
):

    mask = (
        relation_df[
            "transition_type"
        ]
        .eq(transition)
        .to_numpy()
    )

    support = int(
        mask.sum()
    )

    base_acc = (
        semantic_pred[mask]
        ==
        y_true_name[mask]
    ).mean()

    new_acc = (
        meta_pred[mask]
        ==
        y_true_name[mask]
    ).mean()

    rescues_t = int(
        (
            mask
            &
            (effect == "rescue")
        ).sum()
    )

    breaks_t = int(
        (
            mask
            &
            (effect == "break")
        ).sum()
    )

    transition_effect_rows.append({
        "transition_type":
            transition,

        "support":
            support,

        "semantic_accuracy":
            base_acc,

        "meta_accuracy":
            new_acc,

        "accuracy_delta":
            new_acc - base_acc,

        "rescues":
            rescues_t,

        "breaks":
            breaks_t,

        "net":
            rescues_t - breaks_t,
    })


transition_effect_df = (
    pd.DataFrame(
        transition_effect_rows
    )
    .sort_values(
        "accuracy_delta",
        ascending=False,
    )
)

display(
    transition_effect_df.round(4)
)

,transition_type,support,semantic_accuracy,meta_accuracy,accuracy_delta,rescues,breaks,net
2,TOOL_CALL->TOOL_CALL,590,0.6475,0.6881,0.0407,68,44,24
1,ASSISTANT->TOOL_CALL,220,0.5091,0.5136,0.0045,30,29,1
4,NO_HISTORY->TOOL_CALL,22,0.5000,0.5000,0.0000,1,1,0
0,TOOL_CALL->ASSISTANT,397,0.4131,0.4081,-0.0050,54,56,-2
5,ASSISTANT->ASSISTANT,226,0.3451,0.3186,-0.0265,25,31,-6
3,NO_HISTORY->ASSISTANT,34,0.5882,0.4412,-0.1471,1,6,-5


In [ ]:
# ============================================================
# 40. Specialist scores by intervention outcome
# ============================================================

specialist_diagnostic_df = pd.DataFrame({
    "effect":
        effect,

    "transition_type":
        relation_df[
            "transition_type"
        ].to_numpy(),

    "true_family":
        y_true_name,

    "semantic_prediction":
        semantic_pred,

    "meta_prediction":
        meta_pred,

    "tool_use_score":
        X_specialist_probs[:, 0],

    "grounding_score":
        X_specialist_probs[:, 1],

    "constraint_score":
        X_specialist_probs[:, 2],

    "tool_use_active":
        X_specialist_active[:, 0],

    "grounding_active":
        X_specialist_active[:, 1],

    "constraint_active":
        X_specialist_active[:, 2],
})


for transition, score_col in [
    (
        "TOOL_CALL->TOOL_CALL",
        "tool_use_score",
    ),
    (
        "TOOL_CALL->ASSISTANT",
        "grounding_score",
    ),
    (
        "ASSISTANT->TOOL_CALL",
        "constraint_score",
    ),
]:

    print(
        "\n",
        "=" * 80
    )

    print(
        transition,
        "|",
        score_col
    )

    subset = (
        specialist_diagnostic_df[
            specialist_diagnostic_df[
                "transition_type"
            ].eq(transition)
        ]
    )

    display(
        subset
        .groupby("effect")[
            score_col
        ]
        .agg([
            "count",
            "mean",
            "median",
            "std",
        ])
        .round(4)
    )


TOOL_CALL->TOOL_CALL | tool_use_score


,count,mean,median,std
effect,,,,
break,44,0.5733,0.5805,0.2623
rescue,68,0.5551,0.6966,0.3476
unchanged,460,0.2472,0.0565,0.3341
wrong_to_wrong,18,0.3620,0.3400,0.3224



TOOL_CALL->ASSISTANT | grounding_score


,count,mean,median,std
effect,,,,
break,56,0.3415,0.2809,0.2691
rescue,54,0.4131,0.3149,0.3217
unchanged,225,0.4020,0.3707,0.2656
wrong_to_wrong,62,0.3899,0.3020,0.2661



ASSISTANT->TOOL_CALL | constraint_score


,count,mean,median,std
effect,,,,
break,29,0.2973,0.2368,0.2516
rescue,30,0.4897,0.4770,0.3366
unchanged,145,0.3048,0.2209,0.2690
wrong_to_wrong,16,0.3618,0.2961,0.3039


In [ ]:
# ============================================================
# 41. Construct local specialist override proposals
# ============================================================

n = len(train_targets)

override_df = pd.DataFrame({
    "true_family":
        y_true_name,

    "semantic_prediction":
        semantic_pred,

    "transition_type":
        relation_df["transition_type"].to_numpy(),

    "tool_use_score":
        X_specialist_probs[:, 0],

    "grounding_score":
        X_specialist_probs[:, 1],

    "constraint_score":
        X_specialist_probs[:, 2],
})


# Specialist target for each row
override_target = np.full(
    n,
    None,
    dtype=object,
)

override_score = np.full(
    n,
    np.nan,
    dtype=float,
)


mask = (
    override_df["transition_type"]
    .eq("TOOL_CALL->TOOL_CALL")
    .to_numpy()
)

override_target[mask] = "tool_use_error"
override_score[mask] = (
    override_df.loc[
        mask,
        "tool_use_score"
    ].to_numpy()
)


mask = (
    override_df["transition_type"]
    .eq("TOOL_CALL->ASSISTANT")
    .to_numpy()
)

override_target[mask] = "grounding_state_error"
override_score[mask] = (
    override_df.loc[
        mask,
        "grounding_score"
    ].to_numpy()
)


mask = (
    override_df["transition_type"]
    .eq("ASSISTANT->TOOL_CALL")
    .to_numpy()
)

override_target[mask] = "constraint_error"
override_score[mask] = (
    override_df.loc[
        mask,
        "constraint_score"
    ].to_numpy()
)


override_df["override_target"] = (
    override_target
)

override_df["override_score"] = (
    override_score
)

override_df["would_change"] = (
    override_df["override_target"].notna()
    &
    (
        override_df["override_target"]
        !=
        override_df["semantic_prediction"]
    )
)


print("Rows with applicable specialist:",
      override_df["override_target"].notna().sum())

print("Potential prediction changes:",
      override_df["would_change"].sum())

display(
    override_df[
        override_df["would_change"]
    ].head(20)
)

Rows with applicable specialist: 1207
Potential prediction changes: 998


,true_family,semantic_prediction,transition_type,tool_use_score,grounding_score,constraint_score,override_target,override_score,would_change
1,tool_use_error,workflow_error,ASSISTANT->TOOL_CALL,0.500000,0.500000,0.107004,constraint_error,0.107004,True
2,workflow_error,constraint_error,TOOL_CALL->TOOL_CALL,0.226582,0.500000,0.500000,tool_use_error,0.226582,True
3,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,0.500000,0.235636,0.500000,grounding_state_error,0.235636,True
4,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,0.500000,0.390784,0.500000,grounding_state_error,0.390784,True
7,constraint_error,workflow_error,ASSISTANT->TOOL_CALL,0.500000,0.500000,0.103384,constraint_error,0.103384,True
8,constraint_error,constraint_error,TOOL_CALL->ASSISTANT,0.500000,0.109790,0.500000,grounding_state_error,0.109790,True
12,workflow_error,workflow_error,TOOL_CALL->ASSISTANT,0.500000,0.060360,0.500000,grounding_state_error,0.060360,True
13,workflow_error,workflow_error,TOOL_CALL->ASSISTANT,0.500000,0.043660,0.500000,grounding_state_error,0.043660,True
21,grounding_state_error,workflow_error,TOOL_CALL->ASSISTANT,0.500000,0.871244,0.500000,grounding_state_error,0.871244,True
27,workflow_error,constraint_error,TOOL_CALL->ASSISTANT,0.500000,0.473955,0.500000,grounding_state_error,0.473955,True


In [ ]:
# ============================================================
# 42. Characterize possible specialist interventions
# ============================================================

candidate_mask = (
    override_df["would_change"]
    .to_numpy()
)

candidate = (
    override_df[
        candidate_mask
    ].copy()
)

candidate["semantic_correct"] = (
    candidate["semantic_prediction"]
    ==
    candidate["true_family"]
)

candidate["override_correct"] = (
    candidate["override_target"]
    ==
    candidate["true_family"]
)


candidate["effect"] = np.select(
    [
        (~candidate["semantic_correct"])
        &
        candidate["override_correct"],

        candidate["semantic_correct"]
        &
        (~candidate["override_correct"]),

        (~candidate["semantic_correct"])
        &
        (~candidate["override_correct"]),
    ],
    [
        "rescue",
        "break",
        "wrong_to_wrong",
    ],
    default="other",
)


print(
    candidate["effect"]
    .value_counts()
)

display(
    candidate
    .groupby(
        [
            "transition_type",
            "effect",
        ]
    )["override_score"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

effect
break             552
wrong_to_wrong    262
rescue            184
Name: count, dtype: int64


count    mean  median     std     min     max
transition_type      effect                                                       
ASSISTANT->TOOL_CALL break              88  0.2127  0.1473  0.2126  0.0056  0.8761
                     rescue             33  0.4897  0.4637  0.3171  0.0196  0.9556
                     wrong_to_wrong     49  0.2174  0.1311  0.2259  0.0039  0.8980
TOOL_CALL->ASSISTANT break             138  0.3070  0.2697  0.2270  0.0081  0.9171
                     rescue             71  0.4413  0.3797  0.2841  0.0118  0.9533
                     wrong_to_wrong    127  0.3299  0.2746  0.2317  0.0069  0.9530
TOOL_CALL->TOOL_CALL break             326  0.1435  0.0364  0.2300  0.0024  0.9655
                     rescue             80  0.5401  0.6368  0.3242  0.0069  0.9928
                     wrong_to_wrong     86  0.1998  0.0957  0.2247  0.0027  0.9512

In [ ]:
# ============================================================
# 43. Independent local-override threshold sweeps
# ============================================================

thresholds = np.arange(
    0.30,
    0.901,
    0.025,
)

specialist_specs = [
    (
        "tool_to_tool__tool_use",
        "TOOL_CALL->TOOL_CALL",
        "tool_use_error",
    ),
    (
        "tool_to_assistant__grounding",
        "TOOL_CALL->ASSISTANT",
        "grounding_state_error",
    ),
    (
        "assistant_to_tool__constraint",
        "ASSISTANT->TOOL_CALL",
        "constraint_error",
    ),
]


sweep_rows = []


for (
    task,
    transition,
    target_family,
) in specialist_specs:

    transition_mask = (
        override_df["transition_type"]
        .eq(transition)
        .to_numpy()
    )

    for threshold in thresholds:

        accept = (
            transition_mask
            &
            candidate_mask
            &
            (
                override_score
                >= threshold
            )
        )

        pred = (
            semantic_pred.copy()
        )

        pred[accept] = (
            target_family
        )

        before_correct = (
            semantic_pred
            ==
            y_true_name
        )

        after_correct = (
            pred
            ==
            y_true_name
        )

        rescues = int(
            (
                accept
                &
                (~before_correct)
                &
                after_correct
            ).sum()
        )

        breaks = int(
            (
                accept
                &
                before_correct
                &
                (~after_correct)
            ).sum()
        )

        wrong_to_wrong = int(
            (
                accept
                &
                (~before_correct)
                &
                (~after_correct)
            ).sum()
        )

        accepted = int(
            accept.sum()
        )

        precision = (
            rescues / accepted
            if accepted > 0
            else np.nan
        )

        metrics = (
            multiclass_metrics(
                y_true_name,
                pred,
            )
        )

        sweep_rows.append({
            "task":
                task,

            "transition_type":
                transition,

            "target_family":
                target_family,

            "threshold":
                threshold,

            "accepted":
                accepted,

            "coverage":
                accepted / n,

            "rescues":
                rescues,

            "breaks":
                breaks,

            "wrong_to_wrong":
                wrong_to_wrong,

            "net":
                rescues - breaks,

            "accept_precision":
                precision,

            **metrics,
        })


local_sweep_df = pd.DataFrame(
    sweep_rows
)


for task in local_sweep_df["task"].unique():

    print(
        "\n",
        "=" * 90
    )

    print(task)

    display(
        local_sweep_df[
            local_sweep_df["task"]
            == task
        ]
        .sort_values(
            [
                "net",
                "macro_f1",
            ],
            ascending=False,
        )
        .head(15)
        .round(4)
    )


tool_to_tool__tool_use


,task,transition_type,target_family,threshold,accepted,coverage,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
12,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.600,73,0.0490,43,25,5,18,0.5890,0.5272,0.4581,0.4759,0.5166
15,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.675,62,0.0416,38,21,3,17,0.6129,0.5265,0.4612,0.4791,0.5155
11,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.575,77,0.0517,44,27,6,17,0.5714,0.5265,0.4583,0.4757,0.5161
10,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.550,84,0.0564,47,31,6,16,0.5595,0.5259,0.4588,0.4753,0.5154
16,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.700,58,0.0390,35,20,3,15,0.6034,0.5252,0.4590,0.4772,0.5138
13,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.625,69,0.0463,40,25,4,15,0.5797,0.5252,0.4555,0.4737,0.5144
14,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.650,69,0.0463,40,25,4,15,0.5797,0.5252,0.4555,0.4737,0.5144
20,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.800,36,0.0242,24,9,3,15,0.6667,0.5252,0.4530,0.4726,0.5119
17,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.725,49,0.0329,30,16,3,14,0.6122,0.5245,0.4560,0.4748,0.5124
18,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.750,47,0.0316,29,15,3,14,0.6170,0.5245,0.4554,0.4743,0.5122



tool_to_assistant__grounding


,task,transition_type,target_family,threshold,accepted,coverage,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
40,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.675,41,0.0275,17,10,14,7,0.4146,0.5198,0.4451,0.4633,0.5048
39,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.650,44,0.0296,18,11,15,7,0.4091,0.5198,0.4453,0.4633,0.5049
41,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.700,35,0.0235,15,8,12,7,0.4286,0.5198,0.4447,0.4632,0.5046
44,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.775,25,0.0168,12,5,8,7,0.4800,0.5198,0.4438,0.4628,0.5042
42,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.725,33,0.0222,14,7,12,7,0.4242,0.5198,0.4442,0.4628,0.5044
43,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.750,28,0.0188,13,7,8,6,0.4643,0.5191,0.4434,0.4622,0.5036
45,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.800,21,0.0141,10,4,7,6,0.4762,0.5191,0.4425,0.4615,0.5031
47,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.850,16,0.0107,9,3,4,6,0.5625,0.5191,0.4420,0.4613,0.5028
49,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.900,10,0.0067,7,1,2,6,0.7000,0.5191,0.4409,0.4606,0.5022
48,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.875,12,0.0081,7,1,4,6,0.5833,0.5191,0.4409,0.4603,0.5023



assistant_to_tool__constraint


,task,transition_type,target_family,threshold,accepted,coverage,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
65,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.675,17,0.0114,12,3,2,9,0.7059,0.5212,0.4422,0.4608,0.5040
67,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.725,14,0.0094,11,2,1,9,0.7857,0.5212,0.4418,0.4606,0.5038
66,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.700,15,0.0101,11,2,2,9,0.7333,0.5212,0.4418,0.4605,0.5038
68,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.750,13,0.0087,10,2,1,8,0.7692,0.5205,0.4412,0.4600,0.5031
69,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.775,13,0.0087,10,2,1,8,0.7692,0.5205,0.4412,0.4600,0.5031
70,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.800,12,0.0081,9,2,1,7,0.7500,0.5198,0.4406,0.4594,0.5024
71,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.825,12,0.0081,9,2,1,7,0.7500,0.5198,0.4406,0.4594,0.5024
72,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.850,10,0.0067,8,1,1,7,0.8000,0.5198,0.4403,0.4592,0.5022
64,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.650,21,0.0141,12,5,4,7,0.5714,0.5198,0.4405,0.4587,0.5023
73,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.875,9,0.0060,7,1,1,6,0.7778,0.5191,0.4396,0.4586,0.5015


In [ ]:
# ============================================================
# 44. Summarize best local thresholds
# ============================================================

best_rows = []

for task, g in (
    local_sweep_df
    .groupby("task")
):

    positive = (
        g[
            (g["net"] > 0)
            &
            (g["accepted"] >= 5)
        ]
        .copy()
    )

    if len(positive) == 0:
        print(
            task,
            "-> no reliable positive-net threshold"
        )
        continue

    best = (
        positive
        .sort_values(
            [
                "net",
                "macro_f1",
                "balanced_accuracy",
                "accept_precision",
            ],
            ascending=False,
        )
        .iloc[0]
    )

    best_rows.append(
        best
    )


best_local_thresholds = (
    pd.DataFrame(
        best_rows
    )
)

display(
    best_local_thresholds[
        [
            "task",
            "threshold",
            "accepted",
            "coverage",
            "rescues",
            "breaks",
            "wrong_to_wrong",
            "net",
            "accept_precision",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
        ]
    ].round(4)
)

,task,threshold,accepted,coverage,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
65,assistant_to_tool__constraint,0.675,17,0.0114,12,3,2,9,0.7059,0.5212,0.4422,0.4608,0.5040
40,tool_to_assistant__grounding,0.675,41,0.0275,17,10,14,7,0.4146,0.5198,0.4451,0.4633,0.5048
12,tool_to_tool__tool_use,0.600,73,0.0490,43,25,5,18,0.5890,0.5272,0.4581,0.4759,0.5166


In [ ]:
# ============================================================
# 45. Combine selected local specialist overrides
# ============================================================

combined_pred = (
    semantic_pred.copy()
)

combined_accept = np.zeros(
    n,
    dtype=bool,
)

combined_source = np.full(
    n,
    "semantic",
    dtype=object,
)


for _, row in (
    best_local_thresholds.iterrows()
):

    transition = (
        row["transition_type"]
    )

    target = (
        row["target_family"]
    )

    threshold = float(
        row["threshold"]
    )

    accept = (
        override_df["transition_type"]
        .eq(transition)
        .to_numpy()
        &
        candidate_mask
        &
        (
            override_score
            >= threshold
        )
    )

    combined_pred[accept] = (
        target
    )

    combined_accept[accept] = True

    combined_source[accept] = (
        row["task"]
    )


combined_metrics = (
    multiclass_metrics(
        y_true_name,
        combined_pred,
    )
)


combined_correct = (
    combined_pred
    ==
    y_true_name
)

base_correct = (
    semantic_pred
    ==
    y_true_name
)


rescues = int(
    (
        combined_accept
        &
        (~base_correct)
        &
        combined_correct
    ).sum()
)

breaks = int(
    (
        combined_accept
        &
        base_correct
        &
        (~combined_correct)
    ).sum()
)

wrong_to_wrong = int(
    (
        combined_accept
        &
        (~base_correct)
        &
        (~combined_correct)
    ).sum()
)


print(
    "Accepted overrides:",
    int(combined_accept.sum())
)

print(
    "Rescues:",
    rescues
)

print(
    "Breaks:",
    breaks
)

print(
    "Wrong-to-wrong:",
    wrong_to_wrong
)

print(
    "Net:",
    rescues - breaks
)


model_comparison = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",
        **semantic_metrics,
    },
    {
        "model":
            "unrestricted_relation_meta",
        **meta_metrics,
    },
    {
        "model":
            "local_relation_overrides",
        **combined_metrics,
    },
])

display(
    model_comparison.round(4)
)

Accepted overrides: 131
Rescues: 72
Breaks: 38
Wrong-to-wrong: 21
Net: 34


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,unrestricted_relation_meta,0.5232,0.5399,0.4925,0.5299
2,local_relation_overrides,0.5379,0.4743,0.4905,0.5318


In [ ]:
# ============================================================
# 46. Family-level effect of local overrides
# ============================================================

local_family_rows = []

for family in np.unique(
    y_true_name
):

    mask = (
        y_true_name
        ==
        family
    )

    base_recall = (
        semantic_pred[mask]
        ==
        family
    ).mean()

    local_recall = (
        combined_pred[mask]
        ==
        family
    ).mean()

    rescues_f = int(
        (
            mask
            &
            combined_accept
            &
            (~base_correct)
            &
            combined_correct
        ).sum()
    )

    breaks_f = int(
        (
            mask
            &
            combined_accept
            &
            base_correct
            &
            (~combined_correct)
        ).sum()
    )

    local_family_rows.append({
        "failure_family":
            family,

        "support":
            int(mask.sum()),

        "semantic_recall":
            base_recall,

        "local_recall":
            local_recall,

        "recall_delta":
            local_recall
            -
            base_recall,

        "rescues":
            rescues_f,

        "breaks":
            breaks_f,

        "net":
            rescues_f
            -
            breaks_f,
    })


local_family_df = (
    pd.DataFrame(
        local_family_rows
    )
    .sort_values(
        "recall_delta",
        ascending=False,
    )
)

display(
    local_family_df.round(4)
)

,failure_family,support,semantic_recall,local_recall,recall_delta,rescues,breaks,net
3,tool_use_error,237,0.2911,0.4726,0.1814,43,0,43
1,grounding_state_error,244,0.2992,0.3689,0.0697,17,0,17
0,constraint_error,317,0.4038,0.4290,0.0252,12,4,8
2,reasoning_value_error,31,0.4516,0.4194,-0.0323,0,1,-1
4,workflow_error,660,0.7318,0.6818,-0.0500,0,33,-33


In [ ]:
# ============================================================
# 47. Cross-fitted threshold selection for local overrides
# ============================================================

from sklearn.model_selection import StratifiedKFold

# We stratify using the true family so each outer fold has
# approximately similar family composition.
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

threshold_grid = np.arange(
    0.30,
    0.901,
    0.025,
)

crossfit_pred = semantic_pred.copy()

crossfit_accept = np.zeros(
    n,
    dtype=bool,
)

crossfit_source = np.full(
    n,
    "semantic",
    dtype=object,
)

crossfit_threshold = np.full(
    n,
    np.nan,
    dtype=float,
)

threshold_records = []


def evaluate_threshold(
    indices,
    transition,
    target_family,
    threshold,
):
    """
    Evaluate one local override threshold on a specified
    subset of rows.
    """

    indices = np.asarray(indices)

    applicable = (
        override_df["transition_type"]
        .eq(transition)
        .to_numpy()
    )

    accept = (
        applicable
        &
        candidate_mask
        &
        (override_score >= threshold)
    )

    # Restrict to requested subset.
    subset_mask = np.zeros(
        n,
        dtype=bool,
    )
    subset_mask[indices] = True

    accept &= subset_mask

    pred = semantic_pred.copy()
    pred[accept] = target_family

    before_correct = (
        semantic_pred == y_true_name
    )

    after_correct = (
        pred == y_true_name
    )

    rescues = int(
        (
            accept
            &
            (~before_correct)
            &
            after_correct
        ).sum()
    )

    breaks = int(
        (
            accept
            &
            before_correct
            &
            (~after_correct)
        ).sum()
    )

    wrong_to_wrong = int(
        (
            accept
            &
            (~before_correct)
            &
            (~after_correct)
        ).sum()
    )

    accepted = int(
        accept.sum()
    )

    return {
        "accepted": accepted,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong_to_wrong,
        "net": rescues - breaks,
        "accept_precision": (
            rescues / accepted
            if accepted > 0
            else np.nan
        ),
    }


for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(
        np.zeros(n),
        y_true_name,
    ),
    start=1,
):

    print(
        "\n",
        "=" * 80
    )
    print("OUTER FOLD", fold)

    for (
        task,
        transition,
        target_family,
    ) in specialist_specs:

        train_results = []

        for threshold in threshold_grid:

            result = evaluate_threshold(
                train_idx,
                transition,
                target_family,
                threshold,
            )

            train_results.append({
                "threshold": threshold,
                **result,
            })

        train_results = pd.DataFrame(
            train_results
        )

        # Require a minimum number of accepted interventions
        # and positive net benefit on training folds.
        eligible = train_results[
            (train_results["accepted"] >= 5)
            &
            (train_results["net"] > 0)
        ].copy()

        if len(eligible) == 0:

            selected_threshold = np.inf

            print(
                task,
                "-> no positive-net threshold; disabled"
            )

        else:

            # Primary objective:
            # maximize net rescues.
            #
            # Tie breakers:
            # intervention precision, then higher threshold.
            selected = (
                eligible
                .sort_values(
                    [
                        "net",
                        "accept_precision",
                        "threshold",
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

            selected_threshold = float(
                selected["threshold"]
            )

            print(
                task,
                "threshold =",
                round(selected_threshold, 3),
                "| train net =",
                int(selected["net"]),
                "| accepted =",
                int(selected["accepted"]),
                "| precision =",
                round(
                    selected["accept_precision"],
                    3,
                ),
            )

        # Apply selected threshold ONLY to held-out fold.
        test_mask = np.zeros(
            n,
            dtype=bool,
        )
        test_mask[test_idx] = True

        accept_test = (
            test_mask
            &
            override_df["transition_type"]
            .eq(transition)
            .to_numpy()
            &
            candidate_mask
            &
            (
                override_score
                >= selected_threshold
            )
        )

        crossfit_pred[
            accept_test
        ] = target_family

        crossfit_accept[
            accept_test
        ] = True

        crossfit_source[
            accept_test
        ] = task

        crossfit_threshold[
            accept_test
        ] = selected_threshold

        # Record held-out outcome for diagnostics.
        before_correct = (
            semantic_pred == y_true_name
        )

        after_correct = (
            crossfit_pred == y_true_name
        )

        test_rescues = int(
            (
                accept_test
                &
                (~before_correct)
                &
                after_correct
            ).sum()
        )

        test_breaks = int(
            (
                accept_test
                &
                before_correct
                &
                (~after_correct)
            ).sum()
        )

        threshold_records.append({
            "fold": fold,
            "task": task,
            "transition_type": transition,
            "target_family": target_family,
            "selected_threshold":
                selected_threshold,
            "test_accepted":
                int(accept_test.sum()),
            "test_rescues":
                test_rescues,
            "test_breaks":
                test_breaks,
            "test_net":
                test_rescues - test_breaks,
        })


crossfit_threshold_df = pd.DataFrame(
    threshold_records
)

display(
    crossfit_threshold_df.round(4)
)


OUTER FOLD 1
tool_to_tool__tool_use threshold = 0.675 | train net = 20 | accepted = 46 | precision = 0.696
tool_to_assistant__grounding threshold = 0.625 | train net = 7 | accepted = 46 | precision = 0.435
assistant_to_tool__constraint threshold = 0.775 | train net = 5 | accepted = 10 | precision = 0.7

OUTER FOLD 2
tool_to_tool__tool_use threshold = 0.6 | train net = 16 | accepted = 59 | precision = 0.593
tool_to_assistant__grounding threshold = 0.775 | train net = 7 | accepted = 16 | precision = 0.625
assistant_to_tool__constraint threshold = 0.725 | train net = 8 | accepted = 13 | precision = 0.769

OUTER FOLD 3
tool_to_tool__tool_use threshold = 0.675 | train net = 16 | accepted = 50 | precision = 0.64
tool_to_assistant__grounding threshold = 0.9 | train net = 3 | accepted = 7 | precision = 0.571
assistant_to_tool__constraint threshold = 0.725 | train net = 7 | accepted = 9 | precision = 0.889

OUTER FOLD 4
tool_to_tool__tool_use threshold = 0.8 | train net = 10 | accepted = 31 | 

,fold,task,transition_type,target_family,selected_threshold,test_accepted,test_rescues,test_breaks,test_net
0,1,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.675,16,6,9,-3
1,1,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.625,8,2,3,-1
2,1,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.775,3,3,0,3
3,2,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.600,14,8,6,2
4,2,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.775,9,2,2,0
5,2,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.725,1,1,0,1
6,3,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.675,12,6,5,1
7,3,tool_to_assistant__grounding,TOOL_CALL->ASSISTANT,grounding_state_error,0.900,3,3,0,3
8,3,assistant_to_tool__constraint,ASSISTANT->TOOL_CALL,constraint_error,0.725,5,3,1,2
9,4,tool_to_tool__tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.800,5,5,0,5


In [ ]:
# ============================================================
# 48. Evaluate cross-fitted local override policy
# ============================================================

crossfit_metrics = multiclass_metrics(
    y_true_name,
    crossfit_pred,
)

base_correct = (
    semantic_pred == y_true_name
)

crossfit_correct = (
    crossfit_pred == y_true_name
)

cf_rescues = int(
    (
        crossfit_accept
        &
        (~base_correct)
        &
        crossfit_correct
    ).sum()
)

cf_breaks = int(
    (
        crossfit_accept
        &
        base_correct
        &
        (~crossfit_correct)
    ).sum()
)

cf_wrong = int(
    (
        crossfit_accept
        &
        (~base_correct)
        &
        (~crossfit_correct)
    ).sum()
)


print(
    "Cross-fitted accepted overrides:",
    int(crossfit_accept.sum())
)

print("Rescues:", cf_rescues)
print("Breaks:", cf_breaks)
print("Wrong-to-wrong:", cf_wrong)
print("Net:", cf_rescues - cf_breaks)


comparison_cf = pd.DataFrame([
    {
        "model": "semantic_no_t1",
        **semantic_metrics,
    },
    {
        "model": "unrestricted_relation_meta",
        **meta_metrics,
    },
    {
        "model": "optimistic_local_overrides",
        **combined_metrics,
    },
    {
        "model": "crossfit_local_overrides",
        **crossfit_metrics,
    },
])

display(
    comparison_cf.round(4)
)

Cross-fitted accepted overrides: 112
Rescues: 52
Breaks: 39
Wrong-to-wrong: 21
Net: 13


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,unrestricted_relation_meta,0.5232,0.5399,0.4925,0.5299
2,optimistic_local_overrides,0.5379,0.4743,0.4905,0.5318
3,crossfit_local_overrides,0.5238,0.4516,0.4691,0.5159


In [ ]:
# ============================================================
# 49. Cross-fitted intervention outcomes by specialist
# ============================================================

effect = np.full(
    n,
    "unchanged",
    dtype=object,
)

effect[
    crossfit_accept
    &
    (~base_correct)
    &
    crossfit_correct
] = "rescue"

effect[
    crossfit_accept
    &
    base_correct
    &
    (~crossfit_correct)
] = "break"

effect[
    crossfit_accept
    &
    (~base_correct)
    &
    (~crossfit_correct)
] = "wrong_to_wrong"


cf_detail = pd.DataFrame({
    "source": crossfit_source,
    "effect": effect,
    "accepted": crossfit_accept,
})


cf_specialist_summary = []

for task, _, _ in specialist_specs:

    mask = (
        crossfit_source == task
    )

    accepted = int(mask.sum())

    rescues = int(
        (
            mask
            &
            (effect == "rescue")
        ).sum()
    )

    breaks = int(
        (
            mask
            &
            (effect == "break")
        ).sum()
    )

    wrong = int(
        (
            mask
            &
            (effect == "wrong_to_wrong")
        ).sum()
    )

    cf_specialist_summary.append({
        "task": task,
        "accepted": accepted,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "accept_precision": (
            rescues / accepted
            if accepted
            else np.nan
        ),
    })


cf_specialist_summary = pd.DataFrame(
    cf_specialist_summary
)

display(
    cf_specialist_summary.round(4)
)

,task,accepted,rescues,breaks,wrong_to_wrong,net,accept_precision
0,tool_to_tool__tool_use,61,31,25,5,6,0.5082
1,tool_to_assistant__grounding,38,11,12,15,-1,0.2895
2,assistant_to_tool__constraint,13,10,2,1,8,0.7692


In [ ]:
# ============================================================
# 50. Inspect threshold stability across folds
# ============================================================

threshold_stability = (
    crossfit_threshold_df
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .groupby("task")
    .agg(
        folds=("fold", "count"),
        active_folds=(
            "selected_threshold",
            "count",
        ),
        mean_threshold=(
            "selected_threshold",
            "mean",
        ),
        std_threshold=(
            "selected_threshold",
            "std",
        ),
        min_threshold=(
            "selected_threshold",
            "min",
        ),
        max_threshold=(
            "selected_threshold",
            "max",
        ),
        total_accepted=(
            "test_accepted",
            "sum",
        ),
        total_rescues=(
            "test_rescues",
            "sum",
        ),
        total_breaks=(
            "test_breaks",
            "sum",
        ),
        total_net=(
            "test_net",
            "sum",
        ),
    )
    .reset_index()
)

display(
    threshold_stability.round(4)
)

,task,folds,active_folds,mean_threshold,std_threshold,min_threshold,max_threshold,total_accepted,total_rescues,total_breaks,total_net
0,assistant_to_tool__constraint,5,5,0.735,0.0224,0.725,0.775,13,10,2,8
1,tool_to_assistant__grounding,5,5,0.730,0.1110,0.625,0.900,38,11,12,-1
2,tool_to_tool__tool_use,5,5,0.670,0.0818,0.600,0.800,61,31,25,6


The optimistic local policy gave net +34, but after honest threshold selection that falls to net +13. So there was meaningful threshold-selection optimism. However, the entire transition-mechanism hypothesis does not disappear. Instead, the three specialists separate sharply:

Specialist	Accepted	Rescues	Breaks	Net	Rescue / accepted
Assistant → Tool / constraint	13	10	2	+8	76.9%
Tool → Tool / tool-use	61	31	25	+6	50.8%
Tool → Assistant / grounding	38	11	12	−1	28.9%

The strongest result is now actually Assistant→Tool / constraint, not Tool→Tool. Its threshold is also extremely stable: mean 0.735, SD 0.022, range 0.725–0.775, and it is positive in four folds and neutral in one. That's unusually clean.

The Tool→Tool mechanism survives, but more weakly: net +6, with thresholds ranging 0.60–0.80. It is positive in folds 2–5 but fails in fold 1 (−3). So there is genuine signal, but the decision boundary is less stable.

The grounding specialist essentially fails as an intervention. Its PR-AUC experiment showed that previous-tool information predicts grounding better than current semantics, but predictive information ≠ safe override information. Cross-fitted intervention gives 11 rescues / 12 breaks, net −1, with the most unstable threshold (0.625–0.900). That's a useful negative result.

Also, don't choose between the unrestricted meta-model and crossfit overrides just from balanced accuracy yet. They answer different questions. The unrestricted model has balanced_accuracy=.5399, but it rewrites hundreds of examples and badly damages workflow recall. The crossfit policy is a conservative causal-style intervention test: only 112 predictions changed and still produced net +13.

The next experiment should isolate the two surviving mechanisms and determine whether grounding is actually hurting the combined policy.

In [ ]:
# ============================================================
# 51. Cross-fitted specialist ablation
# ============================================================

specialists = [
    "tool_to_tool__tool_use",
    "tool_to_assistant__grounding",
    "assistant_to_tool__constraint",
]

ablation_rows = []

# Evaluate every non-empty subset.
from itertools import combinations

for r in range(1, len(specialists) + 1):

    for subset in combinations(specialists, r):

        subset = set(subset)

        pred = semantic_pred.copy()

        accept = np.zeros(
            n,
            dtype=bool,
        )

        for task in subset:

            mask = (
                crossfit_source == task
            )

            pred[mask] = crossfit_pred[mask]
            accept |= mask

        after_correct = (
            pred == y_true_name
        )

        rescues = int(
            (
                accept
                &
                (~base_correct)
                &
                after_correct
            ).sum()
        )

        breaks = int(
            (
                accept
                &
                base_correct
                &
                (~after_correct)
            ).sum()
        )

        wrong = int(
            (
                accept
                &
                (~base_correct)
                &
                (~after_correct)
            ).sum()
        )

        metrics = multiclass_metrics(
            y_true_name,
            pred,
        )

        ablation_rows.append({
            "specialists": " + ".join(
                sorted(subset)
            ),
            "n_specialists": len(subset),
            "accepted": int(accept.sum()),
            "rescues": rescues,
            "breaks": breaks,
            "wrong_to_wrong": wrong,
            "net": rescues - breaks,
            "accept_precision": (
                rescues / accept.sum()
                if accept.sum()
                else np.nan
            ),
            **metrics,
        })


ablation_df = pd.DataFrame(
    ablation_rows
)

display(
    ablation_df
    .sort_values(
        [
            "net",
            "macro_f1",
        ],
        ascending=False,
    )
    .round(4)
)

,specialists,n_specialists,accepted,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,assistant_to_tool__constraint + tool_to_tool__...,2,74,41,27,6,14,0.5541,0.5245,0.4536,0.4717,0.5139
6,assistant_to_tool__constraint + tool_to_assist...,3,112,52,39,21,13,0.4643,0.5238,0.4516,0.4691,0.5159
2,assistant_to_tool__constraint,1,13,10,2,1,8,0.7692,0.5205,0.4412,0.4600,0.5031
5,assistant_to_tool__constraint + tool_to_assist...,2,51,21,14,16,7,0.4118,0.5198,0.4391,0.4576,0.5052
0,tool_to_tool__tool_use,1,61,31,25,5,6,0.5082,0.5191,0.4479,0.4665,0.5077
3,tool_to_assistant__grounding + tool_to_tool__t...,2,99,42,37,20,5,0.4242,0.5185,0.4459,0.4638,0.5096
1,tool_to_assistant__grounding,1,38,11,12,15,-1,0.2895,0.5144,0.4334,0.4523,0.4989


In [ ]:
# ============================================================
# 52. Two-specialist cross-fitted policy
# ============================================================

keep_specialists = {
    "tool_to_tool__tool_use",
    "assistant_to_tool__constraint",
}

two_spec_mask = np.isin(
    crossfit_source,
    list(keep_specialists),
)

two_spec_pred = semantic_pred.copy()

two_spec_pred[
    two_spec_mask
] = crossfit_pred[
    two_spec_mask
]


two_spec_metrics = multiclass_metrics(
    y_true_name,
    two_spec_pred,
)

print(
    "Two-specialist metrics:",
    two_spec_metrics
)


family_rows = []

for family in np.unique(y_true_name):
    fam = (
        y_true_name == family
    )

    support = int(fam.sum())

    semantic_recall = (
        (semantic_pred[fam] == family).mean()
    )

    two_spec_recall = (
        (two_spec_pred[fam] == family).mean()
    )

    rescue = int(
        (
            fam
            &
            (~base_correct)
            &
            (two_spec_pred == y_true_name)
        ).sum()
    )

    brk = int(
        (
            fam
            &
            base_correct
            &
            (two_spec_pred != y_true_name)
        ).sum()
    )

    family_rows.append({
        "failure_family": family,
        "support": support,
        "semantic_recall": semantic_recall,
        "two_spec_recall": two_spec_recall,
        "recall_delta":
            two_spec_recall - semantic_recall,
        "rescues": rescue,
        "breaks": brk,
        "net": rescue - brk,
    })


two_spec_family_df = pd.DataFrame(
    family_rows
)

display(
    two_spec_family_df
    .sort_values(
        "recall_delta",
        ascending=False,
    )
    .round(4)
)

Two-specialist metrics: {'accuracy': 0.5245130960376091, 'balanced_accuracy': 0.4536463135113452, 'macro_f1': 0.47172254357764337, 'weighted_f1': 0.5138522153740289}


,failure_family,support,semantic_recall,two_spec_recall,recall_delta,rescues,breaks,net
3,tool_use_error,237,0.2911,0.4219,0.1308,31,0,31
0,constraint_error,317,0.4038,0.4353,0.0315,10,0,10
1,grounding_state_error,244,0.2992,0.2992,0.0000,0,0,0
2,reasoning_value_error,31,0.4516,0.4194,-0.0323,0,1,-1
4,workflow_error,660,0.7318,0.6924,-0.0394,0,26,-26


In [ ]:
# ============================================================
# 53. Diagnose cross-fitted specialist breaks
# ============================================================

diagnostic_df = pd.DataFrame({
    "true_family": y_true_name,
    "semantic_prediction": semantic_pred,
    "crossfit_prediction": crossfit_pred,
    "transition_type":
        override_df["transition_type"].to_numpy(),
    "specialist": crossfit_source,
    "threshold": crossfit_threshold,
    "accepted": crossfit_accept,
    "effect": effect,
})


break_df = diagnostic_df[
    diagnostic_df["effect"] == "break"
].copy()


break_summary = (
    break_df
    .groupby(
        [
            "specialist",
            "true_family",
        ]
    )
    .size()
    .rename("breaks")
    .reset_index()
    .sort_values(
        "breaks",
        ascending=False,
    )
)

display(
    break_summary
)


rescue_df = diagnostic_df[
    diagnostic_df["effect"] == "rescue"
].copy()


rescue_summary = (
    rescue_df
    .groupby(
        [
            "specialist",
            "true_family",
        ]
    )
    .size()
    .rename("rescues")
    .reset_index()
    .sort_values(
        "rescues",
        ascending=False,
    )
)

display(
    rescue_summary
)

,specialist,true_family,breaks
5,tool_to_tool__tool_use,workflow_error,24
3,tool_to_assistant__grounding,workflow_error,7
1,tool_to_assistant__grounding,constraint_error,4
0,assistant_to_tool__constraint,workflow_error,2
2,tool_to_assistant__grounding,reasoning_value_error,1
4,tool_to_tool__tool_use,reasoning_value_error,1


,specialist,true_family,rescues
2,tool_to_tool__tool_use,tool_use_error,31
1,tool_to_assistant__grounding,grounding_state_error,11
0,assistant_to_tool__constraint,constraint_error,10


The best cross-fitted intervention is the two-specialist policy:

Assistant→Tool / constraint + Tool→Tool / tool-use

It produces:

74 accepted overrides
41 rescues
27 breaks
Net +14
acceptance precision 55.4%
accuracy 0.5245, versus semantic baseline 0.5151
macro-F1 0.4717, versus 0.4548
weighted-F1 0.5139, versus 0.4969

Grounding contributes negatively. Adding it changes +14 → +13, while grounding alone is −1. More importantly, its 38 interventions contain only 11 rescues versus 12 breaks and 15 wrong-to-wrong changes. It should remain a diagnostic representation result, not a classification override.

The family decomposition tells us exactly where the two-specialist policy succeeds and fails:

Family	Recall before	Recall after	Δ
tool_use_error	.2911	.4219	+.1308
constraint_error	.4038	.4353	+.0315
grounding_state_error	.2992	.2992	0
reasoning_value_error	.4516	.4194	−.0323
workflow_error	.7318	.6924	−.0394

And the break diagnosis is extremely revealing:

Tool→Tool specialist: 24/25 breaks are workflow errors
Assistant→Tool specialist: 2/2 breaks are workflow errors
Therefore 26/27 two-specialist breaks are workflow_error
The remaining break is one reasoning-value error.

So the specialists are not generally unsafe. They have one very specific failure mode:

They confuse local transition failures with workflow-level failures.

That's a much stronger mechanistic finding.

The next experiment should therefore not add more specialists. We should test whether the semantic model's workflow probability can act as a veto/gate on otherwise accepted specialist overrides

In [ ]:
# ============================================================
# 54. Workflow-protection gate for validated specialists
# ============================================================

# Recover class ordering directly from the probability matrix.
# remove_t1_prob columns must correspond to the model's encoded classes.
#
# If your encoder exists, use it. Otherwise inspect candidates below.

candidate_class_vars = [
    "classes",
    "class_names",
    "label_classes",
    "semantic_classes",
    "le_classes",
]

for name in candidate_class_vars:
    if name in globals():
        obj = globals()[name]
        try:
            print(name, "=", list(obj))
        except Exception:
            print(name, "=", obj)

# Also inspect likely encoders
for name, obj in globals().copy().items():
    if hasattr(obj, "classes_"):
        try:
            cls = list(obj.classes_)
            if len(cls) == remove_t1_prob.shape[1]:
                print(
                    "Possible encoder:",
                    name,
                    "classes_ =",
                    cls,
                )
        except Exception:
            pass

class_names = ['constraint_error', 'grounding_state_error', 'reasoning_value_error', 'tool_use_error', 'workflow_error']
Possible encoder: le_meta classes_ = ['constraint_error', 'grounding_state_error', 'reasoning_value_error', 'tool_use_error', 'workflow_error']
Possible encoder: clf classes_ = [0, 1, 2, 3, 4]


In [ ]:
# ============================================================
# 54. Recover probability-column order automatically
# ============================================================

semantic_argmax = np.argmax(
    remove_t1_prob,
    axis=1,
)

print("Probability matrix shape:", remove_t1_prob.shape)
print("Argmax columns:", np.unique(semantic_argmax))
print()


# Recover column -> semantic label mapping
column_to_label = {}

for col in range(remove_t1_prob.shape[1]):

    mask = semantic_argmax == col

    if mask.sum() == 0:
        print(
            f"Column {col}: never wins argmax -- cannot infer directly"
        )
        continue

    labels, counts = np.unique(
        semantic_pred[mask],
        return_counts=True,
    )

    order = np.argsort(counts)[::-1]

    print(
        f"Column {col}:",
        list(
            zip(
                labels[order],
                counts[order],
            )
        )
    )

    # The semantic prediction associated with this winning column
    column_to_label[col] = labels[
        order[0]
    ]


print("\nRecovered mapping:")
print(column_to_label)


# Sanity check: mapping should be one-to-one
assert len(column_to_label) == remove_t1_prob.shape[1], (
    "At least one probability column never wins argmax."
)

assert len(set(column_to_label.values())) == remove_t1_prob.shape[1], (
    "Recovered labels are not one-to-one."
)


# Reconstruct predictions from recovered mapping
reconstructed_semantic_pred = np.array(
    [
        column_to_label[col]
        for col in semantic_argmax
    ]
)

agreement = np.mean(
    reconstructed_semantic_pred
    == semantic_pred
)

print(
    "\nReconstruction agreement:",
    agreement,
)

assert np.isclose(agreement, 1.0), (
    "Probability-column mapping does not perfectly reproduce semantic_pred."
)


# Find workflow column
workflow_idx = [
    col
    for col, label in column_to_label.items()
    if label == "workflow_error"
][0]

print(
    "workflow_error probability column:",
    workflow_idx,
)


workflow_prob = remove_t1_prob[
    :,
    workflow_idx,
]

print(
    "workflow_prob min/max:",
    workflow_prob.min(),
    workflow_prob.max(),
)

Probability matrix shape: (1489, 5)
Argmax columns: [0 1 2 3 4]

Column 0: [('workflow_error', 873)]
Column 1: [('constraint_error', 303)]
Column 2: [('tool_use_error', 132)]
Column 3: [('grounding_state_error', 155)]
Column 4: [('reasoning_value_error', 26)]

Recovered mapping:
{0: 'workflow_error', 1: 'constraint_error', 2: 'tool_use_error', 3: 'grounding_state_error', 4: 'reasoning_value_error'}

Reconstruction agreement: 1.0
workflow_error probability column: 0
workflow_prob min/max: 0.00877697579562664 0.96692955493927


In [ ]:
# ============================================================
# 54b. Workflow-protection gate
# ============================================================

validated_specialists = {
    "tool_to_tool__tool_use",
    "assistant_to_tool__constraint",
}

validated_mask = (
    crossfit_accept
    &
    np.isin(
        crossfit_source,
        list(validated_specialists),
    )
)


workflow_gate_rows = []

for threshold in np.arange(
    0.10,
    0.91,
    0.025,
):

    # Specialist override allowed only when the
    # semantic model is NOT sufficiently confident
    # that this is a workflow error.
    accept = (
        validated_mask
        &
        (workflow_prob < threshold)
    )

    pred = semantic_pred.copy()

    pred[accept] = crossfit_pred[accept]

    after_correct = (
        pred == y_true_name
    )

    rescues = int(
        (
            accept
            &
            (~base_correct)
            &
            after_correct
        ).sum()
    )

    breaks = int(
        (
            accept
            &
            base_correct
            &
            (~after_correct)
        ).sum()
    )

    wrong = int(
        (
            accept
            &
            (~base_correct)
            &
            (~after_correct)
        ).sum()
    )

    metrics = multiclass_metrics(
        y_true_name,
        pred,
    )

    workflow_gate_rows.append({
        "workflow_threshold": threshold,
        "accepted": int(accept.sum()),
        "coverage": accept.mean(),
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "accept_precision": (
            rescues / accept.sum()
            if accept.sum()
            else np.nan
        ),
        **metrics,
    })


workflow_gate_df = pd.DataFrame(
    workflow_gate_rows
)


display(
    workflow_gate_df
    .sort_values(
        [
            "net",
            "macro_f1",
        ],
        ascending=False,
    )
    .head(20)
    .round(4)
)

,workflow_threshold,accepted,coverage,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
28,0.800,73,0.0490,41,26,6,15,0.5616,0.5252,0.4539,0.4721,0.5144
29,0.825,73,0.0490,41,26,6,15,0.5616,0.5252,0.4539,0.4721,0.5144
30,0.850,73,0.0490,41,26,6,15,0.5616,0.5252,0.4539,0.4721,0.5144
31,0.875,73,0.0490,41,26,6,15,0.5616,0.5252,0.4539,0.4721,0.5144
27,0.775,71,0.0477,40,25,6,15,0.5634,0.5252,0.4536,0.4720,0.5143
32,0.900,74,0.0497,41,27,6,14,0.5541,0.5245,0.4536,0.4717,0.5139
21,0.625,56,0.0376,32,18,6,14,0.5714,0.5245,0.4492,0.4683,0.5122
22,0.650,56,0.0376,32,18,6,14,0.5714,0.5245,0.4492,0.4683,0.5122
26,0.750,69,0.0463,38,25,6,13,0.5507,0.5238,0.4519,0.4703,0.5128
23,0.675,57,0.0383,32,19,6,13,0.5614,0.5238,0.4489,0.4680,0.5117


In [ ]:
# ============================================================
# 55. Does workflow confidence distinguish specialist
#     rescues from specialist breaks?
# ============================================================

validated_effect = np.full(
    len(y_true_name),
    "not_accepted",
    dtype=object,
)

validated_after_correct = (
    crossfit_pred == y_true_name
)

validated_effect[
    validated_mask
    &
    (~base_correct)
    &
    validated_after_correct
] = "rescue"

validated_effect[
    validated_mask
    &
    base_correct
    &
    (~validated_after_correct)
] = "break"

validated_effect[
    validated_mask
    &
    (~base_correct)
    &
    (~validated_after_correct)
] = "wrong_to_wrong"


workflow_effect_df = pd.DataFrame({
    "specialist": crossfit_source,
    "true_family": y_true_name,
    "semantic_prediction": semantic_pred,
    "specialist_prediction": crossfit_pred,
    "workflow_prob": workflow_prob,
    "effect": validated_effect,
})


accepted_effect_df = workflow_effect_df[
    workflow_effect_df["effect"] != "not_accepted"
].copy()


print("Overall workflow probability by effect:")
display(
    accepted_effect_df
    .groupby("effect")["workflow_prob"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)


print("\nBy specialist:")
display(
    accepted_effect_df
    .groupby(
        [
            "specialist",
            "effect",
        ]
    )["workflow_prob"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

Overall workflow probability by effect:


,count,mean,median,std,min,max
effect,,,,,,
break,27,0.5789,0.5401,0.1435,0.1273,0.8886
rescue,41,0.5640,0.5559,0.1132,0.2418,0.7849
wrong_to_wrong,6,0.4615,0.4648,0.1503,0.2011,0.6236



By specialist:


count    mean  median     std     min     max
specialist                    effect                                                       
assistant_to_tool__constraint break               2  0.6871  0.6871  0.0235  0.6704  0.7037
                              rescue             10  0.5786  0.5659  0.1132  0.4047  0.7849
                              wrong_to_wrong      1  0.4249  0.4249     NaN  0.4249  0.4249
tool_to_tool__tool_use        break              25  0.5702  0.5245  0.1457  0.1273  0.8886
                              rescue             31  0.5592  0.5411  0.1147  0.2418  0.7570
                              wrong_to_wrong      5  0.4689  0.4879  0.1669  0.2011  0.6236

In [ ]:
# ============================================================
# 56. Rescue-vs-break margin diagnostics
# ============================================================

# Recover the applicable specialist score for each row
specialist_score = np.full(
    len(y_true_name),
    np.nan,
)

specialist_score[
    crossfit_source == "tool_to_tool__tool_use"
] = override_df.loc[
    crossfit_source == "tool_to_tool__tool_use",
    "tool_use_score"
].to_numpy()

specialist_score[
    crossfit_source == "assistant_to_tool__constraint"
] = override_df.loc[
    crossfit_source == "assistant_to_tool__constraint",
    "constraint_score"
].to_numpy()


diag = pd.DataFrame({
    "specialist": crossfit_source,
    "effect": validated_effect,
    "true_family": y_true_name,
    "semantic_prediction": semantic_pred,
    "specialist_prediction": crossfit_pred,
    "specialist_score": specialist_score,
    "workflow_prob": workflow_prob,
})


diag = diag[
    diag["effect"].isin(
        [
            "rescue",
            "break",
            "wrong_to_wrong",
        ]
    )
].copy()


# Useful relational quantities
diag["specialist_minus_workflow"] = (
    diag["specialist_score"]
    - diag["workflow_prob"]
)

diag["specialist_over_workflow"] = (
    diag["specialist_score"]
    /
    (diag["workflow_prob"] + 1e-8)
)


print("Overall:")
display(
    diag
    .groupby("effect")[
        [
            "specialist_score",
            "workflow_prob",
            "specialist_minus_workflow",
            "specialist_over_workflow",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
    .round(4)
)


print("\nBy specialist:")
display(
    diag
    .groupby(
        [
            "specialist",
            "effect",
        ]
    )[
        [
            "specialist_score",
            "workflow_prob",
            "specialist_minus_workflow",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
    .round(4)
)

Overall:


specialist_score                         workflow_prob                         specialist_minus_workflow                         specialist_over_workflow          \
                          count    mean  median     std         count    mean  median     std                     count    mean  median     std                    count    mean   
effect                                                                                                                                                                             
break                        27  0.7872  0.7787  0.0944            27  0.5789  0.5401  0.1435                        27  0.2083  0.2237  0.1542                       27  1.5087   
rescue                       41  0.8551  0.8685  0.0768            41  0.5640  0.5559  0.1132                        41  0.2911  0.3076  0.1292                       41  1.5824   
wrong_to_wrong                6  0.8080  0.8533  0.1488             6  0.4615  0.4648  0.1503                         6  0.3465  0.3159  0.2488                        6  2.0829   

                                
                median     std  
effect                          
break           1.4156  0.7701  
rescue          1.5682  0.3791  
wrong_to_wrong  1.5781  1.3430


By specialist:


specialist_score                         workflow_prob                         specialist_minus_workflow                        
                                                        count    mean  median     std         count    mean  median     std                     count    mean  median     std
specialist                    effect                                                                                                                                         
assistant_to_tool__constraint break                         2  0.8506  0.8506  0.0361             2  0.6871  0.6871  0.0235                         2  0.1635  0.1635  0.0125
                              rescue                       10  0.8845  0.8990  0.0442            10  0.5786  0.5659  0.1132                        10  0.3059  0.3291  0.1149
                              wrong_to_wrong                1  0.8980  0.8980     NaN             1  0.4249  0.4249     NaN                         1  0.4731  0.4731     NaN
tool_to_tool__tool_use        break                        25  0.7821  0.7652  0.0961            25  0.5702  0.5245  0.1457                        25  0.2119  0.2384  0.1600
                              rescue                       31  0.8456  0.8520  0.0831            31  0.5592  0.5411  0.1147                        31  0.2863  0.3075  0.1349
                              wrong_to_wrong                5  0.7900  0.8087  0.1589             5  0.4689  0.4879  0.1669                         5  0.3212  0.3111  0.2694

In [ ]:
# ============================================================
# 56B. Reconstruct the exact outer-fold assignment from Cell 47
# ============================================================

from sklearn.model_selection import StratifiedKFold
import numpy as np

n = len(y_true_name)

outer_cv_rebuild = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Use 1..5 to match the fold numbers printed in Cell 47
fold_assignment = np.zeros(
    n,
    dtype=int,
)

for fold, (_, test_idx) in enumerate(
    outer_cv_rebuild.split(
        np.zeros(n),
        y_true_name,
    ),
    start=1,
):
    fold_assignment[test_idx] = fold


print(
    "Fold counts:"
)

print(
    dict(
        zip(
            *np.unique(
                fold_assignment,
                return_counts=True,
            )
        )
    )
)

assert (fold_assignment > 0).all()
assert len(fold_assignment) == len(y_true_name)

print(
    "Recovered fold assignment:",
    fold_assignment.shape
)

Fold counts:
{1: 298, 2: 298, 3: 298, 4: 298, 5: 297}
Recovered fold assignment: (1489,)


In [ ]:
# ============================================================
# 56C. Verify reconstructed folds against Cell 47
# ============================================================

verification_rows = []

for fold in range(1, 6):

    test_mask = (
        fold_assignment == fold
    )

    for spec in [
        "tool_to_tool__tool_use",
        "tool_to_assistant__grounding",
        "assistant_to_tool__constraint",
    ]:

        actual_accepted = int(
            (
                test_mask
                &
                (crossfit_source == spec)
            ).sum()
        )

        recorded = (
            crossfit_threshold_df[
                (crossfit_threshold_df["fold"] == fold)
                &
                (crossfit_threshold_df["task"] == spec)
            ]
        )

        if len(recorded) == 1:
            expected_accepted = int(
                recorded.iloc[0]["test_accepted"]
            )
        else:
            expected_accepted = np.nan

        verification_rows.append({
            "fold": fold,
            "specialist": spec,
            "reconstructed_accepted":
                actual_accepted,
            "cell47_accepted":
                expected_accepted,
            "match":
                actual_accepted
                == expected_accepted,
        })


fold_verification_df = pd.DataFrame(
    verification_rows
)

display(
    fold_verification_df
)

print(
    "\nAll fold assignments match Cell 47:",
    fold_verification_df["match"].all()
)

,fold,specialist,reconstructed_accepted,cell47_accepted,match
0,1,tool_to_tool__tool_use,0,16,False
1,1,tool_to_assistant__grounding,0,8,False
2,1,assistant_to_tool__constraint,0,3,False
3,2,tool_to_tool__tool_use,0,14,False
4,2,tool_to_assistant__grounding,0,9,False
5,2,assistant_to_tool__constraint,0,1,False
6,3,tool_to_tool__tool_use,0,12,False
7,3,tool_to_assistant__grounding,0,3,False
8,3,assistant_to_tool__constraint,0,5,False
9,4,tool_to_tool__tool_use,0,5,False



All fold assignments match Cell 47: False


In [ ]:
# ============================================================
# 57. Cross-fitted relative-margin gate
# ============================================================

margin = (
    specialist_score
    - workflow_prob
)

margin_grid = np.r_[
    -np.inf,
    np.arange(
        -0.05,
        0.501,
        0.025,
    )
]

margin_crossfit_pred = (
    semantic_pred.copy()
)

margin_crossfit_mask = np.zeros(
    len(y_true_name),
    dtype=bool,
)

selection_rows = []

specialists = [
    "tool_to_tool__tool_use",
    "assistant_to_tool__constraint",
]

for fold in np.unique(
    fold_assignment
):

    train_fold = (
        fold_assignment != fold
    )

    test_fold = (
        fold_assignment == fold
    )

    for spec in specialists:

        train_candidates = (
            train_fold
            &
            validated_mask
            &
            (crossfit_source == spec)
        )

        test_candidates = (
            test_fold
            &
            validated_mask
            &
            (crossfit_source == spec)
        )

        best = None

        for threshold in margin_grid:

            accepted_train = (
                train_candidates
                &
                (margin >= threshold)
            )

            rescues = int(
                (
                    accepted_train
                    &
                    (~base_correct)
                    &
                    (
                        crossfit_pred
                        == y_true_name
                    )
                ).sum()
            )

            breaks = int(
                (
                    accepted_train
                    &
                    base_correct
                    &
                    (
                        crossfit_pred
                        != y_true_name
                    )
                ).sum()
            )

            wrong_to_wrong = int(
                (
                    accepted_train
                    &
                    (~base_correct)
                    &
                    (
                        crossfit_pred
                        != y_true_name
                    )
                ).sum()
            )

            accepted = int(
                accepted_train.sum()
            )

            net = (
                rescues - breaks
            )

            precision = (
                rescues / accepted
                if accepted > 0
                else 0.0
            )

            # Prefer:
            # 1. larger net
            # 2. higher precision
            # 3. larger support
            # 4. stricter margin
            key = (
                net,
                precision,
                accepted,
                threshold,
            )

            if (
                best is None
                or key > best["key"]
            ):

                best = {
                    "key": key,
                    "threshold":
                        threshold,
                    "accepted":
                        accepted,
                    "rescues":
                        rescues,
                    "breaks":
                        breaks,
                    "wrong_to_wrong":
                        wrong_to_wrong,
                    "net":
                        net,
                    "precision":
                        precision,
                }


        selected_threshold = (
            best["threshold"]
        )

        accepted_test = (
            test_candidates
            &
            (
                margin
                >= selected_threshold
            )
        )

        margin_crossfit_mask |= (
            accepted_test
        )

        margin_crossfit_pred[
            accepted_test
        ] = crossfit_pred[
            accepted_test
        ]


        test_rescues = int(
            (
                accepted_test
                &
                (~base_correct)
                &
                (
                    crossfit_pred
                    == y_true_name
                )
            ).sum()
        )

        test_breaks = int(
            (
                accepted_test
                &
                base_correct
                &
                (
                    crossfit_pred
                    != y_true_name
                )
            ).sum()
        )

        test_wrong = int(
            (
                accepted_test
                &
                (~base_correct)
                &
                (
                    crossfit_pred
                    != y_true_name
                )
            ).sum()
        )


        selection_rows.append({
            "fold":
                fold,

            "specialist":
                spec,

            "selected_margin":
                selected_threshold,

            "train_accepted":
                best["accepted"],

            "train_rescues":
                best["rescues"],

            "train_breaks":
                best["breaks"],

            "train_net":
                best["net"],

            "train_precision":
                best["precision"],

            "test_accepted":
                int(
                    accepted_test.sum()
                ),

            "test_rescues":
                test_rescues,

            "test_breaks":
                test_breaks,

            "test_wrong_to_wrong":
                test_wrong,

            "test_net":
                test_rescues
                -
                test_breaks,
        })


margin_selection_df = pd.DataFrame(
    selection_rows
)

display(
    margin_selection_df.round(4)
)

,fold,specialist,selected_margin,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net
0,None,tool_to_tool__tool_use,0.5,0,0,0,0,0.0,5,3,1,1,2
1,None,assistant_to_tool__constraint,0.5,0,0,0,0,0.0,0,0,0,0,0


In [ ]:
# ============================================================
# 57B. Evaluate cross-fitted margin gate
# ============================================================

margin_correct = (
    margin_crossfit_pred
    == y_true_name
)

margin_rescues = int(
    (
        margin_crossfit_mask
        &
        (~base_correct)
        &
        margin_correct
    ).sum()
)

margin_breaks = int(
    (
        margin_crossfit_mask
        &
        base_correct
        &
        (~margin_correct)
    ).sum()
)

margin_wrong = int(
    (
        margin_crossfit_mask
        &
        (~base_correct)
        &
        (~margin_correct)
    ).sum()
)


print(
    "Accepted:",
    int(
        margin_crossfit_mask.sum()
    )
)

print(
    "Rescues:",
    margin_rescues
)

print(
    "Breaks:",
    margin_breaks
)

print(
    "Wrong-to-wrong:",
    margin_wrong
)

print(
    "Net:",
    margin_rescues
    -
    margin_breaks
)


margin_metrics = (
    multiclass_metrics(
        y_true_name,
        margin_crossfit_pred,
    )
)


comparison_margin_df = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",
        **semantic_metrics,
    },
    {
        "model":
            "two_specialist_crossfit",
        **two_spec_metrics,
    },
    {
        "model":
            "margin_gated_crossfit",
        **margin_metrics,
    },
])

display(
    comparison_margin_df.round(4)
)

Accepted: 5
Rescues: 3
Breaks: 1
Wrong-to-wrong: 1
Net: 2


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,two_specialist_crossfit,0.5245,0.4536,0.4717,0.5139
2,margin_gated_crossfit,0.5165,0.4316,0.4522,0.4988


In [ ]:
# ============================================================
# 56D. Recover cross-fitted accepted specialist decisions
# directly from crossfit_pred
# ============================================================

import numpy as np
import pandas as pd

semantic_pred_arr = np.asarray(semantic_pred)
crossfit_pred_arr = np.asarray(crossfit_pred)
transition_arr = np.asarray(transition_type)
y_arr = np.asarray(y_true_name)

# A cross-fitted override is exactly a prediction changed
# relative to the semantic baseline.
crossfit_changed = (
    crossfit_pred_arr != semantic_pred_arr
)

print("Total cross-fitted changed predictions:",
      crossfit_changed.sum())

print("\nChanged predictions by transition:")
print(
    pd.Series(
        transition_arr[crossfit_changed]
    ).value_counts()
)

print("\nChanged predictions by new prediction:")
print(
    pd.Series(
        crossfit_pred_arr[crossfit_changed]
    ).value_counts()
)

print("\nTransition x new prediction:")
display(
    pd.crosstab(
        transition_arr[crossfit_changed],
        crossfit_pred_arr[crossfit_changed],
    )
)

Total cross-fitted changed predictions: 112

Changed predictions by transition:
TOOL_CALL->TOOL_CALL    61
TOOL_CALL->ASSISTANT    38
ASSISTANT->TOOL_CALL    13
Name: count, dtype: int64

Changed predictions by new prediction:
tool_use_error           61
grounding_state_error    38
constraint_error         13
Name: count, dtype: int64

Transition x new prediction:


col_0,constraint_error,grounding_state_error,tool_use_error
row_0,,,
ASSISTANT->TOOL_CALL,13,0,0
TOOL_CALL->ASSISTANT,0,38,0
TOOL_CALL->TOOL_CALL,0,0,61


In [ ]:
# ============================================================
# 56E. Recover specialist identity from transition type
# ============================================================

recovered_source = np.full(
    len(y_arr),
    None,
    dtype=object,
)

recovered_source[
    crossfit_changed
    & (transition_arr == "TOOL_CALL->TOOL_CALL")
] = "tool_to_tool__tool_use"

recovered_source[
    crossfit_changed
    & (transition_arr == "TOOL_CALL->ASSISTANT")
] = "tool_to_assistant__grounding"

recovered_source[
    crossfit_changed
    & (transition_arr == "ASSISTANT->TOOL_CALL")
] = "assistant_to_tool__constraint"


print("Recovered specialist counts:")
print(
    pd.Series(
        recovered_source[crossfit_changed]
    ).value_counts(dropna=False)
)

print(
    "\nUnmapped changed rows:",
    (
        crossfit_changed
        & pd.isna(recovered_source)
    ).sum()
)

Recovered specialist counts:
tool_to_tool__tool_use           61
tool_to_assistant__grounding     38
assistant_to_tool__constraint    13
Name: count, dtype: int64

Unmapped changed rows: 0


Perfect. That reconstruction is clean:

* **112** total cross-fitted interventions
* `TOOL_CALL→TOOL_CALL → tool_use_error`: **61**
* `TOOL_CALL→ASSISTANT → grounding_state_error`: **38**
* `ASSISTANT→TOOL_CALL → constraint_error`: **13**
* **0 unmapped rows**

So we now know exactly what `crossfit_pred` represents: it is the earlier **three-specialist cross-fitted policy**, not the later two-specialist ablation. That also explains why the previous fold-verification attempt failed: `crossfit_source` was not reliable state, but the predictions themselves are.

At this point I would stop trying to repair the margin-gating experiment from partially preserved variables. The scientifically clean next move is to rebuild the whole specialist decision process in one self-contained nested experiment.

The goal of the next block is:

> For each outer fold, train each relation specialist only on outer-train, choose its intervention threshold on outer-train, optionally choose a relative margin gate there too, and then evaluate once on untouched outer-test.

That will answer whether the `specialist_score - workflow_prob` idea actually generalizes.

## Next experiment: fully nested specialist + margin gate

Use only the two mechanisms that survived validation:

* `TOOL_CALL→TOOL_CALL → tool_use_error`
* `ASSISTANT→TOOL_CALL → constraint_error`

Leave grounding out.

The clean structure should be:

```text
Outer fold
│
├─ outer train
│   ├─ build specialist OOF probabilities with inner CV
│   ├─ select specialist threshold
│   ├─ select optional margin threshold
│   └─ fit final specialist on full outer train
│
└─ outer test
    ├─ compute specialist probability
    ├─ apply specialist threshold
    ├─ apply workflow-margin gate
    └─ evaluate
```

The key comparison should be four systems:

```text
semantic baseline
nested specialist threshold only
nested specialist + workflow veto
nested specialist + relative margin
```

If the margin version beats threshold-only cross-fitted intervention consistently, then the hierarchy hypothesis is supported. If not, the right conclusion is that the specialist signal is useful, but hand-designed second-stage gating is not stable enough on this dataset.

Before writing that block, one result is already worth recording as a firm notebook observation:

> **The three relation-specific specialists were reconstructed exactly from their cross-fitted interventions. The validated policy changed 112 predictions, with intervention identity perfectly aligned to transition type and target family. This confirms that the earlier cross-fitted results were genuinely transition-conditional rather than artifacts of label decoding or state misalignment.**

And the current research picture is now:

* **Constraint specialist:** strongest and most stable intervention mechanism.
* **Tool-use specialist:** real but noisier relational mechanism.
* **Grounding specialist:** predictive representation, but not validated as an override.
* **Workflow probability:** not sufficient as a veto.
* **Relative specialist-vs-workflow margin:** promising diagnostically, but not yet validated.

The next notebook code should therefore be the **fully nested two-specialist intervention experiment**, not another patch on the old variables.


In [ ]:
# ============================================================
# 58. Reconstruct previous-role embeddings self-contained
# ============================================================

import numpy as np
import pandas as pd

n = len(train_targets)
embedding_dim = train_event_embeddings.shape[1]

assert len(train_history_indices) == n
assert train_current_embeddings.shape[0] == n


# ------------------------------------------------------------
# Event-role lookup for the full event table
# ------------------------------------------------------------

event_role_col = (
    "event_role"
    if "event_role" in train_events.columns
    else "role"
)

event_role_all = (
    train_events[event_role_col]
    .astype(str)
    .to_numpy()
)


# ------------------------------------------------------------
# Last embedding of a requested role in each target's history
# ------------------------------------------------------------

def last_role_embedding(
    history_indices,
    role_name,
):

    E = np.zeros(
        (len(history_indices), embedding_dim),
        dtype=float,
    )

    present = np.zeros(
        len(history_indices),
        dtype=bool,
    )

    previous_event_idx = np.full(
        len(history_indices),
        -1,
        dtype=int,
    )

    for i, hist in enumerate(
        history_indices
    ):

        if hist is None or len(hist) == 0:
            continue

        # Search backward for latest event of required role
        for event_idx in reversed(hist):

            if (
                event_role_all[event_idx]
                == role_name
            ):

                E[i] = (
                    train_event_embeddings[
                        event_idx
                    ]
                )

                present[i] = True
                previous_event_idx[i] = (
                    event_idx
                )

                break

    return (
        E,
        present,
        previous_event_idx,
    )


E_last_tool, last_tool_present, last_tool_idx = (
    last_role_embedding(
        train_history_indices,
        "TOOL_CALL",
    )
)

E_last_assistant, last_assistant_present, last_assistant_idx = (
    last_role_embedding(
        train_history_indices,
        "ASSISTANT",
    )
)


print(
    "Last tool available:",
    last_tool_present.mean()
)

print(
    "Last assistant available:",
    last_assistant_present.mean()
)

print(
    "Shapes:",
    train_current_embeddings.shape,
    E_last_tool.shape,
    E_last_assistant.shape,
)

Last tool available: 0.9395567494963063
Last assistant available: 0.8629952988582942
Shapes: (1489, 384) (1489, 384) (1489, 384)


In [ ]:
# ============================================================
# 59. Verify previous-role representation applicability
# ============================================================

transition_arr = (
    train_targets["transition_type"].to_numpy()
    if "transition_type" in train_targets.columns
    else transition_structure_df[
        "transition_type"
    ].to_numpy()
)

tool_tool_mask = (
    transition_arr
    ==
    "TOOL_CALL->TOOL_CALL"
)

assistant_tool_mask = (
    transition_arr
    ==
    "ASSISTANT->TOOL_CALL"
)


print(
    "TOOL->TOOL support:",
    tool_tool_mask.sum()
)

print(
    "Previous tool available:",
    last_tool_present[
        tool_tool_mask
    ].mean()
)


print(
    "\nASSISTANT->TOOL support:",
    assistant_tool_mask.sum()
)

print(
    "Previous assistant available:",
    last_assistant_present[
        assistant_tool_mask
    ].mean()
)

TOOL->TOOL support: 590
Previous tool available: 1.0

ASSISTANT->TOOL support: 220
Previous assistant available: 1.0


In [ ]:
# ============================================================
# 60. Define validated relation mechanisms
# ============================================================

true_family_arr = np.asarray(
    y_true_name
)

X_tool_use = np.concatenate(
    [
        train_current_embeddings,
        E_last_tool,
    ],
    axis=1,
)

X_constraint = (
    E_last_assistant.copy()
)


specialist_definitions = {
    "tool_use": {
        "transition":
            "TOOL_CALL->TOOL_CALL",

        "target":
            "tool_use_error",

        "X":
            X_tool_use,
    },

    "constraint": {
        "transition":
            "ASSISTANT->TOOL_CALL",

        "target":
            "constraint_error",

        "X":
            X_constraint,
    },
}


for name, spec in (
    specialist_definitions.items()
):

    mask = (
        transition_arr
        == spec["transition"]
    )

    positives = (
        true_family_arr[mask]
        == spec["target"]
    ).sum()

    print(
        name,
        "| support =", mask.sum(),
        "| positives =", positives,
        "| prevalence =",
        round(
            positives / mask.sum(),
            4
        ),
        "| features =",
        spec["X"].shape[1],
    )

tool_use | support = 590 | positives = 136 | prevalence = 0.2305 | features = 768
constraint | support = 220 | positives = 57 | prevalence = 0.2591 | features = 384


In [ ]:
# ============================================================
# 61. Outer grouped folds
# ============================================================

from sklearn.model_selection import (
    StratifiedGroupKFold
)

if "canonical_group" in train_targets.columns:

    groups_nested = (
        train_targets[
            "canonical_group"
        ]
        .astype(str)
        .to_numpy()
    )

else:

    groups_nested = (
        train_targets[
            "group_id"
        ]
        .astype(str)
        .to_numpy()
    )


outer_cv_nested = (
    StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )
)

outer_splits = list(
    outer_cv_nested.split(
        np.zeros(n),
        true_family_arr,
        groups=groups_nested,
    )
)


print(
    "Outer folds:",
    len(outer_splits)
)

for fold, (tr, va) in enumerate(
    outer_splits,
    start=1,
):

    print(
        fold,
        "train =", len(tr),
        "test =", len(va),
    )

Outer folds: 5
1 train = 1191 test = 298
2 train = 1191 test = 298
3 train = 1191 test = 298
4 train = 1192 test = 297
5 train = 1191 test = 298


In [ ]:
# ============================================================
# 62. Binary specialist model helper
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def make_specialist_model(
    C=0.03,
):

    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        ),
    )

In [ ]:
# ============================================================
# 63. Inner OOF probability generator
# ============================================================

from sklearn.model_selection import (
    StratifiedGroupKFold
)


def inner_oof_specialist(
    X_all,
    y_all,
    global_train_idx,
    relation_mask_all,
    groups_all,
    C=0.03,
):

    # Restrict outer-train to the specialist's relation
    relation_train_idx = (
        global_train_idx[
            relation_mask_all[
                global_train_idx
            ]
        ]
    )

    X = X_all[
        relation_train_idx
    ]

    y = y_all[
        relation_train_idx
    ]

    groups = groups_all[
        relation_train_idx
    ]

    oof = np.full(
        len(relation_train_idx),
        np.nan,
        dtype=float,
    )

    inner_cv = (
        StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=123,
        )
    )

    for inner_tr, inner_va in (
        inner_cv.split(
            X,
            y,
            groups=groups,
        )
    ):

        model = make_specialist_model(
            C=C
        )

        model.fit(
            X[inner_tr],
            y[inner_tr],
        )

        oof[inner_va] = (
            model.predict_proba(
                X[inner_va]
            )[:, 1]
        )

    assert not np.isnan(oof).any()

    return (
        relation_train_idx,
        oof,
    )

In [ ]:
# ============================================================
# 64. Baseline semantic state
# ============================================================

P_semantic = np.asarray(
    remove_t1_prob
)

semantic_pred_arr = np.asarray(
    semantic_pred
)

workflow_prob = (
    P_semantic[:, 0]
)

base_correct = (
    semantic_pred_arr
    == true_family_arr
)


print(
    "Semantic baseline accuracy:",
    base_correct.mean()
)

print(
    "Workflow probability range:",
    workflow_prob.min(),
    workflow_prob.max(),
)

Semantic baseline accuracy: 0.5151108126259234
Workflow probability range: 0.00877697579562664 0.96692955493927


In [ ]:
# ============================================================
# 65. Nested threshold optimization
# ============================================================

specialist_threshold_grid = (
    np.arange(
        0.40,
        0.901,
        0.025,
    )
)

margin_threshold_grid = np.r_[
    -np.inf,
    np.arange(
        -0.10,
        0.501,
        0.025,
    ),
]


def score_local_policy(
    global_indices,
    specialist_prob,
    target_family,
    specialist_threshold,
    margin_threshold=-np.inf,
):

    wf_prob = (
        workflow_prob[
            global_indices
        ]
    )

    margin_local = (
        specialist_prob
        - wf_prob
    )

    accept = (
        (specialist_prob
         >= specialist_threshold)
        &
        (margin_local
         >= margin_threshold)
        &
        (
            semantic_pred_arr[
                global_indices
            ]
            != target_family
        )
    )

    pred = (
        semantic_pred_arr[
            global_indices
        ].copy()
    )

    pred[accept] = (
        target_family
    )

    true = (
        true_family_arr[
            global_indices
        ]
    )

    before_correct = (
        semantic_pred_arr[
            global_indices
        ]
        == true
    )

    after_correct = (
        pred == true
    )

    rescues = int(
        (
            accept
            &
            (~before_correct)
            &
            after_correct
        ).sum()
    )

    breaks = int(
        (
            accept
            &
            before_correct
            &
            (~after_correct)
        ).sum()
    )

    wrong = int(
        (
            accept
            &
            (~before_correct)
            &
            (~after_correct)
        ).sum()
    )

    accepted = int(
        accept.sum()
    )

    return {
        "accepted": accepted,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / accepted
            if accepted
            else 0.0
        ),
    }

In [ ]:
# ============================================================
# 66. Fully nested two-specialist intervention experiment
# ============================================================

nested_threshold_pred = (
    semantic_pred_arr.copy()
)

nested_margin_pred = (
    semantic_pred_arr.copy()
)

nested_threshold_accept = np.zeros(
    n,
    dtype=bool,
)

nested_margin_accept = np.zeros(
    n,
    dtype=bool,
)

nested_records = []


for outer_fold, (
    outer_train_idx,
    outer_test_idx,
) in enumerate(
    outer_splits,
    start=1,
):

    print(
        "\n",
        "=" * 80
    )

    print(
        "OUTER FOLD",
        outer_fold
    )

    for specialist_name, spec in (
        specialist_definitions.items()
    ):

        transition = (
            spec["transition"]
        )

        target_family = (
            spec["target"]
        )

        X_all = spec["X"]

        relation_mask = (
            transition_arr
            == transition
        )

        binary_y = (
            true_family_arr
            == target_family
        ).astype(int)


        # ----------------------------------------------------
        # 1. Inner OOF scores on outer-training relation rows
        # ----------------------------------------------------

        (
            relation_train_idx,
            inner_prob,
        ) = inner_oof_specialist(
            X_all=X_all,
            y_all=binary_y,
            global_train_idx=outer_train_idx,
            relation_mask_all=relation_mask,
            groups_all=groups_nested,
            C=0.03,
        )


        # ----------------------------------------------------
        # 2. Select specialist threshold only
        # ----------------------------------------------------

        threshold_candidates = []

        for s_threshold in (
            specialist_threshold_grid
        ):

            result = score_local_policy(
                global_indices=
                    relation_train_idx,

                specialist_prob=
                    inner_prob,

                target_family=
                    target_family,

                specialist_threshold=
                    s_threshold,

                margin_threshold=
                    -np.inf,
            )

            threshold_candidates.append({
                "specialist_threshold":
                    s_threshold,
                **result,
            })


        threshold_candidates = (
            pd.DataFrame(
                threshold_candidates
            )
        )

        eligible = (
            threshold_candidates[
                threshold_candidates[
                    "accepted"
                ] >= 5
            ]
        )

        if len(eligible):

            best_threshold_row = (
                eligible
                .sort_values(
                    [
                        "net",
                        "precision",
                        "specialist_threshold",
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

        else:

            best_threshold_row = (
                threshold_candidates
                .sort_values(
                    [
                        "net",
                        "precision",
                    ],
                    ascending=False,
                )
                .iloc[0]
            )


        selected_s_threshold = float(
            best_threshold_row[
                "specialist_threshold"
            ]
        )


        # ----------------------------------------------------
        # 3. Select specialist + margin threshold jointly
        # ----------------------------------------------------

        margin_candidates = []

        for s_threshold in (
            specialist_threshold_grid
        ):

            for m_threshold in (
                margin_threshold_grid
            ):

                result = score_local_policy(
                    global_indices=
                        relation_train_idx,

                    specialist_prob=
                        inner_prob,

                    target_family=
                        target_family,

                    specialist_threshold=
                        s_threshold,

                    margin_threshold=
                        m_threshold,
                )

                margin_candidates.append({
                    "specialist_threshold":
                        s_threshold,

                    "margin_threshold":
                        m_threshold,

                    **result,
                })


        margin_candidates = pd.DataFrame(
            margin_candidates
        )

        eligible_margin = (
            margin_candidates[
                margin_candidates[
                    "accepted"
                ] >= 5
            ]
        )

        if len(eligible_margin):

            best_margin_row = (
                eligible_margin
                .sort_values(
                    [
                        "net",
                        "precision",
                        "specialist_threshold",
                        "margin_threshold",
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

        else:

            best_margin_row = (
                margin_candidates
                .sort_values(
                    [
                        "net",
                        "precision",
                    ],
                    ascending=False,
                )
                .iloc[0]
            )


        selected_margin_s = float(
            best_margin_row[
                "specialist_threshold"
            ]
        )

        selected_margin_m = float(
            best_margin_row[
                "margin_threshold"
            ]
        )


        # ----------------------------------------------------
        # 4. Fit final specialist on complete outer-train
        #    relation subset
        # ----------------------------------------------------

        train_relation_idx = (
            outer_train_idx[
                relation_mask[
                    outer_train_idx
                ]
            ]
        )

        test_relation_idx = (
            outer_test_idx[
                relation_mask[
                    outer_test_idx
                ]
            ]
        )


        final_model = (
            make_specialist_model(
                C=0.03
            )
        )

        final_model.fit(
            X_all[
                train_relation_idx
            ],
            binary_y[
                train_relation_idx
            ],
        )


        test_prob = (
            final_model
            .predict_proba(
                X_all[
                    test_relation_idx
                ]
            )[:, 1]
        )


        semantic_test_pred = (
            semantic_pred_arr[
                test_relation_idx
            ]
        )

        wf_test = (
            workflow_prob[
                test_relation_idx
            ]
        )

        test_margin = (
            test_prob - wf_test
        )


        # ----------------------------------------------------
        # 5A. Threshold-only policy
        # ----------------------------------------------------

        accept_threshold_local = (
            (test_prob
             >= selected_s_threshold)
            &
            (
                semantic_test_pred
                != target_family
            )
        )

        accept_threshold_idx = (
            test_relation_idx[
                accept_threshold_local
            ]
        )

        nested_threshold_pred[
            accept_threshold_idx
        ] = target_family

        nested_threshold_accept[
            accept_threshold_idx
        ] = True


        # ----------------------------------------------------
        # 5B. Threshold + relative margin policy
        # ----------------------------------------------------

        accept_margin_local = (
            (test_prob
             >= selected_margin_s)
            &
            (
                test_margin
                >= selected_margin_m
            )
            &
            (
                semantic_test_pred
                != target_family
            )
        )

        accept_margin_idx = (
            test_relation_idx[
                accept_margin_local
            ]
        )

        nested_margin_pred[
            accept_margin_idx
        ] = target_family

        nested_margin_accept[
            accept_margin_idx
        ] = True


        # ----------------------------------------------------
        # 6. Held-out fold diagnostics
        # ----------------------------------------------------

        def heldout_effect(
            accepted_idx,
            prediction,
        ):

            accepted_mask = np.zeros(
                n,
                dtype=bool,
            )

            accepted_mask[
                accepted_idx
            ] = True

            after_correct = (
                prediction
                == true_family_arr
            )

            rescues = int(
                (
                    accepted_mask
                    &
                    (~base_correct)
                    &
                    after_correct
                ).sum()
            )

            breaks = int(
                (
                    accepted_mask
                    &
                    base_correct
                    &
                    (~after_correct)
                ).sum()
            )

            wrong = int(
                (
                    accepted_mask
                    &
                    (~base_correct)
                    &
                    (~after_correct)
                ).sum()
            )

            return (
                rescues,
                breaks,
                wrong,
            )


        thr_rescue, thr_break, thr_wrong = (
            heldout_effect(
                accept_threshold_idx,
                nested_threshold_pred,
            )
        )

        mar_rescue, mar_break, mar_wrong = (
            heldout_effect(
                accept_margin_idx,
                nested_margin_pred,
            )
        )


        nested_records.append({
            "fold":
                outer_fold,

            "specialist":
                specialist_name,

            "transition":
                transition,

            "target":
                target_family,

            "threshold_only_s":
                selected_s_threshold,

            "threshold_only_train_net":
                int(
                    best_threshold_row[
                        "net"
                    ]
                ),

            "threshold_only_test_accepted":
                len(
                    accept_threshold_idx
                ),

            "threshold_only_test_rescues":
                thr_rescue,

            "threshold_only_test_breaks":
                thr_break,

            "threshold_only_test_wrong":
                thr_wrong,

            "threshold_only_test_net":
                thr_rescue - thr_break,

            "margin_s":
                selected_margin_s,

            "margin_m":
                selected_margin_m,

            "margin_train_net":
                int(
                    best_margin_row[
                        "net"
                    ]
                ),

            "margin_test_accepted":
                len(
                    accept_margin_idx
                ),

            "margin_test_rescues":
                mar_rescue,

            "margin_test_breaks":
                mar_break,

            "margin_test_wrong":
                mar_wrong,

            "margin_test_net":
                mar_rescue - mar_break,
        })


nested_records_df = pd.DataFrame(
    nested_records
)

display(
    nested_records_df.round(4)
)


OUTER FOLD 1

OUTER FOLD 2

OUTER FOLD 3

OUTER FOLD 4

OUTER FOLD 5


,fold,specialist,transition,target,threshold_only_s,threshold_only_train_net,threshold_only_test_accepted,threshold_only_test_rescues,threshold_only_test_breaks,threshold_only_test_wrong,threshold_only_test_net,margin_s,margin_m,margin_train_net,margin_test_accepted,margin_test_rescues,margin_test_breaks,margin_test_wrong,margin_test_net
0,1,tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.725,11,8,7,0,1,7,0.750,0.125,11,7,6,0,1,6
1,1,constraint,ASSISTANT->TOOL_CALL,constraint_error,0.750,4,2,1,1,0,0,0.750,-0.025,4,2,1,1,0,0
2,2,tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.625,21,10,3,5,2,-2,0.625,-inf,21,10,3,5,2,-2
3,2,constraint,ASSISTANT->TOOL_CALL,constraint_error,0.625,5,4,3,0,1,3,0.625,-inf,5,4,3,0,1,3
4,3,tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.675,7,13,11,2,0,9,0.625,0.125,7,11,10,1,0,9
5,3,constraint,ASSISTANT->TOOL_CALL,constraint_error,0.775,6,2,1,1,0,0,0.575,0.200,7,0,0,0,0,0
6,4,tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.675,16,19,9,6,4,3,0.675,-inf,16,19,9,6,4,3
7,4,constraint,ASSISTANT->TOOL_CALL,constraint_error,0.675,11,6,2,3,1,-1,0.675,-inf,11,6,2,3,1,-1
8,5,tool_use,TOOL_CALL->TOOL_CALL,tool_use_error,0.550,23,18,8,7,3,1,0.550,-inf,23,18,8,7,3,1
9,5,constraint,ASSISTANT->TOOL_CALL,constraint_error,0.775,7,3,2,0,1,2,0.775,0.125,7,3,2,0,1,2


In [ ]:
# ============================================================
# 67. Evaluate fully nested policies
# ============================================================

def summarize_intervention(
    name,
    pred,
    accept,
):

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            accept
            &
            (~base_correct)
            &
            correct
        ).sum()
    )

    breaks = int(
        (
            accept
            &
            base_correct
            &
            (~correct)
        ).sum()
    )

    wrong = int(
        (
            accept
            &
            (~base_correct)
            &
            (~correct)
        ).sum()
    )

    return {
        "model": name,

        "accepted":
            int(
                accept.sum()
            ),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        "accept_precision":
            (
                rescues
                / accept.sum()
                if accept.sum()
                else np.nan
            ),

        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    }


nested_comparison = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",

        "accepted":
            0,

        "rescues":
            0,

        "breaks":
            0,

        "wrong_to_wrong":
            0,

        "net":
            0,

        "accept_precision":
            np.nan,

        **multiclass_metrics(
            true_family_arr,
            semantic_pred_arr,
        ),
    },

    summarize_intervention(
        "nested_specialist_threshold",
        nested_threshold_pred,
        nested_threshold_accept,
    ),

    summarize_intervention(
        "nested_specialist_plus_margin",
        nested_margin_pred,
        nested_margin_accept,
    ),
])


display(
    nested_comparison.round(4)
)

,model,accepted,rescues,breaks,wrong_to_wrong,net,accept_precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0,0,0,0,0,NaN,0.5151,0.4355,0.4548,0.4969
1,nested_specialist_threshold,85,47,25,13,22,0.5529,0.5299,0.4585,0.4759,0.5194
2,nested_specialist_plus_margin,80,44,23,13,21,0.5500,0.5292,0.4568,0.4744,0.5183


In [ ]:
# ============================================================
# 68. Aggregate nested results by mechanism
# ============================================================

nested_specialist_summary = (
    nested_records_df
    .groupby("specialist")
    .agg(
        folds=(
            "fold",
            "count",
        ),

        mean_threshold=(
            "threshold_only_s",
            "mean",
        ),

        std_threshold=(
            "threshold_only_s",
            "std",
        ),

        threshold_accepted=(
            "threshold_only_test_accepted",
            "sum",
        ),

        threshold_rescues=(
            "threshold_only_test_rescues",
            "sum",
        ),

        threshold_breaks=(
            "threshold_only_test_breaks",
            "sum",
        ),

        threshold_net=(
            "threshold_only_test_net",
            "sum",
        ),

        mean_margin_s=(
            "margin_s",
            "mean",
        ),

        std_margin_s=(
            "margin_s",
            "std",
        ),

        mean_margin_m=(
            "margin_m",
            "mean",
        ),

        std_margin_m=(
            "margin_m",
            "std",
        ),

        margin_accepted=(
            "margin_test_accepted",
            "sum",
        ),

        margin_rescues=(
            "margin_test_rescues",
            "sum",
        ),

        margin_breaks=(
            "margin_test_breaks",
            "sum",
        ),

        margin_net=(
            "margin_test_net",
            "sum",
        ),
    )
    .reset_index()
)


display(
    nested_specialist_summary.round(4)
)

,specialist,folds,mean_threshold,std_threshold,threshold_accepted,threshold_rescues,threshold_breaks,threshold_net,mean_margin_s,std_margin_s,mean_margin_m,std_margin_m,margin_accepted,margin_rescues,margin_breaks,margin_net
0,constraint,5,0.72,0.0671,17,9,5,4,0.680,0.0837,-inf,NaN,15,8,4,4
1,tool_use,5,0.65,0.0661,68,38,20,18,0.645,0.0737,-inf,NaN,65,36,19,17


The fully nested experiment confirms that the two transition mechanisms survive when everything—specialist fitting and threshold selection—is learned strictly inside the outer training folds.

The main result is:

Model	Accepted	Rescues	Breaks	Net	Accuracy	Balanced Acc.	Macro-F1
Semantic no-t1	0	0	0	0	.5151	.4355	.4548
Nested specialists	85	47	25	+22	.5299	.4585	.4759
Nested + margin	80	44	23	+21	.5292	.4568	.4744

So the relative workflow margin hypothesis does not survive strict validation. It removes 5 interventions, 3 rescues, and 2 breaks, changing net +22 → +21. We should stop pursuing that gate.

More importantly, both mechanisms independently remain positive under full nesting:

Tool-use specialist: 68 interventions, 38 rescues, 20 breaks → net +18
Constraint specialist: 17 interventions, 9 rescues, 5 breaks → net +4

That is stronger evidence for the tool-use mechanism than we had from the earlier cross-fitted threshold experiment.

The selected thresholds are also reasonably stable: tool-use mean 0.65 ± 0.066; constraint mean 0.72 ± 0.067. This is no longer a story where one lucky threshold happens to work.

The next research question should therefore change. We have established that role-conditioned antecedent representations work. The remaining problem is:

Why do 25 apparently high-confidence local mechanism detections still break a correct semantic prediction?

We already know most previous breaks were workflow errors. Instead of adding another generic gate, we should inspect whether those breaks differ in what the previous event actually contains.

The next experiment should test event-semantic submechanisms within the two validated transitions.

In [ ]:
# ============================================================
# 69. Final nested intervention diagnostics
# ============================================================

nested_correct = (
    nested_threshold_pred
    == true_family_arr
)

nested_effect = np.full(
    n,
    "not_intervened",
    dtype=object,
)

nested_effect[
    nested_threshold_accept
    &
    (~base_correct)
    &
    nested_correct
] = "rescue"

nested_effect[
    nested_threshold_accept
    &
    base_correct
    &
    (~nested_correct)
] = "break"

nested_effect[
    nested_threshold_accept
    &
    (~base_correct)
    &
    (~nested_correct)
] = "wrong_to_wrong"


nested_source = np.full(
    n,
    "semantic",
    dtype=object,
)

nested_source[
    nested_threshold_accept
    &
    (transition_arr == "TOOL_CALL->TOOL_CALL")
] = "tool_use"

nested_source[
    nested_threshold_accept
    &
    (transition_arr == "ASSISTANT->TOOL_CALL")
] = "constraint"


nested_diag_df = pd.DataFrame({
    "true_family":
        true_family_arr,

    "semantic_prediction":
        semantic_pred_arr,

    "nested_prediction":
        nested_threshold_pred,

    "transition_type":
        transition_arr,

    "specialist":
        nested_source,

    "effect":
        nested_effect,

    "workflow_prob":
        workflow_prob,

    "history_event_count":
        train_targets[
            "history_event_count"
        ].to_numpy(),
})


print(
    nested_diag_df[
        nested_diag_df["effect"]
        != "not_intervened"
    ]["effect"].value_counts()
)


display(
    nested_diag_df[
        nested_diag_df["effect"]
        != "not_intervened"
    ]
    .groupby(
        [
            "specialist",
            "effect",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

effect
rescue            47
break             25
wrong_to_wrong    13
Name: count, dtype: int64


,specialist,effect,count
0,constraint,break,5
1,constraint,rescue,9
2,constraint,wrong_to_wrong,3
3,tool_use,break,20
4,tool_use,rescue,38
5,tool_use,wrong_to_wrong,10


In [ ]:
# ============================================================
# 70. Nested break/rescue family decomposition
# ============================================================

nested_active = nested_diag_df[
    nested_diag_df["effect"]
    != "not_intervened"
].copy()


print("BREAKS")
display(
    nested_active[
        nested_active["effect"]
        == "break"
    ]
    .groupby(
        [
            "specialist",
            "true_family",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
)


print("\nRESCUES")
display(
    nested_active[
        nested_active["effect"]
        == "rescue"
    ]
    .groupby(
        [
            "specialist",
            "true_family",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
)

BREAKS


,specialist,true_family,count
4,tool_use,workflow_error,19
2,constraint,workflow_error,3
0,constraint,reasoning_value_error,1
1,constraint,tool_use_error,1
3,tool_use,grounding_state_error,1



RESCUES


,specialist,true_family,count
1,tool_use,tool_use_error,38
0,constraint,constraint_error,9


In [ ]:
# ============================================================
# 71. Tool identity for TOOL_CALL -> TOOL_CALL transitions
# ============================================================

# Current tool identity
if "primary_tool" not in train_targets.columns:
    raise KeyError(
        "train_targets does not contain primary_tool"
    )

current_tool = (
    train_targets[
        "primary_tool"
    ]
    .fillna("NO_TOOL")
    .astype(str)
    .to_numpy()
)


# Previous event's tool identity
previous_tool_name = np.full(
    n,
    "NO_TOOL",
    dtype=object,
)

# Identify likely tool column in full event table
candidate_tool_cols = [
    "primary_tool",
    "tool_name",
    "tool",
]

event_tool_col = None

for col in candidate_tool_cols:
    if col in train_events.columns:
        event_tool_col = col
        break

print(
    "Event tool column:",
    event_tool_col
)

if event_tool_col is None:
    raise KeyError(
        "Could not find a tool-name column "
        "inside train_events."
    )


event_tool_names = (
    train_events[
        event_tool_col
    ]
    .fillna("NO_TOOL")
    .astype(str)
    .to_numpy()
)


for i in range(n):

    if last_tool_present[i]:

        previous_tool_name[i] = (
            event_tool_names[
                last_tool_idx[i]
            ]
        )


tool_transition_name = np.array([
    f"{prev}->{curr}"
    for prev, curr in zip(
        previous_tool_name,
        current_tool,
    )
])


print(
    pd.Series(
        tool_transition_name[
            tool_tool_mask
        ]
    )
    .value_counts()
    .head(30)
)

Event tool column: primary_tool
get_details_by_id->get_details_by_id                            123
search->search                                                   25
NO_TOOL->NO_TOOL                                                 21
get_customer_by_phone->get_details_by_id                         17
get_details_by_id->get_data_usage                                14
get_details_by_id->get_customer_by_phone                         14
get_flight_cost->book_flight                                     13
check_status_bar->check_network_status                           12
get_data_usage->get_details_by_id                                12
get_details_by_id->NO_TOOL                                       11
get_details_by_id->check_status_bar                              11
check_network_status->check_sim_status                           10
check_data_restriction_status->check_apn_settings                10
touch->echo                                                       8
check_apn_settin

In [ ]:
# ============================================================
# 72. Does specialist utility depend on tool identity change?
# ============================================================

same_tool = (
    previous_tool_name
    == current_tool
)

tool_nested = (
    nested_active[
        nested_active["specialist"]
        == "tool_use"
    ]
    .copy()
)

tool_nested["same_tool"] = (
    same_tool[
        tool_nested.index
    ]
)

tool_nested["previous_tool"] = (
    previous_tool_name[
        tool_nested.index
    ]
)

tool_nested["current_tool"] = (
    current_tool[
        tool_nested.index
    ]
)

tool_nested["tool_transition"] = (
    tool_transition_name[
        tool_nested.index
    ]
)


display(
    tool_nested
    .groupby(
        [
            "same_tool",
            "effect",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)


same_tool_summary = []

for same_value, g in (
    tool_nested.groupby(
        "same_tool"
    )
):

    rescues = int(
        (g["effect"] == "rescue").sum()
    )

    breaks = int(
        (g["effect"] == "break").sum()
    )

    wrong = int(
        (
            g["effect"]
            == "wrong_to_wrong"
        ).sum()
    )

    same_tool_summary.append({
        "same_tool":
            same_value,

        "support":
            len(g),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        "rescue_rate":
            rescues / len(g),
    })


same_tool_summary = pd.DataFrame(
    same_tool_summary
)

display(
    same_tool_summary.round(4)
)

,same_tool,effect,count
0,False,break,18
1,False,rescue,37
2,False,wrong_to_wrong,6
3,True,break,2
4,True,rescue,1
5,True,wrong_to_wrong,4


,same_tool,support,rescues,breaks,wrong_to_wrong,net,rescue_rate
0,False,61,37,18,6,19,0.6066
1,True,7,1,2,4,-1,0.1429


In [ ]:
# ============================================================
# 73. Tool-transition mechanism utility
# ============================================================

tool_transition_summary = (
    tool_nested
    .groupby(
        "tool_transition"
    )
    .agg(
        support=(
            "effect",
            "size",
        ),

        rescues=(
            "effect",
            lambda x:
                (x == "rescue").sum(),
        ),

        breaks=(
            "effect",
            lambda x:
                (x == "break").sum(),
        ),

        wrong_to_wrong=(
            "effect",
            lambda x:
                (
                    x
                    == "wrong_to_wrong"
                ).sum(),
        ),
    )
    .reset_index()
)

tool_transition_summary["net"] = (
    tool_transition_summary["rescues"]
    -
    tool_transition_summary["breaks"]
)

tool_transition_summary[
    "rescue_rate"
] = (
    tool_transition_summary["rescues"]
    /
    tool_transition_summary["support"]
)


display(
    tool_transition_summary[
        tool_transition_summary[
            "support"
        ] >= 3
    ]
    .sort_values(
        [
            "net",
            "support",
        ],
        ascending=False,
    )
    .round(4)
)

,tool_transition,support,rescues,breaks,wrong_to_wrong,net,rescue_rate
13,check_sim_status->check_network_mode_preference,5,4,1,0,3,0.8000
17,check_vpn_status->run_speed_test,3,3,0,0,3,1.0000
11,check_network_status->check_sim_status,7,4,3,0,1,0.5714
21,cp->mv,3,2,1,0,1,0.6667
29,get_details_by_id->get_details_by_id,3,1,1,1,0,0.3333
4,check_apn_settings->check_wifi_status,5,2,3,0,-1,0.4000
6,check_data_restriction_status->check_apn_settings,7,2,4,1,-2,0.2857


In [ ]:
# ============================================================
# 74. Distance × tool-identity interaction
# ============================================================

current_E = train_current_embeddings
previous_tool_E = E_last_tool


tool_l2 = np.linalg.norm(
    current_E
    -
    previous_tool_E,
    axis=1,
)

tool_cosine = np.sum(
    current_E
    *
    previous_tool_E,
    axis=1,
)


tool_nested["tool_l2"] = (
    tool_l2[
        tool_nested.index
    ]
)

tool_nested["tool_cosine"] = (
    tool_cosine[
        tool_nested.index
    ]
)


display(
    tool_nested
    .groupby(
        [
            "same_tool",
            "effect",
        ]
    )[
        [
            "tool_l2",
            "tool_cosine",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
    .round(4)
)

tool_l2                         tool_cosine                        
                           count    mean  median     std       count    mean  median     std
same_tool effect                                                                            
False     break               18  0.8395  0.8356  0.2066          18  0.6275  0.6508  0.1366
          rescue              37  0.9027  0.9590  0.1340          37  0.5839  0.5402  0.1207
          wrong_to_wrong       6  1.0212  1.0276  0.1049           6  0.4740  0.4716  0.1040
True      break                2  0.7561  0.7561  0.1484           2  0.7086  0.7086  0.1122
          rescue               1  0.7063  0.7063     NaN           1  0.7506  0.7506     NaN
          wrong_to_wrong       4  0.6156  0.7239  0.2773           4  0.7817  0.7378  0.1356

The biggest new finding is not simply that tool identity matters. It is that the validated Tool→Tool specialist is useful almost entirely when the tool changes.

For the 68 nested Tool→Tool interventions:

different tool: 61 cases, 37 rescues, 18 breaks, net +19, rescue rate 60.7%
same tool: 7 cases, 1 rescue, 2 breaks, 4 wrong-to-wrong, net −1, rescue rate 14.3%

That is a strong structural asymmetry. The specialist should probably not treat repeated same-tool calls and cross-tool transitions as the same mechanism.

The concrete transitions support this too. Some changed-tool chains are strongly positive, such as check_sim_status → check_network_mode_preference and check_vpn_status → run_speed_test, while others are negative, such as check_data_restriction_status → check_apn_settings. Support is still small, so we should not hard-code individual tool pairs yet.

The distance result is also interesting: among changed-tool rows, rescues have somewhat larger L2 distance than breaks (0.903 vs 0.840), while cosine similarity is actually lower (0.584 vs 0.628). That suggests successful tool-use corrections may correspond to a meaningful semantic/action-state shift rather than near repetition.

So the next research question should be:

Does the Tool→Tool specialist mainly detect incorrect action progression, and can structural action-change features improve it beyond raw current+previous embeddings?

I would test that directly before any more threshold engineering.

In [ ]:
# ============================================================
# 75. Structural ablation: disable same-tool interventions
# ============================================================

structural_pred = nested_threshold_pred.copy()

same_tool_intervention = (
    nested_threshold_accept
    &
    (nested_source == "tool_use")
    &
    same_tool
)

# Revert these interventions to semantic baseline
structural_pred[
    same_tool_intervention
] = semantic_pred_arr[
    same_tool_intervention
]

structural_accept = (
    nested_threshold_accept
    &
    (~same_tool_intervention)
)

structural_correct = (
    structural_pred == true_family_arr
)

rescues = int(
    (
        structural_accept
        &
        (~base_correct)
        &
        structural_correct
    ).sum()
)

breaks = int(
    (
        structural_accept
        &
        base_correct
        &
        (~structural_correct)
    ).sum()
)

wrong = int(
    (
        structural_accept
        &
        (~base_correct)
        &
        (~structural_correct)
    ).sum()
)

print("Removed same-tool interventions:",
      int(same_tool_intervention.sum()))

print("Accepted:", int(structural_accept.sum()))
print("Rescues:", rescues)
print("Breaks:", breaks)
print("Wrong-to-wrong:", wrong)
print("Net:", rescues - breaks)

print(
    "Metrics:",
    multiclass_metrics(
        true_family_arr,
        structural_pred,
    )
)

Removed same-tool interventions: 7
Accepted: 78
Rescues: 46
Breaks: 23
Wrong-to-wrong: 9
Net: 23
Metrics: {'accuracy': 0.5305574210879785, 'balanced_accuracy': 0.4582333176545815, 'macro_f1': 0.4764125269433084, 'weighted_f1': 0.5198250638280503}


In [ ]:
# ============================================================
# 76. Explicit tool-transition structural features
# ============================================================

tool_struct_df = pd.DataFrame({
    "same_tool": same_tool.astype(int),

    "tool_l2": tool_l2,

    "tool_cosine": tool_cosine,

    "history_event_count":
        train_targets[
            "history_event_count"
        ].to_numpy(),
})


# Did the tool identity change?
tool_struct_df["tool_changed"] = (
    1 - tool_struct_df["same_tool"]
)


# Pair frequency across the training data
pair_series = pd.Series(
    tool_transition_name
)

pair_counts = (
    pair_series
    .value_counts()
)

tool_struct_df["tool_pair_frequency"] = (
    pair_series
    .map(pair_counts)
    .to_numpy()
)


# Previous tool frequency
prev_counts = (
    pd.Series(previous_tool_name)
    .value_counts()
)

tool_struct_df[
    "previous_tool_frequency"
] = (
    pd.Series(previous_tool_name)
    .map(prev_counts)
    .to_numpy()
)


# Current tool frequency
curr_counts = (
    pd.Series(current_tool)
    .value_counts()
)

tool_struct_df[
    "current_tool_frequency"
] = (
    pd.Series(current_tool)
    .map(curr_counts)
    .to_numpy()
)


display(
    tool_struct_df[
        tool_tool_mask
    ].describe().T.round(4)
)

,count,mean,std,min,25%,50%,75%,max
same_tool,590.0,0.3271,0.4696,0.0000,0.0000,0.0000,1.0000,1.0000
tool_l2,590.0,0.6604,0.3468,0.0000,0.2623,0.7841,0.9353,1.2504
tool_cosine,590.0,0.7219,0.1981,0.2182,0.5626,0.6926,0.9656,1.0000
history_event_count,590.0,20.5525,20.3564,1.0000,6.0000,12.0000,28.0000,89.0000
tool_changed,590.0,0.6729,0.4696,0.0000,0.0000,1.0000,1.0000,1.0000
tool_pair_frequency,590.0,42.7136,59.1753,1.0000,3.0000,11.0000,114.0000,144.0000
previous_tool_frequency,590.0,125.8712,139.6174,1.0000,13.0000,33.0000,320.0000,320.0000
current_tool_frequency,590.0,115.4881,185.3376,1.0000,10.0000,22.0000,197.0000,717.0000


In [ ]:
# ============================================================
# 77. Structural separation among nested tool-use interventions
# ============================================================

tool_effect_idx = (
    nested_threshold_accept
    &
    (nested_source == "tool_use")
)

tool_struct_diag = (
    tool_struct_df[
        tool_effect_idx
    ]
    .copy()
)

tool_struct_diag["effect"] = (
    nested_effect[
        tool_effect_idx
    ]
)

display(
    tool_struct_diag
    .groupby("effect")
    .agg({
        "same_tool": [
            "count",
            "mean",
        ],
        "tool_changed": [
            "mean",
        ],
        "tool_l2": [
            "mean",
            "median",
            "std",
        ],
        "tool_cosine": [
            "mean",
            "median",
            "std",
        ],
        "tool_pair_frequency": [
            "mean",
            "median",
        ],
        "history_event_count": [
            "mean",
            "median",
        ],
    })
    .round(4)
)

same_tool         tool_changed tool_l2                 tool_cosine                 tool_pair_frequency        history_event_count       
                   count    mean         mean    mean  median     std        mean  median     std                mean median                mean median
effect                                                                                                                                                 
break                 20  0.1000       0.9000  0.8312  0.8356  0.2000      0.6356  0.6508  0.1341             20.8500    8.0              8.5500    9.0
rescue                38  0.0263       0.9737  0.8975  0.9453  0.1359      0.5883  0.5531  0.1221             14.1579    4.5             11.1579    9.0
wrong_to_wrong        10  0.4000       0.6000  0.8590  0.9133  0.2749      0.5971  0.5806  0.1933             31.1000    3.0              9.5000    7.0

In [ ]:
# ============================================================
# 78. Tool-use representation comparison with structure
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

tool_indices = np.where(
    tool_tool_mask
)[0]

y_tool = (
    true_family_arr[
        tool_indices
    ]
    == "tool_use_error"
).astype(int)


X_current_previous = np.concatenate(
    [
        train_current_embeddings[
            tool_indices
        ],
        E_last_tool[
            tool_indices
        ],
    ],
    axis=1,
)


X_structure = (
    tool_struct_df.loc[
        tool_indices,
        [
            "same_tool",
            "tool_l2",
            "tool_cosine",
            "history_event_count",
            "tool_pair_frequency",
            "previous_tool_frequency",
            "current_tool_frequency",
        ],
    ]
    .to_numpy(dtype=float)
)


X_relation_plus_structure = np.concatenate(
    [
        X_current_previous,
        X_structure,
    ],
    axis=1,
)


representation_sets = {
    "current_plus_previous":
        X_current_previous,

    "structure_only":
        X_structure,

    "relation_plus_structure":
        X_relation_plus_structure,
}


rows = []

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

groups_tool = (
    groups_nested[
        tool_indices
    ]
)


for name, X in representation_sets.items():

    oof = np.zeros(
        len(tool_indices),
        dtype=float,
    )

    for tr, va in cv.split(
        X,
        y_tool,
        groups=groups_tool,
    ):

        model = make_specialist_model(
            C=0.03
        )

        model.fit(
            X[tr],
            y_tool[tr],
        )

        oof[va] = (
            model.predict_proba(
                X[va]
            )[:, 1]
        )

    rows.append({
        "representation":
            name,

        "n_features":
            X.shape[1],

        "support":
            len(y_tool),

        "positives":
            int(y_tool.sum()),

        "prevalence":
            y_tool.mean(),

        "pr_auc":
            average_precision_score(
                y_tool,
                oof,
            ),

        "roc_auc":
            roc_auc_score(
                y_tool,
                oof,
            ),
    })


tool_structure_representation_df = (
    pd.DataFrame(rows)
)

display(
    tool_structure_representation_df
    .sort_values(
        "pr_auc",
        ascending=False,
    )
    .round(4)
)

,representation,n_features,support,positives,prevalence,pr_auc,roc_auc
0,current_plus_previous,768,590,136,0.2305,0.7017,0.8638
2,relation_plus_structure,775,590,136,0.2305,0.7012,0.8636
1,structure_only,7,590,136,0.2305,0.3398,0.6997


The simple same-tool veto helps slightly: net goes from +22 to +23, accuracy from 0.5299 → 0.5306, and macro-F1 from 0.4759 → 0.4764. So repeated same-tool transitions are indeed a weakly harmful regime for the tool-use specialist.

But the larger representation experiment says something more important: explicit structural features do not add predictive value beyond the current+previous embeddings.

current_plus_previous gets PR-AUC 0.7017; relation_plus_structure is effectively identical at 0.7012; structure-only is much weaker at 0.3398. So the embedding pair already encodes most of the useful tool-transition structure. The same-tool flag is useful as a tiny policy heuristic, but it is not a new representation breakthrough.

The rescue/break diagnostics also support that. Rescues are overwhelmingly changed-tool transitions: 37/38, while breaks are 18/20. However, changed-tool itself is not enough to distinguish them. The distance differences are modest: rescues have larger L2 (0.8975 vs 0.8312) and lower cosine (0.5883 vs 0.6356), but not enough to justify another hand-built gate.

So I would stop feature engineering around tool identity here.

The next research question should move up one level:

Can we detect when a local specialist intervention is competing with a genuine workflow-level failure using the broader trajectory context, rather than just the immediate previous event?

That is now the main unresolved failure mode, because almost all specialist breaks are workflow errors.

The next experiment should therefore compare local relation representation vs broader history representation specifically on the accepted specialist candidates.

In [ ]:
# ============================================================
# 79. Broader history-state embedding
# ============================================================

history_mean_embedding = np.zeros_like(
    train_current_embeddings,
    dtype=float,
)

history_length = np.zeros(
    n,
    dtype=int,
)

for i, hist in enumerate(
    train_history_indices
):

    if hist is None or len(hist) == 0:
        continue

    hist_idx = np.asarray(
        hist,
        dtype=int,
    )

    history_mean_embedding[i] = (
        train_event_embeddings[
            hist_idx
        ].mean(axis=0)
    )

    history_length[i] = len(hist_idx)


print(
    "History available:",
    (history_length > 0).mean()
)

print(
    "Mean history length:",
    history_length.mean()
)

History available: 0.9623908663532572
Mean history length: 15.756883814640698


In [ ]:
# ============================================================
# 80. Local specialist vs workflow-conflict dataset
# ============================================================

active_mask = (
    nested_threshold_accept
)

conflict_mask = (
    active_mask
    &
    (
        (nested_effect == "rescue")
        |
        (
            (nested_effect == "break")
            &
            (true_family_arr == "workflow_error")
        )
    )
)

conflict_idx = np.where(
    conflict_mask
)[0]

conflict_y = (
    nested_effect[
        conflict_idx
    ]
    == "rescue"
).astype(int)


print(
    "Conflict examples:",
    len(conflict_idx)
)

print(
    "Rescues:",
    conflict_y.sum()
)

print(
    "Workflow breaks:",
    len(conflict_y)
    - conflict_y.sum()
)

print(
    "Rescue prevalence:",
    conflict_y.mean()
)

Conflict examples: 69
Rescues: 47
Workflow breaks: 22
Rescue prevalence: 0.6811594202898551


In [ ]:
# ============================================================
# 81. Local vs broader trajectory representations
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

X_current = (
    train_current_embeddings[
        conflict_idx
    ]
)

X_history = (
    history_mean_embedding[
        conflict_idx
    ]
)

X_current_history = np.concatenate(
    [
        X_current,
        X_history,
    ],
    axis=1,
)

# Immediate previous-role representation
X_prev = np.zeros_like(
    X_current
)

for j, idx in enumerate(
    conflict_idx
):

    if (
        transition_arr[idx]
        == "TOOL_CALL->TOOL_CALL"
    ):
        X_prev[j] = E_last_tool[idx]

    elif (
        transition_arr[idx]
        == "ASSISTANT->TOOL_CALL"
    ):
        X_prev[j] = E_last_assistant[idx]


X_local_relation = np.concatenate(
    [
        X_current,
        X_prev,
    ],
    axis=1,
)

X_local_plus_history = np.concatenate(
    [
        X_current,
        X_prev,
        X_history,
    ],
    axis=1,
)


conflict_representations = {
    "current_only":
        X_current,

    "history_only":
        X_history,

    "current_plus_history":
        X_current_history,

    "local_relation":
        X_local_relation,

    "local_relation_plus_history":
        X_local_plus_history,
}


groups_conflict = (
    groups_nested[
        conflict_idx
    ]
)


conflict_rows = []

for name, X in (
    conflict_representations.items()
):

    oof = np.zeros(
        len(conflict_idx),
        dtype=float,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        conflict_y,
        groups=groups_conflict,
    ):

        model = make_specialist_model(
            C=0.03
        )

        model.fit(
            X[tr],
            conflict_y[tr],
        )

        oof[va] = (
            model
            .predict_proba(
                X[va]
            )[:, 1]
        )

    conflict_rows.append({
        "representation":
            name,

        "n_features":
            X.shape[1],

        "support":
            len(conflict_y),

        "positives":
            int(
                conflict_y.sum()
            ),

        "prevalence":
            conflict_y.mean(),

        "pr_auc":
            average_precision_score(
                conflict_y,
                oof,
            ),

        "roc_auc":
            roc_auc_score(
                conflict_y,
                oof,
            ),
    })


conflict_representation_df = (
    pd.DataFrame(
        conflict_rows
    )
)

display(
    conflict_representation_df
    .sort_values(
        "pr_auc",
        ascending=False,
    )
    .round(4)
)

,representation,n_features,support,positives,prevalence,pr_auc,roc_auc
3,local_relation,768,69,47,0.6812,0.7382,0.5338
1,history_only,384,69,47,0.6812,0.7130,0.5295
2,current_plus_history,768,69,47,0.6812,0.6978,0.4918
0,current_only,384,69,47,0.6812,0.6898,0.4642
4,local_relation_plus_history,1152,69,47,0.6812,0.6841,0.5005


In [ ]:
# ============================================================
# 82. Conflict discrimination by specialist
# ============================================================

specialist_conflict_rows = []

for specialist_name in [
    "tool_use",
    "constraint",
]:

    spec_mask_local = (
        nested_source[
            conflict_idx
        ]
        == specialist_name
    )

    idx_local = (
        conflict_idx[
            spec_mask_local
        ]
    )

    y_local = (
        conflict_y[
            spec_mask_local
        ]
    )

    print(
        "\n",
        "=" * 70,
        specialist_name,
    )

    print(
        "support:",
        len(idx_local),
        "rescues:",
        y_local.sum(),
        "workflow breaks:",
        len(y_local)
        - y_local.sum(),
    )

    # Skip tiny subsets
    if (
        len(np.unique(y_local)) < 2
        or len(idx_local) < 10
    ):
        continue

    X_local = {
        "current":
            train_current_embeddings[
                idx_local
            ],

        "history":
            history_mean_embedding[
                idx_local
            ],

        "current_plus_history":
            np.concatenate(
                [
                    train_current_embeddings[
                        idx_local
                    ],
                    history_mean_embedding[
                        idx_local
                    ],
                ],
                axis=1,
            ),
    }

    groups_local = (
        groups_nested[
            idx_local
        ]
    )

    for rep_name, X in (
        X_local.items()
    ):

        oof = np.zeros(
            len(idx_local),
            dtype=float,
        )

        cv = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=42,
        )

        for tr, va in cv.split(
            X,
            y_local,
            groups=groups_local,
        ):

            model = make_specialist_model(
                C=0.03
            )

            model.fit(
                X[tr],
                y_local[tr],
            )

            oof[va] = (
                model.predict_proba(
                    X[va]
                )[:, 1]
            )

        specialist_conflict_rows.append({
            "specialist":
                specialist_name,

            "representation":
                rep_name,

            "support":
                len(y_local),

            "positives":
                int(y_local.sum()),

            "pr_auc":
                average_precision_score(
                    y_local,
                    oof,
                ),

            "roc_auc":
                roc_auc_score(
                    y_local,
                    oof,
                ),
        })


specialist_conflict_df = (
    pd.DataFrame(
        specialist_conflict_rows
    )
)

display(
    specialist_conflict_df
    .sort_values(
        [
            "specialist",
            "pr_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(4)
)


 ====================================================================== tool_use
support: 57 rescues: 38 workflow breaks: 19

 ====================================================================== constraint
support: 12 rescues: 9 workflow breaks: 3


,specialist,representation,support,positives,pr_auc,roc_auc
4,constraint,history,12,9,0.9437,0.8148
5,constraint,current_plus_history,12,9,0.9329,0.7778
3,constraint,current,12,9,0.8039,0.5556
1,tool_use,history,57,38,0.6496,0.5042
2,tool_use,current_plus_history,57,38,0.6448,0.5256
0,tool_use,current,57,38,0.6335,0.4543


This is a useful negative result. The broader-history hypothesis is not supported for tool-use, and the aggregate result is fairly clear.

The key comparison is:

Representation	PR-AUC	ROC-AUC
local relation	0.7382	0.5338
history only	0.7130	0.5295
current + history	0.6978	0.4918
current only	0.6898	0.4642
local relation + history	0.6841	0.5005

Adding global history actually hurts the best local representation. So I would not build the hierarchical history router yet.

There is one intriguing exception: constraint has history PR-AUC 0.9437, but that's only 12 examples / 3 workflow breaks. That's much too small to make an architectural decision from. Tool-use is the meaningful test here (n=57), and history is essentially at chance by ROC-AUC (0.5042).

One thing is especially important: because rescue prevalence is already 0.681, PR-AUC can look respectable even with weak discrimination. The local_relation ROC-AUC of only 0.534 says that even our best representation has very little ability to rank rescues above workflow breaks. So I would not build a learned rescue-vs-workflow gate from these 69 examples.

We should instead investigate why those 19 tool-use workflow breaks occur. Your earlier result gives us a strong lead: 19/20 tool-use breaks after the same-tool veto are workflow_error. We need to determine whether those workflow breaks form identifiable workflow subtypes, rather than trying more embedding combinations.

In [ ]:
[x for x in globals()
 if any(k in x.lower() for k in [
     "crossfit", "changed", "override", "accept"
 ])]

['changed',
 'override_df',
 'override_target',
 'override_score',
 'accept',
 'accepted',
 'combined_accept',
 'crossfit_pred',
 'crossfit_accept',
 'crossfit_source',
 'crossfit_threshold',
 'accept_test',
 'crossfit_threshold_df',
 'crossfit_metrics',
 'crossfit_correct',
 'accepted_effect_df',
 'actual_accepted',
 'expected_accepted',
 'margin_crossfit_pred',
 'margin_crossfit_mask',
 'accepted_train',
 'accepted_test',
 'crossfit_pred_arr',
 'crossfit_changed',
 'nested_threshold_accept',
 'nested_margin_accept',
 'accept_threshold_local',
 'accept_threshold_idx',
 'accept_margin_local',
 'accept_margin_idx',
 'structural_accept']

In [ ]:
# ============================================================
# 83. Tool-use rescues vs workflow breaks
#     Structural diagnostics, no specialist-score dependency
# ============================================================

import numpy as np
import pandas as pd

# Use the FULLY NESTED intervention state, not the older
# three-specialist crossfit state.
tool_conflict_mask = (
    nested_threshold_accept
    &
    (nested_source == "tool_use")
    &
    (
        (nested_effect == "rescue")
        |
        (
            (nested_effect == "break")
            &
            (true_family_arr == "workflow_error")
        )
    )
)

idx = np.where(tool_conflict_mask)[0]

group = np.where(
    nested_effect[idx] == "rescue",
    "rescue",
    "workflow_break",
)

diag = pd.DataFrame({
    "idx": idx,
    "group": group,
    "true_family": true_family_arr[idx],
    "semantic_pred": semantic_pred_arr[idx],
    "nested_pred": nested_threshold_pred[idx],
    "transition": transition_arr[idx],

    "workflow_prob": workflow_prob[idx],

    "same_tool": same_tool[idx],
    "tool_l2": tool_l2[idx],
    "tool_cosine": tool_cosine[idx],

    "history_len": history_length[idx],

    "previous_tool": previous_tool_name[idx],
    "current_tool": current_tool[idx],
    "tool_transition": tool_transition_name[idx],
})

print("Rows:", len(diag))
print(diag["group"].value_counts())

display(
    diag.groupby("group")[
        [
            "workflow_prob",
            "same_tool",
            "tool_l2",
            "tool_cosine",
            "history_len",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

Rows: 57
group
rescue            38
workflow_break    19
Name: count, dtype: int64


workflow_prob                                         same_tool                                     tool_l2                                         tool_cosine  \
                       count    mean  median     std     min     max     count    mean median     std    min   max   count    mean  median     std     min     max       count   
group                                                                                                                                                                            
rescue                    38  0.5537  0.5231  0.1140  0.2418  0.7925        38  0.0263    0.0  0.1622  False  True      38  0.8975  0.9453  0.1359  0.7063  1.1013          38   
workflow_break            19  0.5666  0.5119  0.1101  0.4818  0.8886        19  0.1053    0.0  0.3153  False  True      19  0.8374  0.8377  0.2034  0.1569  1.0802          19   

                                                       history_len                                   
                  mean  median     std     min     max       count     mean median      std min max  
group                                                                                                
rescue          0.5883  0.5531  0.1221  0.3935  0.7506          38  11.1579    9.0  10.4972   3  62  
workflow_break  0.6297  0.6491  0.1351  0.4166  0.9877          19   8.8421    9.0   3.2018   3  17

In [ ]:
# ============================================================
# 84. Which concrete tool transitions produce rescues/breaks?
# ============================================================

pair_effect = (
    diag
    .groupby(
        [
            "tool_transition",
            "group",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

for col in [
    "rescue",
    "workflow_break",
]:
    if col not in pair_effect.columns:
        pair_effect[col] = 0

pair_effect["support"] = (
    pair_effect["rescue"]
    + pair_effect["workflow_break"]
)

pair_effect["net"] = (
    pair_effect["rescue"]
    - pair_effect["workflow_break"]
)

pair_effect["rescue_rate"] = (
    pair_effect["rescue"]
    / pair_effect["support"]
)

display(
    pair_effect
    .sort_values(
        [
            "support",
            "net",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(30)
    .round(4)
)

group,rescue,workflow_break,support,net,rescue_rate
tool_transition,,,,,
check_network_status->check_sim_status,4,3,7,1,0.5714
check_data_restriction_status->check_apn_settings,2,4,6,-2,0.3333
check_sim_status->check_network_mode_preference,4,1,5,3,0.8000
check_apn_settings->check_wifi_status,2,3,5,-1,0.4000
check_vpn_status->run_speed_test,3,0,3,3,1.0000
check_wifi_status->check_vpn_status,2,0,2,2,1.0000
cp->mv,2,0,2,2,1.0000
get_data_usage->check_network_status,2,0,2,2,1.0000
get_details_by_id->NO_TOOL,2,0,2,2,1.0000


In [ ]:
# ============================================================
# 85. Direct TOOL_USE vs WORKFLOW discrimination
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold

pair_mask = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    &
    np.isin(
        true_family_arr,
        [
            "tool_use_error",
            "workflow_error",
        ],
    )
)

pair_idx = np.where(pair_mask)[0]

pair_y = (
    true_family_arr[pair_idx]
    == "tool_use_error"
).astype(int)

print("Support:", len(pair_idx))
print("Tool-use:", int(pair_y.sum()))
print("Workflow:", int(len(pair_y) - pair_y.sum()))
print("Tool-use prevalence:", pair_y.mean())

Support: 477
Tool-use: 136
Workflow: 341
Tool-use prevalence: 0.2851153039832285


In [ ]:
X_current = (
    train_current_embeddings[
        pair_idx
    ]
)

X_previous_tool = (
    E_last_tool[
        pair_idx
    ]
)

X_history = (
    history_mean_embedding[
        pair_idx
    ]
)

X_relation = np.concatenate(
    [
        X_current,
        X_previous_tool,
    ],
    axis=1,
)

# Explicit relation vectors
X_delta = (
    X_current
    - X_previous_tool
)

X_product = (
    X_current
    * X_previous_tool
)

X_abs_delta = np.abs(
    X_delta
)

# Compact structural features
X_structure_pair = np.column_stack([
    same_tool[pair_idx].astype(float),
    tool_l2[pair_idx],
    tool_cosine[pair_idx],
    history_length[pair_idx],
])

representations = {
    "current":
        X_current,

    "previous_tool":
        X_previous_tool,

    "current_plus_previous":
        X_relation,

    "delta":
        X_delta,

    "product":
        X_product,

    "abs_delta":
        X_abs_delta,

    "history":
        X_history,

    "current_plus_history":
        np.concatenate(
            [
                X_current,
                X_history,
            ],
            axis=1,
        ),

    "relation_plus_history":
        np.concatenate(
            [
                X_current,
                X_previous_tool,
                X_history,
            ],
            axis=1,
        ),

    "structure":
        X_structure_pair,

    "relation_plus_structure":
        np.concatenate(
            [
                X_relation,
                X_structure_pair,
            ],
            axis=1,
        ),
}

In [ ]:
pair_groups = (
    groups_nested[
        pair_idx
    ]
)

direct_rows = []

for rep_name, X in representations.items():

    oof = np.full(
        len(pair_idx),
        np.nan,
        dtype=float,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        pair_y,
        groups=pair_groups,
    ):

        model_local = make_specialist_model(
            C=0.03
        )

        model_local.fit(
            X[tr],
            pair_y[tr],
        )

        oof[va] = (
            model_local
            .predict_proba(
                X[va]
            )[:, 1]
        )

    assert not np.isnan(oof).any()

    direct_rows.append({
        "representation":
            rep_name,

        "n_features":
            X.shape[1],

        "support":
            len(pair_y),

        "positives":
            int(pair_y.sum()),

        "prevalence":
            pair_y.mean(),

        "pr_auc":
            average_precision_score(
                pair_y,
                oof,
            ),

        "roc_auc":
            roc_auc_score(
                pair_y,
                oof,
            ),
    })


direct_tool_workflow_df = pd.DataFrame(
    direct_rows
)

display(
    direct_tool_workflow_df
    .sort_values(
        [
            "roc_auc",
            "pr_auc",
        ],
        ascending=False,
    )
    .round(4)
)

,representation,n_features,support,positives,prevalence,pr_auc,roc_auc
10,relation_plus_structure,772,477,136,0.2851,0.7081,0.8693
2,current_plus_previous,768,477,136,0.2851,0.7059,0.8689
8,relation_plus_history,1152,477,136,0.2851,0.7024,0.8688
5,abs_delta,384,477,136,0.2851,0.7058,0.8648
4,product,384,477,136,0.2851,0.7134,0.8576
7,current_plus_history,768,477,136,0.2851,0.6747,0.8555
0,current,384,477,136,0.2851,0.6252,0.8500
6,history,384,477,136,0.2851,0.6429,0.8101
1,previous_tool,384,477,136,0.2851,0.6357,0.7917
9,structure,4,477,136,0.2851,0.4828,0.7669


The direct pairwise experiment changes the interpretation substantially.

The central finding is:

tool_use_error vs workflow_error is highly separable when we formulate it as the actual competing decision.

On the 477 TOOL_CALL → TOOL_CALL examples belonging to either class:

136 tool_use_error
341 workflow_error
prevalence = 0.285

Yet the best representations reach roughly:

current_plus_previous: PR-AUC 0.7059, ROC-AUC 0.8689
relation_plus_structure: PR-AUC 0.7081, ROC-AUC 0.8693
product: PR-AUC 0.7134, ROC-AUC 0.8576
relation_plus_history: PR-AUC 0.7024, ROC-AUC 0.8688

That is a major difference from the earlier rescue-vs-break gate, where ROC-AUC was around 0.53.

So the problem was not that the representation lacks enough information. The problem is that the original specialist was solving the wrong classification objective:

tool_use_error vs all other families

while the real ambiguity after routing is mostly:

tool_use_error vs workflow_error

This is exactly the setting where a hierarchical / pairwise MoE architecture becomes justified.

The structural features still add almost nothing (0.8689 → 0.8693 ROC-AUC), and global history also adds essentially nothing (0.8689 → 0.8688). So the useful information remains concentrated in the current–previous tool relation itself.

The workflow-break diagnostics reinforce this. Workflow probability is almost identical between rescues and breaks (0.554 vs 0.567). Same-tool is uncommon in both groups after the structural cleanup. Distance differences exist but are modest. There is no single scalar veto. The distinction emerges only when the pair representation is explicitly trained to model the tool_use ↔ workflow boundary.

What this means architecturally

I would now stop thinking of this as:

semantic classifier
      ↓
tool-use specialist
      ↓
confidence gate

and instead move toward:

                         ┌─ local tool-use
semantic / coarse family ─┤
                         └─ workflow
                               ↑
                  pairwise transition arbiter

More generally:

Current event semantics
        +
Transition type
        |
        v
Candidate family set
        |
        +------------------------------+
        |                              |
        v                              v
local mechanism expert          global/base expert
        |                              |
        +--------------+---------------+
                       |
                       v
               pairwise arbiter

For TOOL_CALL → TOOL_CALL, the relevant candidate pair is very clearly:

tool_use_error ↔ workflow_error

For ASSISTANT → TOOL_CALL, we should now test the analogous boundary:

constraint_error ↔ workflow_error

That should be the next experiment before building the full hierarchical model.

In [ ]:
# ============================================================
# 86. Direct CONSTRAINT vs WORKFLOW discrimination
#     on ASSISTANT -> TOOL_CALL transitions
# ============================================================

constraint_workflow_mask = (
    (transition_arr == "ASSISTANT->TOOL_CALL")
    &
    np.isin(
        true_family_arr,
        [
            "constraint_error",
            "workflow_error",
        ],
    )
)

cw_idx = np.where(
    constraint_workflow_mask
)[0]

cw_y = (
    true_family_arr[cw_idx]
    == "constraint_error"
).astype(int)

print("Support:", len(cw_idx))
print("Constraint:", int(cw_y.sum()))
print(
    "Workflow:",
    int(len(cw_y) - cw_y.sum())
)
print(
    "Constraint prevalence:",
    cw_y.mean()
)

Support: 154
Constraint: 57
Workflow: 97
Constraint prevalence: 0.37012987012987014


In [ ]:
X_current_cw = (
    train_current_embeddings[
        cw_idx
    ]
)

X_previous_assistant_cw = (
    E_last_assistant[
        cw_idx
    ]
)

X_history_cw = (
    history_mean_embedding[
        cw_idx
    ]
)

X_delta_cw = (
    X_current_cw
    - X_previous_assistant_cw
)

X_product_cw = (
    X_current_cw
    * X_previous_assistant_cw
)

X_abs_delta_cw = np.abs(
    X_delta_cw
)


cw_representations = {
    "current":
        X_current_cw,

    "previous_assistant":
        X_previous_assistant_cw,

    "current_plus_previous":
        np.concatenate(
            [
                X_current_cw,
                X_previous_assistant_cw,
            ],
            axis=1,
        ),

    "delta":
        X_delta_cw,

    "abs_delta":
        X_abs_delta_cw,

    "product":
        X_product_cw,

    "history":
        X_history_cw,

    "previous_plus_history":
        np.concatenate(
            [
                X_previous_assistant_cw,
                X_history_cw,
            ],
            axis=1,
        ),

    "relation_plus_history":
        np.concatenate(
            [
                X_current_cw,
                X_previous_assistant_cw,
                X_history_cw,
            ],
            axis=1,
        ),
}

In [ ]:
cw_groups = (
    groups_nested[
        cw_idx
    ]
)

cw_rows = []

for rep_name, X in (
    cw_representations.items()
):

    oof = np.full(
        len(cw_idx),
        np.nan,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        cw_y,
        groups=cw_groups,
    ):

        m = make_specialist_model(
            C=0.03
        )

        m.fit(
            X[tr],
            cw_y[tr],
        )

        oof[va] = (
            m.predict_proba(
                X[va]
            )[:, 1]
        )

    cw_rows.append({
        "representation":
            rep_name,

        "n_features":
            X.shape[1],

        "support":
            len(cw_y),

        "positives":
            int(cw_y.sum()),

        "prevalence":
            cw_y.mean(),

        "pr_auc":
            average_precision_score(
                cw_y,
                oof,
            ),

        "roc_auc":
            roc_auc_score(
                cw_y,
                oof,
            ),
    })


constraint_workflow_df = (
    pd.DataFrame(cw_rows)
)

display(
    constraint_workflow_df
    .sort_values(
        [
            "roc_auc",
            "pr_auc",
        ],
        ascending=False,
    )
    .round(4)
)

,representation,n_features,support,positives,prevalence,pr_auc,roc_auc
2,current_plus_previous,768,154,57,0.3701,0.6472,0.7779
1,previous_assistant,384,154,57,0.3701,0.6474,0.7725
5,product,384,154,57,0.3701,0.6150,0.7703
0,current,384,154,57,0.3701,0.6438,0.7562
8,relation_plus_history,1152,154,57,0.3701,0.6190,0.7560
7,previous_plus_history,768,154,57,0.3701,0.5837,0.7410
3,delta,384,154,57,0.3701,0.5656,0.7397
4,abs_delta,384,154,57,0.3701,0.5357,0.7186
6,history,384,154,57,0.3701,0.5324,0.6998


In [ ]:
# ============================================================
# 87. Boundary summary
# ============================================================

tool_best = (
    direct_tool_workflow_df
    .sort_values(
        "roc_auc",
        ascending=False,
    )
    .iloc[0]
)

constraint_best = (
    constraint_workflow_df
    .sort_values(
        "roc_auc",
        ascending=False,
    )
    .iloc[0]
)

boundary_summary = pd.DataFrame([
    {
        "boundary":
            "tool_use_vs_workflow",

        "transition":
            "TOOL_CALL->TOOL_CALL",

        "representation":
            tool_best[
                "representation"
            ],

        "support":
            tool_best["support"],

        "pr_auc":
            tool_best["pr_auc"],

        "roc_auc":
            tool_best["roc_auc"],
    },
    {
        "boundary":
            "constraint_vs_workflow",

        "transition":
            "ASSISTANT->TOOL_CALL",

        "representation":
            constraint_best[
                "representation"
            ],

        "support":
            constraint_best["support"],

        "pr_auc":
            constraint_best["pr_auc"],

        "roc_auc":
            constraint_best["roc_auc"],
    },
])

display(
    boundary_summary.round(4)
)

,boundary,transition,representation,support,pr_auc,roc_auc
0,tool_use_vs_workflow,TOOL_CALL->TOOL_CALL,relation_plus_structure,477,0.7081,0.8693
1,constraint_vs_workflow,ASSISTANT->TOOL_CALL,current_plus_previous,154,0.6472,0.7779


After that, if the constraint/workflow pair is also reasonably separable—say ROC-AUC comfortably above ~0.70—I would build the first true hierarchical router.

The router would work like this:

semantic prediction
      |
      v
transition-type router
      |
      +-- TOOL→TOOL
      |      |
      |      +-- pairwise tool_use vs workflow arbiter
      |
      +-- ASSISTANT→TOOL
             |
             +-- pairwise constraint vs workflow arbiter

Crucially, this is no longer post-hoc threshold hacking. We now have evidence for transition-specific decision boundaries between competing failure mechanisms.

The strongest notebook-level conclusion at this stage is:

Immediate transition semantics contain enough information to distinguish local tool-use failures from workflow failures with ROC-AUC ≈ 0.87, but this information is exposed only when the classifier is trained on the correct pairwise boundary. Generic one-vs-rest specialists and scalar workflow-confidence gates discard much of that discriminative structure. This supports a transition-routed, pairwise mixture-of-experts formulation rather than flat multiclass refinement.

es — this is now strong enough to justify moving from diagnostics into the first actual transition-routed pairwise model.

You now have two independently supported boundaries:

Boundary	Transition	Best representation	Support	PR-AUC	ROC-AUC
tool_use_error vs workflow_error	TOOL_CALL→TOOL_CALL	relation + structure	477	0.7081	0.8693
constraint_error vs workflow_error	ASSISTANT→TOOL_CALL	current + previous	154	0.6472	0.7779

This is not just one anomalous transition. Both local-vs-workflow boundaries are learnable, although Tool→Tool is considerably stronger.

The important conceptual shift is:

The trajectory representation is not primarily useful for predicting five families globally. It is useful for resolving specific competing failure mechanisms conditioned on event transition type.

That is much closer to the MoE/routing hypothesis.

The next experiment should therefore be a true cross-fitted hierarchical arbiter, not another feature comparison.

Notebook hypothesis

For each example:

semantic multiclass prediction
          |
          v
     transition type
          |
    +-----+----------------------+
    |                            |
TOOL→TOOL                  ASSISTANT→TOOL
    |                            |
tool_use vs workflow       constraint vs workflow
pairwise arbiter           pairwise arbiter
    |                            |
    +-------------+--------------+
                  |
             final family

Importantly, we should initially let the arbiter operate only when the semantic model thinks the example is one of the two candidate families. That makes this a genuine pairwise correction rather than an unrestricted override engine.

So for Tool→Tool:

semantic ∈ {workflow_error, tool_use_error}

then use the pairwise model.

For Assistant→Tool:

semantic ∈ {workflow_error, constraint_error}

then use the pairwise model.

Everything else remains untouched.

That gives us a clean test of hierarchical routing.

In [ ]:
# ============================================================
# 88. Pairwise arbitration representations
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# TOOL->TOOL:
# current + previous tool + compact structure
# ------------------------------------------------------------

X_tool_pair_base = np.concatenate(
    [
        train_current_embeddings,
        E_last_tool,
    ],
    axis=1,
)

X_tool_pair_structure = np.column_stack([
    same_tool.astype(float),
    tool_l2.astype(float),
    tool_cosine.astype(float),
    history_length.astype(float),
])

X_tool_pair = np.concatenate(
    [
        X_tool_pair_base,
        X_tool_pair_structure,
    ],
    axis=1,
)


# ------------------------------------------------------------
# ASSISTANT->TOOL:
# current + previous assistant
# ------------------------------------------------------------

X_constraint_pair = np.concatenate(
    [
        train_current_embeddings,
        E_last_assistant,
    ],
    axis=1,
)


print(
    "Tool/workflow representation:",
    X_tool_pair.shape
)

print(
    "Constraint/workflow representation:",
    X_constraint_pair.shape
)

Tool/workflow representation: (1489, 772)
Constraint/workflow representation: (1489, 768)


In [ ]:
# ============================================================
# 89. Pairwise routing definitions
# ============================================================

pairwise_tasks = {
    "tool_vs_workflow": {
        "transition":
            "TOOL_CALL->TOOL_CALL",

        "local_family":
            "tool_use_error",

        "workflow_family":
            "workflow_error",

        "X":
            X_tool_pair,
    },

    "constraint_vs_workflow": {
        "transition":
            "ASSISTANT->TOOL_CALL",

        "local_family":
            "constraint_error",

        "workflow_family":
            "workflow_error",

        "X":
            X_constraint_pair,
    },
}


for task_name, spec in pairwise_tasks.items():

    mask = (
        (transition_arr == spec["transition"])
        &
        np.isin(
            true_family_arr,
            [
                spec["local_family"],
                spec["workflow_family"],
            ],
        )
    )

    y_local = (
        true_family_arr[mask]
        == spec["local_family"]
    )

    print(
        "\n",
        task_name,
    )

    print(
        "support:",
        mask.sum()
    )

    print(
        spec["local_family"],
        ":",
        y_local.sum()
    )

    print(
        spec["workflow_family"],
        ":",
        (~y_local).sum()
    )


 tool_vs_workflow
support: 477
tool_use_error : 136
workflow_error : 341

 constraint_vs_workflow
support: 154
constraint_error : 57
workflow_error : 97


In [ ]:
# ============================================================
# 90. Grouped OOF pairwise arbitration probabilities
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold


pairwise_oof = {}
pairwise_masks = {}


for task_name, spec in pairwise_tasks.items():

    applicable = (
        (transition_arr == spec["transition"])
        &
        np.isin(
            true_family_arr,
            [
                spec["local_family"],
                spec["workflow_family"],
            ],
        )
    )

    idx = np.where(applicable)[0]

    y_binary = (
        true_family_arr[idx]
        == spec["local_family"]
    ).astype(int)

    X = spec["X"][idx]

    groups = groups_nested[idx]

    prob = np.full(
        len(idx),
        np.nan,
        dtype=float,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        y_binary,
        groups=groups,
    ):

        model_pair = make_specialist_model(
            C=0.03
        )

        model_pair.fit(
            X[tr],
            y_binary[tr],
        )

        prob[va] = (
            model_pair
            .predict_proba(
                X[va]
            )[:, 1]
        )

    assert not np.isnan(prob).any()

    global_prob = np.full(
        n,
        np.nan,
        dtype=float,
    )

    global_prob[idx] = prob

    pairwise_oof[task_name] = global_prob
    pairwise_masks[task_name] = applicable


print(
    "Created pairwise models:",
    list(pairwise_oof.keys())
)

Created pairwise models: ['tool_vs_workflow', 'constraint_vs_workflow']


In [ ]:
# ============================================================
# 91. Pairwise OOF quality check
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

pairwise_quality = []

for task_name, spec in pairwise_tasks.items():

    mask = pairwise_masks[
        task_name
    ]

    y = (
        true_family_arr[mask]
        == spec["local_family"]
    ).astype(int)

    p = pairwise_oof[
        task_name
    ][mask]

    pairwise_quality.append({
        "task":
            task_name,

        "transition":
            spec["transition"],

        "local_family":
            spec["local_family"],

        "support":
            len(y),

        "positives":
            int(y.sum()),

        "prevalence":
            y.mean(),

        "pr_auc":
            average_precision_score(
                y,
                p,
            ),

        "roc_auc":
            roc_auc_score(
                y,
                p,
            ),
    })


pairwise_quality_df = pd.DataFrame(
    pairwise_quality
)

display(
    pairwise_quality_df.round(4)
)

,task,transition,local_family,support,positives,prevalence,pr_auc,roc_auc
0,tool_vs_workflow,TOOL_CALL->TOOL_CALL,tool_use_error,477,136,0.2851,0.7081,0.8693
1,constraint_vs_workflow,ASSISTANT->TOOL_CALL,constraint_error,154,57,0.3701,0.6472,0.7779


In [ ]:
# ============================================================
# 92. Conservative hierarchical pairwise router
# ============================================================

hierarchical_pred = (
    semantic_pred_arr.copy()
)

hierarchical_routed = np.zeros(
    n,
    dtype=bool,
)

hierarchical_source = np.full(
    n,
    "semantic",
    dtype=object,
)


for task_name, spec in pairwise_tasks.items():

    pair_prob = (
        pairwise_oof[
            task_name
        ]
    )

    transition_match = (
        transition_arr
        == spec["transition"]
    )

    semantic_candidate = np.isin(
        semantic_pred_arr,
        [
            spec["local_family"],
            spec["workflow_family"],
        ],
    )

    usable = (
        transition_match
        &
        semantic_candidate
        &
        (~np.isnan(pair_prob))
    )

    local_prediction = (
        pair_prob >= 0.5
    )

    hierarchical_pred[
        usable
        &
        local_prediction
    ] = spec["local_family"]

    hierarchical_pred[
        usable
        &
        (~local_prediction)
    ] = spec["workflow_family"]

    hierarchical_routed[
        usable
    ] = True

    hierarchical_source[
        usable
    ] = task_name


print(
    "Routed examples:",
    hierarchical_routed.sum()
)

print(
    pd.Series(
        hierarchical_source[
            hierarchical_routed
        ]
    ).value_counts()
)

Routed examples: 587
tool_vs_workflow          444
constraint_vs_workflow    143
Name: count, dtype: int64


In [ ]:
# ============================================================
# 93. Hierarchical router evaluation
# ============================================================

hierarchical_correct = (
    hierarchical_pred
    == true_family_arr
)

hierarchical_changed = (
    hierarchical_pred
    != semantic_pred_arr
)

rescues = int(
    (
        hierarchical_changed
        &
        (~base_correct)
        &
        hierarchical_correct
    ).sum()
)

breaks = int(
    (
        hierarchical_changed
        &
        base_correct
        &
        (~hierarchical_correct)
    ).sum()
)

wrong_to_wrong = int(
    (
        hierarchical_changed
        &
        (~base_correct)
        &
        (~hierarchical_correct)
    ).sum()
)


hierarchical_metrics = (
    multiclass_metrics(
        true_family_arr,
        hierarchical_pred,
    )
)


print(
    "Changed:",
    hierarchical_changed.sum()
)

print(
    "Rescues:",
    rescues
)

print(
    "Breaks:",
    breaks
)

print(
    "Wrong-to-wrong:",
    wrong_to_wrong
)

print(
    "Net:",
    rescues - breaks
)


hierarchical_comparison_df = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",
        **multiclass_metrics(
            true_family_arr,
            semantic_pred_arr,
        ),
    },
    {
        "model":
            "nested_one_vs_rest_specialists",
        **multiclass_metrics(
            true_family_arr,
            nested_threshold_pred,
        ),
    },
    {
        "model":
            "pairwise_hierarchical_router",
        **hierarchical_metrics,
    },
])

display(
    hierarchical_comparison_df.round(4)
)

Changed: 122
Rescues: 79
Breaks: 43
Wrong-to-wrong: 0
Net: 36


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,nested_one_vs_rest_specialists,0.5299,0.4585,0.4759,0.5194
2,pairwise_hierarchical_router,0.5393,0.4745,0.4918,0.5292


In [ ]:
# ============================================================
# 94. Contribution by pairwise route
# ============================================================

route_rows = []

for task_name, spec in pairwise_tasks.items():

    route = (
        hierarchical_source
        == task_name
    )

    changed = (
        route
        &
        hierarchical_changed
    )

    rescues = int(
        (
            changed
            &
            (~base_correct)
            &
            hierarchical_correct
        ).sum()
    )

    breaks = int(
        (
            changed
            &
            base_correct
            &
            (~hierarchical_correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            &
            (~base_correct)
            &
            (~hierarchical_correct)
        ).sum()
    )

    route_rows.append({
        "route":
            task_name,

        "routed":
            int(route.sum()),

        "changed":
            int(changed.sum()),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        "change_precision":
            (
                rescues / changed.sum()
                if changed.sum()
                else np.nan
            ),
    })


route_results_df = pd.DataFrame(
    route_rows
)

display(
    route_results_df.round(4)
)

,route,routed,changed,rescues,breaks,wrong_to_wrong,net,change_precision
0,tool_vs_workflow,444,87,57,30,0,27,0.6552
1,constraint_vs_workflow,143,35,22,13,0,9,0.6286


In [ ]:
# ============================================================
# 95. Boundary-level before/after accuracy
# ============================================================

boundary_rows = []

for task_name, spec in pairwise_tasks.items():

    true_pair = (
        (transition_arr == spec["transition"])
        &
        np.isin(
            true_family_arr,
            [
                spec["local_family"],
                spec["workflow_family"],
            ],
        )
    )

    sem_acc = (
        semantic_pred_arr[
            true_pair
        ]
        ==
        true_family_arr[
            true_pair
        ]
    ).mean()

    hier_acc = (
        hierarchical_pred[
            true_pair
        ]
        ==
        true_family_arr[
            true_pair
        ]
    ).mean()

    boundary_rows.append({
        "boundary":
            task_name,

        "support":
            int(true_pair.sum()),

        "semantic_accuracy":
            sem_acc,

        "hierarchical_accuracy":
            hier_acc,

        "accuracy_delta":
            hier_acc - sem_acc,
    })


boundary_accuracy_df = pd.DataFrame(
    boundary_rows
)

display(
    boundary_accuracy_df.round(4)
)

,boundary,support,semantic_accuracy,hierarchical_accuracy,accuracy_delta
0,tool_vs_workflow,477,0.7086,0.7652,0.0566
1,constraint_vs_workflow,154,0.6039,0.6623,0.0584


The pairwise hierarchical router clearly beats both previous systems:

Model	Accuracy	Balanced Acc.	Macro-F1	Weighted F1
Semantic no-t1	0.5151	0.4355	0.4548	0.4969
Nested one-vs-rest specialists	0.5299	0.4585	0.4759	0.5194
Pairwise hierarchical router	0.5393	0.4745	0.4918	0.5292

Relative to the semantic baseline, the router gives roughly +2.42 points accuracy, +3.90 points balanced accuracy, and +3.70 points macro-F1.

More importantly, the intervention behavior is much cleaner:

122 changed predictions
79 rescues
43 breaks
0 wrong-to-wrong
net = +36

That wrong-to-wrong = 0 is especially significant. The conservative candidate-set routing is doing exactly what we wanted: it is not arbitrarily moving errors between unrelated classes. It only arbitrates between two plausible competing explanations.

And both boundaries improve by almost exactly the same amount:

TOOL→TOOL
semantic accuracy      0.7086
hierarchical accuracy  0.7652
delta                  +0.0566


ASSISTANT→TOOL
semantic accuracy      0.6039
hierarchical accuracy  0.6623
delta                  +0.0584

So this is not merely the larger Tool→Tool subset carrying everything. The routing principle generalizes to both mechanisms.

At this point the central result of notebook 17 is becoming:

Failure-family prediction benefits from decomposing the problem according to event-transition structure. Immediate event-pair semantics are substantially more useful for resolving specific competing failure families than for globally shifting a flat multiclass prediction. A transition-routed pairwise hierarchy improves both Tool→Tool (tool_use_error vs workflow_error) and Assistant→Tool (constraint_error vs workflow_error) boundaries, producing 79 rescues against 43 breaks with no wrong-to-wrong transitions.

In [ ]:
# ============================================================
# 96. Pairwise router effect decomposition
# ============================================================

router_effect = np.full(
    n,
    "unchanged",
    dtype=object,
)

router_effect[
    hierarchical_changed
    &
    (~base_correct)
    &
    hierarchical_correct
] = "rescue"

router_effect[
    hierarchical_changed
    &
    base_correct
    &
    (~hierarchical_correct)
] = "break"

router_effect[
    hierarchical_changed
    &
    (~base_correct)
    &
    (~hierarchical_correct)
] = "wrong_to_wrong"


router_diag_df = pd.DataFrame({
    "true_family":
        true_family_arr,

    "semantic_prediction":
        semantic_pred_arr,

    "hierarchical_prediction":
        hierarchical_pred,

    "transition_type":
        transition_arr,

    "route":
        hierarchical_source,

    "effect":
        router_effect,
})


display(
    router_diag_df[
        router_diag_df["effect"]
        != "unchanged"
    ]
    .groupby(
        [
            "route",
            "effect",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

,route,effect,count
0,constraint_vs_workflow,break,13
1,constraint_vs_workflow,rescue,22
2,tool_vs_workflow,break,30
3,tool_vs_workflow,rescue,57


In [ ]:
# ============================================================
# 97. Prediction transitions made by the hierarchical router
# ============================================================

changed_router_df = (
    router_diag_df[
        hierarchical_changed
    ]
    .copy()
)

changed_router_df[
    "prediction_transition"
] = (
    changed_router_df[
        "semantic_prediction"
    ]
    .astype(str)
    +
    " -> "
    +
    changed_router_df[
        "hierarchical_prediction"
    ]
    .astype(str)
)


display(
    changed_router_df
    .groupby(
        [
            "route",
            "prediction_transition",
            "effect",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
)

,route,prediction_transition,effect,count
7,tool_vs_workflow,workflow_error -> tool_use_error,rescue,48
6,tool_vs_workflow,workflow_error -> tool_use_error,break,29
3,constraint_vs_workflow,workflow_error -> constraint_error,rescue,12
1,constraint_vs_workflow,constraint_error -> workflow_error,rescue,10
2,constraint_vs_workflow,workflow_error -> constraint_error,break,9
5,tool_vs_workflow,tool_use_error -> workflow_error,rescue,9
0,constraint_vs_workflow,constraint_error -> workflow_error,break,4
4,tool_vs_workflow,tool_use_error -> workflow_error,break,1


In [ ]:
# ============================================================
# 98. Utility by pairwise decision direction
# ============================================================

direction_summary = (
    changed_router_df
    .groupby(
        [
            "route",
            "prediction_transition",
        ]
    )
    .agg(
        support=(
            "effect",
            "size",
        ),

        rescues=(
            "effect",
            lambda x:
                (x == "rescue").sum(),
        ),

        breaks=(
            "effect",
            lambda x:
                (x == "break").sum(),
        ),

        wrong_to_wrong=(
            "effect",
            lambda x:
                (
                    x
                    == "wrong_to_wrong"
                ).sum(),
        ),
    )
    .reset_index()
)

direction_summary["net"] = (
    direction_summary["rescues"]
    -
    direction_summary["breaks"]
)

direction_summary[
    "precision"
] = (
    direction_summary["rescues"]
    /
    direction_summary["support"]
)

display(
    direction_summary
    .sort_values(
        "net",
        ascending=False,
    )
    .round(4)
)

,route,prediction_transition,support,rescues,breaks,wrong_to_wrong,net,precision
3,tool_vs_workflow,workflow_error -> tool_use_error,77,48,29,0,19,0.6234
2,tool_vs_workflow,tool_use_error -> workflow_error,10,9,1,0,8,0.9000
0,constraint_vs_workflow,constraint_error -> workflow_error,14,10,4,0,6,0.7143
1,constraint_vs_workflow,workflow_error -> constraint_error,21,12,9,0,3,0.5714


In [ ]:
# ============================================================
# 99. Pairwise confidence by intervention outcome
# ============================================================

router_score = np.full(
    n,
    np.nan,
    dtype=float,
)

router_score[
    hierarchical_source
    == "tool_vs_workflow"
] = (
    pairwise_oof[
        "tool_vs_workflow"
    ][
        hierarchical_source
        == "tool_vs_workflow"
    ]
)

router_score[
    hierarchical_source
    == "constraint_vs_workflow"
] = (
    pairwise_oof[
        "constraint_vs_workflow"
    ][
        hierarchical_source
        == "constraint_vs_workflow"
    ]
)


changed_router_df[
    "pairwise_local_prob"
] = (
    router_score[
        changed_router_df.index
    ]
)

changed_router_df[
    "distance_from_boundary"
] = np.abs(
    changed_router_df[
        "pairwise_local_prob"
    ]
    - 0.5
)


display(
    changed_router_df
    .groupby(
        [
            "route",
            "prediction_transition",
            "effect",
        ]
    )[
        [
            "pairwise_local_prob",
            "distance_from_boundary",
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

pairwise_local_prob                                         distance_from_boundary                          \
                                                                               count    mean  median     std     min     max                  count    mean  median     std   
route                  prediction_transition              effect                                                                                                              
constraint_vs_workflow constraint_error -> workflow_error break                    4  0.2073  0.1766  0.1271  0.0897  0.3865                      4  0.2927  0.3234  0.1271   
                                                          rescue                  10  0.2560  0.2552  0.1158  0.1101  0.4259                     10  0.2440  0.2448  0.1158   
                       workflow_error -> constraint_error break                    9  0.7661  0.8134  0.1448  0.5364  0.9628                      9  0.2661  0.3134  0.1448   
                                                          rescue                  12  0.7695  0.7067  0.1262  0.6373  0.9772                     12  0.2695  0.2067  0.1262   
tool_vs_workflow       tool_use_error -> workflow_error   break                    1  0.2852  0.2852     NaN  0.2852  0.2852                      1  0.2148  0.2148     NaN   
                                                          rescue                   9  0.2134  0.1948  0.1350  0.0400  0.3879                      9  0.2866  0.3052  0.1350   
                       workflow_error -> tool_use_error   break                   29  0.7721  0.7571  0.1178  0.5159  0.9521                     29  0.2721  0.2571  0.1178   
                                                          rescue                  48  0.7606  0.7770  0.1218  0.5000  0.9961                     48  0.2606  0.2770  0.1218   

                                                                                  
                                                                     min     max  
route                  prediction_transition              effect                  
constraint_vs_workflow constraint_error -> workflow_error break   0.1135  0.4103  
                                                          rescue  0.0741  0.3899  
                       workflow_error -> constraint_error break   0.0364  0.4628  
                                                          rescue  0.1373  0.4772  
tool_vs_workflow       tool_use_error -> workflow_error   break   0.2148  0.2148  
                                                          rescue  0.1121  0.4600  
                       workflow_error -> tool_use_error   break   0.0159  0.4521  
                                                          rescue  0.0000  0.4961

In [ ]:
# ============================================================
# 100. Pairwise boundary-margin diagnostic
# ============================================================

boundary_margin_grid = [
    0.00,
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.20,
]

margin_rows = []

for m in boundary_margin_grid:

    pred = semantic_pred_arr.copy()

    accept = (
        hierarchical_routed
        &
        (
            np.abs(
                router_score - 0.5
            )
            >= m
        )
    )

    pred[accept] = (
        hierarchical_pred[accept]
    )

    correct = (
        pred == true_family_arr
    )

    changed = (
        pred != semantic_pred_arr
    )

    rescues = int(
        (
            changed
            &
            (~base_correct)
            &
            correct
        ).sum()
    )

    breaks = int(
        (
            changed
            &
            base_correct
            &
            (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            &
            (~base_correct)
            &
            (~correct)
        ).sum()
    )

    metrics = multiclass_metrics(
        true_family_arr,
        pred,
    )

    margin_rows.append({
        "margin":
            m,

        "changed":
            int(changed.sum()),

        "coverage":
            changed.mean(),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        "precision":
            (
                rescues / changed.sum()
                if changed.sum()
                else np.nan
            ),

        **metrics,
    })


pairwise_margin_df = pd.DataFrame(
    margin_rows
)

display(
    pairwise_margin_df
    .sort_values(
        [
            "accuracy",
            "macro_f1",
        ],
        ascending=False,
    )
    .round(4)
)

,margin,changed,coverage,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.000,122,0.0819,79,43,0,36,0.6475,0.5393,0.4745,0.4918,0.5292
2,0.050,116,0.0779,76,40,0,36,0.6552,0.5393,0.4728,0.4907,0.5287
3,0.075,114,0.0766,75,39,0,36,0.6579,0.5393,0.4728,0.4907,0.5287
1,0.025,118,0.0792,76,42,0,34,0.6441,0.5379,0.4722,0.4900,0.5276
4,0.100,112,0.0752,73,39,0,34,0.6518,0.5379,0.4717,0.4896,0.5274
5,0.125,107,0.0719,70,37,0,33,0.6542,0.5373,0.4712,0.4890,0.5268
6,0.150,101,0.0678,64,37,0,27,0.6337,0.5332,0.4671,0.4851,0.5225
7,0.200,83,0.0557,51,32,0,19,0.6145,0.5279,0.4588,0.4777,0.5159


These results say the router should now become direction-sensitive, not globally confidence-gated.

The strongest direction is tool_use_error → workflow_error: 9 rescues, 1 break, net +8, precision 0.90. The reverse workflow_error → tool_use_error is still useful but much noisier: 48 rescues, 29 breaks, net +19, precision 0.623. For constraints, constraint_error → workflow_error is also strong at 10 rescues / 4 breaks, while workflow_error → constraint_error is weaker at 12 / 9.

The boundary-margin test also gives a clear negative result. A symmetric margin around 0.5 does not improve net beyond +36. Margins 0.05–0.075 merely remove rescues and breaks in roughly equal numbers; accuracy stays exactly 0.5393 while balanced accuracy and macro-F1 fall. So there is no reason to add a global confidence margin.

The next experiment should test asymmetric decision thresholds by direction. Conceptually:

semantic = workflow
    require stronger evidence before switching to local error


semantic = local error
    permit a different threshold before switching back to workflow

That matches your empirical behavior much better.

In [ ]:
# ============================================================
# 101. Direction-sensitive pairwise thresholds
# ============================================================

upper_grid = np.arange(
    0.50,
    0.901,
    0.025,
)

lower_grid = np.arange(
    0.10,
    0.501,
    0.025,
)


def apply_directional_router(
    route_name,
    lower_threshold,
    upper_threshold,
):
    spec = pairwise_tasks[route_name]

    p_local = pairwise_oof[
        route_name
    ]

    pred = semantic_pred_arr.copy()

    route_mask = (
        (transition_arr == spec["transition"])
        &
        (~np.isnan(p_local))
        &
        np.isin(
            semantic_pred_arr,
            [
                spec["local_family"],
                spec["workflow_family"],
            ],
        )
    )

    # -------------------------------------------
    # workflow -> local
    # -------------------------------------------
    workflow_to_local = (
        route_mask
        &
        (
            semantic_pred_arr
            == spec["workflow_family"]
        )
        &
        (
            p_local
            >= upper_threshold
        )
    )

    pred[
        workflow_to_local
    ] = spec["local_family"]


    # -------------------------------------------
    # local -> workflow
    # -------------------------------------------
    local_to_workflow = (
        route_mask
        &
        (
            semantic_pred_arr
            == spec["local_family"]
        )
        &
        (
            p_local
            <= lower_threshold
        )
    )

    pred[
        local_to_workflow
    ] = spec["workflow_family"]

    changed = (
        pred != semantic_pred_arr
    )

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            changed
            &
            (~base_correct)
            &
            correct
        ).sum()
    )

    breaks = int(
        (
            changed
            &
            base_correct
            &
            (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            &
            (~base_correct)
            &
            (~correct)
        ).sum()
    )

    return {
        "pred": pred,
        "changed": changed,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
    }

In [ ]:
# ============================================================
# 102. Diagnostic threshold surface by route
# ============================================================

directional_rows = []

for route_name in pairwise_tasks:

    for lower in lower_grid:

        for upper in upper_grid:

            # Avoid overlapping decision regions
            if lower >= upper:
                continue

            result = apply_directional_router(
                route_name=route_name,
                lower_threshold=lower,
                upper_threshold=upper,
            )

            pred = result["pred"]

            metrics = multiclass_metrics(
                true_family_arr,
                pred,
            )

            directional_rows.append({
                "route":
                    route_name,

                "lower_threshold":
                    lower,

                "upper_threshold":
                    upper,

                "changed":
                    int(
                        result[
                            "changed"
                        ].sum()
                    ),

                "rescues":
                    result["rescues"],

                "breaks":
                    result["breaks"],

                "wrong_to_wrong":
                    result[
                        "wrong_to_wrong"
                    ],

                "net":
                    result["net"],

                "precision":
                    (
                        result["rescues"]
                        /
                        result["changed"].sum()
                        if result[
                            "changed"
                        ].sum()
                        else np.nan
                    ),

                **metrics,
            })


directional_surface_df = (
    pd.DataFrame(
        directional_rows
    )
)


for route_name in pairwise_tasks:

    print(
        "\n",
        "=" * 80,
        route_name,
    )

    display(
        directional_surface_df[
            directional_surface_df[
                "route"
            ]
            == route_name
        ]
        .sort_values(
            [
                "net",
                "accuracy",
                "macro_f1",
            ],
            ascending=False,
        )
        .head(20)
        .round(4)
    )


 ================================================================================ tool_vs_workflow


,route,lower_threshold,upper_threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
204,tool_vs_workflow,0.400,0.500,87,57,30,0,27,0.6552,0.5332,0.4691,0.4868,0.5228
221,tool_vs_workflow,0.425,0.500,87,57,30,0,27,0.6552,0.5332,0.4691,0.4868,0.5228
238,tool_vs_workflow,0.450,0.500,87,57,30,0,27,0.6552,0.5332,0.4691,0.4868,0.5228
255,tool_vs_workflow,0.475,0.500,87,57,30,0,27,0.6552,0.5332,0.4691,0.4868,0.5228
272,tool_vs_workflow,0.500,0.500,87,57,30,0,27,0.6552,0.5332,0.4691,0.4868,0.5228
206,tool_vs_workflow,0.400,0.550,82,54,28,0,26,0.6585,0.5326,0.4672,0.4853,0.5218
207,tool_vs_workflow,0.400,0.575,82,54,28,0,26,0.6585,0.5326,0.4672,0.4853,0.5218
223,tool_vs_workflow,0.425,0.550,82,54,28,0,26,0.6585,0.5326,0.4672,0.4853,0.5218
224,tool_vs_workflow,0.425,0.575,82,54,28,0,26,0.6585,0.5326,0.4672,0.4853,0.5218
240,tool_vs_workflow,0.450,0.550,82,54,28,0,26,0.6585,0.5326,0.4672,0.4853,0.5218



 ================================================================================ constraint_vs_workflow


,route,lower_threshold,upper_threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
530,constraint_vs_workflow,0.450,0.575,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
531,constraint_vs_workflow,0.450,0.600,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
532,constraint_vs_workflow,0.450,0.625,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
547,constraint_vs_workflow,0.475,0.575,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
548,constraint_vs_workflow,0.475,0.600,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
549,constraint_vs_workflow,0.475,0.625,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
564,constraint_vs_workflow,0.500,0.575,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
565,constraint_vs_workflow,0.500,0.600,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
566,constraint_vs_workflow,0.500,0.625,33,22,11,0,11,0.6667,0.5225,0.4415,0.4605,0.5043
462,constraint_vs_workflow,0.350,0.575,30,20,10,0,10,0.6667,0.5218,0.4415,0.4604,0.5040


In [ ]:
# ============================================================
# 103. Quantiles of pairwise probability by outcome/direction
# ============================================================

direction_prob_summary = (
    changed_router_df
    .groupby(
        [
            "route",
            "prediction_transition",
            "effect",
        ]
    )[
        "pairwise_local_prob"
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .unstack()
    .reset_index()
)

direction_prob_summary.columns = [
    "route",
    "prediction_transition",
    "effect",
    "q10",
    "q25",
    "q50",
    "q75",
    "q90",
]

display(
    direction_prob_summary.round(4)
)

,route,prediction_transition,effect,q10,q25,q50,q75,q90
0,constraint_vs_workflow,constraint_error -> workflow_error,break,0.1104,0.1414,0.1766,0.2425,0.3289
1,constraint_vs_workflow,constraint_error -> workflow_error,rescue,0.1307,0.1485,0.2552,0.3296,0.4163
2,constraint_vs_workflow,workflow_error -> constraint_error,break,0.5654,0.6741,0.8134,0.8544,0.9114
3,constraint_vs_workflow,workflow_error -> constraint_error,rescue,0.6473,0.6797,0.7067,0.8401,0.9623
4,tool_vs_workflow,tool_use_error -> workflow_error,break,0.2852,0.2852,0.2852,0.2852,0.2852
5,tool_vs_workflow,tool_use_error -> workflow_error,rescue,0.0455,0.1419,0.1948,0.3533,0.3815
6,tool_vs_workflow,workflow_error -> tool_use_error,break,0.6480,0.7013,0.7571,0.8326,0.9378
7,tool_vs_workflow,workflow_error -> tool_use_error,rescue,0.6270,0.6723,0.7770,0.8521,0.9041


In [ ]:
# ============================================================
# 104. Direction ablation
# ============================================================

route_direction_defs = {
    "tool_workflow_to_local": {
        "route":
            "tool_vs_workflow",
        "from":
            "workflow_error",
        "to":
            "tool_use_error",
    },

    "tool_local_to_workflow": {
        "route":
            "tool_vs_workflow",
        "from":
            "tool_use_error",
        "to":
            "workflow_error",
    },

    "constraint_workflow_to_local": {
        "route":
            "constraint_vs_workflow",
        "from":
            "workflow_error",
        "to":
            "constraint_error",
    },

    "constraint_local_to_workflow": {
        "route":
            "constraint_vs_workflow",
        "from":
            "constraint_error",
        "to":
            "workflow_error",
    },
}


def evaluate_direction_subset(
    enabled_directions,
):

    pred = semantic_pred_arr.copy()

    for direction_name in (
        enabled_directions
    ):

        spec_direction = (
            route_direction_defs[
                direction_name
            ]
        )

        route_name = (
            spec_direction["route"]
        )

        p = pairwise_oof[
            route_name
        ]

        route_spec = (
            pairwise_tasks[
                route_name
            ]
        )

        route_mask = (
            transition_arr
            == route_spec["transition"]
        )

        from_family = (
            spec_direction["from"]
        )

        to_family = (
            spec_direction["to"]
        )

        if (
            to_family
            == route_spec[
                "local_family"
            ]
        ):
            decision = p >= 0.5

        else:
            decision = p < 0.5

        use = (
            route_mask
            &
            (~np.isnan(p))
            &
            (
                semantic_pred_arr
                == from_family
            )
            &
            decision
        )

        pred[use] = to_family


    changed = (
        pred != semantic_pred_arr
    )

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            changed
            &
            (~base_correct)
            &
            correct
        ).sum()
    )

    breaks = int(
        (
            changed
            &
            base_correct
            &
            (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            &
            (~base_correct)
            &
            (~correct)
        ).sum()
    )

    return {
        "pred": pred,
        "changed": int(
            changed.sum()
        ),
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    }

In [ ]:
from itertools import combinations

direction_names = list(
    route_direction_defs.keys()
)

direction_ablation_rows = []

for k in range(
    1,
    len(direction_names) + 1,
):

    for subset in combinations(
        direction_names,
        k,
    ):

        result = (
            evaluate_direction_subset(
                subset
            )
        )

        direction_ablation_rows.append({
            "directions":
                " + ".join(subset),

            "n_directions":
                k,

            **{
                key: value
                for key, value
                in result.items()
                if key != "pred"
            },
        })


direction_ablation_df = (
    pd.DataFrame(
        direction_ablation_rows
    )
)

display(
    direction_ablation_df
    .sort_values(
        [
            "accuracy",
            "macro_f1",
            "net",
        ],
        ascending=False,
    )
    .head(20)
    .round(4)
)

,directions,n_directions,changed,rescues,breaks,wrong_to_wrong,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
14,tool_workflow_to_local + tool_local_to_workflo...,4,122,79,43,0,36,0.5393,0.4745,0.4918,0.5292
11,tool_workflow_to_local + tool_local_to_workflo...,3,101,67,34,0,33,0.5373,0.4696,0.4875,0.5253
10,tool_workflow_to_local + tool_local_to_workflo...,3,108,69,39,0,30,0.5353,0.4740,0.4909,0.5264
12,tool_workflow_to_local + constraint_workflow_t...,3,112,70,42,0,28,0.5339,0.4726,0.4887,0.5244
4,tool_workflow_to_local + tool_local_to_workflow,2,87,57,30,0,27,0.5332,0.4691,0.4868,0.5228
6,tool_workflow_to_local + constraint_local_to_w...,2,91,58,33,0,25,0.5319,0.4677,0.4845,0.5206
5,tool_workflow_to_local + constraint_workflow_t...,2,98,60,38,0,22,0.5299,0.4721,0.4878,0.5216
0,tool_workflow_to_local,1,77,48,29,0,19,0.5279,0.4672,0.4837,0.5180
13,tool_local_to_workflow + constraint_workflow_t...,3,45,31,14,0,17,0.5265,0.4427,0.4623,0.5074
8,tool_local_to_workflow + constraint_local_to_w...,2,24,19,5,0,14,0.5245,0.4379,0.4580,0.5034


These results are quite decisive: the full four-direction router is currently the best architecture, and neither directional threshold tuning nor direction pruning improves it.

The main result is unchanged and now better supported:

Router	Changed	Rescues	Breaks	Net	Accuracy	Bal. Acc.	Macro-F1
Semantic baseline	0	0	0	0	.5151	.4355	.4548
Tool pair only	87	57	30	+27	.5332	.4691	.4868
Best 3-direction subset	101	67	34	+33	.5373	.4696	.4875
Full 4-direction router	122	79	43	+36	.5393	.4745	.4918

So for now, keep all four directions.

The threshold sweep is also revealing. For Tool↔Workflow, the best diagnostic result is basically the original 0.5 boundary: increasing the workflow→tool threshold removes almost equal numbers of rescues and breaks. The constraint boundary behaves similarly. This matches the probability distributions: rescue and break probabilities overlap heavily inside each direction. For example, workflow→tool rescues have median 0.777, while breaks have median 0.757. There simply is not a clean scalar confidence threshold separating them.

That means we have learned something important:

The pairwise model is good at deciding the boundary globally, but its probability magnitude is not a good estimator of whether an individual intervention will be beneficial relative to the semantic model.

So I would stop threshold tuning here.

The next research question should be different:

Can the pairwise router and semantic model be combined at the probability level instead of replacing one another by hard decisions?

Right now we throw away useful information. If Tool→Tool is applicable, the pairwise model outputs only:

P(tool_use∣tool_use∨workflow)

while the semantic model has probabilities across all five families.

We can use the pairwise model to redistribute probability mass only between the two competing classes, preserving the semantic model's total mass assigned to that pair.

That is much more principled than hard overriding.

For example, if semantic probabilities are:

workflow      0.45
tool_use      0.20
constraint    0.15
grounding     0.15
reasoning     0.05

then the total mass for {workflow, tool_use} is 0.65.

If the pairwise router says:

P(tool_use | tool_use or workflow) = 0.75

we transform that pair to:

tool_use = 0.65 × 0.75 = 0.4875
workflow = 0.65 × 0.25 = 0.1625

while leaving all other probabilities unchanged.

This is effectively conditional probability factorization.

In [ ]:
# ============================================================
# 105. Pairwise conditional probability reconciliation
# ============================================================

# semantic/no-t1 probability matrix
reconciled_prob = remove_t1_prob.copy()

# Mapping already recovered earlier
prob_col = {
    "workflow_error": 0,
    "constraint_error": 1,
    "tool_use_error": 2,
    "grounding_state_error": 3,
    "reasoning_value_error": 4,
}


reconciliation_source = np.full(
    n,
    "semantic",
    dtype=object,
)


for route_name, spec in pairwise_tasks.items():

    p_local = pairwise_oof[
        route_name
    ]

    transition_match = (
        transition_arr
        == spec["transition"]
    )

    usable = (
        transition_match
        &
        (~np.isnan(p_local))
    )

    local_col = prob_col[
        spec["local_family"]
    ]

    workflow_col = prob_col[
        spec["workflow_family"]
    ]

    # Total semantic probability assigned to the pair
    pair_mass = (
        reconciled_prob[
            usable,
            local_col
        ]
        +
        reconciled_prob[
            usable,
            workflow_col
        ]
    )

    # Redistribute that mass according to pairwise conditional probability
    reconciled_prob[
        usable,
        local_col
    ] = (
        pair_mass
        *
        p_local[usable]
    )

    reconciled_prob[
        usable,
        workflow_col
    ] = (
        pair_mass
        *
        (
            1.0
            - p_local[usable]
        )
    )

    reconciliation_source[
        usable
    ] = route_name


print(
    "Reconciled rows:",
    np.sum(
        reconciliation_source
        != "semantic"
    )
)

print(
    "Row sums:",
    reconciled_prob.sum(axis=1).min(),
    reconciled_prob.sum(axis=1).max(),
)

Reconciled rows: 631
Row sums: 0.9999998284038156 1.0000002043088898


In [ ]:
# ============================================================
# 106. Predictions from reconciled probabilities
# ============================================================

col_to_family = {
    0: "workflow_error",
    1: "constraint_error",
    2: "tool_use_error",
    3: "grounding_state_error",
    4: "reasoning_value_error",
}

reconciled_idx = (
    np.argmax(
        reconciled_prob,
        axis=1,
    )
)

reconciled_pred = np.array([
    col_to_family[int(i)]
    for i in reconciled_idx
])


print(
    pd.Series(
        reconciled_pred
    ).value_counts()
)

print(
    "Changed vs semantic:",
    np.sum(
        reconciled_pred
        != semantic_pred_arr
    )
)

workflow_error           799
constraint_error         309
tool_use_error           199
grounding_state_error    156
reasoning_value_error     26
Name: count, dtype: int64
Changed vs semantic: 124


In [ ]:
# ============================================================
# 107. Reconciled router evaluation
# ============================================================

reconciled_correct = (
    reconciled_pred
    == true_family_arr
)

reconciled_changed = (
    reconciled_pred
    != semantic_pred_arr
)

reconciled_rescues = int(
    (
        reconciled_changed
        &
        (~base_correct)
        &
        reconciled_correct
    ).sum()
)

reconciled_breaks = int(
    (
        reconciled_changed
        &
        base_correct
        &
        (~reconciled_correct)
    ).sum()
)

reconciled_wrong = int(
    (
        reconciled_changed
        &
        (~base_correct)
        &
        (~reconciled_correct)
    ).sum()
)

reconciled_metrics = multiclass_metrics(
    true_family_arr,
    reconciled_pred,
)


print(
    "Changed:",
    int(
        reconciled_changed.sum()
    )
)

print(
    "Rescues:",
    reconciled_rescues
)

print(
    "Breaks:",
    reconciled_breaks
)

print(
    "Wrong-to-wrong:",
    reconciled_wrong
)

print(
    "Net:",
    reconciled_rescues
    - reconciled_breaks
)


comparison_107 = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",
        **multiclass_metrics(
            true_family_arr,
            semantic_pred_arr,
        ),
    },

    {
        "model":
            "hard_pairwise_router",
        **multiclass_metrics(
            true_family_arr,
            hierarchical_pred,
        ),
    },

    {
        "model":
            "probability_reconciliation",
        **reconciled_metrics,
    },
])

display(
    comparison_107.round(4)
)

Changed: 124
Rescues: 79
Breaks: 44
Wrong-to-wrong: 1
Net: 35


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,hard_pairwise_router,0.5393,0.4745,0.4918,0.5292
2,probability_reconciliation,0.5386,0.4736,0.4909,0.5284


In [ ]:
# ============================================================
# 108. Soft pairwise probability reconciliation
# ============================================================

alpha_grid = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]

soft_rows = []

soft_predictions = {}


for alpha in alpha_grid:

    P = remove_t1_prob.copy()

    for route_name, spec in (
        pairwise_tasks.items()
    ):

        p_pair = pairwise_oof[
            route_name
        ]

        usable = (
            (transition_arr == spec["transition"])
            &
            (~np.isnan(p_pair))
        )

        local_col = prob_col[
            spec["local_family"]
        ]

        workflow_col = prob_col[
            spec["workflow_family"]
        ]

        local_sem = (
            P[
                usable,
                local_col
            ]
        )

        workflow_sem = (
            P[
                usable,
                workflow_col
            ]
        )

        pair_mass = (
            local_sem
            +
            workflow_sem
        )

        # semantic conditional probability inside the pair
        semantic_conditional = (
            local_sem
            /
            np.maximum(
                pair_mass,
                1e-12,
            )
        )

        q = (
            (1 - alpha)
            *
            semantic_conditional
            +
            alpha
            *
            p_pair[usable]
        )

        P[
            usable,
            local_col
        ] = (
            pair_mass * q
        )

        P[
            usable,
            workflow_col
        ] = (
            pair_mass
            *
            (1 - q)
        )


    pred_idx = np.argmax(
        P,
        axis=1,
    )

    pred = np.array([
        col_to_family[
            int(i)
        ]
        for i in pred_idx
    ])

    soft_predictions[
        alpha
    ] = pred

    correct = (
        pred
        == true_family_arr
    )

    changed = (
        pred
        != semantic_pred_arr
    )

    rescues = int(
        (
            changed
            &
            (~base_correct)
            &
            correct
        ).sum()
    )

    breaks = int(
        (
            changed
            &
            base_correct
            &
            (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            &
            (~base_correct)
            &
            (~correct)
        ).sum()
    )

    soft_rows.append({
        "alpha":
            alpha,

        "changed":
            int(
                changed.sum()
            ),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


soft_reconciliation_df = (
    pd.DataFrame(
        soft_rows
    )
)

display(
    soft_reconciliation_df.round(4)
)

,alpha,changed,rescues,breaks,wrong_to_wrong,net,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.00,0,0,0,0,0,0.5151,0.4355,0.4548,0.4969
1,0.25,56,36,19,1,17,0.5265,0.4539,0.4732,0.5131
2,0.50,95,58,34,3,24,0.5312,0.4628,0.4811,0.5197
3,0.75,116,73,41,2,32,0.5366,0.4697,0.4874,0.5258
4,1.00,124,79,44,1,35,0.5386,0.4736,0.4909,0.5284


This result is pretty conclusive: hard pairwise routing is better than probability reconciliation, and the soft interpolation behaves monotonically toward the hard solution.

The reconciliation experiment does not uncover extra value. Full reconciliation gives net +35, accuracy 0.5386, macro-F1 0.4909, while the hard router remains slightly better at net +36, accuracy 0.5393, macro-F1 0.4918. The interpolation confirms the same story: as alpha increases from 0 → 1, performance steadily improves, so there is no evidence that semantic conditional probabilities should retain weight once the pairwise arbiter is invoked.

That is an important architectural conclusion:

When a transition-specific pairwise boundary is applicable, the pairwise classifier should act as the decision authority between those two candidate families rather than softly blending with the flat semantic classifier.

The semantic model still matters upstream because it defines the candidate set and prevents unrelated-family moves. But inside the candidate pair, the specialist boundary is better treated as a hard arbiter.

Also note the one wrong_to_wrong introduced by probability reconciliation. The hard router had zero. That is another reason to prefer the constrained hard decision: probability redistribution can alter the global argmax in ways that escape the intended pairwise correction logic.

So I would freeze these findings:

flat semantic baseline: 0.5151
nested one-vs-rest specialists: 0.5299
probability reconciliation: 0.5386
hard transition-conditioned pairwise router: 0.5393
all four pairwise directions are useful
confidence/margin gating does not improve it
global history does not improve the local boundary
hand-built structure adds almost nothing beyond the embedding pair

The next question should no longer be “how do we combine these two pairwise experts?” The bigger remaining opportunity is:

Are there additional transition-conditioned pairwise family boundaries beyond the two we manually discovered?

Right now we manually tested:

TOOL_CALL -> TOOL_CALL
    tool_use_error ↔ workflow_error


ASSISTANT -> TOOL_CALL
    constraint_error ↔ workflow_error

But your earlier transition/family tables showed other systematic confusions, especially around TOOL_CALL → ASSISTANT, with grounding, constraint, and workflow. Instead of guessing the next expert, let's systematically search every sufficiently supported pair.

Next experiment: pairwise boundary map

For every transition type and every pair of failure families, measure whether they are learnably separable using the immediate current+previous role-aware representation.

This gives you a data-driven MoE topology.

In [ ]:
# ============================================================
# 109. Discover transition-conditioned pairwise boundaries
# ============================================================

from itertools import combinations
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

families = sorted(np.unique(true_family_arr))

transition_previous_embedding = {
    "TOOL_CALL->TOOL_CALL": E_last_tool,
    "TOOL_CALL->ASSISTANT": E_last_tool,
    "ASSISTANT->TOOL_CALL": E_last_assistant,
    "ASSISTANT->ASSISTANT": E_last_assistant,
}

pairwise_discovery_rows = []

MIN_SUPPORT = 60
MIN_PER_CLASS = 15


for transition_name, E_prev in transition_previous_embedding.items():

    transition_mask = (
        transition_arr == transition_name
    )

    for family_a, family_b in combinations(
        families,
        2
    ):

        mask = (
            transition_mask
            &
            np.isin(
                true_family_arr,
                [family_a, family_b],
            )
        )

        idx = np.where(mask)[0]

        if len(idx) < MIN_SUPPORT:
            continue

        y = (
            true_family_arr[idx]
            == family_a
        ).astype(int)

        class_counts = np.bincount(
            y,
            minlength=2
        )

        if class_counts.min() < MIN_PER_CLASS:
            continue

        X_current = (
            train_current_embeddings[idx]
        )

        X_previous = (
            E_prev[idx]
        )

        X_relation = np.concatenate(
            [
                X_current,
                X_previous,
            ],
            axis=1,
        )

        groups = groups_nested[idx]

        oof = np.full(
            len(idx),
            np.nan,
            dtype=float,
        )

        cv = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=42,
        )

        for tr, va in cv.split(
            X_relation,
            y,
            groups=groups,
        ):

            m = make_specialist_model(
                C=0.03
            )

            m.fit(
                X_relation[tr],
                y[tr],
            )

            oof[va] = (
                m.predict_proba(
                    X_relation[va]
                )[:, 1]
            )

        if np.isnan(oof).any():
            continue

        prevalence = y.mean()

        pr_auc = average_precision_score(
            y,
            oof,
        )

        roc_auc = roc_auc_score(
            y,
            oof,
        )

        # Direction-free ROC separability
        roc_sep = max(
            roc_auc,
            1.0 - roc_auc
        )

        pairwise_discovery_rows.append({
            "transition":
                transition_name,

            "family_a":
                family_a,

            "family_b":
                family_b,

            "support":
                len(idx),

            "family_a_count":
                int(y.sum()),

            "family_b_count":
                int(len(y) - y.sum()),

            "prevalence_a":
                prevalence,

            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "roc_separation":
                roc_sep,
        })


pairwise_boundary_map_df = (
    pd.DataFrame(
        pairwise_discovery_rows
    )
)

display(
    pairwise_boundary_map_df
    .sort_values(
        [
            "roc_separation",
            "support",
        ],
        ascending=False,
    )
    .round(4)
)

,transition,family_a,family_b,support,family_a_count,family_b_count,prevalence_a,pr_auc,roc_auc,roc_separation
7,TOOL_CALL->ASSISTANT,constraint_error,reasoning_value_error,145,128,17,0.8828,0.9960,0.9710,0.9710
13,TOOL_CALL->ASSISTANT,reasoning_value_error,workflow_error,139,17,122,0.1223,0.7698,0.9576,0.9576
3,TOOL_CALL->TOOL_CALL,grounding_state_error,tool_use_error,203,67,136,0.3300,0.8629,0.9497,0.9497
15,ASSISTANT->TOOL_CALL,constraint_error,grounding_state_error,82,57,25,0.6951,0.9771,0.9446,0.9446
2,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,381,40,341,0.1050,0.5236,0.9185,0.9185
1,TOOL_CALL->TOOL_CALL,constraint_error,tool_use_error,176,40,136,0.2273,0.7103,0.9060,0.9060
4,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,408,67,341,0.1642,0.6768,0.8959,0.8959
10,TOOL_CALL->ASSISTANT,grounding_state_error,reasoning_value_error,114,97,17,0.8509,0.9720,0.8775,0.8775
5,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,477,136,341,0.2851,0.7059,0.8689,0.8689
19,ASSISTANT->TOOL_CALL,grounding_state_error,workflow_error,122,25,97,0.2049,0.6246,0.8573,0.8573


In [ ]:
# ============================================================
# 110. Rank pairwise experts by strength × support
# ============================================================

pairwise_boundary_map_df[
    "separation_above_chance"
] = (
    pairwise_boundary_map_df[
        "roc_separation"
    ]
    - 0.5
)

pairwise_boundary_map_df[
    "evidence_score"
] = (
    pairwise_boundary_map_df[
        "separation_above_chance"
    ]
    *
    np.sqrt(
        pairwise_boundary_map_df[
            "support"
        ]
    )
)

display(
    pairwise_boundary_map_df
    .sort_values(
        "evidence_score",
        ascending=False,
    )
    .head(30)
    .round(4)
)

,transition,family_a,family_b,support,family_a_count,family_b_count,prevalence_a,pr_auc,roc_auc,roc_separation,separation_above_chance,evidence_score
2,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,381,40,341,0.1050,0.5236,0.9185,0.9185,0.4185,8.1683
5,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,477,136,341,0.2851,0.7059,0.8689,0.8689,0.3689,8.0566
4,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,408,67,341,0.1642,0.6768,0.8959,0.8959,0.3959,7.9967
3,TOOL_CALL->TOOL_CALL,grounding_state_error,tool_use_error,203,67,136,0.3300,0.8629,0.9497,0.9497,0.4497,6.4078
7,TOOL_CALL->ASSISTANT,constraint_error,reasoning_value_error,145,128,17,0.8828,0.9960,0.9710,0.9710,0.4710,5.6722
13,TOOL_CALL->ASSISTANT,reasoning_value_error,workflow_error,139,17,122,0.1223,0.7698,0.9576,0.9576,0.4576,5.3947
1,TOOL_CALL->TOOL_CALL,constraint_error,tool_use_error,176,40,136,0.2273,0.7103,0.9060,0.9060,0.4060,5.3859
10,TOOL_CALL->ASSISTANT,grounding_state_error,reasoning_value_error,114,97,17,0.8509,0.9720,0.8775,0.8775,0.3775,4.0306
15,ASSISTANT->TOOL_CALL,constraint_error,grounding_state_error,82,57,25,0.6951,0.9771,0.9446,0.9446,0.4446,4.0257
19,ASSISTANT->TOOL_CALL,grounding_state_error,workflow_error,122,25,97,0.2049,0.6246,0.8573,0.8573,0.3573,3.9467


In [ ]:
# ============================================================
# 111. Pairwise learnability × semantic confusion
# ============================================================

confusion_rows = []

for _, row in (
    pairwise_boundary_map_df
    .iterrows()
):

    transition_name = row["transition"]
    a = row["family_a"]
    b = row["family_b"]

    mask = (
        (transition_arr == transition_name)
        &
        np.isin(
            true_family_arr,
            [a, b],
        )
    )

    # Only count mistakes between the two families
    a_to_b = int(
        (
            mask
            &
            (true_family_arr == a)
            &
            (semantic_pred_arr == b)
        ).sum()
    )

    b_to_a = int(
        (
            mask
            &
            (true_family_arr == b)
            &
            (semantic_pred_arr == a)
        ).sum()
    )

    confusion_rows.append({
        **row.to_dict(),

        "a_to_b_confusions":
            a_to_b,

        "b_to_a_confusions":
            b_to_a,

        "pair_confusions":
            a_to_b + b_to_a,
    })


pairwise_opportunity_df = pd.DataFrame(
    confusion_rows
)

pairwise_opportunity_df[
    "opportunity_score"
] = (
    pairwise_opportunity_df[
        "pair_confusions"
    ]
    *
    (
        pairwise_opportunity_df[
            "roc_separation"
        ]
        - 0.5
    )
)


display(
    pairwise_opportunity_df
    .sort_values(
        "opportunity_score",
        ascending=False,
    )
    .head(30)
    .round(4)
)

,transition,family_a,family_b,support,family_a_count,family_b_count,prevalence_a,pr_auc,roc_auc,roc_separation,separation_above_chance,evidence_score,a_to_b_confusions,b_to_a_confusions,pair_confusions,opportunity_score
5,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,477,136,341,0.2851,0.7059,0.8689,0.8689,0.3689,8.0566,72,34,106,39.1020
9,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,250,128,122,0.5120,0.7180,0.7136,0.7136,0.2136,3.3777,51,29,80,17.0902
4,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,408,67,341,0.1642,0.6768,0.8959,0.8959,0.3959,7.9967,31,12,43,17.0235
2,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,381,40,341,0.1050,0.5236,0.9185,0.9185,0.4185,8.1683,22,13,35,14.6466
17,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,154,57,97,0.3701,0.6472,0.7779,0.7779,0.2779,3.4486,30,20,50,13.8949
6,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,225,128,97,0.5689,0.7156,0.7025,0.7025,0.2025,3.0372,15,27,42,8.5042
12,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,219,97,122,0.4429,0.6072,0.6533,0.6533,0.1533,2.2684,39,16,55,8.4308
19,ASSISTANT->TOOL_CALL,grounding_state_error,workflow_error,122,25,97,0.2049,0.6246,0.8573,0.8573,0.3573,3.9467,15,3,18,6.4318
20,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,133,36,97,0.2707,0.4472,0.6546,0.6546,0.1546,1.7834,24,5,29,4.4845
1,TOOL_CALL->TOOL_CALL,constraint_error,tool_use_error,176,40,136,0.2273,0.7103,0.9060,0.9060,0.4060,5.3859,5,4,9,3.6538


In [ ]:
# ============================================================
# 112. Zero-history transition diagnostics
# ============================================================

zero_history_mask = np.isin(
    transition_arr,
    [
        "NO_HISTORY->ASSISTANT",
        "NO_HISTORY->TOOL_CALL",
    ],
)

print("Zero-history support:", int(zero_history_mask.sum()))

display(
    pd.crosstab(
        transition_arr[zero_history_mask],
        true_family_arr[zero_history_mask],
        margins=True,
    )
)

display(
    pd.crosstab(
        transition_arr[zero_history_mask],
        semantic_pred_arr[zero_history_mask],
        margins=True,
    )
)

Zero-history support: 56


col_0,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error,All
row_0,,,,,,
NO_HISTORY->ASSISTANT,9,7,0,7,11,34
NO_HISTORY->TOOL_CALL,4,6,2,7,3,22
All,13,13,2,14,14,56


col_0,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error,All
row_0,,,,,,
NO_HISTORY->ASSISTANT,4,11,2,0,17,34
NO_HISTORY->TOOL_CALL,2,4,1,5,10,22
All,6,15,3,5,27,56


In [ ]:
# ============================================================
# 113. Zero-history semantic confusion
# ============================================================

zero_idx = np.where(
    zero_history_mask
)[0]

zero_confusion_df = pd.DataFrame({
    "transition":
        transition_arr[zero_idx],

    "true_family":
        true_family_arr[zero_idx],

    "semantic_prediction":
        semantic_pred_arr[zero_idx],

    "correct":
        semantic_pred_arr[zero_idx]
        == true_family_arr[zero_idx],

    "history_event_count":
        history_count[zero_idx],
})

display(
    zero_confusion_df
    .groupby(
        [
            "transition",
            "true_family",
            "semantic_prediction",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        "count",
        ascending=False,
    )
)

,transition,true_family,semantic_prediction,count
9,NO_HISTORY->ASSISTANT,workflow_error,workflow_error,10
3,NO_HISTORY->ASSISTANT,grounding_state_error,grounding_state_error,7
2,NO_HISTORY->ASSISTANT,constraint_error,workflow_error,5
0,NO_HISTORY->ASSISTANT,constraint_error,constraint_error,3
19,NO_HISTORY->TOOL_CALL,tool_use_error,workflow_error,3
18,NO_HISTORY->TOOL_CALL,tool_use_error,tool_use_error,3
14,NO_HISTORY->TOOL_CALL,grounding_state_error,workflow_error,3
13,NO_HISTORY->TOOL_CALL,grounding_state_error,grounding_state_error,3
20,NO_HISTORY->TOOL_CALL,workflow_error,workflow_error,3
5,NO_HISTORY->ASSISTANT,tool_use_error,grounding_state_error,3


In [ ]:
# ============================================================
# 114. Zero-history pairwise boundary scan
# ============================================================

from itertools import combinations
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

zero_pair_rows = []

families = sorted(
    np.unique(
        true_family_arr[
            zero_history_mask
        ]
    )
)

for transition_name in [
    "NO_HISTORY->ASSISTANT",
    "NO_HISTORY->TOOL_CALL",
]:

    transition_mask = (
        transition_arr
        == transition_name
    )

    for family_a, family_b in combinations(
        families,
        2,
    ):

        mask = (
            transition_mask
            &
            np.isin(
                true_family_arr,
                [
                    family_a,
                    family_b,
                ],
            )
        )

        idx = np.where(mask)[0]

        # Tiny regime: use conservative minimums
        if len(idx) < 12:
            continue

        y = (
            true_family_arr[idx]
            == family_a
        ).astype(int)

        counts = np.bincount(
            y,
            minlength=2,
        )

        if counts.min() < 4:
            continue

        X = (
            train_current_embeddings[
                idx
            ]
        )

        groups = groups_nested[idx]

        # Cannot always use 5 folds with tiny minority classes
        n_splits = min(
            4,
            int(counts.min()),
        )

        if n_splits < 2:
            continue

        oof = np.full(
            len(idx),
            np.nan,
        )

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=42,
        )

        try:
            for tr, va in cv.split(
                X,
                y,
                groups=groups,
            ):

                m = make_specialist_model(
                    C=0.03
                )

                m.fit(
                    X[tr],
                    y[tr],
                )

                oof[va] = (
                    m.predict_proba(
                        X[va]
                    )[:, 1]
                )

        except ValueError:
            continue

        if np.isnan(oof).any():
            continue

        roc = roc_auc_score(
            y,
            oof,
        )

        zero_pair_rows.append({
            "transition":
                transition_name,

            "family_a":
                family_a,

            "family_b":
                family_b,

            "support":
                len(idx),

            "family_a_count":
                int(y.sum()),

            "family_b_count":
                int((1 - y).sum()),

            "prevalence_a":
                y.mean(),

            "pr_auc":
                average_precision_score(
                    y,
                    oof,
                ),

            "roc_auc":
                roc,

            "roc_separation":
                max(
                    roc,
                    1 - roc,
                ),
        })


zero_history_pairwise_df = (
    pd.DataFrame(
        zero_pair_rows
    )
)

display(
    zero_history_pairwise_df
    .sort_values(
        [
            "roc_separation",
            "support",
        ],
        ascending=False,
    )
    .round(4)
)

,transition,family_a,family_b,support,family_a_count,family_b_count,prevalence_a,pr_auc,roc_auc,roc_separation
4,NO_HISTORY->ASSISTANT,grounding_state_error,workflow_error,18,7,11,0.3889,0.9379,0.9610,0.9610
0,NO_HISTORY->ASSISTANT,constraint_error,grounding_state_error,16,9,7,0.5625,0.8697,0.8095,0.8095
6,NO_HISTORY->TOOL_CALL,grounding_state_error,tool_use_error,13,6,7,0.4615,0.8626,0.7857,0.7857
1,NO_HISTORY->ASSISTANT,constraint_error,tool_use_error,16,9,7,0.5625,0.5156,0.2857,0.7143
3,NO_HISTORY->ASSISTANT,grounding_state_error,tool_use_error,14,7,7,0.5000,0.4486,0.3061,0.6939
5,NO_HISTORY->ASSISTANT,tool_use_error,workflow_error,18,7,11,0.3889,0.5285,0.6494,0.6494
2,NO_HISTORY->ASSISTANT,constraint_error,workflow_error,20,9,11,0.4500,0.5772,0.6061,0.6061


In [ ]:
# ============================================================
# 115. Is zero-history itself predictive?
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

zero_indicator = (
    history_count == 0
).astype(float)[:, None]

current_role_tool = (
    event_roles == "TOOL_CALL"
).astype(float)[:, None]

current_role_assistant = (
    event_roles == "ASSISTANT"
).astype(float)[:, None]

X_semantic_only = (
    train_current_embeddings
)

X_semantic_zero_state = np.concatenate(
    [
        train_current_embeddings,
        zero_indicator,
        current_role_tool,
        current_role_assistant,
    ],
    axis=1,
)

zero_state_reps = {
    "semantic_only":
        X_semantic_only,

    "semantic_plus_zero_state":
        X_semantic_zero_state,
}

In [ ]:
# ============================================================
# 116. Multiclass OOF test of zero-history state
# ============================================================

from sklearn.preprocessing import LabelEncoder

family_order = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

family_to_idx = {
    family: i
    for i, family
    in enumerate(family_order)
}

y_multi = np.array([
    family_to_idx[x]
    for x in true_family_arr
])

zero_state_rows = []

for rep_name, X in zero_state_reps.items():

    pred = np.full(
        n,
        -1,
        dtype=int,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        y_multi,
        groups=groups_nested,
    ):

        m = LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )

        m.fit(
            X[tr],
            y_multi[tr],
        )

        pred[va] = m.predict(
            X[va]
        )

    pred_family = np.array([
        family_order[i]
        for i in pred
    ])

    zero_state_rows.append({
        "representation":
            rep_name,

        "accuracy":
            accuracy_score(
                true_family_arr,
                pred_family,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_family_arr,
                pred_family,
            ),

        "macro_f1":
            f1_score(
                true_family_arr,
                pred_family,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                true_family_arr,
                pred_family,
                average="weighted",
            ),
    })


zero_state_test_df = pd.DataFrame(
    zero_state_rows
)

display(
    zero_state_test_df.round(4)
)

,representation,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_only,0.4909,0.5597,0.4747,0.4961
1,semantic_plus_zero_state,0.4768,0.5489,0.4663,0.4839


In [ ]:
# ============================================================
# 116. Multiclass OOF test of zero-history state
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

family_order = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

family_to_idx = {
    family: i
    for i, family in enumerate(family_order)
}

y_multi = np.array([
    family_to_idx[x]
    for x in true_family_arr
])

zero_state_rows = []

# IMPORTANT: store OOF predictions here
zero_state_predictions = {}

for rep_name, X in zero_state_reps.items():

    pred = np.full(
        n,
        -1,
        dtype=int,
    )

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr, va in cv.split(
        X,
        y_multi,
        groups=groups_nested,
    ):

        m = LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )

        m.fit(
            X[tr],
            y_multi[tr],
        )

        pred[va] = m.predict(
            X[va]
        )

    assert (pred >= 0).all()

    pred_family = np.array([
        family_order[i]
        for i in pred
    ])

    # Store for Cell 117
    zero_state_predictions[
        rep_name
    ] = pred_family

    zero_state_rows.append({
        "representation":
            rep_name,

        "accuracy":
            accuracy_score(
                true_family_arr,
                pred_family,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_family_arr,
                pred_family,
            ),

        "macro_f1":
            f1_score(
                true_family_arr,
                pred_family,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                true_family_arr,
                pred_family,
                average="weighted",
            ),
    })


zero_state_test_df = pd.DataFrame(
    zero_state_rows
)

display(
    zero_state_test_df.round(4)
)

print(
    "Stored predictions:",
    list(zero_state_predictions.keys())
)

,representation,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_only,0.4909,0.5597,0.4747,0.4961
1,semantic_plus_zero_state,0.4768,0.5489,0.4663,0.4839


Stored predictions: ['semantic_only', 'semantic_plus_zero_state']


In [ ]:
# ============================================================
# 117. Zero-history subset effect
# ============================================================

zero_rows = []

mask = (
    history_count == 0
)

print(
    "Zero-history support:",
    int(mask.sum())
)

for rep_name, pred_family in (
    zero_state_predictions.items()
):

    zero_rows.append({
        "model":
            rep_name,

        "support":
            int(mask.sum()),

        "accuracy":
            accuracy_score(
                true_family_arr[mask],
                pred_family[mask],
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_family_arr[mask],
                pred_family[mask],
            ),

        "macro_f1":
            f1_score(
                true_family_arr[mask],
                pred_family[mask],
                average="macro",
            ),
    })


zero_history_model_df = pd.DataFrame(
    zero_rows
)

display(
    zero_history_model_df.round(4)
)

Zero-history support: 56


,model,support,accuracy,balanced_accuracy,macro_f1
0,semantic_only,56,0.3214,0.3571,0.2482
1,semantic_plus_zero_state,56,0.3393,0.3714,0.2613


In [ ]:
# ============================================================
# 118. What did explicit zero-state features change?
# ============================================================

sem_zero_pred = zero_state_predictions[
    "semantic_only"
]

state_zero_pred = zero_state_predictions[
    "semantic_plus_zero_state"
]

mask = (
    history_count == 0
)

changed = (
    state_zero_pred[mask]
    != sem_zero_pred[mask]
)

sem_correct = (
    sem_zero_pred[mask]
    == true_family_arr[mask]
)

state_correct = (
    state_zero_pred[mask]
    == true_family_arr[mask]
)

rescues = (
    changed
    &
    (~sem_correct)
    &
    state_correct
)

breaks = (
    changed
    &
    sem_correct
    &
    (~state_correct)
)

wrong_to_wrong = (
    changed
    &
    (~sem_correct)
    &
    (~state_correct)
)

print(
    "Changed:",
    int(changed.sum())
)

print(
    "Rescues:",
    int(rescues.sum())
)

print(
    "Breaks:",
    int(breaks.sum())
)

print(
    "Wrong-to-wrong:",
    int(wrong_to_wrong.sum())
)

print(
    "Net:",
    int(rescues.sum() - breaks.sum())
)

Changed: 6
Rescues: 1
Breaks: 0
Wrong-to-wrong: 5
Net: 1


In [ ]:
# ============================================================
# 119. Zero-history boundary opportunity
#      separability × actual semantic confusion
# ============================================================

zero_opportunity_rows = []

for _, row in zero_history_pairwise_df.iterrows():

    transition_name = row["transition"]
    family_a = row["family_a"]
    family_b = row["family_b"]

    mask = (
        (transition_arr == transition_name)
        &
        np.isin(
            true_family_arr,
            [family_a, family_b],
        )
    )

    idx = np.where(mask)[0]

    true_local = true_family_arr[idx]
    pred_local = semantic_pred_arr[idx]

    a_to_b = int(
        (
            (true_local == family_a)
            &
            (pred_local == family_b)
        ).sum()
    )

    b_to_a = int(
        (
            (true_local == family_b)
            &
            (pred_local == family_a)
        ).sum()
    )

    pair_confusions = (
        a_to_b + b_to_a
    )

    separation_above_chance = (
        row["roc_separation"] - 0.5
    )

    # Same basic principle as the main opportunity table:
    # strong boundary is useful only if semantic model
    # actually confuses the two families.
    opportunity_score = (
        pair_confusions
        * separation_above_chance
    )

    zero_opportunity_rows.append({
        **row.to_dict(),

        "a_to_b_confusions":
            a_to_b,

        "b_to_a_confusions":
            b_to_a,

        "pair_confusions":
            pair_confusions,

        "separation_above_chance":
            separation_above_chance,

        "opportunity_score":
            opportunity_score,
    })


zero_history_opportunity_df = (
    pd.DataFrame(
        zero_opportunity_rows
    )
    .sort_values(
        [
            "opportunity_score",
            "pair_confusions",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    zero_history_opportunity_df[
        [
            "transition",
            "family_a",
            "family_b",
            "support",
            "roc_auc",
            "roc_separation",
            "a_to_b_confusions",
            "b_to_a_confusions",
            "pair_confusions",
            "opportunity_score",
        ]
    ].round(4)
)

,transition,family_a,family_b,support,roc_auc,roc_separation,a_to_b_confusions,b_to_a_confusions,pair_confusions,opportunity_score
0,NO_HISTORY->ASSISTANT,grounding_state_error,tool_use_error,14,0.3061,0.6939,0,3,3,0.5816
1,NO_HISTORY->ASSISTANT,constraint_error,workflow_error,20,0.6061,0.6061,5,0,5,0.5303
2,NO_HISTORY->ASSISTANT,constraint_error,grounding_state_error,16,0.8095,0.8095,1,0,1,0.3095
3,NO_HISTORY->ASSISTANT,tool_use_error,workflow_error,18,0.6494,0.6494,2,0,2,0.2987
4,NO_HISTORY->ASSISTANT,constraint_error,tool_use_error,16,0.2857,0.7143,0,1,1,0.2143
5,NO_HISTORY->ASSISTANT,grounding_state_error,workflow_error,18,0.9610,0.9610,0,0,0,0.0000
6,NO_HISTORY->TOOL_CALL,grounding_state_error,tool_use_error,13,0.7857,0.7857,0,0,0,0.0000


In [ ]:
# ============================================================
# 120. Full zero-history semantic confusion matrices
# ============================================================

for transition_name in [
    "NO_HISTORY->ASSISTANT",
    "NO_HISTORY->TOOL_CALL",
]:

    mask = (
        transition_arr == transition_name
    )

    print(
        "\n" + "=" * 72
    )
    print(transition_name)
    print(
        "support:",
        int(mask.sum()),
        "accuracy:",
        round(
            (
                semantic_pred_arr[mask]
                == true_family_arr[mask]
            ).mean(),
            4,
        ),
    )

    display(
        pd.crosstab(
            pd.Series(
                true_family_arr[mask],
                name="true",
            ),
            pd.Series(
                semantic_pred_arr[mask],
                name="semantic_pred",
            ),
            margins=True,
        )
    )


NO_HISTORY->ASSISTANT
support: 34 accuracy: 0.5882


semantic_pred,constraint_error,grounding_state_error,reasoning_value_error,workflow_error,All
true,,,,,
constraint_error,3,1,0,5,9
grounding_state_error,0,7,0,0,7
tool_use_error,1,3,1,2,7
workflow_error,0,0,1,10,11
All,4,11,2,17,34



NO_HISTORY->TOOL_CALL
support: 22 accuracy: 0.5


semantic_pred,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error,All
true,,,,,,
constraint_error,1,1,0,2,0,4
grounding_state_error,0,3,0,0,3,6
reasoning_value_error,0,0,1,0,1,2
tool_use_error,1,0,0,3,3,7
workflow_error,0,0,0,0,3,3
All,2,4,1,5,10,22


The spectacular zero-history AUCs are mostly not actionable. In particular, NO_HISTORY→ASSISTANT grounding↔workflow has ROC-AUC 0.961, but zero semantic cross-confusions. Likewise NO_HISTORY→TOOL_CALL grounding↔tool_use has ROC-AUC 0.786 and zero cross-confusions. A specialist there cannot rescue the errors the semantic model is currently making.

The actual zero-history opportunities are tiny:

NO_HISTORY→ASSISTANT, grounding ↔ tool-use: 3 confusions, but the ROC direction is reversed (0.306, separation 0.694).
constraint ↔ workflow: 5 confusions, but only ROC-AUC 0.606.
constraint ↔ grounding: good separation (0.810) but only 1 confusion.
tool-use ↔ workflow: 2 confusions at 0.649.

So I would freeze NO_HISTORY and make no interventions there. With only 56 rows, trying to extract another 1–3 rescues is much more likely to overfit than materially improve the system.

More importantly, the confusion matrices show why. NO_HISTORY→ASSISTANT is already 20/34 = 58.8% accurate, with grounding 7/7 and workflow 10/11 correct. Its weakness is mainly constraint being swallowed by workflow (5/9) and tool-use scattering across other families. NO_HISTORY→TOOL_CALL is only 11/22 = 50%, but its errors are dispersed across several boundaries rather than concentrated into one high-support, high-separation pair.

So I'd now return to the main router and test expert #3:

TOOL_CALL→TOOL_CALL: grounding_state_error ↔ workflow_error

This is far more compelling:

support                 408
grounding                67
workflow                 341
ROC-AUC                0.8959
semantic cross-confusions 43
opportunity score      17.0235

And crucially, it is complementary to your existing tool_use ↔ workflow expert.

Next cell: build the grounding-vs-workflow expert with the same outer folds

We should use the same recovered length-1489 outer fold assignment, so this stays genuinely cross-fitted and directly comparable to the current router.

In [ ]:
# ============================================================
# Recover the 5-fold assignment from notebook state
# ============================================================

import numpy as np
import pandas as pd

fold_candidates = []

for name, obj in list(globals().items()):

    try:
        arr = np.asarray(obj)
    except Exception:
        continue

    if (
        arr.ndim == 1
        and len(arr) == 1489
    ):
        uniq = pd.unique(arr)

        if 2 <= len(uniq) <= 10:
            fold_candidates.append({
                "name": name,
                "dtype": str(arr.dtype),
                "n_unique": len(uniq),
                "unique_values": list(uniq[:10]),
                "counts": pd.Series(arr).value_counts().sort_index().to_dict(),
            })

fold_candidates_df = pd.DataFrame(fold_candidates)

display(
    fold_candidates_df.sort_values(
        ["n_unique", "name"]
    )
)

,name,dtype,n_unique,unique_values,counts
35,accept,bool,2,"[False, True]","{False: 987, True: 502}"
48,accept_test,bool,2,"[False, True]","{False: 1487, True: 2}"
24,active,float64,2,"[0.0, 1.0]","{0.0: 1269, 1.0: 220}"
88,active_mask,bool,2,"[False, True]","{False: 1404, True: 85}"
38,after_correct,bool,2,"[False, True]","{False: 708, True: 781}"
93,applicable,bool,2,"[False, True]","{False: 1335, True: 154}"
71,assistant_tool_mask,bool,2,"[False, True]","{False: 1269, True: 220}"
43,base_correct,bool,2,"[False, True]","{False: 722, True: 767}"
37,before_correct,bool,2,"[False, True]","{False: 722, True: 767}"
78,binary_y,int64,2,"[1, 0]","{0: 1172, 1: 317}"


In [ ]:
# ============================================================
# Auto-select the exact known outer fold assignment
# ============================================================

target_counts = sorted([298, 298, 298, 298, 297])

outer_fold_arr = None
outer_fold_source = None

for name, obj in list(globals().items()):

    try:
        arr = np.asarray(obj)
    except Exception:
        continue

    if (
        arr.ndim != 1
        or len(arr) != 1489
    ):
        continue

    counts = pd.Series(arr).value_counts()

    if (
        len(counts) == 5
        and sorted(counts.tolist()) == target_counts
    ):
        outer_fold_arr = arr.copy()
        outer_fold_source = name
        break

if outer_fold_arr is None:
    raise RuntimeError(
        "Could not find the known 5-fold assignment."
    )

print("Recovered from:", outer_fold_source)
print(
    "Fold counts:",
    pd.Series(outer_fold_arr)
    .value_counts()
    .sort_index()
    .to_dict()
)

RuntimeError: Could not find the known 5-fold assignment.

In [ ]:
# ============================================================
# Find existing previous-event embedding matrices
# ============================================================

embedding_candidates = []

for name, obj in list(globals().items()):

    if not isinstance(obj, np.ndarray):
        continue

    if obj.ndim != 2:
        continue

    if obj.shape[0] != 1489:
        continue

    embedding_candidates.append(
        (name, obj.shape)
    )

embedding_candidates

[('train_current_embeddings', (1489, 384)),
 ('X_sem_train', (1489, 384)),
 ('h', (1489, 384)),
 ('T_distance', (1489, 6)),
 ('full_prob', (1489, 5)),
 ('remove_t1_prob', (1489, 5)),
 ('E_current', (1489, 384)),
 ('E_last_tool', (1489, 384)),
 ('E_last_assistant', (1489, 384)),
 ('E_second_tool', (1489, 384)),
 ('E_second_assistant', (1489, 384)),
 ('X', (1489, 387)),
 ('X_specialist_probs', (1489, 3)),
 ('X_specialist_active', (1489, 3)),
 ('X_specialists', (1489, 6)),
 ('P_semantic', (1489, 5)),
 ('X_meta', (1489, 11)),
 ('meta_oof_prob', (1489, 5)),
 ('X_tool_use', (1489, 768)),
 ('X_constraint', (1489, 384)),
 ('X_all', (1489, 384)),
 ('current_E', (1489, 384)),
 ('previous_tool_E', (1489, 384)),
 ('history_mean_embedding', (1489, 384)),
 ('X_tool_pair_base', (1489, 768)),
 ('X_tool_pair_structure', (1489, 4)),
 ('X_tool_pair', (1489, 772)),
 ('X_constraint_pair', (1489, 768)),
 ('reconciled_prob', (1489, 5)),
 ('P', (1489, 5)),
 ('E_prev', (1489, 384)),
 ('zero_indicator', (1489, 

In [ ]:
# ============================================================
# Grounding-vs-workflow representation
# ============================================================

boundary_mask = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    &
    np.isin(
        true_family_arr,
        [
            "grounding_state_error",
            "workflow_error",
        ]
    )
)

boundary_idx = np.where(boundary_mask)[0]

X_boundary = np.concatenate(
    [
        train_current_embeddings[boundary_idx],
        E_last_tool[boundary_idx],

        # structural variables from the earlier relation model
        np.column_stack([
            same_tool[boundary_idx].astype(float),
            tool_l2[boundary_idx],
            tool_cosine[boundary_idx],
            history_count[boundary_idx].astype(float),
        ]),
    ],
    axis=1,
)

y_boundary = (
    true_family_arr[boundary_idx]
    == "grounding_state_error"
).astype(int)

print("Boundary support:", len(boundary_idx))
print("Features:", X_boundary.shape)
print("Positive grounding:", int(y_boundary.sum()))
print("Workflow:", int((1 - y_boundary).sum()))

Boundary support: 408
Features: (408, 772)
Positive grounding: 67
Workflow: 341


In [ ]:
[
    name
    for name in globals()
    if (
        "fold" in name.lower()
        or "crossfit" in name.lower()
        or "split" in name.lower()
    )
]

['StratifiedKFold',
 'GroupKFold',
 'StratifiedGroupKFold',
 'fold',
 'crossfit_pred',
 'crossfit_accept',
 'crossfit_source',
 'crossfit_threshold',
 'crossfit_threshold_df',
 'crossfit_metrics',
 'crossfit_correct',
 'candidate_fold_names',
 'fold_assignment',
 'fold_variable_name',
 'fold_verification_df',
 'margin_crossfit_pred',
 'margin_crossfit_mask',
 'train_fold',
 'test_fold',
 'crossfit_pred_arr',
 'crossfit_changed',
 'outer_splits',
 'outer_fold',
 'n_splits',
 'fold_candidates',
 'outer_fold_arr',
 'fold_candidates_df',
 'outer_fold_source']

In [ ]:
for name in [
    "crossfit_threshold_df",
    "accepted_effect_df",
    "override_df",
    "train_targets",
]:
    if name in globals():
        obj = globals()[name]

        print("\n", "=" * 70)
        print(name, type(obj))

        if isinstance(obj, pd.DataFrame):
            print(obj.shape)
            print(obj.columns.tolist())


crossfit_threshold_df <class 'pandas.core.frame.DataFrame'>
(15, 9)
['fold', 'task', 'transition_type', 'target_family', 'selected_threshold', 'test_accepted', 'test_rescues', 'test_breaks', 'test_net']

accepted_effect_df <class 'pandas.core.frame.DataFrame'>
(74, 6)
['specialist', 'true_family', 'semantic_prediction', 'specialist_prediction', 'workflow_prob', 'effect']

override_df <class 'pandas.core.frame.DataFrame'>
(1489, 9)
['true_family', 'semantic_prediction', 'transition_type', 'tool_use_score', 'grounding_score', 'constraint_score', 'override_target', 'override_score', 'would_change']

train_targets <class 'pandas.core.frame.DataFrame'>
(1489, 14)
['dataset', 'group_id', 'canonical_group', 'split', 'trajectory_index', 'message_index', 'event_position', 'history_event_count', 'has_history', 'event_role', 'primary_tool', 'content', 'family_label', 'failure_family']


In [ ]:
# ============================================================
# Reconstruct deterministic outer-fold assignment
# only if original fold vector is gone
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

# true class labels
y_for_folds = true_family_arr

# use the same grouping variable used throughout notebook 17
groups_for_folds = groups_nested

cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

outer_fold_arr = np.full(
    n,
    -1,
    dtype=int
)

for fold_id, (_, va) in enumerate(
    cv_outer.split(
        np.zeros(n),
        y_for_folds,
        groups=groups_for_folds,
    ),
    start=1,
):
    outer_fold_arr[va] = fold_id

assert (outer_fold_arr > 0).all()

print(
    "Fold counts:",
    pd.Series(outer_fold_arr)
    .value_counts()
    .sort_index()
    .to_dict()
)

Fold counts: {1: 298, 2: 298, 3: 298, 4: 297, 5: 298}


In [ ]:
# ============================================================
# Reconstruct deterministic outer-fold assignment
# only if original fold vector is gone
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

# true class labels
y_for_folds = true_family_arr

# use the same grouping variable used throughout notebook 17
groups_for_folds = groups_nested

cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

outer_fold_arr = np.full(
    n,
    -1,
    dtype=int
)

for fold_id, (_, va) in enumerate(
    cv_outer.split(
        np.zeros(n),
        y_for_folds,
        groups=groups_for_folds,
    ),
    start=1,
):
    outer_fold_arr[va] = fold_id

assert (outer_fold_arr > 0).all()

print(
    "Fold counts:",
    pd.Series(outer_fold_arr)
    .value_counts()
    .sort_index()
    .to_dict()
)

Fold counts: {1: 298, 2: 298, 3: 298, 4: 297, 5: 298}


In [ ]:
# ============================================================
# 121A. Inspect surviving fold objects directly
# ============================================================

fold_objects = [
    "fold_assignment",
    "outer_fold",
    "train_fold",
    "test_fold",
    "outer_splits",
    "candidate_fold_names",
    "fold_variable_name",
]

for name in fold_objects:

    if name not in globals():
        continue

    obj = globals()[name]

    print("\n" + "=" * 80)
    print(name)
    print("type:", type(obj))

    try:
        print("shape:", np.shape(obj))
    except Exception:
        pass

    try:
        print("len:", len(obj))
    except Exception:
        pass

    try:
        arr = np.asarray(obj)

        print("ndim:", arr.ndim)
        print("dtype:", arr.dtype)

        if arr.ndim == 1:
            print(
                "unique:",
                pd.unique(arr)[:20]
            )

            if len(arr) == 1489:
                print(
                    "counts:",
                    pd.Series(arr)
                    .value_counts(dropna=False)
                    .sort_index()
                    .to_dict()
                )

    except Exception as e:
        print("array inspection failed:", e)


fold_assignment
type: <class 'NoneType'>
shape: ()
ndim: 0
dtype: object

outer_fold
type: <class 'int'>
shape: ()
ndim: 0
dtype: int64

train_fold
type: <class 'bool'>
shape: ()
ndim: 0
dtype: bool

test_fold
type: <class 'bool'>
shape: ()
ndim: 0
dtype: bool

outer_splits
type: <class 'list'>
len: 5
array inspection failed: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (5, 2) + inhomogeneous part.

candidate_fold_names
type: <class 'list'>
shape: (6,)
len: 6
ndim: 1
dtype: <U14
unique: ['outer_fold' 'outer_folds' 'fold_id' 'fold_ids' 'outer_fold_id'
 'outer_fold_ids']

fold_variable_name
type: <class 'NoneType'>
shape: ()
ndim: 0
dtype: object


In [ ]:
# ============================================================
# 121E. Recover exact fold assignment from outer_splits
# ============================================================

assert isinstance(outer_splits, list)
assert len(outer_splits) == 5

outer_fold_arr = np.full(
    len(train_targets),
    -1,
    dtype=int,
)

split_summary = []

for fold_id, split_pair in enumerate(
    outer_splits,
    start=1,
):

    train_idx_fold, test_idx_fold = split_pair

    train_idx_fold = np.asarray(
        train_idx_fold,
        dtype=int,
    )

    test_idx_fold = np.asarray(
        test_idx_fold,
        dtype=int,
    )

    outer_fold_arr[
        test_idx_fold
    ] = fold_id

    split_summary.append({
        "fold": fold_id,
        "train_n": len(train_idx_fold),
        "test_n": len(test_idx_fold),
        "test_min": int(test_idx_fold.min()),
        "test_max": int(test_idx_fold.max()),
    })


assert (
    outer_fold_arr >= 1
).all(), "Some rows were never assigned to a fold."


print(
    "Recovered outer_fold_arr:",
    outer_fold_arr.shape
)

print(
    "Fold counts:",
    pd.Series(
        outer_fold_arr
    )
    .value_counts()
    .sort_index()
    .to_dict()
)

display(
    pd.DataFrame(
        split_summary
    )
)

Recovered outer_fold_arr: (1489,)
Fold counts: {1: 298, 2: 298, 3: 298, 4: 297, 5: 298}


,fold,train_n,test_n,test_min,test_max
0,1,1191,298,9,1485
1,2,1191,298,0,1483
2,3,1191,298,4,1481
3,4,1192,297,15,1486
4,5,1191,298,1,1488


In [ ]:
# ============================================================
# 122. OOF TOOL->TOOL grounding-vs-workflow specialist
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

grounding_workflow_prob = np.full(
    len(train_targets),
    np.nan,
    dtype=float,
)

gw_fold_rows = []


for fold_id in sorted(
    np.unique(
        outer_fold_arr
    )
):

    # fold membership for boundary rows
    boundary_folds = (
        outer_fold_arr[
            boundary_idx
        ]
    )

    tr = np.where(
        boundary_folds
        != fold_id
    )[0]

    va = np.where(
        boundary_folds
        == fold_id
    )[0]


    print(
        f"Fold {fold_id}:",
        "train =", len(tr),
        "| test =", len(va),
        "| train positives =",
        int(
            y_boundary[tr].sum()
        ),
        "| test positives =",
        int(
            y_boundary[va].sum()
        ),
    )


    model_gw = make_specialist_model(
        C=0.03
    )

    model_gw.fit(
        X_boundary[tr],
        y_boundary[tr],
    )

    p_va = (
        model_gw
        .predict_proba(
            X_boundary[va]
        )[:, 1]
    )


    global_va = (
        boundary_idx[va]
    )

    grounding_workflow_prob[
        global_va
    ] = p_va


    gw_fold_rows.append({
        "fold":
            fold_id,

        "train_support":
            len(tr),

        "test_support":
            len(va),

        "train_grounding":
            int(
                y_boundary[tr].sum()
            ),

        "test_grounding":
            int(
                y_boundary[va].sum()
            ),

        "test_workflow":
            int(
                len(va)
                - y_boundary[va].sum()
            ),
    })


display(
    pd.DataFrame(
        gw_fold_rows
    )
)


assert not np.isnan(
    grounding_workflow_prob[
        boundary_idx
    ]
).any()


p_gw_oof = (
    grounding_workflow_prob[
        boundary_idx
    ]
)

gw_pr_auc = (
    average_precision_score(
        y_boundary,
        p_gw_oof,
    )
)

gw_roc_auc = (
    roc_auc_score(
        y_boundary,
        p_gw_oof,
    )
)


print(
    "\nOOF PR-AUC:",
    gw_pr_auc
)

print(
    "OOF ROC-AUC:",
    gw_roc_auc
)

print(
    "Prevalence:",
    y_boundary.mean()
)

print(
    "PR lift:",
    gw_pr_auc
    / y_boundary.mean()
)

Fold 1: train = 330 | test = 78 | train positives = 57 | test positives = 10
Fold 2: train = 326 | test = 82 | train positives = 51 | test positives = 16
Fold 3: train = 316 | test = 92 | train positives = 58 | test positives = 9
Fold 4: train = 339 | test = 69 | train positives = 58 | test positives = 9
Fold 5: train = 321 | test = 87 | train positives = 44 | test positives = 23


,fold,train_support,test_support,train_grounding,test_grounding,test_workflow
0,1,330,78,57,10,68
1,2,326,82,51,16,66
2,3,316,92,58,9,83
3,4,339,69,58,9,60
4,5,321,87,44,23,64



OOF PR-AUC: 0.6871216179922289
OOF ROC-AUC: 0.8987613253381188
Prevalence: 0.1642156862745098
PR lift: 4.184262987176559


In [ ]:
# ============================================================
# 123. Direct grounding↔workflow intervention utility
# ============================================================

gw_candidate_mask = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    &
    np.isin(
        semantic_pred_arr,
        [
            "grounding_state_error",
            "workflow_error",
        ]
    )
    &
    (~np.isnan(
        grounding_workflow_prob
    ))
)


gw_pred = (
    semantic_pred_arr.copy()
)


# p >= .5 means grounding
gw_pred[
    gw_candidate_mask
    &
    (
        grounding_workflow_prob
        >= 0.5
    )
] = "grounding_state_error"


# p < .5 means workflow
gw_pred[
    gw_candidate_mask
    &
    (
        grounding_workflow_prob
        < 0.5
    )
] = "workflow_error"


gw_changed = (
    gw_pred
    != semantic_pred_arr
)

gw_correct = (
    gw_pred
    == true_family_arr
)


gw_rescue = (
    gw_changed
    &
    (~base_correct)
    &
    gw_correct
)

gw_break = (
    gw_changed
    &
    base_correct
    &
    (~gw_correct)
)

gw_wrong = (
    gw_changed
    &
    (~base_correct)
    &
    (~gw_correct)
)


print(
    "Candidates:",
    int(
        gw_candidate_mask.sum()
    )
)

print(
    "Changed:",
    int(
        gw_changed.sum()
    )
)

print(
    "Rescues:",
    int(
        gw_rescue.sum()
    )
)

print(
    "Breaks:",
    int(
        gw_break.sum()
    )
)

print(
    "Wrong-to-wrong:",
    int(
        gw_wrong.sum()
    )
)

print(
    "Net:",
    int(
        gw_rescue.sum()
        -
        gw_break.sum()
    )
)


display(
    pd.DataFrame([
        {
            "model":
                "semantic_no_t1",

            **multiclass_metrics(
                true_family_arr,
                semantic_pred_arr,
            ),
        },

        {
            "model":
                "gw_pairwise_only",

            **multiclass_metrics(
                true_family_arr,
                gw_pred,
            ),
        },
    ]).round(4)
)

Candidates: 355
Changed: 34
Rescues: 18
Breaks: 16
Wrong-to-wrong: 0
Net: 2


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,gw_pairwise_only,0.5165,0.4413,0.4607,0.5008


For the new TOOL_CALL→TOOL_CALL grounding_state_error ↔ workflow_error expert, the classifier itself is strong: ROC-AUC 0.8988, PR-AUC 0.6871, and PR lift 4.18× over prevalence. But when you actually let it override the semantic model at the natural 0.5 boundary, it changes only 34 predictions and yields 18 rescues vs 16 breaks, net +2. Accuracy moves only from 0.5151 → 0.5165, while balanced accuracy and macro-F1 improve modestly.

So this expert is learnable, but most of its predictive signal is not aligned with the specific mistakes that the semantic model needs corrected. That is exactly why the earlier opportunity_score was necessary.

I would not add it blindly as expert #3 yet. The next question should be narrower:

Can we identify a safe subset of this grounding↔workflow expert where its intervention precision is meaningfully above 50%?

For this expert, thresholding may actually matter because the AUC is high but the decision boundary is producing nearly equal rescues and breaks. Let's inspect rescue/break probability distributions and then test a small cross-fitted threshold grid.

In [ ]:
# ============================================================
# 124. Grounding↔workflow intervention diagnostics
# ============================================================

gw_diag_idx = np.where(gw_changed)[0]

gw_diag_df = pd.DataFrame({
    "idx": gw_diag_idx,
    "true_family": true_family_arr[gw_diag_idx],
    "semantic_prediction": semantic_pred_arr[gw_diag_idx],
    "gw_prediction": gw_pred[gw_diag_idx],
    "gw_prob_grounding": grounding_workflow_prob[gw_diag_idx],
})

gw_diag_df["effect"] = np.select(
    [
        (~base_correct[gw_diag_idx]) & gw_correct[gw_diag_idx],
        base_correct[gw_diag_idx] & (~gw_correct[gw_diag_idx]),
    ],
    [
        "rescue",
        "break",
    ],
    default="wrong_to_wrong",
)

display(
    gw_diag_df.groupby("effect")[
        "gw_prob_grounding"
    ].agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ]).round(4)
)

display(
    gw_diag_df.groupby(
        [
            "semantic_prediction",
            "gw_prediction",
            "effect",
        ]
    ).size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

,count,mean,median,std,min,max
effect,,,,,,
break,16,0.4629,0.5384,0.3134,0.0226,0.9050
rescue,18,0.7921,0.8053,0.2127,0.1240,0.9984


,semantic_prediction,gw_prediction,effect,count
3,workflow_error,grounding_state_error,rescue,17
2,workflow_error,grounding_state_error,break,9
0,grounding_state_error,workflow_error,break,7
1,grounding_state_error,workflow_error,rescue,1


In [ ]:
# ============================================================
# 125. Direction-specific utility
# ============================================================

gw_diag_df["direction"] = (
    gw_diag_df["semantic_prediction"]
    + " -> "
    + gw_diag_df["gw_prediction"]
)

direction_summary = (
    gw_diag_df
    .groupby("direction")
    .agg(
        support=("effect", "size"),
        rescues=("effect", lambda x: int((x == "rescue").sum())),
        breaks=("effect", lambda x: int((x == "break").sum())),
        wrong_to_wrong=("effect", lambda x: int((x == "wrong_to_wrong").sum())),
    )
    .reset_index()
)

direction_summary["net"] = (
    direction_summary["rescues"]
    - direction_summary["breaks"]
)

direction_summary["precision"] = (
    direction_summary["rescues"]
    / direction_summary["support"]
)

display(
    direction_summary.sort_values(
        "net",
        ascending=False
    ).round(4)
)

,direction,support,rescues,breaks,wrong_to_wrong,net,precision
1,workflow_error -> grounding_state_error,26,17,9,0,8,0.6538
0,grounding_state_error -> workflow_error,8,1,7,0,-6,0.1250


In [ ]:
# ============================================================
# 126. Confidence-distance sweep by intervention direction
# ============================================================

gw_candidate_idx = np.where(
    gw_candidate_mask
)[0]

rows = []

for min_distance in [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]:

    pred = semantic_pred_arr.copy()

    p = grounding_workflow_prob

    # Only intervene when pairwise model is sufficiently
    # far from its 0.5 decision boundary.
    confident = (
        np.abs(p - 0.5)
        >= min_distance
    )

    use = (
        gw_candidate_mask
        & confident
    )

    pred[
        use & (p >= 0.5)
    ] = "grounding_state_error"

    pred[
        use & (p < 0.5)
    ] = "workflow_error"

    changed = (
        pred != semantic_pred_arr
    )

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            changed
            & (~base_correct)
            & correct
        ).sum()
    )

    breaks = int(
        (
            changed
            & base_correct
            & (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            & (~base_correct)
            & (~correct)
        ).sum()
    )

    rows.append({
        "min_distance":
            min_distance,

        "changed":
            int(changed.sum()),

        "rescues":
            rescues,

        "breaks":
            breaks,

        "wrong_to_wrong":
            wrong,

        "net":
            rescues - breaks,

        "precision":
            rescues / max(
                int(changed.sum()),
                1,
            ),

        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


gw_distance_sweep_df = pd.DataFrame(rows)

display(
    gw_distance_sweep_df
    .sort_values(
        ["net", "macro_f1"],
        ascending=False
    )
    .round(4)
)

,min_distance,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,0.20,26,16,10,0,6,0.6154,0.5191,0.4425,0.4621,0.5031
3,0.15,29,17,12,0,5,0.5862,0.5185,0.4422,0.4618,0.5025
2,0.10,30,17,13,0,4,0.5667,0.5178,0.4414,0.4609,0.5017
1,0.05,33,18,15,0,3,0.5455,0.5171,0.4416,0.4610,0.5014
5,0.25,19,11,8,0,3,0.5789,0.5171,0.4390,0.4586,0.5000
0,0.00,34,18,16,0,2,0.5294,0.5165,0.4413,0.4607,0.5008
6,0.30,18,10,8,0,2,0.5556,0.5165,0.4382,0.4577,0.4992


In [ ]:
# ============================================================
# 127. One-way workflow -> grounding intervention sweep
# ============================================================

rows = []

p = grounding_workflow_prob

for grounding_threshold in [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]:

    pred = semantic_pred_arr.copy()

    # ONLY allow:
    # semantic workflow -> specialist grounding
    use = (
        (transition_arr == "TOOL_CALL->TOOL_CALL")
        &
        (semantic_pred_arr == "workflow_error")
        &
        (~np.isnan(p))
        &
        (p >= grounding_threshold)
    )

    pred[use] = "grounding_state_error"

    changed = (
        pred != semantic_pred_arr
    )

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            changed
            & (~base_correct)
            & correct
        ).sum()
    )

    breaks = int(
        (
            changed
            & base_correct
            & (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            & (~base_correct)
            & (~correct)
        ).sum()
    )

    changed_n = int(changed.sum())

    rows.append({
        "threshold": grounding_threshold,
        "changed": changed_n,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / changed_n
            if changed_n
            else np.nan
        ),
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


gw_oneway_sweep_df = pd.DataFrame(rows)

display(
    gw_oneway_sweep_df
    .sort_values(
        ["net", "precision", "macro_f1"],
        ascending=False,
    )
    .round(4)
)

,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,0.70,20,15,5,0,10,0.7500,0.5218,0.4463,0.4660,0.5066
2,0.60,22,16,6,0,10,0.7273,0.5218,0.4468,0.4665,0.5068
3,0.65,22,16,6,0,10,0.7273,0.5218,0.4468,0.4665,0.5068
1,0.55,25,17,8,0,9,0.6800,0.5212,0.4470,0.4665,0.5065
0,0.50,26,17,9,0,8,0.6538,0.5205,0.4467,0.4661,0.5059
7,0.85,9,8,1,0,7,0.8889,0.5198,0.4418,0.4615,0.5031
5,0.75,13,10,3,0,7,0.7692,0.5198,0.4428,0.4625,0.5036
8,0.90,8,7,1,0,6,0.8750,0.5191,0.4409,0.4607,0.5023
6,0.80,12,9,3,0,6,0.7500,0.5191,0.4420,0.4617,0.5028


In [ ]:
# ============================================================
# 128. Incremental value on top of existing hard router
# ============================================================

# Change this if your saved +36 router has another name.
existing_router_pred = hierarchical_pred.copy()

p = grounding_workflow_prob

gw_use = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    &
    (existing_router_pred == "workflow_error")
    &
    (~np.isnan(p))
    &
    (p >= 0.50)
)

combined_pred = existing_router_pred.copy()

combined_pred[
    gw_use
] = "grounding_state_error"


existing_correct = (
    existing_router_pred
    == true_family_arr
)

combined_correct = (
    combined_pred
    == true_family_arr
)

incremental_changed = (
    combined_pred
    != existing_router_pred
)

incremental_rescue = (
    incremental_changed
    &
    (~existing_correct)
    &
    combined_correct
)

incremental_break = (
    incremental_changed
    &
    existing_correct
    &
    (~combined_correct)
)

incremental_wrong = (
    incremental_changed
    &
    (~existing_correct)
    &
    (~combined_correct)
)


print(
    "Incremental changed:",
    int(incremental_changed.sum())
)

print(
    "Incremental rescues:",
    int(incremental_rescue.sum())
)

print(
    "Incremental breaks:",
    int(incremental_break.sum())
)

print(
    "Incremental wrong-to-wrong:",
    int(incremental_wrong.sum())
)

print(
    "Incremental net:",
    int(
        incremental_rescue.sum()
        - incremental_break.sum()
    )
)


display(
    pd.DataFrame([
        {
            "model": "existing_hard_router",
            **multiclass_metrics(
                true_family_arr,
                existing_router_pred,
            ),
        },
        {
            "model": "router_plus_gw_oneway",
            **multiclass_metrics(
                true_family_arr,
                combined_pred,
            ),
        },
    ]).round(4)
)

Incremental changed: 28
Incremental rescues: 17
Incremental breaks: 11
Incremental wrong-to-wrong: 0
Incremental net: 6


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,existing_hard_router,0.5393,0.4745,0.4918,0.5292
1,router_plus_gw_oneway,0.5433,0.4851,0.5024,0.5370


Yes — this is a meaningful improvement. The grounding specialist is adding new signal on top of the existing router, not merely rediscovering its corrections.

The threshold sweep also gives a much cleaner picture. The sweet spot is around 0.60–0.70: both give net +10 in the standalone one-way experiment, versus +8 at 0.50. At 0.70 you make only 20 interventions with 15 rescues and 5 breaks, giving 75% intervention precision. Above 0.70, precision rises but you start throwing away too many useful rescues.

More importantly, the combined result moves every metric in the right direction:

Model	Accuracy	Balanced acc.	Macro-F1	Weighted-F1
Semantic	.5151	.4355	.4548	.4969
Existing hard router	.5393	.4745	.4918	.5292
+ grounding one-way	.5433	.4851	.5024	.5370

The jump in macro-F1 from .4918 → .5024 is particularly useful because this isn't merely exploiting the dominant class. Balanced accuracy similarly moves .4745 → .4851.

There is one issue with the incremental experiment, though: it appears you used p >= 0.50, while the sweep now says 0.60–0.70 is better. So I would rerun the incremental experiment across thresholds rather than choosing 0.50.

In [ ]:
# ============================================================
# Incremental grounding threshold sweep on existing router
# ============================================================

rows = []

p = grounding_workflow_prob
existing_router_pred = hierarchical_pred.copy()

existing_correct = (
    existing_router_pred == true_family_arr
)

for threshold in [
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90,
]:

    pred = existing_router_pred.copy()

    # Important:
    # Gate on the EXISTING ROUTER prediction, not semantic prediction.
    use = (
        (transition_arr == "TOOL_CALL->TOOL_CALL")
        &
        (existing_router_pred == "workflow_error")
        &
        (~np.isnan(p))
        &
        (p >= threshold)
    )

    pred[use] = "grounding_state_error"

    changed = pred != existing_router_pred
    correct = pred == true_family_arr

    rescues = int(
        (
            changed
            & (~existing_correct)
            & correct
        ).sum()
    )

    breaks = int(
        (
            changed
            & existing_correct
            & (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            & (~existing_correct)
            & (~correct)
        ).sum()
    )

    changed_n = int(changed.sum())

    rows.append({
        "threshold": threshold,
        "changed": changed_n,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / changed_n
            if changed_n
            else np.nan
        ),
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


incremental_gw_sweep_df = pd.DataFrame(rows)

display(
    incremental_gw_sweep_df
    .sort_values(
        ["net", "macro_f1", "precision"],
        ascending=False,
    )
    .round(4)
)

,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,0.70,19,15,4,0,11,0.7895,0.5467,0.4855,0.5034,0.5395
2,0.60,22,16,6,0,10,0.7273,0.5460,0.4858,0.5035,0.5391
3,0.65,22,16,6,0,10,0.7273,0.5460,0.4858,0.5035,0.5391
1,0.55,25,17,8,0,9,0.6800,0.5453,0.4860,0.5035,0.5387
5,0.75,13,10,3,0,7,0.7692,0.5440,0.4817,0.4995,0.5359
7,0.85,9,8,1,0,7,0.8889,0.5440,0.4807,0.4986,0.5354
0,0.50,28,17,11,0,6,0.6071,0.5433,0.4851,0.5024,0.5370
6,0.80,12,9,3,0,6,0.7500,0.5433,0.4809,0.4987,0.5351
8,0.90,8,7,1,0,6,0.8750,0.5433,0.4799,0.4977,0.5346


This confirms the hypothesis. **0.70 is the strongest operating point for the grounding override**, with 0.60–0.65 essentially tied if macro-F1/balanced accuracy are the optimization target.

At **0.70**, you get 19 interventions, **15 rescues / 4 breaks**, net **+11**, and **78.95% intervention precision**. Compared with the existing hard router (`.5393 accuracy / .4745 balanced accuracy / .4918 macro-F1 / .5292 weighted-F1`), that produces:

| Metric            | Hard router | + GW @ 0.70 |          Δ |
| ----------------- | ----------: | ----------: | ---------: |
| Accuracy          |       .5393 |   **.5467** | **+.0074** |
| Balanced accuracy |       .4745 |   **.4855** | **+.0110** |
| Macro-F1          |       .4918 |   **.5034** | **+.0116** |
| Weighted-F1       |       .5292 |   **.5395** | **+.0103** |

There is an interesting distinction in the plateau. **0.55–0.65 gives marginally higher balanced accuracy/macro-F1** (`.4860/.5035` at .55 and `.4858/.5035` at .60/.65), whereas **0.70 maximizes accuracy, weighted-F1, net rescues, and intervention precision**. The macro-F1 difference is only `.0001`, so I would not treat it as evidence for the lower threshold.

The more important result is the shape:

```text
threshold    rescues / breaks    net    precision
0.50           17 / 11           +6      .607
0.55           17 /  8           +9      .680
0.60           16 /  6          +10      .727
0.65           16 /  6          +10      .727
0.70           15 /  4          +11      .789   ← strongest tradeoff
0.75           10 /  3           +7      .769
0.80            9 /  3           +6      .750
0.85            8 /  1           +7      .889
0.90            7 /  1           +6      .875
```

So this isn't a fragile single-threshold spike. There is a sensible **0.55–0.70 operating region**, with performance deteriorating once you become too conservative above .70. That's much more encouraging than finding one lucky threshold.

### I would now freeze this experiment

I wouldn't do another global threshold search. We've extracted the useful exploratory conclusion:

> For `TOOL_CALL→TOOL_CALL` examples currently classified as `workflow_error`, the grounding-vs-workflow specialist contains useful incremental information, particularly at high grounding confidence.

And importantly, the directionality is justified by the earlier result:

* `workflow → grounding`: **17 rescue / 9 break**
* `grounding → workflow`: **1 rescue / 7 break**

So the expert should be **one-way**, not symmetric.

The next experiment should answer the harder question: *does the threshold survive honest threshold selection?*

Use your recovered `outer_splits` directly, which avoids all of the fold-ID problems from earlier. For every outer fold, train the GW specialist on the outer-training data, obtain **inner OOF probabilities within that training set**, select the threshold from a prespecified grid such as `{.50,.55,.60,.65,.70,.75,.80,.85,.90}`, then fit on all outer-training GW examples and apply the selected threshold to the untouched outer test fold **on top of the hard router**.

I'd select the inner threshold by **net rescues (`rescues - breaks`)**, with deterministic tie-breaks of higher intervention precision, then higher threshold. That matches what the router is actually supposed to accomplish and avoids optimizing tiny macro-F1 fluctuations over the whole multiclass dataset.

The key output I'd want next is something like:

```text
fold  selected_t  eligible  changed  rescue  break  net
1       .70          ...       ...      ...     ...   ...
2       .60          ...       ...      ...     ...   ...
3       .65          ...       ...      ...     ...   ...
4       .70          ...       ...      ...     ...   ...
5       .60          ...       ...      ...     ...   ...
---------------------------------------------------------
OOF                               R       B    R-B
```

plus the four final multiclass metrics against the **same hard-router OOF baseline**.

One additional diagnostic is important: report the **selected threshold per fold**. If they cluster around `.60–.70`, that's excellent evidence that the operating region is real. If they jump wildly between `.50` and `.90`, the +11 global result is much less trustworthy.

At this point, I would **not add another pairwise expert yet**. This grounding expert has now earned the nested-validation test; establishing that the incremental gain survives leakage-free threshold selection is more valuable than continuing to expand the router.


In [ ]:
# ============================================================
# 129. Nested threshold selection for
#      TOOL_CALL->TOOL_CALL
#      workflow_error -> grounding_state_error
#
# Outer test folds stay untouched.
# Threshold is selected using INNER OOF predictions
# from the outer-training portion only.
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Frozen threshold grid
# ------------------------------------------------------------

GW_THRESHOLDS = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

# Existing hard pairwise router.
# Change this one line only if your saved router
# has a different variable name.
base_router_pred = hierarchical_pred.copy()

assert len(base_router_pred) == len(train_targets)
assert len(outer_splits) == 5

base_router_correct = (
    base_router_pred
    == true_family_arr
)

# Final nested prediction:
nested_gw_pred = base_router_pred.copy()

# Store selected threshold for each outer fold
nested_gw_threshold = np.full(
    len(train_targets),
    np.nan,
)

nested_gw_accept = np.zeros(
    len(train_targets),
    dtype=bool,
)

nested_gw_prob = np.full(
    len(train_targets),
    np.nan,
)

fold_rows = []
threshold_search_rows = []


# ============================================================
# Outer CV
# ============================================================

for outer_fold_id, (
    outer_train_idx,
    outer_test_idx,
) in enumerate(
    outer_splits,
    start=1,
):

    outer_train_idx = np.asarray(
        outer_train_idx,
        dtype=int,
    )

    outer_test_idx = np.asarray(
        outer_test_idx,
        dtype=int,
    )

    # --------------------------------------------------------
    # Grounding/workflow TRUE-family boundary
    # inside outer training set
    # --------------------------------------------------------

    train_boundary_global = np.intersect1d(
        outer_train_idx,
        boundary_idx,
    )

    test_boundary_global = np.intersect1d(
        outer_test_idx,
        boundary_idx,
    )

    # Convert global row IDs -> positions inside boundary_idx
    boundary_position = {
        global_idx: local_idx
        for local_idx, global_idx
        in enumerate(boundary_idx)
    }

    train_boundary_local = np.array([
        boundary_position[i]
        for i in train_boundary_global
    ])

    test_boundary_local = np.array([
        boundary_position[i]
        for i in test_boundary_global
    ])

    X_outer_train = X_boundary[
        train_boundary_local
    ]

    y_outer_train = y_boundary[
        train_boundary_local
    ]

    X_outer_test = X_boundary[
        test_boundary_local
    ]

    # --------------------------------------------------------
    # Inner groups
    # --------------------------------------------------------

    inner_groups = np.asarray(
        groups_nested
    )[train_boundary_global]

    # Number of inner folds must be valid for minority class
    class_counts = np.bincount(
        y_outer_train,
        minlength=2,
    )

    n_inner = min(
        4,
        int(class_counts.min()),
    )

    if n_inner < 2:
        print(
            f"Outer fold {outer_fold_id}: "
            "not enough minority examples."
        )
        continue

    # --------------------------------------------------------
    # INNER OOF probabilities
    # --------------------------------------------------------

    inner_prob = np.full(
        len(train_boundary_global),
        np.nan,
    )

    inner_cv = StratifiedGroupKFold(
        n_splits=n_inner,
        shuffle=True,
        random_state=42 + outer_fold_id,
    )

    for inner_tr, inner_va in inner_cv.split(
        X_outer_train,
        y_outer_train,
        groups=inner_groups,
    ):

        inner_model = make_specialist_model(
            C=0.03
        )

        inner_model.fit(
            X_outer_train[inner_tr],
            y_outer_train[inner_tr],
        )

        inner_prob[
            inner_va
        ] = inner_model.predict_proba(
            X_outer_train[inner_va]
        )[:, 1]


    assert not np.isnan(
        inner_prob
    ).any()


    # ========================================================
    # Threshold selection on OUTER TRAIN only
    #
    # IMPORTANT:
    # intervention is one-way:
    # existing router workflow -> grounding
    # ========================================================

    best_threshold = None
    best_key = None
    best_stats = None

    for threshold in GW_THRESHOLDS:

        # We need specialist probabilities only for rows
        # whose TRUE family is grounding/workflow.
        #
        # To evaluate intervention utility, map those
        # probabilities back to outer-training global rows.

        train_prob_global = np.full(
            len(train_targets),
            np.nan,
        )

        train_prob_global[
            train_boundary_global
        ] = inner_prob

        use_train = (
            np.isin(
                np.arange(len(train_targets)),
                outer_train_idx,
            )
            &
            (
                transition_arr
                == "TOOL_CALL->TOOL_CALL"
            )
            &
            (
                base_router_pred
                == "workflow_error"
            )
            &
            (~np.isnan(
                train_prob_global
            ))
            &
            (
                train_prob_global
                >= threshold
            )
        )

        candidate_pred = (
            base_router_pred.copy()
        )

        candidate_pred[
            use_train
        ] = "grounding_state_error"

        changed = (
            candidate_pred
            != base_router_pred
        ) & np.isin(
            np.arange(len(train_targets)),
            outer_train_idx,
        )

        candidate_correct = (
            candidate_pred
            == true_family_arr
        )

        rescues = int(
            (
                changed
                & (~base_router_correct)
                & candidate_correct
            ).sum()
        )

        breaks = int(
            (
                changed
                & base_router_correct
                & (~candidate_correct)
            ).sum()
        )

        wrong = int(
            (
                changed
                & (~base_router_correct)
                & (~candidate_correct)
            ).sum()
        )

        accepted = int(
            changed.sum()
        )

        net = rescues - breaks

        precision = (
            rescues / accepted
            if accepted > 0
            else 0.0
        )

        threshold_search_rows.append({
            "outer_fold":
                outer_fold_id,

            "threshold":
                threshold,

            "train_accepted":
                accepted,

            "train_rescues":
                rescues,

            "train_breaks":
                breaks,

            "train_wrong":
                wrong,

            "train_net":
                net,

            "train_precision":
                precision,
        })

        # ----------------------------------------------------
        # Selection criterion:
        #
        # 1. maximize rescue-break net
        # 2. maximize precision
        # 3. prefer higher threshold
        #
        # ----------------------------------------------------

        selection_key = (
            net,
            precision,
            threshold,
        )

        if (
            best_key is None
            or selection_key > best_key
        ):
            best_key = selection_key
            best_threshold = threshold
            best_stats = {
                "accepted": accepted,
                "rescues": rescues,
                "breaks": breaks,
                "wrong": wrong,
                "net": net,
                "precision": precision,
            }


    # ========================================================
    # Fit specialist on ALL outer-training boundary examples
    # ========================================================

    final_model = make_specialist_model(
        C=0.03
    )

    final_model.fit(
        X_outer_train,
        y_outer_train,
    )

    test_prob = final_model.predict_proba(
        X_outer_test
    )[:, 1]

    nested_gw_prob[
        test_boundary_global
    ] = test_prob

    nested_gw_threshold[
        outer_test_idx
    ] = best_threshold


    # ========================================================
    # Apply selected threshold to UNTOUCHED outer test fold
    # ========================================================

    test_prob_global = np.full(
        len(train_targets),
        np.nan,
    )

    test_prob_global[
        test_boundary_global
    ] = test_prob

    test_membership = np.zeros(
        len(train_targets),
        dtype=bool,
    )

    test_membership[
        outer_test_idx
    ] = True

    use_test = (
        test_membership
        &
        (
            transition_arr
            == "TOOL_CALL->TOOL_CALL"
        )
        &
        (
            base_router_pred
            == "workflow_error"
        )
        &
        (~np.isnan(
            test_prob_global
        ))
        &
        (
            test_prob_global
            >= best_threshold
        )
    )

    nested_gw_accept[
        use_test
    ] = True

    nested_gw_pred[
        use_test
    ] = "grounding_state_error"


    # --------------------------------------------------------
    # Outer-test utility
    # --------------------------------------------------------

    test_changed = (
        nested_gw_pred
        != base_router_pred
    ) & test_membership

    nested_correct_now = (
        nested_gw_pred
        == true_family_arr
    )

    test_rescues = int(
        (
            test_changed
            & (~base_router_correct)
            & nested_correct_now
        ).sum()
    )

    test_breaks = int(
        (
            test_changed
            & base_router_correct
            & (~nested_correct_now)
        ).sum()
    )

    test_wrong = int(
        (
            test_changed
            & (~base_router_correct)
            & (~nested_correct_now)
        ).sum()
    )

    test_accepted = int(
        test_changed.sum()
    )

    fold_rows.append({
        "fold":
            outer_fold_id,

        "selected_threshold":
            best_threshold,

        "train_accepted":
            best_stats["accepted"],

        "train_rescues":
            best_stats["rescues"],

        "train_breaks":
            best_stats["breaks"],

        "train_net":
            best_stats["net"],

        "train_precision":
            best_stats["precision"],

        "test_accepted":
            test_accepted,

        "test_rescues":
            test_rescues,

        "test_breaks":
            test_breaks,

        "test_wrong_to_wrong":
            test_wrong,

        "test_net":
            test_rescues
            - test_breaks,

        "test_precision":
            (
                test_rescues
                / test_accepted
                if test_accepted > 0
                else np.nan
            ),
    })


nested_gw_fold_df = pd.DataFrame(
    fold_rows
)

nested_gw_threshold_search_df = pd.DataFrame(
    threshold_search_rows
)

display(
    nested_gw_fold_df.round(4)
)

,fold,selected_threshold,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net,test_precision
0,1,0.65,16,13,3,10,0.8125,6,3,3,0,0,0.5000
1,2,0.55,22,16,6,10,0.7273,3,2,1,0,1,0.6667
2,3,0.80,14,13,1,12,0.9286,0,0,0,0,0,NaN
3,4,0.75,17,12,5,7,0.7059,3,3,0,0,3,1.0000
4,5,0.90,6,6,0,6,1.0000,1,1,0,0,1,1.0000


In [ ]:
# ============================================================
# 130. Final nested GW evaluation
# ============================================================

nested_changed = (
    nested_gw_pred
    != base_router_pred
)

nested_correct = (
    nested_gw_pred
    == true_family_arr
)

nested_rescues = int(
    (
        nested_changed
        & (~base_router_correct)
        & nested_correct
    ).sum()
)

nested_breaks = int(
    (
        nested_changed
        & base_router_correct
        & (~nested_correct)
    ).sum()
)

nested_wrong = int(
    (
        nested_changed
        & (~base_router_correct)
        & (~nested_correct)
    ).sum()
)

print(
    "Nested accepted:",
    int(nested_changed.sum())
)

print(
    "Nested rescues:",
    nested_rescues
)

print(
    "Nested breaks:",
    nested_breaks
)

print(
    "Nested wrong-to-wrong:",
    nested_wrong
)

print(
    "Nested net:",
    nested_rescues
    - nested_breaks
)

print(
    "Nested intervention precision:",
    round(
        nested_rescues
        / max(
            int(nested_changed.sum()),
            1,
        ),
        4,
    )
)


nested_metrics_df = pd.DataFrame([
    {
        "model":
            "semantic_no_t1",

        **multiclass_metrics(
            true_family_arr,
            semantic_pred_arr,
        ),
    },

    {
        "model":
            "existing_hard_router",

        **multiclass_metrics(
            true_family_arr,
            base_router_pred,
        ),
    },

    {
        "model":
            "hard_router_plus_nested_gw",

        **multiclass_metrics(
            true_family_arr,
            nested_gw_pred,
        ),
    },
])

display(
    nested_metrics_df.round(4)
)

Nested accepted: 13
Nested rescues: 9
Nested breaks: 4
Nested wrong-to-wrong: 0
Nested net: 5
Nested intervention precision: 0.6923


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_no_t1,0.5151,0.4355,0.4548,0.4969
1,existing_hard_router,0.5393,0.4745,0.4918,0.5292
2,hard_router_plus_nested_gw,0.5426,0.4806,0.4983,0.5345


In [ ]:
# ============================================================
# 131. Threshold stability
# ============================================================

display(
    nested_gw_fold_df[
        [
            "fold",
            "selected_threshold",
            "train_net",
            "train_precision",
            "test_accepted",
            "test_rescues",
            "test_breaks",
            "test_net",
            "test_precision",
        ]
    ].round(4)
)

print(
    "\nSelected thresholds:",
    nested_gw_fold_df[
        "selected_threshold"
    ].tolist()
)

print(
    "Mean threshold:",
    round(
        nested_gw_fold_df[
            "selected_threshold"
        ].mean(),
        4,
    )
)

print(
    "Std threshold:",
    round(
        nested_gw_fold_df[
            "selected_threshold"
        ].std(),
        4,
    )
)

,fold,selected_threshold,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_net,test_precision
0,1,0.65,10,0.8125,6,3,3,0,0.5000
1,2,0.55,10,0.7273,3,2,1,1,0.6667
2,3,0.80,12,0.9286,0,0,0,0,NaN
3,4,0.75,7,0.7059,3,3,0,3,1.0000
4,5,0.90,6,1.0000,1,1,0,1,1.0000



Selected thresholds: [0.65, 0.55, 0.8, 0.75, 0.9]
Mean threshold: 0.73
Std threshold: 0.1351


This is the result we wanted from the nested test: **the grounding expert survives leakage-free validation**, although its effect is smaller than the globally tuned estimate.

Your clean comparison is:

| Model                       |  Accuracy | Balanced acc. |  Macro-F1 | Weighted-F1 |
| --------------------------- | --------: | ------------: | --------: | ----------: |
| Semantic                    |     .5151 |         .4355 |     .4548 |       .4969 |
| Hard router                 |     .5393 |         .4745 |     .4918 |       .5292 |
| **Hard router + nested GW** | **.5426** |     **.4806** | **.4983** |   **.5345** |

The incremental nested intervention produced **13 changes, 9 rescues, 4 breaks, net +5, precision 69.2%**. So I would keep this expert.

There is one caveat: threshold stability is only moderate. The selected thresholds `[.65, .55, .80, .75, .90]` are clearly shifted toward conservative intervention, but they don't concentrate tightly around one value. Fold 3 selecting `.80` and then making zero test interventions is especially informative. This says the *directional signal is real*, but the exact cutoff remains noisy because this boundary has relatively little grounding support.

So I would record the conclusion as:

> **Validated specialist:** `TOOL_CALL→TOOL_CALL`, `workflow_error → grounding_state_error`. Nested OOF validation gives +5 net corrections (9 rescues, 4 breaks), improving macro-F1 from .4918 to .4983 and balanced accuracy from .4745 to .4806.

## What I would do next

Now move to the next highest-value unresolved boundary rather than tuning GW further.

From your opportunity analysis, after tool-vs-workflow the strongest remaining confusion opportunities were roughly:

1. `TOOL_CALL→ASSISTANT`: **constraint vs workflow** — 80 pair confusions, opportunity ≈17.09
2. `TOOL_CALL→TOOL_CALL`: grounding vs workflow — the one we just validated
3. `TOOL_CALL→TOOL_CALL`: constraint vs workflow — 35 confusions
4. `ASSISTANT→TOOL_CALL`: constraint vs workflow — 50 confusions

The first one is especially attractive because your earlier representation study already showed:

```text
TOOL_CALL->ASSISTANT
constraint_error vs workflow_error

support = 250
constraint = 128
workflow = 122
ROC-AUC = .7136
PR-AUC  = .7180
confusions = 80
```

Unlike GW, it's also almost perfectly balanced. That makes it a much healthier specialist-training problem.

### But first determine the useful direction

We shouldn't assume this specialist should operate symmetrically. The GW experiment showed exactly why.

The immediate next cell should therefore answer:

> Among `TOOL_CALL→ASSISTANT` constraint/workflow conflicts, does changing **workflow → constraint**, **constraint → workflow**, or both actually rescue the hard router?

Train an **OOF pairwise constraint-vs-workflow specialist** on this transition and analyze the two directions independently before applying anything.

Do **not threshold-tune yet**. First generate the leakage-free pairwise probabilities and produce:

```text
direction                         support  rescue  break  net
workflow -> constraint             ...
constraint -> workflow             ...
```

along with probability distributions for rescues versus breaks in each direction.

That will tell us whether expert #4 should exist at all and, if so, which direction deserves nested threshold validation.

I would leave the validated GW expert frozen now. Don't optimize its `.73` mean threshold or revisit its features. We have enough evidence to include it as a validated component while we test whether `TOOL_CALL→ASSISTANT` constraint/workflow can contribute another independent gain.


In [ ]:
# ============================================================
# 132. OOF TOOL_CALL->ASSISTANT
#      constraint_error vs workflow_error specialist
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# ------------------------------------------------------------
# Boundary rows
# ------------------------------------------------------------

cw_mask = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    &
    np.isin(
        true_family_arr,
        [
            "constraint_error",
            "workflow_error",
        ],
    )
)

cw_idx = np.where(cw_mask)[0]

y_cw = (
    true_family_arr[cw_idx]
    == "constraint_error"
).astype(int)

print("Support:", len(cw_idx))
print("Constraint:", int(y_cw.sum()))
print("Workflow:", int((1 - y_cw).sum()))
print("Prevalence:", round(y_cw.mean(), 4))

Support: 250
Constraint: 128
Workflow: 122
Prevalence: 0.512


In [ ]:
# ============================================================
# 133. Build CW representation
# ============================================================

X_cw = np.concatenate(
    [
        E_current[cw_idx],
        E_last_tool[cw_idx],
    ],
    axis=1,
)

print("X_cw:", X_cw.shape)

X_cw: (250, 768)


In [ ]:
# ============================================================
# 134. Exact outer-fold OOF probabilities
# ============================================================

cw_prob = np.full(
    len(train_targets),
    np.nan,
    dtype=float,
)

cw_fold_rows = []

# map global row -> position inside cw_idx
cw_pos = {
    global_idx: local_idx
    for local_idx, global_idx in enumerate(cw_idx)
}

for fold_id, (
    outer_train_idx,
    outer_test_idx,
) in enumerate(
    outer_splits,
    start=1,
):

    outer_train_idx = np.asarray(
        outer_train_idx,
        dtype=int,
    )

    outer_test_idx = np.asarray(
        outer_test_idx,
        dtype=int,
    )

    train_global = np.intersect1d(
        outer_train_idx,
        cw_idx,
    )

    test_global = np.intersect1d(
        outer_test_idx,
        cw_idx,
    )

    tr = np.array([
        cw_pos[i]
        for i in train_global
    ])

    va = np.array([
        cw_pos[i]
        for i in test_global
    ])

    m = make_specialist_model(
        C=0.03
    )

    m.fit(
        X_cw[tr],
        y_cw[tr],
    )

    p = m.predict_proba(
        X_cw[va]
    )[:, 1]

    cw_prob[
        test_global
    ] = p

    cw_fold_rows.append({
        "fold": fold_id,
        "train_support": len(tr),
        "test_support": len(va),
        "train_constraint": int(y_cw[tr].sum()),
        "test_constraint": int(y_cw[va].sum()),
        "test_workflow": int(len(va) - y_cw[va].sum()),
    })

display(
    pd.DataFrame(
        cw_fold_rows
    )
)

assert not np.isnan(
    cw_prob[cw_idx]
).any()

p_cw_oof = cw_prob[cw_idx]

print(
    "OOF PR-AUC:",
    average_precision_score(
        y_cw,
        p_cw_oof,
    )
)

print(
    "OOF ROC-AUC:",
    roc_auc_score(
        y_cw,
        p_cw_oof,
    )
)

print(
    "PR lift:",
    average_precision_score(
        y_cw,
        p_cw_oof,
    ) / y_cw.mean()
)

,fold,train_support,test_support,train_constraint,test_constraint,test_workflow
0,1,191,59,99,29,30
1,2,199,51,101,27,24
2,3,217,33,107,21,12
3,4,202,48,104,24,24
4,5,191,59,101,27,32


OOF PR-AUC: 0.6985306625854015
OOF ROC-AUC: 0.7042776639344263
PR lift: 1.3643177003621123


In [ ]:
# ============================================================
# 135. CW pairwise intervention on top of hard router
# ============================================================

base_router_pred = hierarchical_pred.copy()

base_router_correct = (
    base_router_pred
    == true_family_arr
)

cw_candidate = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    &
    np.isin(
        base_router_pred,
        [
            "constraint_error",
            "workflow_error",
        ]
    )
    &
    (~np.isnan(cw_prob))
)

cw_pair_pred = base_router_pred.copy()

# p >= .5 -> constraint
cw_pair_pred[
    cw_candidate
    &
    (cw_prob >= 0.5)
] = "constraint_error"

# p < .5 -> workflow
cw_pair_pred[
    cw_candidate
    &
    (cw_prob < 0.5)
] = "workflow_error"

cw_changed = (
    cw_pair_pred
    != base_router_pred
)

cw_correct = (
    cw_pair_pred
    == true_family_arr
)

cw_rescue = (
    cw_changed
    &
    (~base_router_correct)
    &
    cw_correct
)

cw_break = (
    cw_changed
    &
    base_router_correct
    &
    (~cw_correct)
)

cw_wrong = (
    cw_changed
    &
    (~base_router_correct)
    &
    (~cw_correct)
)

print("Candidates:", int(cw_candidate.sum()))
print("Changed:", int(cw_changed.sum()))
print("Rescues:", int(cw_rescue.sum()))
print("Breaks:", int(cw_break.sum()))
print("Wrong-to-wrong:", int(cw_wrong.sum()))
print(
    "Net:",
    int(cw_rescue.sum() - cw_break.sum())
)

Candidates: 213
Changed: 55
Rescues: 29
Breaks: 26
Wrong-to-wrong: 0
Net: 3


In [ ]:
# ============================================================
# 136. Direction-specific CW utility
# ============================================================

idx = np.where(cw_changed)[0]

cw_effect_df = pd.DataFrame({
    "idx": idx,
    "true_family": true_family_arr[idx],
    "router_prediction": base_router_pred[idx],
    "cw_prediction": cw_pair_pred[idx],
    "constraint_prob": cw_prob[idx],
})

cw_effect_df["effect"] = np.select(
    [
        (
            (~base_router_correct[idx])
            &
            cw_correct[idx]
        ),
        (
            base_router_correct[idx]
            &
            (~cw_correct[idx])
        ),
    ],
    [
        "rescue",
        "break",
    ],
    default="wrong_to_wrong",
)

cw_effect_df["direction"] = (
    cw_effect_df["router_prediction"]
    + " -> "
    + cw_effect_df["cw_prediction"]
)

direction_cw_df = (
    cw_effect_df
    .groupby("direction")
    .agg(
        support=("effect", "size"),
        rescues=(
            "effect",
            lambda x: int((x == "rescue").sum()),
        ),
        breaks=(
            "effect",
            lambda x: int((x == "break").sum()),
        ),
        wrong_to_wrong=(
            "effect",
            lambda x: int((x == "wrong_to_wrong").sum()),
        ),
    )
    .reset_index()
)

direction_cw_df["net"] = (
    direction_cw_df["rescues"]
    - direction_cw_df["breaks"]
)

direction_cw_df["precision"] = (
    direction_cw_df["rescues"]
    / direction_cw_df["support"]
)

display(
    direction_cw_df
    .sort_values(
        "net",
        ascending=False,
    )
    .round(4)
)

,direction,support,rescues,breaks,wrong_to_wrong,net,precision
0,constraint_error -> workflow_error,16,11,5,0,6,0.6875
1,workflow_error -> constraint_error,39,18,21,0,-3,0.4615


In [ ]:
# ============================================================
# 137. CW score distributions by direction and effect
# ============================================================

display(
    cw_effect_df
    .groupby(
        [
            "direction",
            "effect",
        ]
    )[
        "constraint_prob"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

count    mean  median     std     min     max
direction                          effect                                               
constraint_error -> workflow_error break       5  0.3435  0.3783  0.1673  0.0726  0.4970
                                   rescue     11  0.2826  0.2744  0.1529  0.0928  0.4710
workflow_error -> constraint_error break      21  0.7161  0.7153  0.1461  0.5112  0.9461
                                   rescue     18  0.7821  0.7998  0.1244  0.5045  0.9592

In [ ]:
# ============================================================
# 138. One-way threshold sweep:
#      TOOL_CALL->ASSISTANT
#      constraint_error -> workflow_error
#
# cw_prob = P(constraint_error)
# therefore LOWER p = stronger workflow evidence
# ============================================================

rows = []

base_pred = hierarchical_pred.copy()
base_correct = (
    base_pred == true_family_arr
)

# Only allow this directional correction.
eligible = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    &
    (base_pred == "constraint_error")
    &
    (~np.isnan(cw_prob))
)

# Sweep upper bounds on P(constraint).
#
# Smaller threshold = more conservative:
# only override constraint when specialist is
# highly confident that it is workflow.
CW_THRESHOLDS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]

for threshold in CW_THRESHOLDS:

    pred = base_pred.copy()

    use = (
        eligible
        &
        (cw_prob <= threshold)
    )

    pred[use] = "workflow_error"

    changed = (
        pred != base_pred
    )

    correct = (
        pred == true_family_arr
    )

    rescues = int(
        (
            changed
            & (~base_correct)
            & correct
        ).sum()
    )

    breaks = int(
        (
            changed
            & base_correct
            & (~correct)
        ).sum()
    )

    wrong = int(
        (
            changed
            & (~base_correct)
            & (~correct)
        ).sum()
    )

    changed_n = int(
        changed.sum()
    )

    rows.append({
        "threshold": threshold,
        "changed": changed_n,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / changed_n
            if changed_n
            else np.nan
        ),
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


cw_oneway_sweep_df = pd.DataFrame(rows)

display(
    cw_oneway_sweep_df
    .sort_values(
        [
            "net",
            "macro_f1",
            "precision",
        ],
        ascending=False,
    )
    .round(4)
)

,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
7,0.45,12,9,3,0,6,0.7500,0.5433,0.4753,0.4930,0.5320
8,0.50,16,11,5,0,6,0.6875,0.5433,0.4746,0.4924,0.5317
4,0.30,7,6,1,0,5,0.8571,0.5426,0.4756,0.4932,0.5318
2,0.20,6,5,1,0,4,0.8333,0.5420,0.4753,0.4929,0.5312
3,0.25,6,5,1,0,4,0.8333,0.5420,0.4753,0.4929,0.5312
5,0.35,8,6,2,0,4,0.7500,0.5420,0.4750,0.4926,0.5311
6,0.40,10,7,3,0,4,0.7000,0.5420,0.4747,0.4923,0.5309
1,0.15,4,3,1,0,2,0.7500,0.5406,0.4747,0.4922,0.5301
0,0.10,2,1,1,0,0,0.5000,0.5393,0.4741,0.4916,0.5290


In [ ]:
# ============================================================
# 139. Score quantiles for the useful direction
# ============================================================

useful_direction = cw_effect_df[
    cw_effect_df["direction"]
    ==
    "constraint_error -> workflow_error"
].copy()

display(
    useful_direction
    .groupby("effect")[
        "constraint_prob"
    ]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
    .rename(
        columns={
            0.10: "q10",
            0.25: "q25",
            0.50: "q50",
            0.75: "q75",
            0.90: "q90",
        }
    )
    .round(4)
)

,q10,q25,q50,q75,q90
effect,,,,,
break,0.1685,0.3123,0.3783,0.4571,0.4810
rescue,0.1054,0.1447,0.2744,0.4320,0.4539


In [ ]:
# ============================================================
# Nested threshold selection:
# TOOL_CALL->ASSISTANT
# constraint_error -> workflow_error
# ============================================================

CW_THRESHOLDS = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]

base_pred = hierarchical_pred.copy()
base_correct = (base_pred == true_family_arr)

eligible = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    & (base_pred == "constraint_error")
    & (~np.isnan(cw_prob))
)

nested_cw_pred = base_pred.copy()
nested_rows = []

for fold in sorted(pd.unique(outer_fold_arr)):

    train_mask = (outer_fold_arr != fold)
    test_mask  = (outer_fold_arr == fold)

    # ------------------------------------------
    # Select threshold using TRAIN portion only
    # ------------------------------------------
    threshold_rows = []

    for threshold in CW_THRESHOLDS:

        use = (
            train_mask
            & eligible
            & (cw_prob <= threshold)
        )

        pred = base_pred.copy()
        pred[use] = "workflow_error"

        changed = use
        correct = (pred == true_family_arr)

        rescues = int(
            (
                changed
                & (~base_correct)
                & correct
            ).sum()
        )

        breaks = int(
            (
                changed
                & base_correct
                & (~correct)
            ).sum()
        )

        accepted = int(changed.sum())
        net = rescues - breaks

        precision = (
            rescues / accepted
            if accepted
            else np.nan
        )

        threshold_rows.append({
            "threshold": threshold,
            "accepted": accepted,
            "rescues": rescues,
            "breaks": breaks,
            "net": net,
            "precision": precision,
        })

    threshold_df = pd.DataFrame(threshold_rows)

    # Primary objective: net
    # Tie-break: precision
    # Final tie-break: more conservative threshold
    best = (
        threshold_df
        .sort_values(
            ["net", "precision", "threshold"],
            ascending=[False, False, True],
        )
        .iloc[0]
    )

    selected_threshold = float(best["threshold"])

    # ------------------------------------------
    # Apply selected threshold to TEST fold
    # ------------------------------------------
    test_use = (
        test_mask
        & eligible
        & (cw_prob <= selected_threshold)
    )

    nested_cw_pred[test_use] = "workflow_error"

    test_correct = (
        nested_cw_pred == true_family_arr
    )

    test_rescues = int(
        (
            test_use
            & (~base_correct)
            & test_correct
        ).sum()
    )

    test_breaks = int(
        (
            test_use
            & base_correct
            & (~test_correct)
        ).sum()
    )

    test_wrong = int(
        (
            test_use
            & (~base_correct)
            & (~test_correct)
        ).sum()
    )

    test_accepted = int(test_use.sum())

    nested_rows.append({
        "fold": fold,
        "selected_threshold": selected_threshold,

        "train_accepted": int(best["accepted"]),
        "train_rescues": int(best["rescues"]),
        "train_breaks": int(best["breaks"]),
        "train_net": int(best["net"]),
        "train_precision": best["precision"],

        "test_accepted": test_accepted,
        "test_rescues": test_rescues,
        "test_breaks": test_breaks,
        "test_wrong_to_wrong": test_wrong,
        "test_net": test_rescues - test_breaks,
        "test_precision": (
            test_rescues / test_accepted
            if test_accepted
            else np.nan
        ),
    })


nested_cw_df = pd.DataFrame(nested_rows)

display(
    nested_cw_df.round(4)
)

,fold,selected_threshold,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net,test_precision
0,1,0.50,15,11,4,7,0.7333,1,0,1,0,-1,0.000
1,2,0.45,10,7,3,4,0.7000,2,2,0,0,2,1.000
2,3,0.30,6,5,1,4,0.8333,1,1,0,0,1,1.000
3,4,0.45,11,9,2,7,0.8182,1,0,1,0,-1,0.000
4,5,0.50,8,6,2,4,0.7500,8,5,3,0,2,0.625


In [ ]:
# ============================================================
# Evaluate nested CW specialist
# ============================================================

nested_changed = (
    nested_cw_pred != base_pred
)

nested_correct = (
    nested_cw_pred == true_family_arr
)

nested_rescues = int(
    (
        nested_changed
        & (~base_correct)
        & nested_correct
    ).sum()
)

nested_breaks = int(
    (
        nested_changed
        & base_correct
        & (~nested_correct)
    ).sum()
)

nested_wrong = int(
    (
        nested_changed
        & (~base_correct)
        & (~nested_correct)
    ).sum()
)

nested_n = int(nested_changed.sum())

print("Nested accepted:", nested_n)
print("Nested rescues:", nested_rescues)
print("Nested breaks:", nested_breaks)
print("Nested wrong-to-wrong:", nested_wrong)
print("Nested net:", nested_rescues - nested_breaks)
print(
    "Nested intervention precision:",
    nested_rescues / nested_n if nested_n else np.nan
)

comparison = pd.DataFrame([
    {
        "model": "existing_hard_router",
        **multiclass_metrics(
            true_family_arr,
            base_pred,
        ),
    },
    {
        "model": "hard_router_plus_nested_cw",
        **multiclass_metrics(
            true_family_arr,
            nested_cw_pred,
        ),
    },
])

display(comparison.round(4))

print(
    "\nSelected thresholds:",
    nested_cw_df["selected_threshold"].tolist()
)
print(
    "Mean threshold:",
    round(nested_cw_df["selected_threshold"].mean(), 4)
)
print(
    "Std threshold:",
    round(nested_cw_df["selected_threshold"].std(), 4)
)

Nested accepted: 13
Nested rescues: 8
Nested breaks: 5
Nested wrong-to-wrong: 0
Nested net: 3
Nested intervention precision: 0.6153846153846154


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,existing_hard_router,0.5393,0.4745,0.4918,0.5292
1,hard_router_plus_nested_cw,0.5413,0.4737,0.4915,0.5300



Selected thresholds: [0.5, 0.45, 0.3, 0.45, 0.5]
Mean threshold: 0.44
Std threshold: 0.0822


In [ ]:
# ============================================================
# Compose nested GW + CW specialists
# ============================================================

base_pred = hierarchical_pred.copy()

gw_changed = (nested_gw_pred != base_pred)
cw_changed = (nested_cw_pred != base_pred)

print("GW changed:", gw_changed.sum())
print("CW changed:", cw_changed.sum())
print("Overlap:", (gw_changed & cw_changed).sum())

conflict = (
    gw_changed
    & cw_changed
    & (nested_gw_pred != nested_cw_pred)
)

print("Conflicts:", conflict.sum())

# Inspect any overlap/conflicts
overlap_df = pd.DataFrame({
    "true": true_family_arr,
    "base": base_pred,
    "gw": nested_gw_pred,
    "cw": nested_cw_pred,
    "gw_changed": gw_changed,
    "cw_changed": cw_changed,
})

display(
    overlap_df[
        gw_changed & cw_changed
    ]
)

GW changed: 13
CW changed: 13
Overlap: 0
Conflicts: 0


,true,base,gw,cw,gw_changed,cw_changed


In [ ]:
# ============================================================
# Order 1: GW then CW
# ============================================================

gw_then_cw = base_pred.copy()

gw_then_cw[gw_changed] = nested_gw_pred[gw_changed]
gw_then_cw[cw_changed] = nested_cw_pred[cw_changed]


# ============================================================
# Order 2: CW then GW
# ============================================================

cw_then_gw = base_pred.copy()

cw_then_gw[cw_changed] = nested_cw_pred[cw_changed]
cw_then_gw[gw_changed] = nested_gw_pred[gw_changed]


# ============================================================
# Evaluation
# ============================================================

comparison = pd.DataFrame([
    {
        "model": "existing_hard_router",
        **multiclass_metrics(
            true_family_arr,
            base_pred,
        ),
    },
    {
        "model": "nested_gw_only",
        **multiclass_metrics(
            true_family_arr,
            nested_gw_pred,
        ),
    },
    {
        "model": "nested_cw_only",
        **multiclass_metrics(
            true_family_arr,
            nested_cw_pred,
        ),
    },
    {
        "model": "gw_then_cw",
        **multiclass_metrics(
            true_family_arr,
            gw_then_cw,
        ),
    },
    {
        "model": "cw_then_gw",
        **multiclass_metrics(
            true_family_arr,
            cw_then_gw,
        ),
    },
])

display(comparison.round(4))

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,existing_hard_router,0.5393,0.4745,0.4918,0.5292
1,nested_gw_only,0.5426,0.4806,0.4983,0.5345
2,nested_cw_only,0.5413,0.4737,0.4915,0.5300
3,gw_then_cw,0.5447,0.4799,0.4979,0.5354
4,cw_then_gw,0.5447,0.4799,0.4979,0.5354


In [ ]:
def intervention_summary(name, pred):
    changed = (pred != base_pred)

    base_correct = (base_pred == true_family_arr)
    new_correct = (pred == true_family_arr)

    rescues = int(
        (
            changed
            & (~base_correct)
            & new_correct
        ).sum()
    )

    breaks = int(
        (
            changed
            & base_correct
            & (~new_correct)
        ).sum()
    )

    wrong_to_wrong = int(
        (
            changed
            & (~base_correct)
            & (~new_correct)
        ).sum()
    )

    n = int(changed.sum())

    return {
        "model": name,
        "changed": n,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong_to_wrong,
        "net": rescues - breaks,
        "precision": rescues / n if n else np.nan,
    }


intervention_df = pd.DataFrame([
    intervention_summary("nested_gw_only", nested_gw_pred),
    intervention_summary("nested_cw_only", nested_cw_pred),
    intervention_summary("gw_then_cw", gw_then_cw),
    intervention_summary("cw_then_gw", cw_then_gw),
])

display(intervention_df.round(4))

,model,changed,rescues,breaks,wrong_to_wrong,net,precision
0,nested_gw_only,13,9,4,0,5,0.6923
1,nested_cw_only,13,8,5,0,3,0.6154
2,gw_then_cw,26,17,9,0,8,0.6538
3,cw_then_gw,26,17,9,0,8,0.6538


This is a very clean result. The two specialists are complementary rather than competing: zero overlap, zero conflicts, and their gains are exactly additive: +5 + +3 = +8.

The combined router reaches 0.5447 accuracy, up from 0.5393, and weighted F1 rises from 0.5292 → 0.5354. However, GW alone remains slightly better on balanced accuracy (0.4806 vs 0.4799) and macro-F1 (0.4983 vs 0.4979). So CW adds useful raw corrections, but slightly shifts the class-level tradeoff in the wrong direction.

I would keep both variants now rather than immediately discard CW:

nested_gw_only = conservative / macro-metric champion.
nested_gw_plus_cw = maximum-correct-predictions / accuracy champion.

The next thing I would do is not add another boundary yet. We've reached the point where we should characterize exactly which classes the +8 rescues and 9 breaks affect. That will tell us whether CW's macro-F1 decrease is merely noise or whether it systematically harms a minority class.

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

combined_pred = gw_then_cw

labels = sorted(pd.unique(true_family_arr))

# ------------------------------------------------------------
# Per-class metrics
# ------------------------------------------------------------

base_report = pd.DataFrame(
    classification_report(
        true_family_arr,
        base_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )
).T

combined_report = pd.DataFrame(
    classification_report(
        true_family_arr,
        combined_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )
).T

per_class = pd.DataFrame({
    "support": [
        (true_family_arr == label).sum()
        for label in labels
    ],
    "base_precision": base_report.loc[labels, "precision"],
    "new_precision": combined_report.loc[labels, "precision"],
    "base_recall": base_report.loc[labels, "recall"],
    "new_recall": combined_report.loc[labels, "recall"],
    "base_f1": base_report.loc[labels, "f1-score"],
    "new_f1": combined_report.loc[labels, "f1-score"],
}, index=labels)

per_class["delta_precision"] = (
    per_class["new_precision"]
    - per_class["base_precision"]
)

per_class["delta_recall"] = (
    per_class["new_recall"]
    - per_class["base_recall"]
)

per_class["delta_f1"] = (
    per_class["new_f1"]
    - per_class["base_f1"]
)

display(per_class.round(4))

,support,base_precision,new_precision,base_recall,new_recall,base_f1,new_f1,delta_precision,delta_recall,delta_f1
constraint_error,317,0.4387,0.4411,0.4290,0.4132,0.4338,0.4267,0.0024,-0.0158,-0.0071
grounding_state_error,244,0.4710,0.4881,0.2992,0.3361,0.3659,0.3981,0.0171,0.0369,0.0321
reasoning_value_error,31,0.5385,0.5385,0.4516,0.4516,0.4912,0.4912,0.0000,0.0000,0.0000
tool_use_error,237,0.5829,0.5829,0.4895,0.4895,0.5321,0.5321,0.0000,0.0000,0.0000
workflow_error,660,0.5807,0.5857,0.7030,0.7091,0.6361,0.6415,0.0050,0.0061,0.0055


In [ ]:
changed = combined_pred != base_pred

change_df = pd.DataFrame({
    "true_family": true_family_arr[changed],
    "base_prediction": base_pred[changed],
    "new_prediction": combined_pred[changed],
})

change_df["effect"] = np.select(
    [
        (
            change_df["base_prediction"]
            != change_df["true_family"]
        )
        & (
            change_df["new_prediction"]
            == change_df["true_family"]
        ),

        (
            change_df["base_prediction"]
            == change_df["true_family"]
        )
        & (
            change_df["new_prediction"]
            != change_df["true_family"]
        ),
    ],
    [
        "rescue",
        "break",
    ],
    default="wrong_to_wrong",
)

flow = (
    change_df
    .groupby(
        [
            "true_family",
            "base_prediction",
            "new_prediction",
            "effect",
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["effect", "count"],
        ascending=[True, False],
    )
)

display(flow)

,true_family,base_prediction,new_prediction,effect,count
0,constraint_error,constraint_error,workflow_error,break,5
3,workflow_error,workflow_error,grounding_state_error,break,4
1,grounding_state_error,workflow_error,grounding_state_error,rescue,9
2,workflow_error,constraint_error,workflow_error,rescue,8


In [ ]:
effect_by_true = pd.crosstab(
    change_df["true_family"],
    change_df["effect"],
    margins=True,
)

display(effect_by_true)

effect,break,rescue,All
true_family,,,
constraint_error,5,0,5
grounding_state_error,0,9,9
workflow_error,4,8,12
All,9,17,26


This resolves the question pretty cleanly: I would promote GW+CW to the new working baseline.

The macro-F1 concern is real but very small and completely interpretable. CW converts 13 constraint → workflow: 8 are correct workflow rescues and 5 are constraint breaks. That costs constraint recall 0.4290 → 0.4132 and F1 0.4338 → 0.4267, while improving workflow recall 0.7030 → 0.7091. GW is much stronger: its 13 changes give 9 grounding rescues versus 4 workflow breaks, producing a substantial grounding F1 improvement of +0.0321.

So the combined intervention has an unusually simple structure:

Specialist	Direction	Rescues	Breaks	Net
GW	workflow → grounding	9	4	+5
CW	constraint → workflow	8	5	+3
Combined	—	17	9	+8

There are no wrong-to-wrong changes, no overlap, and no conflicts. That's strong evidence that the architecture is doing something meaningful rather than merely perturbing predictions.

In [ ]:
final_pairwise_pred = gw_then_cw.copy()

In [ ]:
# ============================================================
# Remaining errors after GW + CW
# ============================================================

new_base_pred = gw_then_cw.copy()

remaining_errors = pd.DataFrame({
    "true_family": true_family_arr,
    "prediction": new_base_pred,
    "transition": transition_arr,
})

remaining_errors = remaining_errors[
    remaining_errors["true_family"]
    != remaining_errors["prediction"]
].copy()

# Overall confusion directions
remaining_confusions = (
    remaining_errors
    .groupby(
        ["true_family", "prediction"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(remaining_confusions.head(30))

,true_family,prediction,count
3,constraint_error,workflow_error,143
7,grounding_state_error,workflow_error,98
15,tool_use_error,workflow_error,84
16,workflow_error,constraint_error,81
19,workflow_error,tool_use_error,62
4,grounding_state_error,constraint_error,53
17,workflow_error,grounding_state_error,45
0,constraint_error,grounding_state_error,30
12,tool_use_error,constraint_error,25
2,constraint_error,tool_use_error,12


In [ ]:
# ============================================================
# Remaining confusion opportunities by transition
# ============================================================

remaining_by_transition = (
    remaining_errors
    .groupby(
        [
            "transition",
            "true_family",
            "prediction",
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(
    remaining_by_transition.head(50)
)

,transition,true_family,prediction,count
41,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,56
68,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,54
45,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,39
2,ASSISTANT->ASSISTANT,constraint_error,workflow_error,38
11,ASSISTANT->ASSISTANT,workflow_error,constraint_error,28
42,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,27
65,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,25
22,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,24
51,TOOL_CALL->ASSISTANT,tool_use_error,workflow_error,23
16,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,22


This changes the priority. The largest remaining errors are highly concentrated, and the best third specialist is now fairly clear.

The biggest raw confusion is constraint → workflow (143), but it's spread across transitions and we've already attacked one important slice with CW. I would not immediately build another constraint/workflow specialist because that risks repeatedly optimizing the same family boundary.

Instead, the standout untouched opportunity is:

TOOL_CALL → TOOL_CALL: workflow ↔ tool_use

It has 79 remaining errors in the two directions: 54 workflow → tool_use and 25 tool_use → workflow. More importantly, your earlier pairwise analysis already showed this boundary was strong: support 477, ROC-AUC about 0.869, PR-AUC about 0.706, and it was your highest opportunity-score pair. So we have both error mass and demonstrated separability.

I'd test this third specialist next, but against the new GW+CW baseline, and keep it fully OOF.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# ============================================================
# TOOL_CALL -> TOOL_CALL
# tool_use_error vs workflow_error
# ============================================================

base_pred = gw_then_cw.copy()

boundary_mask = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    & np.isin(
        true_family_arr,
        ["tool_use_error", "workflow_error"]
    )
)

boundary_idx = np.where(boundary_mask)[0]

y_tw = (
    true_family_arr[boundary_idx]
    == "tool_use_error"
).astype(int)

X_tw = X_tool_pair[boundary_idx]
fold_tw = outer_fold_arr[boundary_idx]

tw_oof_prob = np.full(
    len(boundary_idx),
    np.nan,
    dtype=float,
)

rows = []

for fold in sorted(pd.unique(fold_tw)):

    tr = fold_tw != fold
    te = fold_tw == fold

    clf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
    )

    clf.fit(
        X_tw[tr],
        y_tw[tr],
    )

    tw_oof_prob[te] = clf.predict_proba(
        X_tw[te]
    )[:, 1]

    rows.append({
        "fold": fold,
        "train_support": int(tr.sum()),
        "test_support": int(te.sum()),
        "train_tool": int(y_tw[tr].sum()),
        "test_tool": int(y_tw[te].sum()),
        "test_workflow": int((1 - y_tw[te]).sum()),
    })

display(pd.DataFrame(rows))

print(
    "OOF PR-AUC:",
    average_precision_score(
        y_tw,
        tw_oof_prob,
    )
)

print(
    "OOF ROC-AUC:",
    roc_auc_score(
        y_tw,
        tw_oof_prob,
    )
)

print(
    "Prevalence:",
    y_tw.mean()
)

,fold,train_support,test_support,train_tool,test_tool,test_workflow
0,1,382,95,109,27,68
1,2,384,93,109,27,66
2,3,364,113,106,30,83
3,4,391,86,110,26,60
4,5,387,90,110,26,64


OOF PR-AUC: 0.6872797829182273
OOF ROC-AUC: 0.8433349146110056
Prevalence: 0.2851153039832285


In [ ]:
# Map local OOF probabilities back to all 1489 examples
tw_prob = np.full(
    len(true_family_arr),
    np.nan,
)

tw_prob[boundary_idx] = tw_oof_prob

base_correct = (
    base_pred == true_family_arr
)

eligible_tw = (
    (transition_arr == "TOOL_CALL->TOOL_CALL")
    & np.isin(
        base_pred,
        ["tool_use_error", "workflow_error"]
    )
    & (~np.isnan(tw_prob))
)

# Naive 0.5 specialist decision
specialist_pred = np.where(
    tw_prob >= 0.5,
    "tool_use_error",
    "workflow_error",
)

would_change = (
    eligible_tw
    & (specialist_pred != base_pred)
)

candidate_df = pd.DataFrame({
    "true": true_family_arr[would_change],
    "base": base_pred[would_change],
    "specialist": specialist_pred[would_change],
    "tool_prob": tw_prob[would_change],
})

candidate_df["effect"] = np.select(
    [
        candidate_df["specialist"]
        == candidate_df["true"],

        candidate_df["base"]
        == candidate_df["true"],
    ],
    [
        "rescue",
        "break",
    ],
    default="wrong_to_wrong",
)

direction_summary = (
    candidate_df
    .assign(
        direction=lambda d:
            d["base"] + " -> " + d["specialist"]
    )
    .groupby(["direction", "effect"])
    .size()
    .unstack(fill_value=0)
)

display(direction_summary)

effect,break,rescue
direction,,
tool_use_error -> workflow_error,7,9
workflow_error -> tool_use_error,22,7


In [ ]:
candidate_df["distance"] = np.abs(
    candidate_df["tool_prob"] - 0.5
)

display(
    candidate_df
    .groupby(
        [
            candidate_df["base"]
            + " -> "
            + candidate_df["specialist"],
            "effect",
        ]
    )["tool_prob"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

count    mean  median     std     min     max
                                 effect                                               
tool_use_error -> workflow_error break       7  0.2546  0.3427  0.1799  0.0247  0.4455
                                 rescue      9  0.2917  0.3256  0.1694  0.0708  0.4746
workflow_error -> tool_use_error break      22  0.6052  0.6076  0.0543  0.5141  0.7243
                                 rescue      7  0.6607  0.6270  0.0772  0.5938  0.7905

In [ ]:
display(
    candidate_df
    .groupby(
        [
            candidate_df["base"]
            + " -> "
            + candidate_df["specialist"],
            "effect",
        ]
    )["tool_prob"]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
    .round(4)
)

0.10    0.25    0.50    0.75    0.90
                                 effect                                        
tool_use_error -> workflow_error break   0.0498  0.0882  0.3427  0.3964  0.4267
                                 rescue  0.0737  0.0819  0.3256  0.4238  0.4513
workflow_error -> tool_use_error break   0.5491  0.5673  0.6076  0.6299  0.6745
                                 rescue  0.6013  0.6091  0.6270  0.6977  0.7656

In [ ]:
# ============================================================
# Independent threshold sweeps for TW specialist
#
# p = P(tool_use_error)
#
# tool_use -> workflow:
#     override when p <= lower_threshold
#
# workflow -> tool_use:
#     override when p >= upper_threshold
# ============================================================

base_pred = gw_then_cw.copy()
base_correct = (base_pred == true_family_arr)

rows = []

# ------------------------------------------------------------
# Direction A: tool_use -> workflow
# ------------------------------------------------------------

for threshold in np.arange(0.05, 0.501, 0.025):

    use = (
        (transition_arr == "TOOL_CALL->TOOL_CALL")
        & (base_pred == "tool_use_error")
        & (~np.isnan(tw_prob))
        & (tw_prob <= threshold)
    )

    pred = base_pred.copy()
    pred[use] = "workflow_error"

    new_correct = (pred == true_family_arr)

    rescues = int(
        (use & ~base_correct & new_correct).sum()
    )
    breaks = int(
        (use & base_correct & ~new_correct).sum()
    )
    wrong = int(
        (use & ~base_correct & ~new_correct).sum()
    )

    changed = int(use.sum())

    rows.append({
        "direction": "tool_to_workflow",
        "threshold": threshold,
        "changed": changed,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / changed
            if changed else np.nan
        ),
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


# ------------------------------------------------------------
# Direction B: workflow -> tool_use
# ------------------------------------------------------------

for threshold in np.arange(0.50, 0.951, 0.025):

    use = (
        (transition_arr == "TOOL_CALL->TOOL_CALL")
        & (base_pred == "workflow_error")
        & (~np.isnan(tw_prob))
        & (tw_prob >= threshold)
    )

    pred = base_pred.copy()
    pred[use] = "tool_use_error"

    new_correct = (pred == true_family_arr)

    rescues = int(
        (use & ~base_correct & new_correct).sum()
    )
    breaks = int(
        (use & base_correct & ~new_correct).sum()
    )
    wrong = int(
        (use & ~base_correct & ~new_correct).sum()
    )

    changed = int(use.sum())

    rows.append({
        "direction": "workflow_to_tool",
        "threshold": threshold,
        "changed": changed,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / changed
            if changed else np.nan
        ),
        **multiclass_metrics(
            true_family_arr,
            pred,
        ),
    })


tw_threshold_df = pd.DataFrame(rows)

In [ ]:
for direction in [
    "tool_to_workflow",
    "workflow_to_tool",
]:
    print("\n" + "=" * 80)
    print(direction)

    display(
        tw_threshold_df[
            tw_threshold_df["direction"] == direction
        ]
        .sort_values(
            ["net", "precision"],
            ascending=False,
        )
        .head(15)
        .round(4)
    )


tool_to_workflow


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
17,tool_to_workflow,0.475,16,9,7,0,2,0.5625,0.5460,0.4767,0.4964,0.5356
18,tool_to_workflow,0.500,16,9,7,0,2,0.5625,0.5460,0.4767,0.4964,0.5356
2,tool_to_workflow,0.100,5,3,2,0,1,0.6000,0.5453,0.4791,0.4977,0.5357
11,tool_to_workflow,0.325,7,4,3,0,1,0.5714,0.5453,0.4786,0.4973,0.5356
12,tool_to_workflow,0.350,9,5,4,0,1,0.5556,0.5453,0.4780,0.4970,0.5354
13,tool_to_workflow,0.375,9,5,4,0,1,0.5556,0.5453,0.4780,0.4970,0.5354
15,tool_to_workflow,0.425,13,7,6,0,1,0.5385,0.5453,0.4770,0.4963,0.5351
16,tool_to_workflow,0.450,15,8,7,0,1,0.5333,0.5453,0.4764,0.4959,0.5350
1,tool_to_workflow,0.075,4,2,2,0,0,0.5000,0.5447,0.4788,0.4973,0.5351
3,tool_to_workflow,0.125,6,3,3,0,0,0.5000,0.5447,0.4783,0.4969,0.5350



workflow_to_tool


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
28,workflow_to_tool,0.725,2,2,0,0,2,1.0000,0.5460,0.4816,0.4994,0.5368
29,workflow_to_tool,0.750,1,1,0,0,1,1.0000,0.5453,0.4807,0.4987,0.5361
30,workflow_to_tool,0.775,1,1,0,0,1,1.0000,0.5453,0.4807,0.4987,0.5361
27,workflow_to_tool,0.700,4,2,2,0,0,0.5000,0.5447,0.4810,0.4986,0.5356
31,workflow_to_tool,0.800,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354
32,workflow_to_tool,0.825,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354
33,workflow_to_tool,0.850,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354
34,workflow_to_tool,0.875,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354
35,workflow_to_tool,0.900,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354
36,workflow_to_tool,0.925,0,0,0,0,0,NaN,0.5447,0.4799,0.4979,0.5354


In [ ]:
candidate_df["distance_from_boundary"] = np.abs(
    candidate_df["tool_prob"] - 0.5
)

display(
    candidate_df
    .assign(
        direction=lambda d:
            d["base"] + " -> " + d["specialist"]
    )
    .groupby(["direction", "effect"])[
        "distance_from_boundary"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

count    mean  median     std     min     max
direction                        effect                                               
tool_use_error -> workflow_error break       7  0.2454  0.1573  0.1799  0.0545  0.4753
                                 rescue      9  0.2083  0.1744  0.1694  0.0254  0.4292
workflow_error -> tool_use_error break      22  0.1052  0.1076  0.0543  0.0141  0.2243
                                 rescue      7  0.1607  0.1270  0.0772  0.0938  0.2905

In [ ]:
# ============================================================
# Nested threshold selection:
# workflow_error -> tool_use_error ONLY
#
# Base = GW + CW nested router
# Specialist = TW OOF probability
#
# Threshold selected on outer-train;
# evaluated only on outer-test.
# ============================================================

base_pred = gw_then_cw.copy()
y = np.asarray(true_family_arr)
folds = np.asarray(outer_fold_arr)

threshold_grid = np.arange(0.60, 0.776, 0.025)

nested_tw_pred = base_pred.copy()
fold_rows = []

for fold in sorted(np.unique(folds)):

    train_mask = folds != fold
    test_mask = folds == fold

    # Candidate population: TT transition currently predicted workflow
    train_candidate = (
        train_mask
        & (transition_arr == "TOOL_CALL->TOOL_CALL")
        & (base_pred == "workflow_error")
        & ~np.isnan(tw_prob)
    )

    # ---------------------------------------------
    # Select threshold using TRAIN portion only
    # ---------------------------------------------
    threshold_rows = []

    for threshold in threshold_grid:

        use = (
            train_candidate
            & (tw_prob >= threshold)
        )

        old_correct = base_pred == y

        trial_pred = base_pred.copy()
        trial_pred[use] = "tool_use_error"

        new_correct = trial_pred == y

        changed = int(use.sum())

        rescues = int(
            (use & ~old_correct & new_correct).sum()
        )
        breaks = int(
            (use & old_correct & ~new_correct).sum()
        )
        wrong = int(
            (use & ~old_correct & ~new_correct).sum()
        )

        net = rescues - breaks
        precision = (
            rescues / changed
            if changed
            else np.nan
        )

        threshold_rows.append({
            "threshold": threshold,
            "changed": changed,
            "rescues": rescues,
            "breaks": breaks,
            "wrong_to_wrong": wrong,
            "net": net,
            "precision": precision,
        })

    threshold_df = pd.DataFrame(threshold_rows)

    # Require at least a little evidence.
    #
    # Selection priority:
    #   1. positive net
    #   2. highest net
    #   3. highest precision
    #   4. more accepted examples
    #
    eligible = threshold_df[
        (threshold_df["net"] > 0)
        & (threshold_df["changed"] >= 2)
    ].copy()

    if len(eligible) == 0:
        selected_threshold = np.inf
        train_stats = {
            "changed": 0,
            "rescues": 0,
            "breaks": 0,
            "net": 0,
            "precision": np.nan,
        }

    else:
        best = (
            eligible
            .sort_values(
                ["net", "precision", "changed"],
                ascending=[False, False, False],
            )
            .iloc[0]
        )

        selected_threshold = float(
            best["threshold"]
        )

        train_stats = {
            "changed": int(best["changed"]),
            "rescues": int(best["rescues"]),
            "breaks": int(best["breaks"]),
            "net": int(best["net"]),
            "precision": float(best["precision"]),
        }

    # ---------------------------------------------
    # Apply selected threshold to TEST fold only
    # ---------------------------------------------
    test_use = (
        test_mask
        & (transition_arr == "TOOL_CALL->TOOL_CALL")
        & (base_pred == "workflow_error")
        & ~np.isnan(tw_prob)
        & (tw_prob >= selected_threshold)
    )

    old_correct = base_pred == y

    nested_tw_pred[test_use] = "tool_use_error"

    new_correct = nested_tw_pred == y

    test_changed = int(test_use.sum())

    test_rescues = int(
        (test_use & ~old_correct & new_correct).sum()
    )
    test_breaks = int(
        (test_use & old_correct & ~new_correct).sum()
    )
    test_wrong = int(
        (test_use & ~old_correct & ~new_correct).sum()
    )

    fold_rows.append({
        "fold": fold,
        "selected_threshold": (
            selected_threshold
            if np.isfinite(selected_threshold)
            else np.nan
        ),

        "train_accepted": train_stats["changed"],
        "train_rescues": train_stats["rescues"],
        "train_breaks": train_stats["breaks"],
        "train_net": train_stats["net"],
        "train_precision": train_stats["precision"],

        "test_accepted": test_changed,
        "test_rescues": test_rescues,
        "test_breaks": test_breaks,
        "test_wrong_to_wrong": test_wrong,
        "test_net": test_rescues - test_breaks,
        "test_precision": (
            test_rescues / test_changed
            if test_changed
            else np.nan
        ),
    })


nested_tw_fold_df = pd.DataFrame(fold_rows)

display(
    nested_tw_fold_df.round(4)
)

,fold,selected_threshold,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net,test_precision
0,1,0.725,2,2,0,2,1.0,0,0,0,0,0,NaN
1,2,NaN,0,0,0,0,NaN,0,0,0,0,0,NaN
2,3,0.725,2,2,0,2,1.0,0,0,0,0,0,NaN
3,4,0.725,2,2,0,2,1.0,0,0,0,0,0,NaN
4,5,NaN,0,0,0,0,NaN,0,0,0,0,0,NaN


In [ ]:
# ============================================================
# Aggregate nested TW result
# ============================================================

base_correct = base_pred == y
new_correct = nested_tw_pred == y

changed = nested_tw_pred != base_pred

rescues = (
    changed
    & ~base_correct
    & new_correct
)

breaks = (
    changed
    & base_correct
    & ~new_correct
)

wrong_to_wrong = (
    changed
    & ~base_correct
    & ~new_correct
)

print("Nested accepted:", int(changed.sum()))
print("Nested rescues:", int(rescues.sum()))
print("Nested breaks:", int(breaks.sum()))
print(
    "Nested wrong-to-wrong:",
    int(wrong_to_wrong.sum()),
)
print(
    "Nested net:",
    int(rescues.sum() - breaks.sum()),
)

if changed.sum():
    print(
        "Nested intervention precision:",
        rescues.sum() / changed.sum(),
    )


comparison = pd.DataFrame([
    {
        "model": "gw_then_cw",
        **multiclass_metrics(y, base_pred),
    },
    {
        "model": "gw_cw_plus_nested_tw",
        **multiclass_metrics(y, nested_tw_pred),
    },
])

display(comparison.round(4))

print("\nSelected thresholds:")
print(
    nested_tw_fold_df[
        ["fold", "selected_threshold",
         "train_net", "test_net"]
    ].to_string(index=False)
)

Nested accepted: 0
Nested rescues: 0
Nested breaks: 0
Nested wrong-to-wrong: 0
Nested net: 0


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,gw_then_cw,0.5447,0.4799,0.4979,0.5354
1,gw_cw_plus_nested_tw,0.5447,0.4799,0.4979,0.5354



Selected thresholds:
 fold  selected_threshold  train_net  test_net
    1               0.725          2         0
    2                 NaN          0         0
    3               0.725          2         0
    4               0.725          2         0
    5                 NaN          0         0


Yes — effectively no test data above the selected threshold, rather than a bug.

What happened is quite informative. On folds 1, 3, and 4, the training split found exactly 2 qualifying examples at threshold 0.725, both rescues. But when that threshold was applied to each held-out fold, zero examples qualified. Folds 2 and 5 couldn't even select a threshold under the changed >= 2 && net > 0 rule.

So the nested result is:

TW actionable support = 0 / 1489 held-out observations.

That tells us the apparent full-OOF result at workflow → tool_use, threshold=0.725:

changed = 2
rescues = 2
breaks = 0

comes from the same two observations repeatedly appearing in the training portions of folds 1/3/4. Whichever outer fold contains those observations apparently fails to select the rule from its remaining training data. Thus they never become legitimate held-out interventions.

This is actually a useful negative result: TW ranks the pair reasonably well (ROC-AUC 0.843), but there isn't enough stable high-confidence support to turn it into an override.

I would stop TW here

Don't lower the threshold just to force nested test coverage. Your earlier sweep already tells us what happens:

workflow → tool_use deteriorates rapidly below 0.70, while tool_use → workflow never develops a convincing high-precision region.

So our current validated router remains:

existing hard router → nested GW → nested CW

with:

Model	Accuracy	Balanced acc.	Macro F1	Weighted F1
Existing hard router	.5393	.4745	.4918	.5292
+ GW	.5426	.4806	.4983	.5345
+ GW + CW	.5447	.4799	.4979	.5354

Notice GW alone still has slightly higher balanced accuracy and macro-F1 than GW+CW, but CW raises overall accuracy and weighted F1. So CW is useful, though weaker than GW.

Next target: constraint → workflow

This is now the largest obvious remaining opportunity.

You currently have 143 constraint→workflow errors overall, and the transition decomposition shows a particularly large pocket:

TOOL_CALL→ASSISTANT: constraint → workflow = 56

That's much larger than the tiny TW tail we were just trying to exploit.

More importantly, we already know a generic CW specialist has some signal, but its benefit gets diluted by transition. So rather than build another global pairwise classifier, I'd test a transition-specific constraint-vs-workflow specialist for TOOL_CALL->ASSISTANT.

That follows the pattern that worked for GW: use transition structure to turn a broad ambiguous family pair into a narrower classification problem.

In [ ]:
# ============================================================
# Diagnostic:
# TOOL_CALL->ASSISTANT
# constraint_error vs workflow_error
# ============================================================

transition_target = "TOOL_CALL->ASSISTANT"

mask = (
    (transition_arr == transition_target)
    & np.isin(
        y,
        ["constraint_error", "workflow_error"]
    )
)

print("Support:", mask.sum())

print(
    pd.Series(y[mask])
    .value_counts()
    .rename("count")
)

# Current router behavior on this boundary
diag = pd.DataFrame({
    "true": y[mask],
    "base": gw_then_cw[mask],
})

print("\nCurrent predictions:")
display(
    pd.crosstab(
        diag["true"],
        diag["base"],
        margins=True,
    )
)

# Exact pairwise confusions
pair_conf = diag[
    (
        (diag["true"] == "constraint_error")
        & (diag["base"] == "workflow_error")
    )
    |
    (
        (diag["true"] == "workflow_error")
        & (diag["base"] == "constraint_error")
    )
]

print("\nPairwise confusion support:", len(pair_conf))

display(
    pair_conf
    .value_counts(["true", "base"])
    .rename("count")
    .reset_index()
)

Support: 250
constraint_error    128
workflow_error      122
Name: count, dtype: int64

Current predictions:


base,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error,All
true,,,,,,
constraint_error,55,15,0,2,56,128
workflow_error,21,16,2,2,81,122
All,76,31,2,4,137,250



Pairwise confusion support: 77


,true,base,count
0,constraint_error,workflow_error,56
1,workflow_error,constraint_error,21


In [ ]:
cw_errors = pd.DataFrame({
    "transition": transition_arr,
    "true": y,
    "prediction": gw_then_cw,
})

cw_errors = cw_errors[
    (cw_errors["true"] == "constraint_error")
    & (cw_errors["prediction"] == "workflow_error")
]

display(
    cw_errors["transition"]
    .value_counts()
    .rename("constraint_to_workflow")
    .to_frame()
)

,constraint_to_workflow
transition,
TOOL_CALL->ASSISTANT,56
ASSISTANT->ASSISTANT,38
ASSISTANT->TOOL_CALL,22
TOOL_CALL->TOOL_CALL,22
NO_HISTORY->ASSISTANT,5


In [ ]:
# ============================================================
# TOOL_CALL -> ASSISTANT
# constraint_error vs workflow_error specialist
#
# OOF probabilities using existing outer folds.
# Positive class = constraint_error
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
import numpy as np
import pandas as pd


transition_target = "TOOL_CALL->ASSISTANT"

pair_mask = (
    (transition_arr == transition_target)
    & np.isin(
        y,
        ["constraint_error", "workflow_error"],
    )
)

pair_idx = np.where(pair_mask)[0]

print("Pair support:", len(pair_idx))
print(
    pd.Series(y[pair_idx])
    .value_counts()
)


# ------------------------------------------------------------
# Features
#
# Start with current + previous tool embedding.
# For TOOL_CALL -> ASSISTANT this is structurally meaningful:
# current assistant content + immediately preceding tool state.
# ------------------------------------------------------------

X_pair = np.hstack([
    current_E,
    previous_tool_E,
])

print("Feature shape:", X_pair.shape)


# ------------------------------------------------------------
# Binary target
# constraint = 1
# workflow   = 0
# ------------------------------------------------------------

y_pair = (
    y == "constraint_error"
).astype(int)


cw_tca_oof_prob = np.full(
    len(y),
    np.nan,
    dtype=float,
)


# ------------------------------------------------------------
# Cross-fit using the SAME outer folds
# ------------------------------------------------------------

fold_rows = []

for fold in sorted(np.unique(outer_fold_arr)):

    train_mask = (
        pair_mask
        & (outer_fold_arr != fold)
    )

    test_mask = (
        pair_mask
        & (outer_fold_arr == fold)
    )

    print(
        f"Fold {fold}:",
        "train =", train_mask.sum(),
        "| test =", test_mask.sum(),
        "| train constraint =", y_pair[train_mask].sum(),
        "| test constraint =", y_pair[test_mask].sum(),
    )

    model = LogisticRegression(
        C=1.0,
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
    )

    model.fit(
        X_pair[train_mask],
        y_pair[train_mask],
    )

    cw_tca_oof_prob[test_mask] = (
        model.predict_proba(
            X_pair[test_mask]
        )[:, 1]
    )


valid = pair_mask & ~np.isnan(cw_tca_oof_prob)

pr_auc = average_precision_score(
    y_pair[valid],
    cw_tca_oof_prob[valid],
)

roc_auc = roc_auc_score(
    y_pair[valid],
    cw_tca_oof_prob[valid],
)

prevalence = y_pair[valid].mean()

print("\nOOF PR-AUC:", pr_auc)
print("OOF ROC-AUC:", roc_auc)
print("Prevalence:", prevalence)
print("PR lift:", pr_auc / prevalence)

Pair support: 250
constraint_error    128
workflow_error      122
Name: count, dtype: int64
Feature shape: (1489, 768)
Fold 1: train = 191 | test = 59 | train constraint = 99 | test constraint = 29
Fold 2: train = 199 | test = 51 | train constraint = 101 | test constraint = 27
Fold 3: train = 217 | test = 33 | train constraint = 107 | test constraint = 21
Fold 4: train = 202 | test = 48 | train constraint = 104 | test constraint = 24
Fold 5: train = 191 | test = 59 | train constraint = 101 | test constraint = 27

OOF PR-AUC: 0.681368479990236
OOF ROC-AUC: 0.7090804303278689
Prevalence: 0.512
PR lift: 1.3307978124809297


In [ ]:
# ============================================================
# Intervention diagnostic:
# workflow prediction -> constraint
# ============================================================

base_pred = gw_then_cw.copy()

candidate = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    & (base_pred == "workflow_error")
    & ~np.isnan(cw_tca_oof_prob)
)

print("Candidate support:", candidate.sum())


diag = pd.DataFrame({
    "true": y[candidate],
    "base": base_pred[candidate],
    "constraint_prob": cw_tca_oof_prob[candidate],
})

diag["effect"] = np.where(
    diag["true"] == "constraint_error",
    "rescue",
    np.where(
        diag["true"] == "workflow_error",
        "break",
        "other",
    ),
)

print("\nEffect counts:")
display(
    diag["effect"]
    .value_counts()
    .to_frame("count")
)


print("\nProbability distributions:")

display(
    diag[
        diag["effect"].isin(["rescue", "break"])
    ]
    .groupby("effect")["constraint_prob"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)


print("\nQuantiles:")

display(
    diag[
        diag["effect"].isin(["rescue", "break"])
    ]
    .groupby("effect")["constraint_prob"]
    .quantile([.10, .25, .50, .75, .90])
    .unstack()
    .round(4)
)

Candidate support: 137

Effect counts:


,count
effect,
break,81
rescue,56



Probability distributions:


,count,mean,median,std,min,max
effect,,,,,,
break,81,0.3949,0.3833,0.1485,0.1420,0.7557
rescue,56,0.4702,0.4836,0.1315,0.1442,0.7376



Quantiles:


,0.10,0.25,0.50,0.75,0.90
effect,,,,,
break,0.2339,0.2895,0.3833,0.4839,0.614
rescue,0.3233,0.3705,0.4836,0.5768,0.630


In [ ]:
# ============================================================
# Reverse direction:
# constraint prediction -> workflow
#
# workflow score = 1 - constraint probability
# ============================================================

reverse_candidate = (
    (transition_arr == "TOOL_CALL->ASSISTANT")
    & (base_pred == "constraint_error")
    & ~np.isnan(cw_tca_oof_prob)
)

reverse_diag = pd.DataFrame({
    "true": y[reverse_candidate],
    "workflow_prob":
        1.0 - cw_tca_oof_prob[reverse_candidate],
})

reverse_diag["effect"] = np.where(
    reverse_diag["true"] == "workflow_error",
    "rescue",
    np.where(
        reverse_diag["true"] == "constraint_error",
        "break",
        "other",
    ),
)

display(
    reverse_diag[
        reverse_diag["effect"].isin(["rescue", "break"])
    ]
    .groupby("effect")["workflow_prob"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

display(
    reverse_diag[
        reverse_diag["effect"].isin(["rescue", "break"])
    ]
    .groupby("effect")["workflow_prob"]
    .quantile([.10, .25, .50, .75, .90])
    .unstack()
    .round(4)
)

,count,mean,median,std,min,max
effect,,,,,,
break,55,0.2997,0.3021,0.0900,0.1447,0.4900
rescue,21,0.3657,0.3466,0.1481,0.1429,0.6521


,0.10,0.25,0.50,0.75,0.90
effect,,,,,
break,0.1796,0.2301,0.3021,0.3538,0.4263
rescue,0.1862,0.2579,0.3466,0.4645,0.5723


In [ ]:
# ============================================================
# TCA constraint/workflow specialist:
# threshold sweep in BOTH directions
#
# Positive specialist probability:
#     cw_tca_oof_prob = P(constraint_error)
#
# Base:
#     gw_then_cw
# ============================================================

base_pred = gw_then_cw.copy()
y_arr = np.asarray(y)

rows = []

# Fine enough grid to see the useful tail.
threshold_grid = np.arange(0.30, 0.801, 0.025)

directions = [
    "workflow_to_constraint",
    "constraint_to_workflow",
]

for direction in directions:

    for threshold in threshold_grid:

        trial = base_pred.copy()

        if direction == "workflow_to_constraint":

            use = (
                (transition_arr == "TOOL_CALL->ASSISTANT")
                & (base_pred == "workflow_error")
                & ~np.isnan(cw_tca_oof_prob)
                & (cw_tca_oof_prob >= threshold)
            )

            trial[use] = "constraint_error"

        else:

            # P(workflow) = 1 - P(constraint)
            workflow_prob = 1.0 - cw_tca_oof_prob

            use = (
                (transition_arr == "TOOL_CALL->ASSISTANT")
                & (base_pred == "constraint_error")
                & ~np.isnan(cw_tca_oof_prob)
                & (workflow_prob >= threshold)
            )

            trial[use] = "workflow_error"

        old_correct = base_pred == y_arr
        new_correct = trial == y_arr

        changed = trial != base_pred

        rescues = (
            changed
            & ~old_correct
            & new_correct
        )

        breaks = (
            changed
            & old_correct
            & ~new_correct
        )

        wrong_to_wrong = (
            changed
            & ~old_correct
            & ~new_correct
        )

        n_changed = int(changed.sum())
        n_rescues = int(rescues.sum())
        n_breaks = int(breaks.sum())

        metrics = multiclass_metrics(
            y_arr,
            trial,
        )

        rows.append({
            "direction": direction,
            "threshold": threshold,
            "changed": n_changed,
            "rescues": n_rescues,
            "breaks": n_breaks,
            "wrong_to_wrong":
                int(wrong_to_wrong.sum()),
            "net": n_rescues - n_breaks,
            "precision": (
                n_rescues / n_changed
                if n_changed
                else np.nan
            ),
            **metrics,
        })


tca_sweep_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Show best results separately by direction
# ------------------------------------------------------------

for direction in directions:

    print("\n" + "=" * 80)
    print(direction)

    display(
        tca_sweep_df[
            tca_sweep_df["direction"] == direction
        ]
        .sort_values(
            ["net", "precision", "macro_f1"],
            ascending=[False, False, False],
        )
        .head(15)
        .round(4)
    )


workflow_to_constraint


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
9,workflow_to_constraint,0.525,37,22,15,0,7,0.5946,0.5494,0.4892,0.5057,0.5426
8,workflow_to_constraint,0.500,39,23,16,0,7,0.5897,0.5494,0.4896,0.5059,0.5427
7,workflow_to_constraint,0.475,51,29,22,0,7,0.5686,0.5494,0.4915,0.5072,0.5434
6,workflow_to_constraint,0.450,61,33,28,0,5,0.5410,0.5480,0.4922,0.5074,0.5426
11,workflow_to_constraint,0.575,27,15,12,0,3,0.5556,0.5467,0.4857,0.5028,0.5394
5,workflow_to_constraint,0.425,65,34,31,0,3,0.5231,0.5467,0.4920,0.5069,0.5415
10,workflow_to_constraint,0.550,32,17,15,0,2,0.5312,0.5460,0.4861,0.5029,0.5391
1,workflow_to_constraint,0.325,96,48,48,0,0,0.5000,0.5447,0.4956,0.5084,0.5407
2,workflow_to_constraint,0.350,88,44,44,0,0,0.5000,0.5447,0.4943,0.5077,0.5405
3,workflow_to_constraint,0.375,82,41,41,0,0,0.5000,0.5447,0.4933,0.5072,0.5403



constraint_to_workflow


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
29,constraint_to_workflow,0.500,3,3,0,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
30,constraint_to_workflow,0.525,3,3,0,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
31,constraint_to_workflow,0.550,3,3,0,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
28,constraint_to_workflow,0.475,7,5,2,0,3,0.7143,0.5467,0.4801,0.4983,0.5367
27,constraint_to_workflow,0.450,11,7,4,0,3,0.6364,0.5467,0.4795,0.4978,0.5363
26,constraint_to_workflow,0.425,15,9,6,0,3,0.6000,0.5467,0.4788,0.4972,0.5359
32,constraint_to_workflow,0.575,2,2,0,0,2,1.0000,0.5460,0.4805,0.4986,0.5365
33,constraint_to_workflow,0.600,1,1,0,0,1,1.0000,0.5453,0.4802,0.4983,0.5359
34,constraint_to_workflow,0.625,1,1,0,0,1,1.0000,0.5453,0.4802,0.4983,0.5359
35,constraint_to_workflow,0.650,1,1,0,0,1,1.0000,0.5453,0.4802,0.4983,0.5359


In [ ]:
positive_tca = (
    tca_sweep_df[
        tca_sweep_df["net"] > 0
    ]
    .sort_values(
        [
            "direction",
            "net",
            "precision",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)

display(
    positive_tca[
        [
            "direction",
            "threshold",
            "changed",
            "rescues",
            "breaks",
            "net",
            "precision",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
        ]
    ].round(4)
)

,direction,threshold,changed,rescues,breaks,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
29,constraint_to_workflow,0.500,3,3,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
30,constraint_to_workflow,0.525,3,3,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
31,constraint_to_workflow,0.550,3,3,0,3,1.0000,0.5467,0.4808,0.4989,0.5370
28,constraint_to_workflow,0.475,7,5,2,3,0.7143,0.5467,0.4801,0.4983,0.5367
27,constraint_to_workflow,0.450,11,7,4,3,0.6364,0.5467,0.4795,0.4978,0.5363
26,constraint_to_workflow,0.425,15,9,6,3,0.6000,0.5467,0.4788,0.4972,0.5359
32,constraint_to_workflow,0.575,2,2,0,2,1.0000,0.5460,0.4805,0.4986,0.5365
33,constraint_to_workflow,0.600,1,1,0,1,1.0000,0.5453,0.4802,0.4983,0.5359
34,constraint_to_workflow,0.625,1,1,0,1,1.0000,0.5453,0.4802,0.4983,0.5359
35,constraint_to_workflow,0.650,1,1,0,1,1.0000,0.5453,0.4802,0.4983,0.5359


In [ ]:
# ============================================================
# Nested threshold selection:
# TOOL_CALL->ASSISTANT
# workflow_error -> constraint_error
#
# Base router: gw_then_cw
# Specialist probability:
#     cw_tca_oof_prob = P(constraint_error)
# ============================================================

y_arr = np.asarray(y)
base_pred = np.asarray(gw_then_cw).copy()
prob_arr = np.asarray(cw_tca_oof_prob, dtype=float)

# Candidate threshold region.
# Include the whole plausible region rather than just the
# three thresholds that looked best globally.
threshold_grid = np.arange(0.40, 0.651, 0.025)

nested_tca_pred = base_pred.copy()

rows = []

for fold in sorted(pd.unique(outer_fold_arr)):

    train_mask = outer_fold_arr != fold
    test_mask = outer_fold_arr == fold

    # --------------------------------------------------------
    # Select threshold using TRAIN folds only
    # --------------------------------------------------------

    candidates = []

    for threshold in threshold_grid:

        eligible_train = (
            train_mask
            & (transition_arr == "TOOL_CALL->ASSISTANT")
            & (base_pred == "workflow_error")
            & ~np.isnan(prob_arr)
            & (prob_arr >= threshold)
        )

        trial_train = base_pred.copy()
        trial_train[eligible_train] = "constraint_error"

        old_correct = base_pred[train_mask] == y_arr[train_mask]
        new_correct = trial_train[train_mask] == y_arr[train_mask]

        changed = (
            trial_train[train_mask]
            != base_pred[train_mask]
        )

        rescues = changed & ~old_correct & new_correct
        breaks = changed & old_correct & ~new_correct

        accepted = int(changed.sum())
        n_rescues = int(rescues.sum())
        n_breaks = int(breaks.sum())

        net = n_rescues - n_breaks

        precision = (
            n_rescues / accepted
            if accepted
            else np.nan
        )

        candidates.append({
            "threshold": threshold,
            "accepted": accepted,
            "rescues": n_rescues,
            "breaks": n_breaks,
            "net": net,
            "precision": precision,
        })

    candidate_df = pd.DataFrame(candidates)

    # --------------------------------------------------------
    # Threshold-selection rule
    #
    # Require a non-trivial number of interventions.
    # Then maximize:
    #   1. net rescues
    #   2. precision
    #   3. higher threshold (conservative tie-break)
    # --------------------------------------------------------

    viable = candidate_df[
        candidate_df["accepted"] >= 10
    ].copy()

    if len(viable) == 0:

        selected_threshold = np.nan

    else:

        best = (
            viable
            .sort_values(
                [
                    "net",
                    "precision",
                    "threshold",
                ],
                ascending=[
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

        # Don't deploy a threshold that failed to help
        # on the training folds.
        if best["net"] <= 0:
            selected_threshold = np.nan
        else:
            selected_threshold = float(
                best["threshold"]
            )

    # --------------------------------------------------------
    # Apply selected threshold to held-out fold
    # --------------------------------------------------------

    if np.isnan(selected_threshold):

        test_use = np.zeros(
            len(y_arr),
            dtype=bool,
        )

    else:

        test_use = (
            test_mask
            & (transition_arr == "TOOL_CALL->ASSISTANT")
            & (base_pred == "workflow_error")
            & ~np.isnan(prob_arr)
            & (prob_arr >= selected_threshold)
        )

    nested_tca_pred[test_use] = "constraint_error"

    # --------------------------------------------------------
    # Train diagnostics at selected threshold
    # --------------------------------------------------------

    if np.isnan(selected_threshold):

        train_accepted = 0
        train_rescues = 0
        train_breaks = 0
        train_net = 0
        train_precision = np.nan

    else:

        selected_row = candidate_df[
            np.isclose(
                candidate_df["threshold"],
                selected_threshold,
            )
        ].iloc[0]

        train_accepted = int(
            selected_row["accepted"]
        )
        train_rescues = int(
            selected_row["rescues"]
        )
        train_breaks = int(
            selected_row["breaks"]
        )
        train_net = int(
            selected_row["net"]
        )
        train_precision = float(
            selected_row["precision"]
        )

    # --------------------------------------------------------
    # Test diagnostics
    # --------------------------------------------------------

    old_correct_test = (
        base_pred[test_mask]
        == y_arr[test_mask]
    )

    new_correct_test = (
        nested_tca_pred[test_mask]
        == y_arr[test_mask]
    )

    changed_test = (
        nested_tca_pred[test_mask]
        != base_pred[test_mask]
    )

    rescues_test = (
        changed_test
        & ~old_correct_test
        & new_correct_test
    )

    breaks_test = (
        changed_test
        & old_correct_test
        & ~new_correct_test
    )

    wrong_test = (
        changed_test
        & ~old_correct_test
        & ~new_correct_test
    )

    test_accepted = int(changed_test.sum())
    test_rescues = int(rescues_test.sum())
    test_breaks = int(breaks_test.sum())

    rows.append({
        "fold": fold,
        "selected_threshold":
            selected_threshold,

        "train_accepted":
            train_accepted,
        "train_rescues":
            train_rescues,
        "train_breaks":
            train_breaks,
        "train_net":
            train_net,
        "train_precision":
            train_precision,

        "test_accepted":
            test_accepted,
        "test_rescues":
            test_rescues,
        "test_breaks":
            test_breaks,
        "test_wrong_to_wrong":
            int(wrong_test.sum()),
        "test_net":
            test_rescues - test_breaks,
        "test_precision": (
            test_rescues / test_accepted
            if test_accepted
            else np.nan
        ),
    })


nested_tca_df = pd.DataFrame(rows)

display(
    nested_tca_df.round(4)
)

# ============================================================
# Overall intervention diagnostics
# ============================================================

changed = nested_tca_pred != base_pred

old_correct = base_pred == y_arr
new_correct = nested_tca_pred == y_arr

rescues = (
    changed
    & ~old_correct
    & new_correct
)

breaks = (
    changed
    & old_correct
    & ~new_correct
)

wrong_to_wrong = (
    changed
    & ~old_correct
    & ~new_correct
)

print(
    "\nNested accepted:",
    int(changed.sum()),
)

print(
    "Nested rescues:",
    int(rescues.sum()),
)

print(
    "Nested breaks:",
    int(breaks.sum()),
)

print(
    "Nested wrong-to-wrong:",
    int(wrong_to_wrong.sum()),
)

print(
    "Nested net:",
    int(rescues.sum() - breaks.sum()),
)

print(
    "Nested intervention precision:",
    (
        rescues.sum() / changed.sum()
        if changed.sum()
        else np.nan
    ),
)

# ============================================================
# Full metrics
# ============================================================

comparison = []

for name, pred in [
    ("gw_then_cw", base_pred),
    ("gw_cw_plus_nested_tca", nested_tca_pred),
]:

    comparison.append({
        "model": name,
        **multiclass_metrics(
            y_arr,
            pred,
        ),
    })

display(
    pd.DataFrame(comparison).round(4)
)

print(
    "\nSelected thresholds:",
    nested_tca_df[
        "selected_threshold"
    ].tolist()
)

print(
    "Mean threshold:",
    np.nanmean(
        nested_tca_df[
            "selected_threshold"
        ]
    ),
)

print(
    "Std threshold:",
    np.nanstd(
        nested_tca_df[
            "selected_threshold"
        ]
    ),
)

,fold,selected_threshold,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net,test_precision
0,1,NaN,0,0,0,0,NaN,0,0,0,0,0,NaN
1,2,0.475,43,26,17,9,0.6047,8,3,5,0,-2,0.3750
2,3,0.475,44,24,20,4,0.5455,7,5,2,0,3,0.7143
3,4,0.525,34,20,14,6,0.5882,3,2,1,0,1,0.6667
4,5,0.475,41,27,14,13,0.6585,10,2,8,0,-6,0.2000



Nested accepted: 28
Nested rescues: 12
Nested breaks: 16
Nested wrong-to-wrong: 0
Nested net: -4
Nested intervention precision: 0.42857142857142855


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,gw_then_cw,0.5447,0.4799,0.4979,0.5354
1,gw_cw_plus_nested_tca,0.5420,0.4826,0.4997,0.5350



Selected thresholds: [nan, 0.4750000000000001, 0.4750000000000001, 0.5250000000000001, 0.4750000000000001]
Mean threshold: 0.4875000000000001
Std threshold: 0.021650635094610987


In [ ]:
# ============================================================
# Residual error map AFTER the accepted GW + CW router
# ============================================================

y_arr = np.asarray(y)
pred_arr = np.asarray(gw_then_cw)

residual = pd.DataFrame({
    "true_family": y_arr,
    "prediction": pred_arr,
    "transition": transition_arr,
})

residual["correct"] = (
    residual["true_family"]
    == residual["prediction"]
)

errors = residual.loc[
    ~residual["correct"]
].copy()

print("Total support:", len(residual))
print("Correct:", residual["correct"].sum())
print("Residual errors:", len(errors))
print(
    "Accuracy:",
    residual["correct"].mean(),
)

# ------------------------------------------------------------
# 1. Remaining family-level confusions
# ------------------------------------------------------------

family_confusions = (
    errors
    .groupby(
        ["true_family", "prediction"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("TOP RESIDUAL FAMILY CONFUSIONS")

display(
    family_confusions.head(30)
)

# ------------------------------------------------------------
# 2. Remaining transition-specific confusions
# ------------------------------------------------------------

transition_confusions = (
    errors
    .groupby(
        [
            "transition",
            "true_family",
            "prediction",
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("TOP RESIDUAL TRANSITION × CONFUSION CELLS")

display(
    transition_confusions.head(50)
)

# ------------------------------------------------------------
# 3. Aggregate each unordered family pair
#
# This tells us where there is enough two-way confusion
# to potentially justify a pairwise specialist.
# ------------------------------------------------------------

errors["family_a"] = errors[
    ["true_family", "prediction"]
].min(axis=1)

errors["family_b"] = errors[
    ["true_family", "prediction"]
].max(axis=1)

pair_support = (
    errors
    .groupby(
        ["family_a", "family_b"]
    )
    .size()
    .reset_index(name="error_support")
    .sort_values(
        "error_support",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("RESIDUAL ERROR SUPPORT BY FAMILY PAIR")

display(
    pair_support.head(20)
)

# ------------------------------------------------------------
# 4. Same thing, but transition-specific
# ------------------------------------------------------------

transition_pair_support = (
    errors
    .groupby(
        [
            "transition",
            "family_a",
            "family_b",
        ]
    )
    .size()
    .reset_index(name="error_support")
    .sort_values(
        "error_support",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("RESIDUAL ERROR SUPPORT BY TRANSITION × FAMILY PAIR")

display(
    transition_pair_support.head(40)
)

# ------------------------------------------------------------
# 5. Full support for each candidate cell
#
# Error count alone can be misleading.
# Calculate total examples belonging to either family
# for the same transition.
# ------------------------------------------------------------

rows = []

for _, r in transition_pair_support.iterrows():

    transition = r["transition"]
    family_a = r["family_a"]
    family_b = r["family_b"]

    mask = (
        (residual["transition"] == transition)
        & residual["true_family"].isin(
            [family_a, family_b]
        )
    )

    support = int(mask.sum())

    error_support = int(
        r["error_support"]
    )

    rows.append({
        "transition": transition,
        "family_a": family_a,
        "family_b": family_b,
        "support": support,
        "error_support": error_support,
        "error_rate": (
            error_support / support
            if support
            else np.nan
        ),
    })

opportunity_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "error_support",
            "error_rate",
        ],
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("CANDIDATE SPECIALIST OPPORTUNITIES")

display(
    opportunity_df.head(40).round(4)
)

# ------------------------------------------------------------
# 6. Conservative shortlist
#
# Require:
#   >= 20 relevant examples
#   >= 8 residual errors
#
# These aren't magic thresholds; they just prevent us from
# chasing tiny cells.
# ------------------------------------------------------------

shortlist = opportunity_df[
    (opportunity_df["support"] >= 20)
    & (opportunity_df["error_support"] >= 8)
].copy()

print("\n" + "=" * 80)
print("SHORTLIST")

display(
    shortlist.head(25).round(4)
)

Total support: 1489
Correct: 811
Residual errors: 678
Accuracy: 0.544660846205507

TOP RESIDUAL FAMILY CONFUSIONS


,true_family,prediction,count
3,constraint_error,workflow_error,143
7,grounding_state_error,workflow_error,98
15,tool_use_error,workflow_error,84
16,workflow_error,constraint_error,81
19,workflow_error,tool_use_error,62
4,grounding_state_error,constraint_error,53
17,workflow_error,grounding_state_error,45
0,constraint_error,grounding_state_error,30
12,tool_use_error,constraint_error,25
2,constraint_error,tool_use_error,12



TOP RESIDUAL TRANSITION × CONFUSION CELLS


,transition,true_family,prediction,count
41,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,56
68,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,54
45,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,39
2,ASSISTANT->ASSISTANT,constraint_error,workflow_error,38
11,ASSISTANT->ASSISTANT,workflow_error,constraint_error,28
42,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,27
65,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,25
22,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,24
51,TOOL_CALL->ASSISTANT,tool_use_error,workflow_error,23
16,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,22



RESIDUAL ERROR SUPPORT BY FAMILY PAIR


...

,transition,family_a,family_b,error_support
44,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,79
32,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,77
3,ASSISTANT->ASSISTANT,constraint_error,workflow_error,66
35,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,55
29,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,42
12,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,41
42,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,38
40,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,35
6,ASSISTANT->ASSISTANT,grounding_state_error,workflow_error,29
16,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,29



CANDIDATE SPECIALIST OPPORTUNITIES


,transition,family_a,family_b,support,error_support,error_rate
0,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,477,79,0.1656
1,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,250,77,0.3080
2,ASSISTANT->ASSISTANT,constraint_error,workflow_error,165,66,0.4000
3,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,219,55,0.2511
4,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,225,42,0.1867
5,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,154,41,0.2662
6,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,408,38,0.0931
7,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,381,35,0.0919
8,ASSISTANT->ASSISTANT,grounding_state_error,workflow_error,128,29,0.2266
9,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,133,29,0.2180



SHORTLIST


,transition,family_a,family_b,support,error_support,error_rate
0,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,477,79,0.1656
1,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,250,77,0.3080
2,ASSISTANT->ASSISTANT,constraint_error,workflow_error,165,66,0.4000
3,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,219,55,0.2511
4,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,225,42,0.1867
5,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,154,41,0.2662
6,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,408,38,0.0931
7,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,381,35,0.0919
8,ASSISTANT->ASSISTANT,grounding_state_error,workflow_error,128,29,0.2266
9,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,133,29,0.2180


In [ ]:
# ============================================================
# ASSISTANT->ASSISTANT
# constraint_error vs workflow_error
#
# Diagnostic only:
#   Can a pairwise classifier separate this boundary OOF?
#
# No routing / threshold selection yet.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TRANSITION = "ASSISTANT->ASSISTANT"

FAMILY_A = "constraint_error"
FAMILY_B = "workflow_error"

# Positive class = constraint_error
POSITIVE_FAMILY = FAMILY_A


# ------------------------------------------------------------
# Base arrays
# ------------------------------------------------------------

y_arr = np.asarray(y)
base_pred = np.asarray(gw_then_cw)
fold_arr = np.asarray(outer_fold_arr)

# Pairwise semantic representation.
#
# Use current + previous-assistant embeddings because this
# transition explicitly contains an assistant-history state.
#
# E_current and E_last_assistant were already present in
# your notebook.
# ------------------------------------------------------------

X_pair = np.hstack([
    np.asarray(E_current),
    np.asarray(E_last_assistant),
])

print("Feature shape:", X_pair.shape)


# ------------------------------------------------------------
# Relevant examples
# ------------------------------------------------------------

pair_mask = (
    (transition_arr == TRANSITION)
    & np.isin(
        y_arr,
        [FAMILY_A, FAMILY_B],
    )
)

pair_idx = np.where(pair_mask)[0]

y_pair = (
    y_arr[pair_idx] == POSITIVE_FAMILY
).astype(int)

print("Pair support:", len(pair_idx))

print(
    pd.Series(y_arr[pair_idx])
    .value_counts()
)


# ------------------------------------------------------------
# OOF predictions using EXISTING outer folds
# ------------------------------------------------------------

oof_prob = np.full(
    len(y_arr),
    np.nan,
    dtype=float,
)

fold_rows = []

for fold in sorted(pd.unique(fold_arr)):

    train_idx = np.where(
        pair_mask
        & (fold_arr != fold)
    )[0]

    test_idx = np.where(
        pair_mask
        & (fold_arr == fold)
    )[0]

    y_train = (
        y_arr[train_idx]
        == POSITIVE_FAMILY
    ).astype(int)

    y_test = (
        y_arr[test_idx]
        == POSITIVE_FAMILY
    ).astype(int)

    print(
        f"Fold {fold}:",
        "train =", len(train_idx),
        "| test =", len(test_idx),
        "| train constraint =", y_train.sum(),
        "| test constraint =", y_test.sum(),
    )

    # Guard against pathological fold
    if (
        len(train_idx) == 0
        or len(test_idx) == 0
        or len(np.unique(y_train)) < 2
    ):
        continue

    clf = LogisticRegression(
        C=1.0,
        max_iter=3000,
        class_weight="balanced",
    )

    clf.fit(
        X_pair[train_idx],
        y_train,
    )

    oof_prob[test_idx] = (
        clf.predict_proba(
            X_pair[test_idx]
        )[:, 1]
    )

    fold_rows.append({
        "fold": fold,
        "train_support": len(train_idx),
        "test_support": len(test_idx),
        "train_constraint": int(
            y_train.sum()
        ),
        "test_constraint": int(
            y_test.sum()
        ),
        "test_workflow": int(
            len(y_test) - y_test.sum()
        ),
    })


fold_df = pd.DataFrame(fold_rows)

display(fold_df)


# ------------------------------------------------------------
# OOF quality
# ------------------------------------------------------------

valid = (
    pair_mask
    & ~np.isnan(oof_prob)
)

y_valid = (
    y_arr[valid]
    == POSITIVE_FAMILY
).astype(int)

p_valid = oof_prob[valid]

prevalence = y_valid.mean()

pr_auc = average_precision_score(
    y_valid,
    p_valid,
)

roc_auc = roc_auc_score(
    y_valid,
    p_valid,
)

print()
print("OOF PR-AUC:", pr_auc)
print("OOF ROC-AUC:", roc_auc)
print("Prevalence:", prevalence)
print(
    "PR lift:",
    pr_auc / prevalence,
)


# ------------------------------------------------------------
# Candidate interventions relative to CURRENT router
#
# Separate the two directions because they can behave
# completely differently.
# ------------------------------------------------------------

candidate = (
    valid
    & np.isin(
        base_pred,
        [FAMILY_A, FAMILY_B],
    )
)

rows = []

for idx in np.where(candidate)[0]:

    true_family = y_arr[idx]
    old_pred = base_pred[idx]

    if old_pred == FAMILY_B:

        # workflow -> constraint
        direction = (
            "workflow_error -> constraint_error"
        )

        # Higher probability means more evidence
        # for constraint.
        score = oof_prob[idx]

        proposed = FAMILY_A

    else:

        # constraint -> workflow
        direction = (
            "constraint_error -> workflow_error"
        )

        # Higher score means more evidence for workflow.
        score = 1.0 - oof_prob[idx]

        proposed = FAMILY_B

    old_correct = (
        old_pred == true_family
    )

    new_correct = (
        proposed == true_family
    )

    if (
        not old_correct
        and new_correct
    ):
        effect = "rescue"

    elif (
        old_correct
        and not new_correct
    ):
        effect = "break"

    else:
        effect = "other"

    rows.append({
        "index": idx,
        "true_family": true_family,
        "base_prediction": old_pred,
        "proposed_prediction": proposed,
        "direction": direction,
        "score": score,
        "effect": effect,
    })


aa_cw_effect_df = pd.DataFrame(rows)

print("\nEffect counts:")

display(
    pd.crosstab(
        aa_cw_effect_df["direction"],
        aa_cw_effect_df["effect"],
    )
)


# ------------------------------------------------------------
# Score distributions: rescue vs break
# ------------------------------------------------------------

distribution = (
    aa_cw_effect_df[
        aa_cw_effect_df["effect"].isin(
            ["rescue", "break"]
        )
    ]
    .groupby(
        ["direction", "effect"]
    )["score"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
)

print(
    "\nScore distributions:"
)

display(
    distribution.round(4)
)


# ------------------------------------------------------------
# Quantiles
# ------------------------------------------------------------

quantiles = (
    aa_cw_effect_df[
        aa_cw_effect_df["effect"].isin(
            ["rescue", "break"]
        )
    ]
    .groupby(
        ["direction", "effect"]
    )["score"]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
)

print("\nQuantiles:")

display(
    quantiles.round(4)
)

Feature shape: (1489, 768)
Pair support: 165
workflow_error      86
constraint_error    79
Name: count, dtype: int64
Fold 1: train = 146 | test = 19 | train constraint = 70 | test constraint = 9
Fold 2: train = 124 | test = 41 | train constraint = 64 | test constraint = 15
Fold 3: train = 123 | test = 42 | train constraint = 53 | test constraint = 26
Fold 4: train = 127 | test = 38 | train constraint = 63 | test constraint = 16
Fold 5: train = 140 | test = 25 | train constraint = 66 | test constraint = 13


,fold,train_support,test_support,train_constraint,test_constraint,test_workflow
0,1,146,19,70,9,10
1,2,124,41,64,15,26
2,3,123,42,53,26,16
3,4,127,38,63,16,22
4,5,140,25,66,13,12



OOF PR-AUC: 0.4675264407431741
OOF ROC-AUC: 0.5223726817780394
Prevalence: 0.47878787878787876
PR lift: 0.9764792749699207

Effect counts:


effect,break,rescue
direction,,
constraint_error -> workflow_error,30,28
workflow_error -> constraint_error,46,38



Score distributions:


count    mean  median     std     min     max
direction                          effect                                               
constraint_error -> workflow_error break      30  0.4302  0.4337  0.1088  0.2228  0.6835
                                   rescue     28  0.4184  0.3958  0.1428  0.2124  0.6991
workflow_error -> constraint_error break      46  0.4239  0.4105  0.1515  0.1664  0.7855
                                   rescue     38  0.4281  0.4231  0.1224  0.2465  0.6956


Quantiles:


0.10    0.25    0.50    0.75    0.90
direction                          effect                                        
constraint_error -> workflow_error break   0.3041  0.3386  0.4337  0.4830  0.5427
                                   rescue  0.2778  0.3024  0.3958  0.5269  0.6320
workflow_error -> constraint_error break   0.2412  0.3327  0.4105  0.4800  0.6463
                                   rescue  0.2618  0.3426  0.4231  0.5185  0.5872

In [ ]:
# ============================================================
# TOOL_CALL->ASSISTANT
# constraint_error vs grounding_state_error
#
# Diagnostic only — no threshold selection yet.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import pandas as pd


TRANSITION = "TOOL_CALL->ASSISTANT"

FAMILY_A = "constraint_error"
FAMILY_B = "grounding_state_error"

# Positive = grounding.
# Conceptually:
#   "Does the assistant response fail to ground itself in the
#    immediately preceding tool result?"
POSITIVE_FAMILY = FAMILY_B


# ------------------------------------------------------------
# Representation
#
# Current assistant message + immediately preceding tool.
# ------------------------------------------------------------

X_pair = np.hstack([
    np.asarray(E_current),
    np.asarray(E_last_tool),
])

print("Feature shape:", X_pair.shape)


y_arr = np.asarray(y)
base_pred = np.asarray(gw_then_cw)
fold_arr = np.asarray(outer_fold_arr)


# ------------------------------------------------------------
# Pair support
# ------------------------------------------------------------

pair_mask = (
    (transition_arr == TRANSITION)
    & np.isin(
        y_arr,
        [FAMILY_A, FAMILY_B],
    )
)

pair_idx = np.where(pair_mask)[0]

y_pair = (
    y_arr[pair_idx] == POSITIVE_FAMILY
).astype(int)

print("Pair support:", len(pair_idx))

print(
    pd.Series(y_arr[pair_idx])
    .value_counts()
)


# ------------------------------------------------------------
# OOF classifier
# ------------------------------------------------------------

oof_prob = np.full(
    len(y_arr),
    np.nan,
    dtype=float,
)

fold_rows = []

for fold in sorted(pd.unique(fold_arr)):

    train_idx = np.where(
        pair_mask
        & (fold_arr != fold)
    )[0]

    test_idx = np.where(
        pair_mask
        & (fold_arr == fold)
    )[0]

    y_train = (
        y_arr[train_idx] == POSITIVE_FAMILY
    ).astype(int)

    y_test = (
        y_arr[test_idx] == POSITIVE_FAMILY
    ).astype(int)

    print(
        f"Fold {fold}:",
        "train =", len(train_idx),
        "| test =", len(test_idx),
        "| train grounding =", y_train.sum(),
        "| test grounding =", y_test.sum(),
    )

    if (
        len(train_idx) == 0
        or len(test_idx) == 0
        or len(np.unique(y_train)) < 2
    ):
        continue

    clf = LogisticRegression(
        C=1.0,
        max_iter=3000,
        class_weight="balanced",
    )

    clf.fit(
        X_pair[train_idx],
        y_train,
    )

    oof_prob[test_idx] = (
        clf.predict_proba(
            X_pair[test_idx]
        )[:, 1]
    )

    fold_rows.append({
        "fold": fold,
        "train_support": len(train_idx),
        "test_support": len(test_idx),
        "train_grounding": int(y_train.sum()),
        "test_grounding": int(y_test.sum()),
        "test_constraint": int(
            len(y_test) - y_test.sum()
        ),
    })


display(pd.DataFrame(fold_rows))


# ------------------------------------------------------------
# OOF discrimination
# ------------------------------------------------------------

valid = (
    pair_mask
    & ~np.isnan(oof_prob)
)

y_valid = (
    y_arr[valid] == POSITIVE_FAMILY
).astype(int)

p_valid = oof_prob[valid]

prevalence = y_valid.mean()

pr_auc = average_precision_score(
    y_valid,
    p_valid,
)

roc_auc = roc_auc_score(
    y_valid,
    p_valid,
)

print()
print("OOF PR-AUC:", pr_auc)
print("OOF ROC-AUC:", roc_auc)
print("Prevalence:", prevalence)
print("PR lift:", pr_auc / prevalence)


# ------------------------------------------------------------
# Evaluate potential interventions relative to current router
# ------------------------------------------------------------

candidate = (
    valid
    & np.isin(
        base_pred,
        [FAMILY_A, FAMILY_B],
    )
)

rows = []

for idx in np.where(candidate)[0]:

    true_family = y_arr[idx]
    old_pred = base_pred[idx]

    if old_pred == FAMILY_A:

        # constraint -> grounding
        direction = (
            "constraint_error -> grounding_state_error"
        )

        score = oof_prob[idx]
        proposed = FAMILY_B

    else:

        # grounding -> constraint
        direction = (
            "grounding_state_error -> constraint_error"
        )

        score = 1.0 - oof_prob[idx]
        proposed = FAMILY_A

    old_correct = (
        old_pred == true_family
    )

    new_correct = (
        proposed == true_family
    )

    if not old_correct and new_correct:
        effect = "rescue"

    elif old_correct and not new_correct:
        effect = "break"

    else:
        effect = "other"

    rows.append({
        "index": idx,
        "true_family": true_family,
        "base_prediction": old_pred,
        "proposed_prediction": proposed,
        "direction": direction,
        "score": score,
        "effect": effect,
    })


tcg_effect_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# Effect counts
# ------------------------------------------------------------

print("\nEffect counts:")

display(
    pd.crosstab(
        tcg_effect_df["direction"],
        tcg_effect_df["effect"],
    )
)


# ------------------------------------------------------------
# Rescue / break score distributions
# ------------------------------------------------------------

effect_subset = tcg_effect_df[
    tcg_effect_df["effect"].isin(
        ["rescue", "break"]
    )
]

distribution = (
    effect_subset
    .groupby(
        ["direction", "effect"]
    )["score"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
)

print("\nScore distributions:")

display(
    distribution.round(4)
)


# ------------------------------------------------------------
# Quantiles
# ------------------------------------------------------------

quantiles = (
    effect_subset
    .groupby(
        ["direction", "effect"]
    )["score"]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
)

print("\nQuantiles:")

display(
    quantiles.round(4)
)

Feature shape: (1489, 768)
Pair support: 225
constraint_error         128
grounding_state_error     97
Name: count, dtype: int64
Fold 1: train = 178 | test = 47 | train grounding = 79 | test grounding = 18
Fold 2: train = 180 | test = 45 | train grounding = 79 | test grounding = 18
Fold 3: train = 183 | test = 42 | train grounding = 76 | test grounding = 21
Fold 4: train = 178 | test = 47 | train grounding = 74 | test grounding = 23
Fold 5: train = 181 | test = 44 | train grounding = 80 | test grounding = 17


,fold,train_support,test_support,train_grounding,test_grounding,test_constraint
0,1,178,47,79,18,29
1,2,180,45,79,18,27
2,3,183,42,76,21,21
3,4,178,47,74,23,24
4,5,181,44,80,17,27



OOF PR-AUC: 0.6380515142661531
OOF ROC-AUC: 0.726159793814433
Prevalence: 0.4311111111111111
PR lift: 1.4800163990709738

Effect counts:


effect,break,rescue
direction,,
constraint_error -> grounding_state_error,55,27
grounding_state_error -> constraint_error,26,15



Score distributions:


count    mean  median     std     min     max
direction                                 effect                                               
constraint_error -> grounding_state_error break      55  0.3354  0.3012  0.1319  0.1412  0.6569
                                          rescue     27  0.4160  0.4063  0.1591  0.1303  0.7556
grounding_state_error -> constraint_error break      26  0.2489  0.2376  0.0788  0.1147  0.4922
                                          rescue     15  0.3804  0.3086  0.2196  0.0927  0.7110


Quantiles:


0.10    0.25    0.50    0.75    0.90
direction                                 effect                                        
constraint_error -> grounding_state_error break   0.1872  0.2381  0.3012  0.4025  0.5283
                                          rescue  0.2486  0.3058  0.4063  0.4920  0.6428
grounding_state_error -> constraint_error break   0.1734  0.2034  0.2376  0.2882  0.3194
                                          rescue  0.1526  0.1799  0.3086  0.5795  0.6726

In [ ]:
# ============================================================
# TOOL_CALL->ASSISTANT
# constraint <-> grounding
#
# Direction-specific OOF threshold sweep
# Baseline: gw_then_cw
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import numpy as np
import pandas as pd


base_pred = np.asarray(gw_then_cw).copy()
true_arr = np.asarray(y)

FAMILY_C = "constraint_error"
FAMILY_G = "grounding_state_error"


def evaluate_prediction(pred):
    return {
        "accuracy": accuracy_score(true_arr, pred),
        "balanced_accuracy": balanced_accuracy_score(
            true_arr, pred
        ),
        "macro_f1": f1_score(
            true_arr,
            pred,
            average="macro",
        ),
        "weighted_f1": f1_score(
            true_arr,
            pred,
            average="weighted",
        ),
    }


def sweep_direction(
    direction,
    thresholds,
):
    rows = []

    if direction == "constraint_to_grounding":

        direction_mask = (
            candidate
            & (base_pred == FAMILY_C)
        )

        score = oof_prob
        target = FAMILY_G

    elif direction == "grounding_to_constraint":

        direction_mask = (
            candidate
            & (base_pred == FAMILY_G)
        )

        # Probability of constraint.
        score = 1.0 - oof_prob
        target = FAMILY_C

    else:
        raise ValueError(direction)

    for threshold in thresholds:

        accept = (
            direction_mask
            & (score >= threshold)
        )

        pred = base_pred.copy()
        pred[accept] = target

        changed_idx = np.where(accept)[0]

        rescues = np.sum(
            (base_pred[changed_idx] != true_arr[changed_idx])
            & (pred[changed_idx] == true_arr[changed_idx])
        )

        breaks = np.sum(
            (base_pred[changed_idx] == true_arr[changed_idx])
            & (pred[changed_idx] != true_arr[changed_idx])
        )

        wrong_to_wrong = np.sum(
            (base_pred[changed_idx] != true_arr[changed_idx])
            & (pred[changed_idx] != true_arr[changed_idx])
        )

        changed = len(changed_idx)
        net = rescues - breaks

        precision = (
            rescues / changed
            if changed
            else np.nan
        )

        metrics = evaluate_prediction(pred)

        rows.append({
            "direction": direction,
            "threshold": threshold,
            "changed": changed,
            "rescues": int(rescues),
            "breaks": int(breaks),
            "wrong_to_wrong": int(wrong_to_wrong),
            "net": int(net),
            "precision": precision,
            **metrics,
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Sweep ranges
#
# Use reasonably fine resolution because support is not huge.
# ------------------------------------------------------------

cg_thresholds = np.arange(
    0.30,
    0.81,
    0.025,
)

gc_thresholds = np.arange(
    0.25,
    0.81,
    0.025,
)


cg_sweep = sweep_direction(
    "constraint_to_grounding",
    cg_thresholds,
)

gc_sweep = sweep_direction(
    "grounding_to_constraint",
    gc_thresholds,
)


# ------------------------------------------------------------
# Display ranked results
# ------------------------------------------------------------

for name, df in [
    ("constraint_to_grounding", cg_sweep),
    ("grounding_to_constraint", gc_sweep),
]:

    print("\n" + "=" * 80)
    print(name)

    display(
        df.sort_values(
            [
                "net",
                "precision",
                "macro_f1",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .head(15)
        .round(4)
    )


constraint_to_grounding


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
14,constraint_to_grounding,0.650,4,3,1,0,2,0.7500,0.5460,0.4817,0.5000,0.5370
13,constraint_to_grounding,0.625,6,4,2,0,2,0.6667,0.5460,0.4819,0.5001,0.5371
12,constraint_to_grounding,0.600,8,5,3,0,2,0.6250,0.5460,0.4821,0.5003,0.5371
11,constraint_to_grounding,0.575,10,6,4,0,2,0.6000,0.5460,0.4823,0.5005,0.5372
15,constraint_to_grounding,0.675,1,1,0,0,1,1.0000,0.5453,0.4807,0.4988,0.5361
16,constraint_to_grounding,0.700,1,1,0,0,1,1.0000,0.5453,0.4807,0.4988,0.5361
17,constraint_to_grounding,0.725,1,1,0,0,1,1.0000,0.5453,0.4807,0.4988,0.5361
18,constraint_to_grounding,0.750,1,1,0,0,1,1.0000,0.5453,0.4807,0.4988,0.5361
10,constraint_to_grounding,0.550,11,6,5,0,1,0.5455,0.5453,0.4817,0.4998,0.5364
4,constraint_to_grounding,0.400,28,14,14,0,0,0.5000,0.5447,0.4825,0.5001,0.5358



grounding_to_constraint


,direction,threshold,changed,rescues,breaks,wrong_to_wrong,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
6,grounding_to_constraint,0.400,8,7,1,0,6,0.8750,0.5487,0.4835,0.5019,0.5395
10,grounding_to_constraint,0.500,5,5,0,0,5,1.0000,0.5480,0.4830,0.5014,0.5389
11,grounding_to_constraint,0.525,5,5,0,0,5,1.0000,0.5480,0.4830,0.5014,0.5389
12,grounding_to_constraint,0.550,5,5,0,0,5,1.0000,0.5480,0.4830,0.5014,0.5389
7,grounding_to_constraint,0.425,7,6,1,0,5,0.8571,0.5480,0.4829,0.5012,0.5388
8,grounding_to_constraint,0.450,7,6,1,0,5,0.8571,0.5480,0.4829,0.5012,0.5388
9,grounding_to_constraint,0.475,7,6,1,0,5,0.8571,0.5480,0.4829,0.5012,0.5388
4,grounding_to_constraint,0.350,9,7,2,0,5,0.7778,0.5480,0.4827,0.5010,0.5387
5,grounding_to_constraint,0.375,9,7,2,0,5,0.7778,0.5480,0.4827,0.5010,0.5387
13,grounding_to_constraint,0.575,4,4,0,0,4,1.0000,0.5473,0.4824,0.5007,0.5382


In [ ]:
# ============================================================
# NESTED THRESHOLD SELECTION
#
# TOOL_CALL->ASSISTANT
# grounding_state_error -> constraint_error
#
# Baseline: gw_then_cw
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


true_arr = np.asarray(y)
base_pred = np.asarray(gw_then_cw).copy()
fold_arr = np.asarray(outer_fold_arr)

FAMILY_G = "grounding_state_error"
FAMILY_C = "constraint_error"

TRANSITION = "TOOL_CALL->ASSISTANT"


# ------------------------------------------------------------
# Candidate rows
#
# Only rows where the CURRENT router predicts grounding.
# Specialist score = probability of constraint.
# ------------------------------------------------------------

direction_mask = (
    (np.asarray(transition_arr) == TRANSITION)
    & (base_pred == FAMILY_G)
    & ~np.isnan(oof_prob)
)

constraint_score = 1.0 - np.asarray(oof_prob)


# ------------------------------------------------------------
# Threshold grid
# ------------------------------------------------------------

thresholds = np.arange(
    0.30,
    0.701,
    0.025,
)


# ------------------------------------------------------------
# Selection constraints
#
# Avoid selecting thresholds from tiny lucky samples.
# ------------------------------------------------------------

MIN_TRAIN_ACCEPTED = 3
MIN_TRAIN_PRECISION = 0.60


def intervention_stats(mask, baseline):
    idx = np.where(mask)[0]

    if len(idx) == 0:
        return {
            "accepted": 0,
            "rescues": 0,
            "breaks": 0,
            "wrong_to_wrong": 0,
            "net": 0,
            "precision": np.nan,
        }

    proposed = baseline.copy()
    proposed[idx] = FAMILY_C

    old_correct = (
        baseline[idx] == true_arr[idx]
    )

    new_correct = (
        proposed[idx] == true_arr[idx]
    )

    rescues = int(
        np.sum(
            (~old_correct) & new_correct
        )
    )

    breaks = int(
        np.sum(
            old_correct & (~new_correct)
        )
    )

    wrong_to_wrong = int(
        np.sum(
            (~old_correct) & (~new_correct)
        )
    )

    accepted = len(idx)

    return {
        "accepted": accepted,
        "rescues": rescues,
        "breaks": breaks,
        "wrong_to_wrong": wrong_to_wrong,
        "net": rescues - breaks,
        "precision": (
            rescues / accepted
            if accepted
            else np.nan
        ),
    }


# ------------------------------------------------------------
# Nested threshold selection
# ------------------------------------------------------------

nested_pred = base_pred.copy()

fold_rows = []


for fold in sorted(pd.unique(fold_arr)):

    train_region = (
        direction_mask
        & (fold_arr != fold)
    )

    test_region = (
        direction_mask
        & (fold_arr == fold)
    )

    threshold_rows = []

    # ----------------------------------------
    # Select threshold using TRAIN folds only
    # ----------------------------------------

    for threshold in thresholds:

        train_accept = (
            train_region
            & (constraint_score >= threshold)
        )

        stats = intervention_stats(
            train_accept,
            base_pred,
        )

        threshold_rows.append({
            "threshold": threshold,
            **stats,
        })


    threshold_df = pd.DataFrame(
        threshold_rows
    )


    # ----------------------------------------
    # Eligibility
    # ----------------------------------------

    eligible = threshold_df[
        (threshold_df["accepted"] >= MIN_TRAIN_ACCEPTED)
        & (
            threshold_df["precision"]
            >= MIN_TRAIN_PRECISION
        )
        & (threshold_df["net"] > 0)
    ].copy()


    if len(eligible) == 0:

        selected_threshold = np.nan

        train_stats = {
            "accepted": 0,
            "rescues": 0,
            "breaks": 0,
            "wrong_to_wrong": 0,
            "net": 0,
            "precision": np.nan,
        }

        test_stats = {
            "accepted": 0,
            "rescues": 0,
            "breaks": 0,
            "wrong_to_wrong": 0,
            "net": 0,
            "precision": np.nan,
        }

    else:

        # ----------------------------------------
        # Primary objective: maximum net rescue.
        #
        # Tie-break:
        #   1. precision
        #   2. fewer interventions
        #   3. higher threshold
        # ----------------------------------------

        eligible = eligible.sort_values(
            [
                "net",
                "precision",
                "accepted",
                "threshold",
            ],
            ascending=[
                False,
                False,
                True,
                False,
            ],
        )

        best = eligible.iloc[0]

        selected_threshold = float(
            best["threshold"]
        )

        train_accept = (
            train_region
            & (
                constraint_score
                >= selected_threshold
            )
        )

        test_accept = (
            test_region
            & (
                constraint_score
                >= selected_threshold
            )
        )

        train_stats = intervention_stats(
            train_accept,
            base_pred,
        )

        test_stats = intervention_stats(
            test_accept,
            base_pred,
        )

        # Apply ONLY to held-out fold.
        nested_pred[test_accept] = FAMILY_C


    fold_rows.append({
        "fold": fold,
        "selected_threshold": selected_threshold,

        "train_accepted":
            train_stats["accepted"],
        "train_rescues":
            train_stats["rescues"],
        "train_breaks":
            train_stats["breaks"],
        "train_net":
            train_stats["net"],
        "train_precision":
            train_stats["precision"],

        "test_accepted":
            test_stats["accepted"],
        "test_rescues":
            test_stats["rescues"],
        "test_breaks":
            test_stats["breaks"],
        "test_wrong_to_wrong":
            test_stats["wrong_to_wrong"],
        "test_net":
            test_stats["net"],
        "test_precision":
            test_stats["precision"],
    })


nested_gc_df = pd.DataFrame(
    fold_rows
)

display(
    nested_gc_df.round(4)
)


# ------------------------------------------------------------
# Aggregate intervention statistics
# ------------------------------------------------------------

changed = (
    nested_pred != base_pred
)

changed_idx = np.where(changed)[0]

rescues = int(np.sum(
    (base_pred[changed_idx] != true_arr[changed_idx])
    & (nested_pred[changed_idx] == true_arr[changed_idx])
))

breaks = int(np.sum(
    (base_pred[changed_idx] == true_arr[changed_idx])
    & (nested_pred[changed_idx] != true_arr[changed_idx])
))

wrong_to_wrong = int(np.sum(
    (base_pred[changed_idx] != true_arr[changed_idx])
    & (nested_pred[changed_idx] != true_arr[changed_idx])
))

net = rescues - breaks

precision = (
    rescues / len(changed_idx)
    if len(changed_idx)
    else np.nan
)


print()
print("Nested accepted:", len(changed_idx))
print("Nested rescues:", rescues)
print("Nested breaks:", breaks)
print(
    "Nested wrong-to-wrong:",
    wrong_to_wrong,
)
print("Nested net:", net)
print(
    "Nested intervention precision:",
    precision,
)


# ------------------------------------------------------------
# Overall model metrics
# ------------------------------------------------------------

def metrics(pred):
    return {
        "accuracy":
            accuracy_score(true_arr, pred),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                pred,
            ),

        "macro_f1":
            f1_score(
                true_arr,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                true_arr,
                pred,
                average="weighted",
            ),
    }


comparison = pd.DataFrame([
    {
        "model": "gw_then_cw",
        **metrics(base_pred),
    },
    {
        "model":
            "gw_cw_plus_nested_gc",
        **metrics(nested_pred),
    },
])

display(
    comparison.round(4)
)


# ------------------------------------------------------------
# Threshold stability
# ------------------------------------------------------------

print("\nSelected thresholds:")

display(
    nested_gc_df[
        [
            "fold",
            "selected_threshold",
            "train_net",
            "train_precision",
            "test_accepted",
            "test_rescues",
            "test_breaks",
            "test_net",
            "test_precision",
        ]
    ].round(4)
)

selected = (
    nested_gc_df[
        "selected_threshold"
    ]
    .dropna()
    .to_numpy()
)

if len(selected):

    print(
        "\nMean threshold:",
        selected.mean(),
    )

    print(
        "Std threshold:",
        selected.std(),
    )

,fold,selected_threshold,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_wrong_to_wrong,test_net,test_precision
0,1,0.40,5,4,1,3,0.800,3,3,0,0,3,1.0
1,2,0.65,3,3,0,3,1.000,0,0,0,0,0,NaN
2,3,0.40,8,7,1,6,0.875,0,0,0,0,0,NaN
3,4,0.40,7,7,0,7,1.000,1,0,1,0,-1,0.0
4,5,0.55,5,5,0,5,1.000,0,0,0,0,0,NaN



Nested accepted: 4
Nested rescues: 3
Nested breaks: 1
Nested wrong-to-wrong: 0
Nested net: 2
Nested intervention precision: 0.75


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,gw_then_cw,0.5447,0.4799,0.4979,0.5354
1,gw_cw_plus_nested_gc,0.5460,0.4810,0.4991,0.5367



Selected thresholds:


,fold,selected_threshold,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_net,test_precision
0,1,0.40,3,0.800,3,3,0,3,1.0
1,2,0.65,3,1.000,0,0,0,0,NaN
2,3,0.40,6,0.875,0,0,0,0,NaN
3,4,0.40,7,1.000,1,0,1,-1,0.0
4,5,0.55,5,1.000,0,0,0,0,NaN



Mean threshold: 0.48000000000000026
Std threshold: 0.10295630140987012


In [ ]:
# ============================================================
# FIND EXISTING PREDICTION VARIABLES
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

true_arr = np.asarray(y)
n = len(true_arr)

print("Target length:", n)
print("\nCandidate prediction variables:\n")

candidates = []

for name, obj in list(globals().items()):
    if name.startswith("_") or name in {"y", "true_arr"}:
        continue

    try:
        arr = np.asarray(obj)

        if arr.ndim != 1 or len(arr) != n:
            continue

        # Only consider things that look like class predictions
        unique = np.unique(arr)

        if len(unique) > 20:
            continue

        acc = accuracy_score(true_arr, arr)

        candidates.append({
            "name": name,
            "accuracy": acc,
            "n_unique": len(unique),
            "unique": unique.tolist()
        })

    except Exception:
        pass

results = pd.DataFrame(candidates)

if len(results):
    results["distance_to_expected"] = results["accuracy"].apply(
        lambda x: min(
            abs(x - 0.5393),   # original hard router
            abs(x - 0.5447),   # GW + CW
            abs(x - 0.5460),   # GW + CW + GC
        )
    )

    results = results.sort_values(
        ["distance_to_expected", "accuracy"]
    )

    display(results.head(30))
else:
    print("No candidate prediction arrays found.")

Target length: 1489

Candidate prediction variables:



,name,accuracy,n_unique,unique,distance_to_expected
58,nested_pred,0.546004,5,"[constraint_error, grounding_state_error, reas...",0.000004
30,hierarchical_pred,0.539288,5,"[constraint_error, grounding_state_error, reas...",0.000012
40,existing_router_pred,0.539288,5,"[constraint_error, grounding_state_error, reas...",0.000012
41,base_router_pred,0.539288,5,"[constraint_error, grounding_state_error, reas...",0.000012
12,combined_pred,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039
45,base_pred,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039
47,gw_then_cw,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039
48,cw_then_gw,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039
49,final_pairwise_pred,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039
50,new_base_pred,0.544661,5,"[constraint_error, grounding_state_error, reas...",0.000039


In [ ]:
print("Expected correct counts:")
print("baseline :", round(0.5393 * 1489))
print("GW+CW    :", round(0.5447 * 1489))
print("GC final :", round(0.5460 * 1489))

print("\nExact candidate counts:")
if len(results):
    results["correct"] = (
        results["accuracy"] * n
    ).round().astype(int)

    display(
        results[
            results["correct"].between(800, 820)
        ][["name", "accuracy", "correct", "n_unique"]]
        .sort_values("correct")
    )

Expected correct counts:
baseline : 803
GW+CW    : 811
GC final : 813

Exact candidate counts:


,name,accuracy,correct,n_unique
4,trajectory_pred,0.537945,801,5
34,reconciled_pred,0.538617,802,5
30,hierarchical_pred,0.539288,803,5
40,existing_router_pred,0.539288,803,5
41,base_router_pred,0.539288,803,5
46,nested_cw_pred,0.541303,806,5
44,cw_pair_pred,0.541303,806,5
55,nested_tca_pred,0.541974,807,5
11,pred,0.541974,807,5
42,nested_gw_pred,0.542646,808,5


In [ ]:
FINAL_ROUTER_PRED = np.asarray(nested_pred).copy()
BASE_ROUTER_PRED = np.asarray(existing_router_pred).copy()
GW_CW_ROUTER_PRED = np.asarray(gw_then_cw).copy()

assert (FINAL_ROUTER_PRED == nested_pred).all()
assert (BASE_ROUTER_PRED == existing_router_pred).all()

print(
    "Base accuracy:",
    np.mean(BASE_ROUTER_PRED == true_arr)
)

print(
    "GW+CW accuracy:",
    np.mean(GW_CW_ROUTER_PRED == true_arr)
)

print(
    "Final accuracy:",
    np.mean(FINAL_ROUTER_PRED == true_arr)
)

Base accuracy: 0.539288112827401
GW+CW accuracy: 0.544660846205507
Final accuracy: 0.5460040295500336


In [ ]:
# ============================================================
# CONSOLIDATED ROUTER AUDIT
#
# Current system:
#   existing hard router
#       + nested GW
#       + nested CW
#       + nested GC
#
# Goal:
#   1. quantify cumulative gain
#   2. inspect per-family effects
#   3. inspect what errors remain
#   4. identify whether another specialist is justified
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)


true_arr = np.asarray(y)

base_router = np.asarray(existing_router_pred).copy()
gw_cw_pred = np.asarray(gw_then_cw).copy()
final_pred = np.asarray(nested_pred).copy()


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(true_arr) == len(base_router)
assert len(true_arr) == len(gw_cw_pred)
assert len(true_arr) == len(final_pred)

print("Total support:", len(true_arr))

print(
    "Changes vs existing hard router:",
    np.sum(final_pred != base_router),
)

print(
    "GC incremental changes:",
    np.sum(final_pred != gw_cw_pred),
)


# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

def model_metrics(name, pred):

    return {
        "model": name,

        "accuracy":
            accuracy_score(
                true_arr,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                pred,
            ),

        "macro_f1":
            f1_score(
                true_arr,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                true_arr,
                pred,
                average="weighted",
            ),
    }


comparison = pd.DataFrame([
    model_metrics(
        "existing_hard_router",
        base_router,
    ),
    model_metrics(
        "nested_gw_plus_cw",
        gw_cw_pred,
    ),
    model_metrics(
        "nested_gw_plus_cw_plus_gc",
        final_pred,
    ),
])

print("\n" + "=" * 80)
print("OVERALL METRICS")

display(
    comparison.round(4)
)


# ------------------------------------------------------------
# Cumulative intervention accounting
#
# Compare final system directly against original hard router.
# ------------------------------------------------------------

changed = (
    final_pred != base_router
)

idx = np.where(changed)[0]

old_correct = (
    base_router[idx] == true_arr[idx]
)

new_correct = (
    final_pred[idx] == true_arr[idx]
)

rescues = int(np.sum(
    (~old_correct) & new_correct
))

breaks = int(np.sum(
    old_correct & (~new_correct)
))

wrong_to_wrong = int(np.sum(
    (~old_correct) & (~new_correct)
))

net = rescues - breaks

precision = (
    rescues / len(idx)
    if len(idx)
    else np.nan
)

print("\n" + "=" * 80)
print("CUMULATIVE INTERVENTION EFFECT")

print("Changed:", len(idx))
print("Rescues:", rescues)
print("Breaks:", breaks)
print(
    "Wrong-to-wrong:",
    wrong_to_wrong,
)
print("Net:", net)
print(
    "Intervention precision:",
    precision,
)


# ------------------------------------------------------------
# Per-family metrics
# ------------------------------------------------------------

labels = sorted(
    pd.unique(true_arr)
)

base_p, base_r, base_f, base_s = (
    precision_recall_fscore_support(
        true_arr,
        base_router,
        labels=labels,
        zero_division=0,
    )
)

new_p, new_r, new_f, new_s = (
    precision_recall_fscore_support(
        true_arr,
        final_pred,
        labels=labels,
        zero_division=0,
    )
)


family_df = pd.DataFrame({
    "family": labels,
    "support": new_s,

    "base_precision": base_p,
    "new_precision": new_p,

    "base_recall": base_r,
    "new_recall": new_r,

    "base_f1": base_f,
    "new_f1": new_f,
})

family_df["delta_precision"] = (
    family_df["new_precision"]
    - family_df["base_precision"]
)

family_df["delta_recall"] = (
    family_df["new_recall"]
    - family_df["base_recall"]
)

family_df["delta_f1"] = (
    family_df["new_f1"]
    - family_df["base_f1"]
)


print("\n" + "=" * 80)
print("PER-FAMILY EFFECT")

display(
    family_df.round(4)
)


# ------------------------------------------------------------
# Exact cumulative intervention types
# ------------------------------------------------------------

intervention_df = pd.DataFrame({
    "true_family": true_arr[idx],
    "base_prediction": base_router[idx],
    "new_prediction": final_pred[idx],
})

intervention_df["effect"] = np.where(
    (
        intervention_df["base_prediction"]
        != intervention_df["true_family"]
    )
    & (
        intervention_df["new_prediction"]
        == intervention_df["true_family"]
    ),
    "rescue",
    np.where(
        (
            intervention_df["base_prediction"]
            == intervention_df["true_family"]
        )
        & (
            intervention_df["new_prediction"]
            != intervention_df["true_family"]
        ),
        "break",
        "wrong_to_wrong",
    )
)


effect_table = (
    intervention_df
    .groupby([
        "true_family",
        "base_prediction",
        "new_prediction",
        "effect",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
)


print("\n" + "=" * 80)
print("INTERVENTION TYPES")

display(effect_table)


# ------------------------------------------------------------
# Final residual confusion matrix
# ------------------------------------------------------------

residual_mask = (
    final_pred != true_arr
)

residual_df = pd.DataFrame({
    "true_family":
        true_arr[residual_mask],

    "prediction":
        final_pred[residual_mask],

    "transition":
        np.asarray(
            transition_arr
        )[residual_mask],
})


print("\n" + "=" * 80)
print(
    "FINAL RESIDUAL ERRORS:",
    residual_mask.sum(),
)


family_confusions = (
    residual_df
    .groupby([
        "true_family",
        "prediction",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
)


print("\nTOP RESIDUAL FAMILY CONFUSIONS")

display(
    family_confusions.head(20)
)


# ------------------------------------------------------------
# Transition × confusion residuals
# ------------------------------------------------------------

transition_confusions = (
    residual_df
    .groupby([
        "transition",
        "true_family",
        "prediction",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
)


print(
    "\nTOP RESIDUAL "
    "TRANSITION × CONFUSION CELLS"
)

display(
    transition_confusions.head(40)
)


# ------------------------------------------------------------
# Residual support by unordered family pair
# ------------------------------------------------------------

tmp = residual_df.copy()

tmp["family_a"] = tmp[
    ["true_family", "prediction"]
].min(axis=1)

tmp["family_b"] = tmp[
    ["true_family", "prediction"]
].max(axis=1)


pair_residual = (
    tmp
    .groupby([
        "family_a",
        "family_b",
    ])
    .size()
    .reset_index(
        name="error_support"
    )
    .sort_values(
        "error_support",
        ascending=False,
    )
)


print(
    "\nRESIDUAL ERROR SUPPORT "
    "BY FAMILY PAIR"
)

display(
    pair_residual.head(20)
)


# ------------------------------------------------------------
# Transition × unordered pair
# ------------------------------------------------------------

transition_pair_residual = (
    tmp
    .groupby([
        "transition",
        "family_a",
        "family_b",
    ])
    .size()
    .reset_index(
        name="error_support"
    )
    .sort_values(
        "error_support",
        ascending=False,
    )
)


print(
    "\nRESIDUAL ERROR SUPPORT BY "
    "TRANSITION × FAMILY PAIR"
)

display(
    transition_pair_residual.head(30)
)

Total support: 1489
Changes vs existing hard router: 30
GC incremental changes: 4

OVERALL METRICS


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,existing_hard_router,0.5393,0.4745,0.4918,0.5292
1,nested_gw_plus_cw,0.5447,0.4799,0.4979,0.5354
2,nested_gw_plus_cw_plus_gc,0.5460,0.4810,0.4991,0.5367



CUMULATIVE INTERVENTION EFFECT
Changed: 30
Rescues: 20
Breaks: 10
Wrong-to-wrong: 0
Net: 10
Intervention precision: 0.6666666666666666

PER-FAMILY EFFECT


,family,support,base_precision,new_precision,base_recall,new_recall,base_f1,new_f1,delta_precision,delta_recall,delta_f1
0,constraint_error,317,0.4387,0.4452,0.4290,0.4227,0.4338,0.4337,0.0065,-0.0063,-0.0002
1,grounding_state_error,244,0.4710,0.4939,0.2992,0.3320,0.3659,0.3971,0.0229,0.0328,0.0311
2,reasoning_value_error,31,0.5385,0.5385,0.4516,0.4516,0.4912,0.4912,0.0000,0.0000,0.0000
3,tool_use_error,237,0.5829,0.5829,0.4895,0.4895,0.5321,0.5321,0.0000,0.0000,0.0000
4,workflow_error,660,0.5807,0.5857,0.7030,0.7091,0.6361,0.6415,0.0050,0.0061,0.0055



INTERVENTION TYPES


...

,transition,true_family,prediction,count
41,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,56
68,TOOL_CALL->TOOL_CALL,workflow_error,tool_use_error,54
45,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,39
2,ASSISTANT->ASSISTANT,constraint_error,workflow_error,38
42,TOOL_CALL->ASSISTANT,grounding_state_error,constraint_error,28
11,ASSISTANT->ASSISTANT,workflow_error,constraint_error,28
65,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,25
22,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,24
51,TOOL_CALL->ASSISTANT,tool_use_error,workflow_error,23
16,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,22



RESIDUAL ERROR SUPPORT BY FAMILY PAIR


,family_a,family_b,error_support
3,constraint_error,workflow_error,224
9,tool_use_error,workflow_error,146
6,grounding_state_error,workflow_error,143
0,constraint_error,grounding_state_error,81
2,constraint_error,tool_use_error,37
5,grounding_state_error,tool_use_error,16
8,reasoning_value_error,workflow_error,10
1,constraint_error,reasoning_value_error,8
4,grounding_state_error,reasoning_value_error,6
7,reasoning_value_error,tool_use_error,5



RESIDUAL ERROR SUPPORT BY TRANSITION × FAMILY PAIR


,transition,family_a,family_b,error_support
44,TOOL_CALL->TOOL_CALL,tool_use_error,workflow_error,79
32,TOOL_CALL->ASSISTANT,constraint_error,workflow_error,77
3,ASSISTANT->ASSISTANT,constraint_error,workflow_error,66
35,TOOL_CALL->ASSISTANT,grounding_state_error,workflow_error,55
12,ASSISTANT->TOOL_CALL,constraint_error,workflow_error,41
29,TOOL_CALL->ASSISTANT,constraint_error,grounding_state_error,40
42,TOOL_CALL->TOOL_CALL,grounding_state_error,workflow_error,38
40,TOOL_CALL->TOOL_CALL,constraint_error,workflow_error,35
6,ASSISTANT->ASSISTANT,grounding_state_error,workflow_error,29
16,ASSISTANT->TOOL_CALL,tool_use_error,workflow_error,29


In [ ]:
# ============================================================
# TRANSITION-CONDITIONED CALIBRATION DIAGNOSTIC
#
# Question:
# Are the remaining errors mainly due to incorrectly calibrated
# semantic probabilities across transition types?
#
# OOF only. No specialist threshold tuning.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


# ------------------------------------------------------------
# Stable state
# ------------------------------------------------------------

true_arr = np.asarray(y)
fold_arr = np.asarray(outer_fold_arr)

BASE_ROUTER_PRED = np.asarray(
    existing_router_pred
).copy()

FINAL_ROUTER_PRED = np.asarray(
    nested_pred
).copy()


# ------------------------------------------------------------
# Semantic probability matrix
#
# Recovered mapping:
#
# 0 workflow_error
# 1 constraint_error
# 2 tool_use_error
# 3 grounding_state_error
# 4 reasoning_value_error
# ------------------------------------------------------------

P_sem = np.asarray(
    remove_t1_prob,
    dtype=float,
)

prob_class_order = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

assert P_sem.shape == (len(true_arr), 5)


# ------------------------------------------------------------
# Structural dataframe
# ------------------------------------------------------------

cal_df = pd.DataFrame({
    "transition": np.asarray(
        transition_arr
    ),

    "history_count": np.asarray(
        train_targets[
            "history_event_count"
        ]
    ),

    "has_history": np.asarray(
        train_targets[
            "has_history"
        ]
    ).astype(int),

    "event_role": np.asarray(
        train_targets[
            "event_role"
        ]
    ),
})


# Semantic probabilities as explicit columns
for j, cls in enumerate(prob_class_order):
    cal_df[f"p_{cls}"] = P_sem[:, j]


print("Calibration feature frame:")
display(cal_df.head())

print("\nShape:", cal_df.shape)

Calibration feature frame:


,transition,history_count,has_history,event_role,p_workflow_error,p_constraint_error,p_tool_use_error,p_grounding_state_error,p_reasoning_value_error
0,TOOL_CALL->ASSISTANT,2,1,ASSISTANT,0.032803,0.035806,0.007295,0.923623,0.000474
1,ASSISTANT->TOOL_CALL,1,1,TOOL_CALL,0.857731,0.060988,0.063957,0.016366,0.000959
2,TOOL_CALL->TOOL_CALL,2,1,TOOL_CALL,0.294385,0.407353,0.251713,0.043991,0.002558
3,TOOL_CALL->ASSISTANT,3,1,ASSISTANT,0.060847,0.874368,0.017621,0.042206,0.004959
4,TOOL_CALL->ASSISTANT,3,1,ASSISTANT,0.118948,0.678562,0.017623,0.180420,0.004446



Shape: (1489, 9)


In [ ]:
# ============================================================
# OOF transition-conditioned calibrator
# ============================================================

categorical_cols = [
    "transition",
    "event_role",
]

numeric_cols = [
    "history_count",
    "has_history",
] + [
    f"p_{c}"
    for c in prob_class_order
]


preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_cols,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_cols,
        ),
    ],
    remainder="drop",
)


calibrator = Pipeline([
    (
        "features",
        preprocess,
    ),
    (
        "model",
        LogisticRegression(
            C=0.5,
            max_iter=5000,
            class_weight=None,
            solver="lbfgs",
        ),
    ),
])


calibrated_oof_pred = np.empty(
    len(true_arr),
    dtype=object,
)

calibrated_oof_prob = np.zeros(
    (len(true_arr), 5),
    dtype=float,
)


fold_rows = []


for fold in sorted(
    pd.unique(fold_arr)
):

    train_idx = np.where(
        fold_arr != fold
    )[0]

    test_idx = np.where(
        fold_arr == fold
    )[0]

    model = Pipeline([
        (
            "features",
            preprocess,
        ),
        (
            "model",
            LogisticRegression(
                C=0.5,
                max_iter=5000,
                solver="lbfgs",
            ),
        ),
    ])

    model.fit(
        cal_df.iloc[train_idx],
        true_arr[train_idx],
    )

    pred = model.predict(
        cal_df.iloc[test_idx]
    )

    prob = model.predict_proba(
        cal_df.iloc[test_idx]
    )

    calibrated_oof_pred[
        test_idx
    ] = pred

    # Align model probability columns to our fixed order
    local_classes = list(
        model.named_steps[
            "model"
        ].classes_
    )

    for j, cls in enumerate(
        prob_class_order
    ):
        calibrated_oof_prob[
            test_idx,
            j
        ] = prob[
            :,
            local_classes.index(cls)
        ]

    fold_rows.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "accuracy": accuracy_score(
            true_arr[test_idx],
            pred,
        ),
    })


display(
    pd.DataFrame(
        fold_rows
    ).round(4)
)

,fold,train_n,test_n,accuracy
0,1,1191,298,0.4933
1,2,1191,298,0.5201
2,3,1191,298,0.5235
3,4,1192,297,0.5354
4,5,1191,298,0.5470


In [ ]:
# ============================================================
# Comparison
# ============================================================

def metric_row(name, pred):

    return {
        "model": name,

        "accuracy":
            accuracy_score(
                true_arr,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                pred,
            ),

        "macro_f1":
            f1_score(
                true_arr,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                true_arr,
                pred,
                average="weighted",
            ),
    }


comparison = pd.DataFrame([
    metric_row(
        "existing_hard_router",
        BASE_ROUTER_PRED,
    ),

    metric_row(
        "GW_CW_GC_router",
        FINAL_ROUTER_PRED,
    ),

    metric_row(
        "transition_calibrator",
        calibrated_oof_pred,
    ),
])


display(
    comparison.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,existing_hard_router,0.5393,0.4745,0.4918,0.5292
1,GW_CW_GC_router,0.5460,0.4810,0.4991,0.5367
2,transition_calibrator,0.5238,0.4461,0.4750,0.5111


In [ ]:
# ============================================================
# Effect relative to current best router
# ============================================================

changed = (
    calibrated_oof_pred
    != FINAL_ROUTER_PRED
)

old_correct = (
    FINAL_ROUTER_PRED
    == true_arr
)

new_correct = (
    calibrated_oof_pred
    == true_arr
)


effect = np.full(
    len(true_arr),
    "unchanged",
    dtype=object,
)

effect[
    changed
    & ~old_correct
    & new_correct
] = "rescue"

effect[
    changed
    & old_correct
    & ~new_correct
] = "break"

effect[
    changed
    & ~old_correct
    & ~new_correct
] = "wrong_to_wrong"


print(
    "Changed:",
    int(changed.sum()),
)

print(
    pd.Series(
        effect[changed]
    ).value_counts()
)


effect_df = pd.DataFrame({
    "transition":
        np.asarray(transition_arr),

    "true_family":
        true_arr,

    "old_prediction":
        FINAL_ROUTER_PRED,

    "new_prediction":
        calibrated_oof_pred,

    "effect":
        effect,
})


print(
    "\nEffect by transition:"
)

display(
    pd.crosstab(
        effect_df.loc[
            changed,
            "transition"
        ],
        effect_df.loc[
            changed,
            "effect"
        ],
    )
)


print(
    "\nMost common prediction changes:"
)

display(
    effect_df.loc[changed]
    .groupby([
        "old_prediction",
        "new_prediction",
        "effect",
    ])
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .head(30)
)

Changed: 303
break             138
rescue            105
wrong_to_wrong     60
Name: count, dtype: int64

Effect by transition:


effect,break,rescue,wrong_to_wrong
transition,,,
ASSISTANT->ASSISTANT,10,12,19
ASSISTANT->TOOL_CALL,27,16,3
NO_HISTORY->ASSISTANT,7,0,4
NO_HISTORY->TOOL_CALL,2,2,4
TOOL_CALL->ASSISTANT,24,32,18
TOOL_CALL->TOOL_CALL,68,43,12



Most common prediction changes:


,old_prediction,new_prediction,effect,count
20,tool_use_error,workflow_error,break,38
23,workflow_error,constraint_error,break,33
5,constraint_error,workflow_error,break,27
6,constraint_error,workflow_error,rescue,26
24,workflow_error,constraint_error,rescue,24
21,tool_use_error,workflow_error,rescue,21
25,workflow_error,constraint_error,wrong_to_wrong,16
27,workflow_error,grounding_state_error,rescue,14
7,constraint_error,workflow_error,wrong_to_wrong,13
11,grounding_state_error,workflow_error,break,13


In [ ]:
# ============================================================
# RESIDUAL REPRESENTATION TEST
#
# Cell:
# TOOL_CALL -> ASSISTANT
# constraint_error vs workflow_error
#
# Compare:
#   A. semantic probabilities only
#   B. embedding only
#   C. semantic probabilities + compressed embedding
#
# OOF using the EXISTING outer folds.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
)


true_arr = np.asarray(y)
fold_arr = np.asarray(outer_fold_arr)
transition_np = np.asarray(transition_arr)

P_sem = np.asarray(remove_t1_prob, dtype=float)

# Replace this ONLY if your embedding matrix has a different name.
X_emb = np.asarray(X)

print("Embedding shape:", X_emb.shape)

assert X_emb.shape[0] == len(true_arr)
assert P_sem.shape == (len(true_arr), 5)


# ------------------------------------------------------------
# Target cell
# ------------------------------------------------------------

A = "constraint_error"
B = "workflow_error"
TRANSITION = "TOOL_CALL->ASSISTANT"

cell_mask = (
    (transition_np == TRANSITION)
    & np.isin(true_arr, [A, B])
)

cell_idx = np.where(cell_mask)[0]

print("Cell support:", len(cell_idx))

print(
    pd.Series(
        true_arr[cell_idx]
    ).value_counts()
)


# Binary target:
# 1 = constraint
# 0 = workflow
binary_y = (
    true_arr[cell_idx] == A
).astype(int)

cell_folds = fold_arr[cell_idx]

Embedding shape: (1489, 387)
Cell support: 250
constraint_error    128
workflow_error      122
Name: count, dtype: int64


In [ ]:
# ============================================================
# OOF comparison
# ============================================================

results = {}
oof_scores = {}


# Semantic probabilities alone.
# Only use the relevant pair probabilities + useful context
# from the other semantic classes.
X_sem = P_sem[cell_idx]


# Full embedding for this cell.
X_embedding = X_emb[cell_idx]


for mode in [
    "semantic",
    "embedding",
    "semantic_plus_embedding",
]:

    scores = np.full(
        len(cell_idx),
        np.nan,
        dtype=float,
    )

    fold_stats = []

    for fold in sorted(
        pd.unique(cell_folds)
    ):

        tr = np.where(
            cell_folds != fold
        )[0]

        te = np.where(
            cell_folds == fold
        )[0]

        y_tr = binary_y[tr]
        y_te = binary_y[te]


        if mode == "semantic":

            model = Pipeline([
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "clf",
                    LogisticRegression(
                        C=0.5,
                        max_iter=5000,
                    ),
                ),
            ])

            model.fit(
                X_sem[tr],
                y_tr,
            )

            fold_score = model.predict_proba(
                X_sem[te]
            )[:, 1]


        elif mode == "embedding":

            model = Pipeline([
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "pca",
                    PCA(
                        n_components=0.90,
                        svd_solver="full",
                    ),
                ),
                (
                    "clf",
                    LogisticRegression(
                        C=0.1,
                        max_iter=5000,
                    ),
                ),
            ])

            model.fit(
                X_embedding[tr],
                y_tr,
            )

            fold_score = model.predict_proba(
                X_embedding[te]
            )[:, 1]


        else:

            # IMPORTANT:
            # PCA must be fit only on training data.

            scaler = StandardScaler()

            emb_tr = scaler.fit_transform(
                X_embedding[tr]
            )

            emb_te = scaler.transform(
                X_embedding[te]
            )


            pca = PCA(
                n_components=0.90,
                svd_solver="full",
            )

            emb_tr_pca = pca.fit_transform(
                emb_tr
            )

            emb_te_pca = pca.transform(
                emb_te
            )


            combined_tr = np.column_stack([
                X_sem[tr],
                emb_tr_pca,
            ])

            combined_te = np.column_stack([
                X_sem[te],
                emb_te_pca,
            ])


            final_scaler = StandardScaler()

            combined_tr = (
                final_scaler.fit_transform(
                    combined_tr
                )
            )

            combined_te = (
                final_scaler.transform(
                    combined_te
                )
            )


            clf = LogisticRegression(
                C=0.1,
                max_iter=5000,
            )

            clf.fit(
                combined_tr,
                y_tr,
            )

            fold_score = clf.predict_proba(
                combined_te
            )[:, 1]


        scores[te] = fold_score

        fold_stats.append({
            "fold": fold,
            "train_n": len(tr),
            "test_n": len(te),
            "test_constraint":
                int(y_te.sum()),
        })


    assert not np.isnan(scores).any()

    oof_scores[mode] = scores

    results[mode] = {
        "roc_auc":
            roc_auc_score(
                binary_y,
                scores,
            ),

        "pr_auc":
            average_precision_score(
                binary_y,
                scores,
            ),
    }


display(
    pd.DataFrame(results)
    .T
    .round(4)
)

,roc_auc,pr_auc
semantic,0.6100,0.6474
embedding,0.6613,0.6883
semantic_plus_embedding,0.6557,0.6814


In [ ]:
# ============================================================
# RANKING QUALITY ON CURRENT ROUTER'S RESIDUAL ERRORS
# ============================================================

current_pred = np.asarray(nested_pred)

cell_current = current_pred[cell_idx]

residual_mask = (
    cell_current != true_arr[cell_idx]
)

print(
    "Residual examples in cell:",
    int(residual_mask.sum()),
)


residual_rows = []

for mode, scores in oof_scores.items():

    if residual_mask.sum() == 0:
        continue

    y_res = binary_y[residual_mask]
    s_res = scores[residual_mask]

    # Only meaningful if both labels remain.
    if len(np.unique(y_res)) < 2:
        continue

    residual_rows.append({
        "representation": mode,

        "n": len(y_res),

        "roc_auc":
            roc_auc_score(
                y_res,
                s_res,
            ),

        "pr_auc":
            average_precision_score(
                y_res,
                s_res,
            ),
    })


display(
    pd.DataFrame(
        residual_rows
    ).round(4)
)

Residual examples in cell: 111


,representation,n,roc_auc,pr_auc
0,semantic,111,0.2195,0.4732
1,embedding,111,0.3700,0.5452
2,semantic_plus_embedding,111,0.3868,0.5753


In [ ]:
# ============================================================
# INTERVENTION RANKING TEST
#
# Does the OOF embedding score identify profitable flips,
# conditional on the current router prediction?
#
# score = P(constraint_error)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score


current_pred = np.asarray(nested_pred)
scores = np.asarray(oof_scores["embedding"])

cell_true = true_arr[cell_idx]
cell_current = current_pred[cell_idx]


rows = []

for current_class in [A, B]:

    m = cell_current == current_class

    yy = cell_true[m]
    ss = scores[m]

    # "flip is correct" means current prediction is wrong.
    flip_correct = (yy != current_class).astype(int)

    # Convert P(constraint) into evidence FOR flipping.
    #
    # If current = workflow:
    #   high P(constraint) supports flip.
    #
    # If current = constraint:
    #   low P(constraint) supports flip.
    if current_class == B:
        flip_score = ss
    else:
        flip_score = 1.0 - ss

    row = {
        "current_prediction": current_class,
        "n": len(yy),
        "currently_wrong": int(flip_correct.sum()),
        "error_rate": flip_correct.mean(),
    }

    if len(np.unique(flip_correct)) == 2:
        row["flip_roc_auc"] = roc_auc_score(
            flip_correct,
            flip_score,
        )
        row["flip_pr_auc"] = average_precision_score(
            flip_correct,
            flip_score,
        )
        row["pr_lift"] = (
            row["flip_pr_auc"] / row["error_rate"]
        )

    rows.append(row)


display(
    pd.DataFrame(rows).round(4)
)

,current_prediction,n,currently_wrong,error_rate,flip_roc_auc,flip_pr_auc,pr_lift
0,constraint_error,79,21,0.2658,0.6552,0.4944,1.8599
1,workflow_error,137,56,0.4088,0.5822,0.4847,1.1859


In [ ]:
# ============================================================
# OOF FLIP PRECISION AT DIFFERENT COVERAGES
# ============================================================

coverage_levels = [
    0.02, 0.05, 0.10, 0.15,
    0.20, 0.25, 0.30,
]

tail_rows = []

for current_class in [A, B]:

    m = cell_current == current_class

    yy = cell_true[m]
    ss = scores[m]

    flip_correct = (
        yy != current_class
    ).astype(int)

    if current_class == B:
        flip_score = ss
    else:
        flip_score = 1.0 - ss

    order = np.argsort(-flip_score)

    n = len(order)

    for coverage in coverage_levels:

        k = max(
            1,
            int(np.ceil(n * coverage)),
        )

        chosen = order[:k]

        rescues = int(
            flip_correct[chosen].sum()
        )

        breaks = k - rescues
        net = rescues - breaks

        tail_rows.append({
            "current_prediction":
                current_class,
            "coverage":
                coverage,
            "accepted":
                k,
            "rescues":
                rescues,
            "breaks":
                breaks,
            "net":
                net,
            "precision":
                rescues / k,
            "baseline_error_rate":
                flip_correct.mean(),
        })


tail_df = pd.DataFrame(tail_rows)

display(
    tail_df.round(4)
)

,current_prediction,coverage,accepted,rescues,breaks,net,precision,baseline_error_rate
0,constraint_error,0.02,2,2,0,2,1.0000,0.2658
1,constraint_error,0.05,4,3,1,2,0.7500,0.2658
2,constraint_error,0.10,8,6,2,4,0.7500,0.2658
3,constraint_error,0.15,12,7,5,2,0.5833,0.2658
4,constraint_error,0.20,16,7,9,-2,0.4375,0.2658
5,constraint_error,0.25,20,8,12,-4,0.4000,0.2658
6,constraint_error,0.30,24,8,16,-8,0.3333,0.2658
7,workflow_error,0.02,3,1,2,-1,0.3333,0.4088
8,workflow_error,0.05,7,3,4,-1,0.4286,0.4088
9,workflow_error,0.10,14,8,6,2,0.5714,0.4088


In [ ]:
# ============================================================
# NESTED OOF COVERAGE SELECTION
#
# Target:
# TOOL_CALL->ASSISTANT
# current prediction = constraint_error
#
# Proposed intervention:
# constraint_error -> workflow_error
#
# Uses EMBEDDING-ONLY OOF flip scores already computed.
# ============================================================

import numpy as np
import pandas as pd


scores = np.asarray(oof_scores["embedding"])

cell_true = true_arr[cell_idx]
cell_current = np.asarray(nested_pred)[cell_idx]
cell_folds = fold_arr[cell_idx]


# Only examples eligible for this ONE-DIRECTION intervention.
eligible = cell_current == A

yy = cell_true[eligible]
ss = scores[eligible]
ff = cell_folds[eligible]

# Since current=A=constraint, evidence for flipping to workflow
# is LOW P(constraint).
flip_score = 1.0 - ss

# 1 means flipping would rescue the example.
flip_correct = (yy == B).astype(int)


coverage_grid = np.array([
    0.02,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
])

MIN_TRAIN_ACCEPTED = 4
MIN_TRAIN_PRECISION = 0.65
MIN_TRAIN_NET = 2


nested_rows = []
accepted_global = []


for fold in sorted(pd.unique(ff)):

    tr = np.where(ff != fold)[0]
    te = np.where(ff == fold)[0]

    candidates = []

    # ------------------------------------------
    # Select coverage using TRAIN ONLY
    # ------------------------------------------
    for coverage in coverage_grid:

        k = max(
            1,
            int(np.ceil(len(tr) * coverage))
        )

        order = tr[
            np.argsort(-flip_score[tr])
        ]

        chosen = order[:k]

        rescues = int(
            flip_correct[chosen].sum()
        )
        breaks = k - rescues
        net = rescues - breaks
        precision = rescues / k

        candidates.append({
            "coverage": coverage,
            "accepted": k,
            "rescues": rescues,
            "breaks": breaks,
            "net": net,
            "precision": precision,
        })


    cand_df = pd.DataFrame(candidates)

    valid = cand_df[
        (cand_df["accepted"] >= MIN_TRAIN_ACCEPTED)
        & (cand_df["precision"] >= MIN_TRAIN_PRECISION)
        & (cand_df["net"] >= MIN_TRAIN_NET)
    ].copy()


    if len(valid) == 0:

        selected_coverage = np.nan

        nested_rows.append({
            "fold": fold,
            "selected_coverage": np.nan,
            "train_accepted": 0,
            "train_rescues": 0,
            "train_breaks": 0,
            "train_net": 0,
            "train_precision": np.nan,
            "test_accepted": 0,
            "test_rescues": 0,
            "test_breaks": 0,
            "test_net": 0,
            "test_precision": np.nan,
        })

        continue


    # Prefer maximum net.
    # Tie-break:
    #   higher precision,
    #   then LOWER coverage.
    selected = (
        valid
        .sort_values(
            ["net", "precision", "coverage"],
            ascending=[False, False, True],
        )
        .iloc[0]
    )

    selected_coverage = float(
        selected["coverage"]
    )


    # ------------------------------------------
    # Apply selected coverage to TEST fold
    # ------------------------------------------
    k_test = max(
        1,
        int(np.ceil(
            len(te) * selected_coverage
        ))
    )

    test_order = te[
        np.argsort(-flip_score[te])
    ]

    chosen_test = test_order[:k_test]

    test_rescues = int(
        flip_correct[chosen_test].sum()
    )

    test_breaks = (
        len(chosen_test) - test_rescues
    )

    test_net = (
        test_rescues - test_breaks
    )

    test_precision = (
        test_rescues / len(chosen_test)
        if len(chosen_test)
        else np.nan
    )


    # Convert back to global dataset indices.
    eligible_cell_positions = np.where(
        eligible
    )[0]

    chosen_cell_positions = (
        eligible_cell_positions[chosen_test]
    )

    chosen_global = cell_idx[
        chosen_cell_positions
    ]

    accepted_global.extend(
        chosen_global.tolist()
    )


    nested_rows.append({
        "fold": fold,
        "selected_coverage":
            selected_coverage,

        "train_accepted":
            int(selected["accepted"]),
        "train_rescues":
            int(selected["rescues"]),
        "train_breaks":
            int(selected["breaks"]),
        "train_net":
            int(selected["net"]),
        "train_precision":
            float(selected["precision"]),

        "test_accepted":
            len(chosen_test),
        "test_rescues":
            test_rescues,
        "test_breaks":
            test_breaks,
        "test_net":
            test_net,
        "test_precision":
            test_precision,
    })


nested_cov_df = pd.DataFrame(
    nested_rows
)

display(
    nested_cov_df.round(4)
)

,fold,selected_coverage,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_net,test_precision
0,1,0.050,4,4,0,4,1.0000,1,1,0,1,1.0
1,2,0.125,9,7,2,5,0.7778,2,1,1,0,0.5
2,3,0.100,7,5,2,3,0.7143,2,2,0,2,1.0
3,4,0.100,7,5,2,3,0.7143,2,2,0,2,1.0
4,5,0.150,9,7,2,5,0.7778,3,0,3,-3,0.0


In [ ]:
# ============================================================
# NESTED RESULT
# ============================================================

print(
    "Nested accepted:",
    nested_cov_df["test_accepted"].sum()
)

print(
    "Nested rescues:",
    nested_cov_df["test_rescues"].sum()
)

print(
    "Nested breaks:",
    nested_cov_df["test_breaks"].sum()
)

print(
    "Nested net:",
    nested_cov_df["test_net"].sum()
)

accepted = (
    nested_cov_df["test_accepted"].sum()
)

if accepted:
    print(
        "Nested intervention precision:",
        nested_cov_df["test_rescues"].sum()
        / accepted
    )


# Apply nested held-out interventions.
embedding_override_pred = np.asarray(
    nested_pred
).copy()

accepted_global = np.asarray(
    accepted_global,
    dtype=int,
)

embedding_override_pred[
    accepted_global
] = B


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

comparison = pd.DataFrame([
    {
        "model": "GW_CW_GC",
        "accuracy":
            accuracy_score(
                true_arr,
                nested_pred,
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                nested_pred,
            ),
        "macro_f1":
            f1_score(
                true_arr,
                nested_pred,
                average="macro",
            ),
        "weighted_f1":
            f1_score(
                true_arr,
                nested_pred,
                average="weighted",
            ),
    },
    {
        "model": "GW_CW_GC_plus_embedding_override",
        "accuracy":
            accuracy_score(
                true_arr,
                embedding_override_pred,
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                embedding_override_pred,
            ),
        "macro_f1":
            f1_score(
                true_arr,
                embedding_override_pred,
                average="macro",
            ),
        "weighted_f1":
            f1_score(
                true_arr,
                embedding_override_pred,
                average="weighted",
            ),
    },
])

display(
    comparison.round(4)
)

print(
    "\nCorrect before:",
    int(
        (np.asarray(nested_pred)
         == true_arr).sum()
    )
)

print(
    "Correct after:",
    int(
        (embedding_override_pred
         == true_arr).sum()
    )
)

Nested accepted: 10
Nested rescues: 6
Nested breaks: 4
Nested net: 2
Nested intervention precision: 0.6


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,GW_CW_GC,0.5460,0.4810,0.4991,0.5367
1,GW_CW_GC_plus_embedding_override,0.5473,0.4803,0.4987,0.5371



Correct before: 813
Correct after: 815


In [ ]:
# ============================================================
# CONSERVATIVE NESTED EMBEDDING OVERRIDE
#
# Same experiment, but:
#   coverage <= 10%
#   train precision >= 75%
#
# Goal: test robustness of ONLY the high-confidence tail.
# ============================================================

coverage_grid = np.array([
    0.02,
    0.05,
    0.075,
    0.10,
])

MIN_TRAIN_ACCEPTED = 4
MIN_TRAIN_PRECISION = 0.75
MIN_TRAIN_NET = 2

nested_rows = []
accepted_global = []

eligible_cell_positions = np.where(eligible)[0]


for fold in sorted(pd.unique(ff)):

    tr = np.where(ff != fold)[0]
    te = np.where(ff == fold)[0]

    candidates = []

    # -------------------------------
    # TRAIN selection
    # -------------------------------
    for coverage in coverage_grid:

        k = max(
            1,
            int(np.ceil(len(tr) * coverage))
        )

        order = tr[
            np.argsort(-flip_score[tr])
        ]

        chosen = order[:k]

        rescues = int(
            flip_correct[chosen].sum()
        )
        breaks = k - rescues
        net = rescues - breaks
        precision = rescues / k

        candidates.append({
            "coverage": coverage,
            "accepted": k,
            "rescues": rescues,
            "breaks": breaks,
            "net": net,
            "precision": precision,
        })


    cand_df = pd.DataFrame(candidates)

    valid = cand_df[
        (cand_df["accepted"] >= MIN_TRAIN_ACCEPTED)
        & (cand_df["precision"] >= MIN_TRAIN_PRECISION)
        & (cand_df["net"] >= MIN_TRAIN_NET)
    ].copy()


    if valid.empty:

        nested_rows.append({
            "fold": fold,
            "selected_coverage": np.nan,
            "train_accepted": 0,
            "train_rescues": 0,
            "train_breaks": 0,
            "train_net": 0,
            "train_precision": np.nan,
            "test_accepted": 0,
            "test_rescues": 0,
            "test_breaks": 0,
            "test_net": 0,
            "test_precision": np.nan,
        })

        continue


    # Maximum net, then maximum precision,
    # then smallest coverage.
    selected = (
        valid
        .sort_values(
            ["net", "precision", "coverage"],
            ascending=[False, False, True],
        )
        .iloc[0]
    )

    coverage = float(
        selected["coverage"]
    )


    # -------------------------------
    # TEST application
    # -------------------------------
    k_test = max(
        1,
        int(np.ceil(len(te) * coverage))
    )

    test_order = te[
        np.argsort(-flip_score[te])
    ]

    chosen_test = test_order[:k_test]

    test_rescues = int(
        flip_correct[chosen_test].sum()
    )
    test_breaks = (
        len(chosen_test) - test_rescues
    )

    test_net = (
        test_rescues - test_breaks
    )

    test_precision = (
        test_rescues / len(chosen_test)
        if len(chosen_test)
        else np.nan
    )


    chosen_cell_positions = (
        eligible_cell_positions[chosen_test]
    )

    chosen_global = cell_idx[
        chosen_cell_positions
    ]

    accepted_global.extend(
        chosen_global.tolist()
    )


    nested_rows.append({
        "fold": fold,
        "selected_coverage": coverage,

        "train_accepted":
            int(selected["accepted"]),
        "train_rescues":
            int(selected["rescues"]),
        "train_breaks":
            int(selected["breaks"]),
        "train_net":
            int(selected["net"]),
        "train_precision":
            float(selected["precision"]),

        "test_accepted":
            len(chosen_test),
        "test_rescues":
            test_rescues,
        "test_breaks":
            test_breaks,
        "test_net":
            test_net,
        "test_precision":
            test_precision,
    })


conservative_df = pd.DataFrame(
    nested_rows
)

display(
    conservative_df.round(4)
)


# ============================================================
# SUMMARY
# ============================================================

accepted = int(
    conservative_df["test_accepted"].sum()
)
rescues = int(
    conservative_df["test_rescues"].sum()
)
breaks = int(
    conservative_df["test_breaks"].sum()
)
net = rescues - breaks

print("Accepted:", accepted)
print("Rescues:", rescues)
print("Breaks:", breaks)
print("Net:", net)

print(
    "Precision:",
    rescues / accepted
    if accepted else np.nan
)


# ============================================================
# APPLY TO 813-CORRECT ROUTER
# ============================================================

conservative_pred = np.asarray(
    nested_pred
).copy()

accepted_global = np.asarray(
    accepted_global,
    dtype=int,
)

conservative_pred[
    accepted_global
] = B


comparison = pd.DataFrame([
    {
        "model": "GW_CW_GC",
        "accuracy":
            accuracy_score(true_arr, nested_pred),
        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr, nested_pred
            ),
        "macro_f1":
            f1_score(
                true_arr,
                nested_pred,
                average="macro",
            ),
        "weighted_f1":
            f1_score(
                true_arr,
                nested_pred,
                average="weighted",
            ),
        "correct":
            int(
                (np.asarray(nested_pred)
                 == true_arr).sum()
            ),
    },
    {
        "model":
            "plus_conservative_embedding",
        "accuracy":
            accuracy_score(
                true_arr,
                conservative_pred,
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                true_arr,
                conservative_pred,
            ),
        "macro_f1":
            f1_score(
                true_arr,
                conservative_pred,
                average="macro",
            ),
        "weighted_f1":
            f1_score(
                true_arr,
                conservative_pred,
                average="weighted",
            ),
        "correct":
            int(
                (conservative_pred
                 == true_arr).sum()
            ),
    },
])

display(
    comparison.round(4)
)

,fold,selected_coverage,train_accepted,train_rescues,train_breaks,train_net,train_precision,test_accepted,test_rescues,test_breaks,test_net,test_precision
0,1,0.05,4,4,0,4,1.00,1,1,0,1,1.0
1,2,0.05,4,3,1,2,0.75,1,0,1,-1,0.0
2,3,NaN,0,0,0,0,NaN,0,0,0,0,NaN
3,4,NaN,0,0,0,0,NaN,0,0,0,0,NaN
4,5,NaN,0,0,0,0,NaN,0,0,0,0,NaN


Accepted: 2
Rescues: 1
Breaks: 1
Net: 0
Precision: 0.5


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,correct
0,GW_CW_GC,0.546,0.4810,0.4991,0.5367,813
1,plus_conservative_embedding,0.546,0.4806,0.4988,0.5365,813


In [ ]:
np.save(
    "../data/processed/train_current_embeddings.npy",
    np.asarray(E_current)
)

# If you also have the test equivalent:
np.save(
    "../data/processed/test_current_embeddings.npy",
    np.asarray(E_current_test)
)

NameError: name 'np' is not defined

Yes. I would **stop optimizing this MoE/router branch and move to the next research direction**.

The embedding experiment was worth doing because it answered an important question rather cleanly. There is representation-level signal: embedding-only reached ROC-AUC **0.661** versus **0.610** for semantic probabilities in the constraint/workflow cell, and the initial tail looked promising. But once you imposed conservative nested selection, it collapsed to **1 rescue / 1 break, net 0**. So the apparent `813 → 815` improvement isn't robust enough to claim.

I would therefore freeze the current result as:

**Existing hard router:** 803/1489 = 53.93%
**GW + CW:** 811/1489 = 54.47%
**GW + CW + GC:** **813/1489 = 54.60%**
**Embedding override:** no robust incremental gain

That's actually a useful endpoint. The specialists have demonstrated that targeted corrections can work, but you're now extracting single-digit improvements from increasingly small subsets. Continuing to add specialists risks turning the validation/OOF predictions into an implicit tuning set.

### Where I would go next

The residual structure says the next research should address the **representation/objective**, rather than add another routing layer. You still have 676 residual errors, heavily concentrated in three pairs: constraint↔workflow (**224**), tool-use↔workflow (**146**), and grounding↔workflow (**143**). That's 513/676, about **76% of all remaining errors**. Workflow is the common boundary problem.

So my next experiment would target a different question:

> **Is `workflow_error` acting as an overly broad class whose internal subtypes are semantically separable?**

That's more fundamental than another binary specialist. Your evidence points in that direction: pairwise classifiers can obtain ranking signal, yet reliably deciding when to override the existing classifier remains difficult. That often happens when one label encompasses multiple mechanisms.

I'd inspect the **660 true workflow examples** and characterize them by transition, semantic embedding neighborhoods/clusters, and which competing family they are confused with. Specifically, we'd ask whether true workflow examples confused with `constraint`, `grounding`, and `tool_use` form different regions of representation space.

If they do, the promising architecture becomes something like:

**family classifier → workflow subtype/contrastive representation → collapse subtypes back to workflow**

rather than:

**family classifier → specialist → specialist → specialist → override**

That gives us a route to improve the underlying decision boundary instead of patching individual errors.

So yes: **freeze 813 as the current MoE result.** I wouldn't spend more research budget squeezing this particular embedding gate.

The next experiment I'd run is a **workflow heterogeneity analysis**, with no classifier changes initially. We can determine whether workflow is one coherent class or effectively several latent subclasses, and use that to choose between hierarchical labels, contrastive training, or better features.


## Summary — `17_relation_specific_evidence_representations.ipynb`

This notebook investigates whether **relation-specific evidence representations** can improve failure-family classification beyond a semantic baseline. The central hypothesis is that the current event embedding alone does not fully capture failure mechanisms, and that information about the relationship between the current event and relevant prior events—especially previous tool or assistant states—can provide additional discriminative signal.

The experiments focus on five failure families:

`workflow_error`, `constraint_error`, `tool_use_error`, `grounding_state_error`, and `reasoning_value_error`.

The semantic/no-T1 baseline achieved:

| Model             | Accuracy | Balanced Accuracy | Macro-F1 | Weighted-F1 |
| ----------------- | -------: | ----------------: | -------: | ----------: |
| Semantic baseline |   0.5151 |            0.4355 |   0.4548 |      0.4969 |

The notebook then evaluates relation-aware representations and specialist models conditioned on transition type.

### 1. Relation-specific representations contain useful signal

For several transition-specific binary tasks, incorporating the correct previous-role embedding substantially improved discrimination over the current embedding alone.

For `TOOL_CALL → TOOL_CALL`, predicting `tool_use_error`, the strongest representation was `current_plus_previous`:

* support: 590
* positives: 136
* prevalence: 0.2305
* PR-AUC: **0.7019**
* PR lift: **3.05×**
* ROC-AUC: **0.8638**

This substantially outperformed current semantic representation alone:

* PR-AUC: 0.6142
* ROC-AUC: 0.8471

For `ASSISTANT → TOOL_CALL`, predicting `constraint_error`, the previous assistant embedding was especially informative:

* PR-AUC: **0.6039**
* ROC-AUC: **0.7770**

compared with the current embedding:

* PR-AUC: 0.4786
* ROC-AUC: 0.7489

For `TOOL_CALL → ASSISTANT`, predicting `grounding_state_error`, the previous tool embedding also improved performance:

* PR-AUC: **0.4957**
* ROC-AUC: **0.6852**

versus:

* current PR-AUC: 0.3820
* current ROC-AUC: 0.6484

A control experiment using the **wrong previous-role embedding** degraded performance, confirming that the gain was not simply caused by adding another embedding. The role-correct relational state matters.

### 2. Relation information is most useful for specific transition/failure boundaries

The strongest relation-specific effects were observed for:

* `TOOL_CALL → TOOL_CALL` → `tool_use_error`
* `ASSISTANT → TOOL_CALL` → `constraint_error`
* `TOOL_CALL → ASSISTANT` → `grounding_state_error`

The gain was particularly strong for tool-use prediction. Relation features raised PR-AUC by approximately **+0.088** over the current representation.

For grounding and constraint errors, the previous-role semantic embedding alone was often stronger than explicit delta/product representations. This suggests that the **semantic state of the previous event** is more useful than simple geometric operations such as subtraction or element-wise products.

Scalar relation features alone performed poorly and were close to chance for some tasks. Thus, relation evidence is largely represented in the embedding space rather than simple scalar similarity measures.

---

## Specialist meta-model experiments

The notebook next used the strongest transition-specific classifiers as specialists layered over the semantic baseline.

An unrestricted relation-specialist meta-model improved minority-class detection substantially:

| Model                    |   Accuracy | Balanced Accuracy |   Macro-F1 | Weighted-F1 |
| ------------------------ | ---------: | ----------------: | ---------: | ----------: |
| Semantic baseline        |     0.5151 |            0.4355 |     0.4548 |      0.4969 |
| Relation specialist meta | **0.5232** |        **0.5399** | **0.4925** |  **0.5299** |

The improvement in balanced accuracy was particularly large:

**+0.1044**

However, the unrestricted meta-model made 473 prediction changes and produced only a small net accuracy gain. It substantially increased recall for minority error families while reducing workflow recall.

Examples:

* `tool_use_error` recall: 0.2911 → **0.5148**
* `grounding_state_error`: 0.2992 → **0.4590**
* `constraint_error`: 0.4038 → **0.5489**
* `reasoning_value_error`: 0.4516 → **0.6452**
* `workflow_error`: 0.7318 → **0.5318**

This showed that relation specialists contained strong information, but naive global fusion over-corrected the dominant workflow class.

---

## Conservative local overrides

The notebook therefore shifted from global fusion to **high-confidence local overrides**.

Thresholded specialist interventions improved the semantic baseline while changing only a small fraction of examples.

An optimistic threshold search gave:

* 131 accepted overrides
* 72 rescues
* 38 breaks
* net: **+34**

with accuracy increasing to 0.5379.

Because this used globally optimized thresholds, nested/cross-fitted threshold selection was introduced to prevent optimistic bias.

Cross-fitted local overrides produced:

* 112 interventions
* 52 rescues
* 39 breaks
* net: **+13**
* accuracy: **0.5238**

The specialists behaved very differently:

* constraint specialist: positive and relatively precise
* tool-use specialist: positive but less stable
* grounding specialist: approximately neutral or slightly harmful

This motivated removing weaker specialists and focusing on the most robust boundaries.

---

## Pairwise hierarchical routing

A major improvement came from reformulating some specialist tasks as **pairwise classification problems**, especially against `workflow_error`.

Two useful boundaries were identified:

### Tool-use vs workflow

`TOOL_CALL → TOOL_CALL`

* support: 477
* tool-use positives: 136
* PR-AUC: **0.7081**
* ROC-AUC: **0.8693**

### Constraint vs workflow

`ASSISTANT → TOOL_CALL`

* support: 154
* constraint positives: 57
* PR-AUC: **0.6472**
* ROC-AUC: **0.7779**

A pairwise hierarchical router using these boundaries achieved:

| Model                          |   Accuracy | Balanced Accuracy |   Macro-F1 | Weighted-F1 |
| ------------------------------ | ---------: | ----------------: | ---------: | ----------: |
| Semantic baseline              |     0.5151 |            0.4355 |     0.4548 |      0.4969 |
| Nested one-vs-rest specialists |     0.5299 |            0.4585 |     0.4759 |      0.5194 |
| Pairwise hierarchical router   | **0.5393** |        **0.4745** | **0.4918** |  **0.5292** |

The pairwise router changed 122 predictions:

* 79 rescues
* 43 breaks
* net: **+36**

Pairwise probability reconciliation gave similar but slightly lower performance, suggesting that the hard routing decisions were more effective than globally blending probabilities.

---

## Nested specialist additions

The notebook then tested whether additional nested specialists could improve the pairwise router.

### GW specialist

A grounding/workflow correction was validated using nested threshold selection.

It produced a small but positive result:

* nested interventions: 13
* rescues: 9
* breaks: 4
* net: **+5**

### CW specialist

A constraint/workflow correction also produced:

* 13 interventions
* 8 rescues
* 5 breaks
* net: **+3**

Importantly, the GW and CW corrections had:

* overlap: 0
* conflicts: 0

Combining them therefore produced an additive gain.

The combined router achieved:

| Model                |   Accuracy | Balanced Accuracy |   Macro-F1 | Weighted-F1 |
| -------------------- | ---------: | ----------------: | ---------: | ----------: |
| Existing hard router |     0.5393 |            0.4745 |     0.4918 |      0.5292 |
| GW only              |     0.5426 |            0.4806 |     0.4983 |      0.5345 |
| CW only              |     0.5413 |            0.4737 |     0.4915 |      0.5300 |
| **GW + CW**          | **0.5447** |        **0.4799** | **0.4979** |  **0.5354** |

This corresponded to **811 correct predictions out of 1489**, compared with 803 for the original hard pairwise router.

---

## Grounding → constraint specialist

The next promising residual boundary was between grounding and constraint errors.

Exploratory results showed that the `grounding → constraint` direction was considerably stronger than the reverse direction.

Nested validation yielded:

* 4 interventions
* 3 rescues
* 1 break
* net: **+2**
* intervention precision: 0.75

Adding this GC specialist produced the final robust router:

| Model                |   Accuracy | Balanced Accuracy |   Macro-F1 | Weighted-F1 |
| -------------------- | ---------: | ----------------: | ---------: | ----------: |
| Existing hard router |     0.5393 |            0.4745 |     0.4918 |      0.5292 |
| GW + CW              |     0.5447 |            0.4799 |     0.4979 |      0.5354 |
| **GW + CW + GC**     | **0.5460** |        **0.4810** | **0.4991** |  **0.5367** |

The final system correctly classified:

**813 / 1489 examples**

versus:

**803 / 1489** for the initial hard relation router.

Relative to that router, the final specialist layer made:

* 30 prediction changes
* 20 rescues
* 10 breaks
* 0 wrong-to-wrong changes
* net: **+10**
* intervention precision: **66.7%**

---

# Final class-level effects

The cumulative router primarily benefited `grounding_state_error` and slightly improved workflow performance.

Notable changes relative to the original hard router:

* grounding recall: **0.2992 → 0.3320**
* grounding F1: **0.3659 → 0.3971**
* workflow recall: **0.7030 → 0.7091**
* workflow F1: **0.6361 → 0.6415**

Constraint performance was approximately unchanged overall, despite individual constraint-related rescues and breaks balancing each other.

Tool-use and reasoning-value metrics were effectively unchanged by the final specialist additions.

---

# Transition-conditioned calibration experiment

A multinomial calibrator using:

* semantic probabilities,
* transition type,
* event role,
* history availability,
* history length

was tested as an alternative to hand-routed specialists.

It performed substantially worse:

| Model                 |   Accuracy | Balanced Accuracy |   Macro-F1 | Weighted-F1 |
| --------------------- | ---------: | ----------------: | ---------: | ----------: |
| GW + CW + GC          | **0.5460** |        **0.4810** | **0.4991** |  **0.5367** |
| Transition calibrator |     0.5238 |            0.4461 |     0.4750 |      0.5111 |

It changed 303 predictions but generated:

* 105 rescues
* 138 breaks
* 60 wrong-to-wrong changes

for a net loss of **33 correct predictions**.

This indicates that transition-level structural information is useful but too coarse for unrestricted global recalibration. The benefit comes from **specific local boundaries**, not wholesale transition-conditioned probability adjustment.

---

# Embedding residual-gating experiment

The notebook also tested whether embeddings contain residual signal beyond semantic probabilities for the difficult:

`TOOL_CALL → ASSISTANT`, `constraint_error` vs `workflow_error`

boundary.

Across all 250 examples:

| Representation         |    ROC-AUC |     PR-AUC |
| ---------------------- | ---------: | ---------: |
| Semantic probabilities |     0.6100 |     0.6474 |
| **Embedding**          | **0.6613** | **0.6883** |
| Semantic + embedding   |     0.6557 |     0.6814 |

Thus, embeddings contain additional information that is not fully captured by semantic probabilities.

For examples currently predicted as `constraint_error`, an embedding-derived flip score showed:

* ROC-AUC: **0.6552**
* PR-AUC: **0.4944**
* error prevalence: 0.2658
* PR lift: **1.86×**

The top 10% exploratory tail achieved 6 rescues and 2 breaks.

However, nested validation weakened this result. A flexible nested coverage rule produced only:

* 6 rescues
* 4 breaks
* net +2

and a more conservative nested test produced:

* 1 rescue
* 1 break
* net **0**

Therefore the embedding signal is real for class discrimination, but the apparent high-confidence intervention tail is **not sufficiently stable to add to the final router**.

---

# Residual error structure

Even after the final GW + CW + GC router, 676 errors remain.

The dominant residual family pairs are:

| Family pair            | Residual errors |
| ---------------------- | --------------: |
| constraint ↔ workflow  |         **224** |
| tool-use ↔ workflow    |         **146** |
| grounding ↔ workflow   |         **143** |
| constraint ↔ grounding |              81 |
| constraint ↔ tool-use  |              37 |

The first three workflow-related boundaries account for approximately **76% of all remaining errors**.

The largest transition-specific residual cells include:

* `TOOL_CALL → TOOL_CALL`: tool-use ↔ workflow — 79
* `TOOL_CALL → ASSISTANT`: constraint ↔ workflow — 77
* `ASSISTANT → ASSISTANT`: constraint ↔ workflow — 66
* `TOOL_CALL → ASSISTANT`: grounding ↔ workflow — 55

This concentration suggests that `workflow_error` is the main remaining decision-boundary problem.

---

# Main conclusions

The notebook establishes several important findings.

**First, relation-specific representations are genuinely useful.** The identity and semantic state of the correct previous event improve failure-family prediction, especially for tool-use, constraint, and grounding errors.

**Second, the useful information is highly local.** Global meta-models and transition-conditioned calibration tend to over-correct workflow examples. The best results come from transition-specific pairwise boundaries and conservative specialist interventions.

**Third, pairwise routing is more effective than unrestricted multiclass fusion.** Reframing difficult decisions as local family-vs-family boundaries raised accuracy from the semantic baseline of **51.51%** to **53.93%** before additional specialists.

**Fourth, nested specialist corrections provide a small but robust additional gain.** GW, CW, and GC together increased accuracy to **54.60%**, or **813/1489 correct examples**, a net improvement of 10 examples relative to the initial hard relation router.

**Fifth, the notebook reaches clear diminishing returns.** Additional specialist searches and embedding overrides yield only very small or unstable gains. The conservative embedding experiment returned net zero, indicating that further hand-built override layers are unlikely to be the best use of research effort.

---

## Final result

The strongest defensible configuration from this notebook is:

**Pairwise hierarchical relation router + nested GW + CW + GC specialists**

with:

* **Accuracy: 0.5460**
* **Balanced accuracy: 0.4810**
* **Macro-F1: 0.4991**
* **Weighted-F1: 0.5367**
* **Correct predictions: 813 / 1489**

Compared with the semantic/no-T1 baseline:

* accuracy: **0.5151 → 0.5460**
* balanced accuracy: **0.4355 → 0.4810**
* macro-F1: **0.4548 → 0.4991**
* weighted-F1: **0.4969 → 0.5367**

The notebook therefore supports the conclusion that **relation-specific evidence provides real complementary information**, but that its strongest use is through carefully targeted local decision boundaries rather than unrestricted fusion.

The next research direction should move away from adding more MoE specialists and investigate the underlying representation and label structure—especially the apparent heterogeneity of `workflow_error` and why it remains the dominant source of residual confusion.
